# 🛢️ Geosteering EDA + Target-Free Stratigraphic Alignment for TVT Recovery

This notebook studies how horizontal well logs can be aligned to formation-aware TVT structure without directly observing the hidden interval.

- Use observed gamma-ray patterns, formation context, and geometric priors to build TVT estimates.
- Compare complementary alignment signals: local planes, dense analogs, beam-style paths, PF-style priors, and DTW-like sequence matches.
- Keep the final correction conservative: when two estimates disagree strongly, the blend retreats toward the more stable base trajectory.

For a more detailed walkthrough of the notebook, see this post: [ROGII Target-Free Stratigraphic Alignment for TVT](https://pilkwangkim.github.io/posts/ROGII-Target-Free-Stratigraphic-Alignment-for-TVT/).


In [ ]:
from pathlib import Path
from IPython.display import Image, display

cover_image_path = Path("/kaggle/input/datasets/pilkwang/pilkwang-public-dataset-for-notebooks-figures/ROGII70.png")
if cover_image_path.exists():
    display(Image(filename=str(cover_image_path)))

In [ ]:
from pathlib import Path
from IPython.display import Image, display

cover_image_path = Path("/kaggle/input/datasets/pilkwang/pilkwang-public-dataset-for-notebooks-figures/ROGII_Graph_FigMain.png")
if cover_image_path.exists():
    display(Image(filename=str(cover_image_path)))

In [ ]:
from pathlib import Path
from IPython.display import HTML, Markdown, display
from base64 import b64encode


def display_rogii_figure(filename: str, title: str, caption: str, width: int = 1120) -> None:
    kaggle_dir = Path("/kaggle/input/datasets/pilkwang/pilkwang-public-dataset-for-notebooks-figures")
    fallback_dirs = [
        Path("./Figs"),
        Path("Figs"),
    ]
    image_path = kaggle_dir / filename
    if not image_path.exists():
        image_path = next((d / filename for d in fallback_dirs if (d / filename).exists()), None)
    if image_path is None or not image_path.exists():
        display(Markdown(f"> Figure unavailable: `{filename}`."))
        return
    data = b64encode(image_path.read_bytes()).decode("ascii")
    html = f"""
    <div style="margin: 18px 0 22px 0;">
      <h3 style="margin:0 0 6px 0;">{title}</h3>
      <p style="margin:0 0 12px 0; color:#5b6470; font-style:italic;">{caption}</p>
      <img src="data:image/png;base64,{data}" style="max-width:100%; width:{int(width)}px; height:auto; display:block; border-radius:10px;" />
    </div>
    """
    display(HTML(html))


display_rogii_figure(
    "ROGII_Fig1.png",
    "Figure 1 — EDA, Leakage, and Blend Strategy",
    "One-page overview of the prediction problem, leakage boundary, residual anchor, formation/physics signals, and final stack logic.",
    width=1120,
)


## 📌 Modeling Summary

> **Main idea:** turn each hidden row into a compact collection of target-free geological hypotheses, then learn which hypotheses are reliable in context.

### 🎯 Task Snapshot

| Item | Value | Note |
|---|---:|---|
| Train wells | 773 | spatial references and validation must respect well identity |
| Test wells | 3 | submission predictions are produced only for hidden `TVT_input` rows |
| Submission rows | 14,151 | final output must exactly match these ids |
| Train tail rows | 3,783,989 | long hidden tails provide most of the row-level training signal |
| v7 final matrix | Section 18 DTW super-stack table | built directly from target-free physics, GR, and spatial priors |

### 🧬 Feature / Signal Map

| Signal family | Examples | Inputs | Use |
|---|---|---:|---|
| ⚓ Anchor & prefix TVT | `last_known_TVT`, prefix slopes, prefix range | strict | residual reference and local drift prior |
| 🛤️ Trajectory | `MD`, `X/Y/Z`, local steps, tail position | strict / offline | geometry and smoothness context |
| 🌋 GR texture | trailing rolls, centered rolls, lead/lag GR | strict / offline | local stratigraphic events |
| 🧭 Typewell / physics paths | candidate endpoints, beam paths, DTW paths, self-correlation, PF-lite | offline-heavy | plausible TVT paths from observed GR and typewell shape |
| 🏗️ Formation plane | six formation formulas, plane distance | offline | spatial geology prior |
| 🌐 Row ANCC / dense surface | row-level ANCC, dense distance/std/bias | offline | local surface correction |
| 🏁 Super stack | LGB seeds + CatBoost + Ridge/Hill-climb | model | residual prediction with model diversity |


In [ ]:
from pathlib import Path
from IPython.display import Image, display

figure_image_path = Path("/kaggle/input/datasets/pilkwang/pilkwang-public-dataset-for-notebooks-figures/ROGII_Graph_Fig1.png")
if figure_image_path.exists():
    display(Image(filename=str(figure_image_path)))


**Figure 2. Horizontal well geometry and hidden-tail prediction setup.**  
The observed prefix contains known `TVT_input` values. The prediction interval begins after the last known TVT anchor $T_L$. The model predicts the hidden-tail stratigraphic coordinate by estimating drift away from this anchor.


## Final Prediction Controls

This notebook exposes a small set of submission profiles. Each profile owns the final `submission.csv` path, so different engines do not accidentally stack on top of each other.

| Profile | Final prediction | Use |
|---|---|---|
| `ridge_artifact_exact` | artifact Ridge reference plus physical/PF heuristic reference | closest reproduction of the ridge reference path |
| `ridge_artifact_push` | same ridge/heuristic engine with a different ridge-vs-heuristic weight | lightweight public-score probe around the ridge reference |
| `ridge_artifact_pf_boost` | same ridge reference with a larger heuristic PF ensemble | pushes the observed 500/128 to 600/150 improvement direction further |
| `ridge_artifact_model_blend` | `ridge_artifact_exact` mixed with PF-residual GBDT | score-push probe using two independent model families |
| `pf_residual_gbdt_exact` | PF trajectory plus LGB/XGB/CatBoost residual correction | closest reproduction of the public GBDT ensemble path |
| `fast_pf_selector` | target-free PF/beam selector only | quick physical selector reproduction |
| `full_stack_postproc` | local alignment stack with postprocessing | heavier local-stack diagnostic path |

References:

- PF selector: https://www.kaggle.com/code/aiwody/physical-model-less-overfitting-noise
- Ridge artifact reference: https://www.kaggle.com/code/overvalueawareness/wellbore-geology-prediction-ridge/notebook
- Ridge artifact reference: https://www.kaggle.com/code/ravaghi/wellbore-geology-prediction-ridge

### PF selector and same-well overlap

The PF selector predicts absolute TVT from the observed trajectory, GR shape, prefix TVT, and formation/typewell hypotheses. It does not use hidden-tail TVT values.

`PF_SELECTOR_USE_SAME_WELL_PHYSICAL` controls one important shortcut:

| Setting | Behavior | Score interpretation |
|---|---|---|
| `True` | if a test `well_id` also exists in train, use the same-well physical/contact path before PF/beam fallback | strongest public-reproduction setting when train/test wells overlap |
| `False` | always use the PF/beam selector path | cleaner unseen-well robustness probe, often weaker on overlap-heavy public samples |

This is not a target leak by itself. It is a distribution bet. If the evaluation sample contains continuation rows from wells whose prefix appears in train, `True` can be strong. If the private sample is mostly unseen wells, `False` is the more conservative stress test.

### Prediction formulas

Let:

| Symbol | Meaning |
|---|---|
| $T_i^{\mathrm{ridge}}$ | artifact Ridge prediction |
| $T_i^{\mathrm{heur}}$ | physical/PF heuristic prediction from the ridge reference path |
| $T_i^{\mathrm{pfgbdt}}$ | PF-residual GBDT ensemble prediction |
| $T_i^{\mathrm{stack}}$ | local alignment-stack prediction |

The exact ridge reference profile writes:

$$
T_i^{\mathrm{ridge\_ref}} = 0.30\,T_i^{\mathrm{ridge}} + 0.70\,T_i^{\mathrm{heur}}.
$$

The ridge weight-probe profile writes:

$$
T_i^{\mathrm{final}} = w\,T_i^{\mathrm{ridge}} + (1-w)\,T_i^{\mathrm{heur}}.
$$

The model-blend probe keeps the exact ridge reference as the anchor and mixes in the independent PF-residual GBDT estimate:

$$
T_i^{\mathrm{final}} = (1-\lambda)\,T_i^{\mathrm{ridge\_ref}} + \lambda\,T_i^{\mathrm{pfgbdt}}.
$$

Candidate files are written for both configured ridge weights and configured model-mix weights, so `submission.csv` can be compared against nearby alternatives.

### Recommended usage

Start with `ridge_artifact_exact` when the goal is exact reference reproduction. Use `ridge_artifact_push` for a cheap weight probe. Use `ridge_artifact_pf_boost` to test whether a larger PF ensemble improves the heuristic estimate. Use `ridge_artifact_model_blend` when it is worth running both the ridge artifact path and the PF-residual GBDT path.


In [ ]:
# Submission profile.
# Recommended score paths:
# - ridge_artifact_exact: reproduce the artifact-backed Ridge + heuristic reference blend.
# - ridge_artifact_push: change only the ridge-vs-heuristic weight in the same engine.
# - ridge_artifact_pf_boost: use a larger PF ensemble for the ridge heuristic path.
# - ridge_artifact_model_blend: mix the ridge reference with PF-residual GBDT predictions.
# - pf_residual_gbdt_exact: reproduce the PF + LGB/XGB/CatBoost residual ensemble.
# - fast_pf_selector: target-free PF/beam selector-only reproduction.
# - full_stack_postproc: original heavier EDA stack with postprocessing.
SUBMISSION_PROFILE = 'ridge_artifact_push'

# Target-free PF/beam selector settings.
PF_SELECTOR_N_PARTICLES = 500
PF_SELECTOR_N_SEEDS = 64
PF_SELECTOR_SCALES = (3.0, 5.0, 8.0, 12.0)
PF_SELECTOR_AS_AUX_GATED_MAX_WEIGHT = 0.015
PF_SELECTOR_AS_AUX_GATED_SCALE = 4.0
# True enables the overlap-aware selector shortcut; False disables the same-well physical shortcut.
PF_SELECTOR_USE_SAME_WELL_PHYSICAL = True

# PF-residual GBDT settings. This profile predicts TVT as PF path + learned PF error.
PF_RESIDUAL_GBDT_N_PARTICLES = 500
PF_RESIDUAL_GBDT_N_SEEDS = 32
PF_RESIDUAL_GBDT_LIKELIHOOD_SCALE = 5.0
PF_RESIDUAL_GBDT_N_SPLITS = 5
PF_RESIDUAL_GBDT_NUM_BOOST_ROUND = 1000
PF_RESIDUAL_GBDT_USE_GPU = True
PF_RESIDUAL_GBDT_USE_MULTIPROCESS = True
PF_RESIDUAL_GBDT_WEIGHTS = {'lgb': 0.40, 'xgb': 0.40, 'catboost': 0.20}

# Artifact-backed ridge reference settings.
RIDGE_ARTIFACT_ROOTS = [
    '/kaggle/input/datasets/ravaghi/wellbore-geology-prediction-artifacts',
    '/kaggle/input/wellbore-geology-prediction-artifacts',
]
RIDGE_ARTIFACT_EXACT_RIDGE_WEIGHT = 0.30
RIDGE_ARTIFACT_PUSH_RIDGE_WEIGHT = 0.33
RIDGE_ARTIFACT_CANDIDATE_RIDGE_WEIGHTS = [0.20, 0.25, 0.30, 0.35, 0.40]

# Heuristic PF ensemble size for ridge artifact profiles.
# (1) used 500/128, the stronger reference uses 600/150; boost mode pushes this direction further.
RIDGE_ARTIFACT_DEFAULT_PF_N_PARTICLES = 600
RIDGE_ARTIFACT_DEFAULT_PF_N_SEEDS = 150
RIDGE_ARTIFACT_PF_N_PARTICLES = RIDGE_ARTIFACT_DEFAULT_PF_N_PARTICLES
RIDGE_ARTIFACT_PF_N_SEEDS = RIDGE_ARTIFACT_DEFAULT_PF_N_SEEDS
RIDGE_ARTIFACT_BOOST_PF_N_PARTICLES = 650
RIDGE_ARTIFACT_BOOST_PF_N_SEEDS = 165

# Model-mix probe: ridge reference anchor + PF-residual GBDT correction.
RIDGE_ARTIFACT_MODEL_BLEND_PF_GBDT_WEIGHT = 0.05
RIDGE_ARTIFACT_MODEL_BLEND_WEIGHTS = [0.00, 0.03, 0.05, 0.07, 0.10]

# Data roots.
COMPETITION_DATA_ROOTS = [
    '/kaggle/input/rogii-wellbore-geology-prediction',
    '/kaggle/input/competitions/rogii-wellbore-geology-prediction',
]

# Optional TVT clipping. Keep None unless calibrated bounds are known.
TVT_CLIP_MIN = None
TVT_CLIP_MAX = None


In [ ]:
_profile = str(SUBMISSION_PROFILE).strip().lower()
_valid_profiles = {
    'ridge_artifact_exact',
    'ridge_artifact_push',
    'ridge_artifact_pf_boost',
    'ridge_artifact_model_blend',
    'pf_residual_gbdt_exact',
    'fast_pf_selector',
    'full_stack_postproc',
}
if _profile not in _valid_profiles:
    raise ValueError(f'SUBMISSION_PROFILE must be one of {sorted(_valid_profiles)}')

RUN_FAST_PF_SELECTOR_ONLY = _profile == 'fast_pf_selector'
RUN_FAST_PF_SELECTOR_128 = False
RUN_RIDGE_ARTIFACT_EXACT = _profile == 'ridge_artifact_exact'
RUN_RIDGE_ARTIFACT_PUSH = _profile == 'ridge_artifact_push'
RUN_RIDGE_ARTIFACT_PF_BOOST = _profile == 'ridge_artifact_pf_boost'
RUN_RIDGE_ARTIFACT_MODEL_BLEND = _profile == 'ridge_artifact_model_blend'
RUN_RIDGE_ARTIFACT_PROFILE = _profile in {'ridge_artifact_exact', 'ridge_artifact_push', 'ridge_artifact_pf_boost', 'ridge_artifact_model_blend'}
RUN_PF_RESIDUAL_GBDT_EXACT = _profile in {'pf_residual_gbdt_exact', 'ridge_artifact_model_blend'}
RUN_PF_RESIDUAL_GBDT_ONLY = RUN_PF_RESIDUAL_GBDT_EXACT
RUN_TARGET_FREE_SELECTOR_CANDIDATE = _profile == 'fast_pf_selector'


PF_RESIDUAL_GBDT_FILL_MODE = 'median_guarded'
RIDGE_ARTIFACT_PF_N_PARTICLES = int(globals().get('RIDGE_ARTIFACT_DEFAULT_PF_N_PARTICLES', 600))
RIDGE_ARTIFACT_PF_N_SEEDS = int(globals().get('RIDGE_ARTIFACT_DEFAULT_PF_N_SEEDS', 150))

if _profile == 'fast_pf_selector':
    FINAL_V7_CANDIDATE = 'pf_selector'
elif _profile == 'ridge_artifact_exact':
    FINAL_V7_CANDIDATE = 'ridge_artifact_exact'
elif _profile == 'ridge_artifact_push':
    FINAL_V7_CANDIDATE = 'ridge_artifact_push'
elif _profile == 'ridge_artifact_pf_boost':
    FINAL_V7_CANDIDATE = 'ridge_artifact_pf_boost'
    RIDGE_ARTIFACT_PF_N_PARTICLES = int(globals().get('RIDGE_ARTIFACT_BOOST_PF_N_PARTICLES', 700))
    RIDGE_ARTIFACT_PF_N_SEEDS = int(globals().get('RIDGE_ARTIFACT_BOOST_PF_N_SEEDS', 180))
elif _profile == 'ridge_artifact_model_blend':
    FINAL_V7_CANDIDATE = 'ridge_artifact_model_blend'
    PF_RESIDUAL_GBDT_FILL_MODE = 'public_native'
elif _profile == 'pf_residual_gbdt_exact':
    FINAL_V7_CANDIDATE = 'pf_residual_gbdt_exact'
    PF_RESIDUAL_GBDT_FILL_MODE = 'public_native'
else:
    FINAL_V7_CANDIDATE = 'postproc'

if RUN_PF_RESIDUAL_GBDT_EXACT:
    PF_RESIDUAL_GBDT_N_PARTICLES = 500
    PF_RESIDUAL_GBDT_N_SEEDS = 32
    PF_RESIDUAL_GBDT_LIKELIHOOD_SCALE = 5.0
    PF_RESIDUAL_GBDT_N_SPLITS = 5
    PF_RESIDUAL_GBDT_NUM_BOOST_ROUND = 1000
    PF_RESIDUAL_GBDT_USE_GPU = True
    PF_RESIDUAL_GBDT_USE_MULTIPROCESS = True
    PF_RESIDUAL_GBDT_WEIGHTS = {'lgb': 0.40, 'xgb': 0.40, 'catboost': 0.20}

profile_summary = {
    'submission_profile': SUBMISSION_PROFILE,
    'final_candidate': FINAL_V7_CANDIDATE,
    'active_engine': (
        'artifact ridge exact reproduction' if RUN_RIDGE_ARTIFACT_EXACT else
        'artifact ridge weight probe' if RUN_RIDGE_ARTIFACT_PUSH else
        'artifact ridge boosted-PF heuristic probe' if RUN_RIDGE_ARTIFACT_PF_BOOST else
        'artifact ridge plus PF-residual GBDT model blend' if RUN_RIDGE_ARTIFACT_MODEL_BLEND else
        'PF-residual GBDT exact reproduction' if _profile == 'pf_residual_gbdt_exact' else
        'PF selector only' if RUN_FAST_PF_SELECTOR_ONLY else
        'full stack postprocessed'
    ),
}
print(profile_summary)


In [ ]:
from pathlib import Path
from IPython.display import Image, display

figure_image_path = Path("/kaggle/input/datasets/pilkwang/pilkwang-public-dataset-for-notebooks-figures/ROGII_Graph_Fig12.png")
if figure_image_path.exists():
    display(Image(filename=str(figure_image_path)))


**Figure 3. Submission profile choices.**  
The fast PF selector submits a direct physical estimate. The ridge artifact profiles combine a saved Ridge estimate with a physical/PF heuristic. The GBDT profile learns residuals around a PF trajectory. The model-blend profile mixes two separately computed estimates instead of changing only one internal weight.


In [ ]:
# Shared path normalization.
from pathlib import Path

COMPETITION_DATA_ROOTS = [Path(p) for p in COMPETITION_DATA_ROOTS]
FINAL_SIDECAR_SOURCE_LABEL = str(globals().get('FINAL_BASE_SOURCE_LABEL', 'base_only'))
FINAL_SIDECAR_AVAILABLE = False
FINAL_SIDECAR_AUTO_DISABLED_REASON = ''


## 🧭 Notebook Roadmap

<div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(260px,1fr)); gap:12px; margin:14px 0;">
  <div style="border-left:5px solid #3b82f6; padding:10px 14px; background:#f8fbff; color:#1f2933; border-radius:10px; box-shadow:0 1px 2px rgba(31,41,51,0.08);">
    <b>🔒 0-3. Information boundary</b><br>
    Define observable columns, prediction rows, and leakage boundaries.
  </div>
  <div style="border-left:5px solid #10b981; padding:10px 14px; background:#f8fffb; color:#1f2933; border-radius:10px; box-shadow:0 1px 2px rgba(31,41,51,0.08);">
    <b>📊 4-14. Data signals</b><br>
    Inspect tail difficulty, typewell alignment, smoothness, spatial priors, and representative wells.
  </div>
  <div style="border-left:5px solid #f59e0b; padding:10px 14px; background:#fffaf0; color:#1f2933; border-radius:10px; box-shadow:0 1px 2px rgba(31,41,51,0.08);">
    <b>🧱 15-17. Feature framework</b><br>
    Build residual features and keep strict/offline policies explicit.
  </div>
  <div style="border-left:5px solid #8b5cf6; padding:10px 14px; background:#fbf8ff; color:#1f2933; border-radius:10px; box-shadow:0 1px 2px rgba(31,41,51,0.08);">
    <b>🚀 18-19. v7 final path</b><br>
    Run the DTW super-stack and validate the submission contract.
  </div>
</div>


## 🔒 0. Information Policies and Leakage Rules

### 🧭 Two-track information model

| Policy | Uses | Does **not** use | Best use |
|---|---|---|---|
| **Strict drilling-time** | prefix + current/trailing row evidence | future tail shape, centered windows, tail length | conservative geosteering-style validation |
| **Offline batch** | full provided test CSV covariates | future `TVT`, target-derived summaries | Kaggle submission candidates |

> **Practical rule:** offline features may look at future **GR / trajectory** rows because they are provided in test files, but they must never look at future **TVT**.

### ✅ Allowed vs 🚫 excluded

| Feature family | Strict | Offline | Reason |
|---|---:|---:|---|
| current `MD/X/Y/Z/GR` | ✅ | ✅ | observed covariates |
| prefix `TVT_input` | ✅ | ✅ | known target prefix |
| trailing GR windows | ✅ | ✅ | no future row access |
| centered GR / lead-lag GR | 🚫 | ✅ | future covariates, target-free |
| tail length / tail fraction | 🚫 | ✅ | known only in batch mode |
| candidate-path typewell features | 🚫 | ✅ | path uses full tail position |
| beam alignment | 🚫 | ✅ | sequence feature from hidden GR |
| direct train-only surfaces | 🚫 | 🚫 | hidden-test columns unavailable |
| fold-safe formation imputer outputs | 🚫 | ✅ | reproducible spatial reference model |
| tail `TVT` labels | 🚫 | 🚫 | direct target leakage |


In [ ]:
display_rogii_figure(
    "ROGII_Fig3.png",
    "Figure 4 — Leakage Boundary: Unsafe vs Fold-Aware",
    "Why spatial/formation evidence must be separated into submission-batch and fold-safe interpretations.",
    width=1120,
)


## 🧯 0.1 Leakage Risk Table

| Risk source | What can go wrong | Guardrail |
|---|---|---|
| Same well id in train/test | same-well spatial hints can look stronger than unseen-well generalization | compare overlap-enabled and overlap-disabled selector runs |
| PF selector same-well physical branch | public score can be dominated by a valid but overlap-dependent shortcut | treat `PF_SELECTOR_USE_SAME_WELL_PHYSICAL=True` as public-aggressive; use `False` for robustness diagnostics |
| Row random split | same-well autocorrelation leaks across folds | use `GroupKFold(well_id)` |
| Train-only surfaces | hidden test does not contain these columns | use them only through spatial imputers |
| Prefix `TVT_input` | valid only before Prediction Start | use prefix rows only |
| Tail length / centered windows | future covariates | offline features only |
| True tail TVT alignment | direct target leakage | never use hidden-tail target values |
| GR calibration | tail GR can overfit local noise | fit affine GR calibration on prefix only |

💡 **Anchor sanity check**

$$
H_0: y_{w,i}=y_{w,PS-1}
$$

- ⚓ The flat anchor is hard to beat on stable wells.
- 📈 Residual features should help drifting wells without making flat wells worse.


### 0.2 Public/Private Feature Policy

The same feature can be safe in one evaluation setting and risky in another. The useful distinction is not just *target leakage* vs *no leakage*, but also whether a signal depends on train/test overlap.

| Signal family | Direct target leakage? | Public/PB behavior | Private/generalization risk |
|---|---:|---|---|
| Prefix `TVT_input` statistics | No | stable anchor | low, if only prefix rows are used |
| Hidden GR full sequence | No | strong batch signal | medium; assumes the full hidden log is available at inference |
| Centered GR windows | No | useful for offline batch | not real-time geosteering, but acceptable for Kaggle batch inference |
| Formation columns copied directly | Yes for hidden inference | not reproducible on test | exclude |
| Spatially imputed formation surfaces | No, if fit without validation/test targets | useful geology prior | depends on spatial extrapolation quality |
| Same-well physical/contact path | No hidden-tail TVT read | can be very strong when public test overlaps train wells | high if private wells are non-overlapping or structurally different |
| PF/beam selector from GR/typewell | No hidden-tail TVT read | strong target-free alignment signal | depends on typewell correlation and GR gap quality |

A high score from `PF_SELECTOR_USE_SAME_WELL_PHYSICAL=True` should therefore be interpreted as an overlap-enabled geological shortcut, not as a universal unseen-well guarantee. The useful diagnostic pair is:

```python
# Public-overlap reproduction
SUBMISSION_PROFILE = 'fast_pf_selector'
PF_SELECTOR_USE_SAME_WELL_PHYSICAL = True

# Unseen-well robustness probe
SUBMISSION_PROFILE = 'fast_pf_selector'
PF_SELECTOR_USE_SAME_WELL_PHYSICAL = False
```


In [ ]:
from pathlib import Path
from IPython.display import Image, display

figure_image_path = Path("/kaggle/input/datasets/pilkwang/pilkwang-public-dataset-for-notebooks-figures/ROGII_Graph_Fig8.png")
if figure_image_path.exists():
    display(Image(filename=str(figure_image_path)))


**Figure 5. Public-aggressive versus private-safe feature policies.**  
Target-free PF and beam tracking use only observed test logs and typewell information. Same-well physical contact estimates can exploit train/test well overlap and are therefore separated as a public-aggressive policy.


In [ ]:
from pathlib import Path
from IPython.display import Image, display

figure_image_path = Path("/kaggle/input/datasets/pilkwang/pilkwang-public-dataset-for-notebooks-figures/ROGII_Graph_Fig11.png")
if figure_image_path.exists():
    display(Image(filename=str(figure_image_path)))


**Figure 6. Same-well physical contact estimate.**  
When a test well is also present in the training set, formation-contact geometry can produce a strong physical TVT estimate. This is treated as a public-aggressive overlap policy and is separated from private-safe target-free tracking.


In [ ]:
# Configure imports, data locations, and display settings.

from pathlib import Path
from collections import Counter
import json
import gc
import math
import re
import zipfile
import xml.etree.ElementTree as ET
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter, MaxNLocator
from pandas.errors import PerformanceWarning

try:
    from scipy.spatial import cKDTree
except Exception:
    cKDTree = None

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:.5g}')
sns.set_theme(style='whitegrid')
warnings.filterwarnings('ignore', category=PerformanceWarning)


# Compact, readable numeric formatting for EDA plots.
ACRONYM_LABELS = {
    'gr': 'GR',
    'tvt': 'TVT',
    'rmse': 'RMSE',
    'mae': 'MAE',
    'md': 'MD',
    'xy': 'XY',
    'pf': 'PF',
    'ancc': 'ANCC',
    'astnu': 'ASTNU',
    'astnl': 'ASTNL',
    'egfdu': 'EGFDU',
    'egfdl': 'EGFDL',
    'buda': 'BUDA',
    'loo': 'LOO',
    'knn': 'KNN',
}


def pretty_label(text: str | None) -> str:
    if text is None:
        return ''
    raw = str(text).strip()
    if not raw:
        return raw
    if raw.lower() == 'count':
        return 'Count'
    parts = raw.replace('__', '_').replace('_', ' ').split()
    return ' '.join(ACRONYM_LABELS.get(part.lower(), part) for part in parts)


def compact_number(value, pos=None, span: float | None = None, mode: str = 'auto') -> str:
    try:
        value = float(value)
    except Exception:
        return str(value)
    if not np.isfinite(value):
        return ''
    if mode == 'count':
        if abs(value - round(value)) < 1e-8:
            return str(int(round(value)))
        return f'{value:.1f}'.rstrip('0').rstrip('.')
    if mode == 'rate':
        if abs(value) < 1e-12:
            return '0'
        if abs(value - 1.0) < 1e-12:
            return '1'
        return f'{value:.3f}'.rstrip('0').rstrip('.')
    if mode == 'corr':
        if abs(value) < 1e-12:
            return '0'
        return f'{value:.3f}'.rstrip('0').rstrip('.')
    av = abs(value)
    span = abs(float(span)) if span is not None and np.isfinite(span) else None
    if av < 1e-12:
        return '0'
    if av >= 1_000_000:
        if span is not None and span < 20_000:
            return f'{value / 1_000:.1f}k'
        return f'{value / 1_000_000:.3g}M'
    if av >= 10_000:
        if span is not None and span < 500:
            return f'{value:.0f}'
        if span is not None and span < 20_000:
            return f'{value / 1_000:.2f}'.rstrip('0').rstrip('.') + 'k'
        return f'{value / 1_000:.3g}k'
    if av >= 100:
        return f'{value:.0f}'
    if av >= 10:
        return f'{value:.1f}'.rstrip('0').rstrip('.')
    if av >= 1:
        return f'{value:.2f}'.rstrip('0').rstrip('.')
    if av >= 0.01:
        return f'{value:.3f}'.rstrip('0').rstrip('.')
    return f'{value:.2g}'


def axis_looks_categorical(axis) -> bool:
    labels = [tick.get_text() for tick in axis.get_ticklabels()]
    return any(re.search(r'[A-Za-z가-힣_]', label) for label in labels if label)


def axis_span(axis) -> float:
    lo, hi = axis.get_view_interval()
    return float(abs(hi - lo))


def axis_format_mode(axis, label: str) -> str:
    low = label.lower()
    if low == 'count' or low.endswith(' count') or low in {'well count', 'row count'}:
        return 'count'
    if any(token in low for token in ['missing rate', 'rate', 'fraction', 'share']):
        return 'rate'
    if any(token in low for token in ['corr', 'correlation']):
        return 'corr'
    return 'auto'


def shorten_legend_text(text: str) -> str:
    text = str(text)
    text = text.replace('hidden_gr_missing_rate', 'hidden GR missing rate')
    text = text.replace('prefix_gr_missing_rate', 'prefix GR missing rate')
    text = text.replace('selector_variant', 'selector')
    text = text.replace('constant_tail_rmse', 'constant RMSE')
    text = text.replace('tail_end_delta_from_last_known', 'tail-end drift')
    
    cleaned = pretty_label(text)
    if cleaned.lower() == 'selector label':
        return 'selector'
    return cleaned


def polish_legend(ax, mode: str = 'right') -> bool:
    legend = ax.get_legend()
    if legend is None:
        return False
    title = legend.get_title()
    if title is not None:
        title.set_text(shorten_legend_text(title.get_text()))
        title.set_fontsize(8)
    for text in legend.get_texts():
        text.set_text(shorten_legend_text(text.get_text()))
        text.set_fontsize(8)
    legend.get_frame().set_alpha(0.88)
    legend.get_frame().set_linewidth(0.8)
    try:
        if mode == 'below':
            label_count = max(1, len(legend.get_texts()))
            ncol = min(3, label_count)
            sns.move_legend(
                ax,
                'upper center',
                bbox_to_anchor=(0.5, -0.20),
                borderaxespad=0.0,
                frameon=True,
                fontsize=7,
                title_fontsize=7,
                ncol=ncol,
            )
        else:
            sns.move_legend(
                ax,
                'upper left',
                bbox_to_anchor=(1.02, 1.0),
                borderaxespad=0.0,
                frameon=True,
                fontsize=8,
                title_fontsize=8,
            )
    except Exception:
        if mode == 'below':
            legend.set_bbox_to_anchor((0.5, -0.20))
            legend._loc = 9
        else:
            legend.set_bbox_to_anchor((1.02, 1.0))
            legend._loc = 2
    return True


def polish_axis(ax, nbins: int = 6, move_legend: bool = True, legend_mode: str = 'right') -> bool:
    ax.tick_params(axis='both', labelsize=9)
    ax.grid(True, alpha=0.20, linewidth=0.8)
    for spine in ax.spines.values():
        spine.set_alpha(0.35)

    x_label = pretty_label(ax.get_xlabel())
    y_label = pretty_label(ax.get_ylabel())
    ax.set_xlabel(x_label, fontsize=10)
    ax.set_ylabel(y_label, fontsize=10)

    if not axis_looks_categorical(ax.xaxis):
        x_span = axis_span(ax.xaxis)
        x_mode = axis_format_mode(ax.xaxis, x_label)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=nbins, integer=(x_mode == 'count'), prune=None))
        ax.xaxis.set_major_formatter(FuncFormatter(lambda value, pos: compact_number(value, pos, x_span, x_mode)))
    else:
        ax.tick_params(axis='x', labelrotation=20)

    if not axis_looks_categorical(ax.yaxis):
        y_span = axis_span(ax.yaxis)
        y_mode = axis_format_mode(ax.yaxis, y_label)
        ax.yaxis.set_major_locator(MaxNLocator(nbins=nbins, integer=(y_mode == 'count'), prune=None))
        ax.yaxis.set_major_formatter(FuncFormatter(lambda value, pos: compact_number(value, pos, y_span, y_mode)))

    title = ax.get_title()
    if title:
        ax.set_title(title, fontsize=11, pad=8)
    return polish_legend(ax, mode=legend_mode) if move_legend else False


def polish_current_figure(fig=None, nbins: int = 6, move_legend: bool = True) -> None:
    fig = fig or plt.gcf()
    data_axes = [ax for ax in fig.axes if ax.has_data()]
    legend_mode = 'below' if len(data_axes) > 1 else 'right'
    moved_any = False
    for ax in fig.axes:
        moved_any = polish_axis(ax, nbins=nbins, move_legend=move_legend, legend_mode=legend_mode) or moved_any
    if moved_any:
        if legend_mode == 'below':
            fig.subplots_adjust(bottom=0.28, hspace=0.42, wspace=0.28)
        else:
            fig.subplots_adjust(right=0.78)

# Resolve competition data. Kaggle input mounts are preferred; local runs fall back to ./.
KAGGLE_DATA_DIRS = [
    Path('/kaggle/input/rogii-wellbore-geology-prediction'),
    Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction'),
]
LOCAL_DATA_DIR = Path('.')
CANDIDATE_DATA_DIRS = KAGGLE_DATA_DIRS + [LOCAL_DATA_DIR]
DATA_DIR = next(
    (
        p
        for p in CANDIDATE_DATA_DIRS
        if (p / 'train').exists() and (p / 'sample_submission.csv').exists()
    ),
    LOCAL_DATA_DIR,
)
TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR = DATA_DIR / 'test'
SAMPLE_SUBMISSION = DATA_DIR / 'sample_submission.csv'
PPTX_PATH = DATA_DIR / 'AI_wellbore_geology_prediction_task_en.pptx'
KAGGLE_WORKING_DIR = Path('/kaggle/working')
KAGGLE_NOTEBOOK_RUN = KAGGLE_WORKING_DIR.exists()
OUTPUT_DIR = KAGGLE_WORKING_DIR if KAGGLE_NOTEBOOK_RUN else DATA_DIR
FINAL_SUBMISSION_OUTPUT = OUTPUT_DIR / 'submission.csv'
LIGHTGBM_DEVICE_TYPE = 'gpu' if KAGGLE_NOTEBOOK_RUN else 'cpu'


def lightgbm_accelerator_params(device_type: str | None = None) -> dict:
    device_type = device_type or LIGHTGBM_DEVICE_TYPE
    if device_type == 'gpu':
        memory_safe = bool(globals().get('KAGGLE_MEMORY_SAFE_MODE', False))
        return {
            'device_type': 'gpu',
            'gpu_use_dp': False,
            'max_bin': 127 if memory_safe else 255,
        }
    return {
        'device_type': 'cpu',
        'force_col_wise': True,
    }


DATA_DIR_LABEL = './' if DATA_DIR == LOCAL_DATA_DIR else DATA_DIR.as_posix()
print('DATA_DIR:', DATA_DIR_LABEL)
print('train exists:', TRAIN_DIR.exists())
print('test exists:', TEST_DIR.exists())
print('sample_submission exists:', SAMPLE_SUBMISSION.exists())
print('OUTPUT_DIR:', OUTPUT_DIR.as_posix() if OUTPUT_DIR != DATA_DIR else DATA_DIR_LABEL)
print('FINAL_SUBMISSION_OUTPUT:', FINAL_SUBMISSION_OUTPUT.as_posix())
print('LightGBM device type:', LIGHTGBM_DEVICE_TYPE)


## 1. File Inventory

- 🧾 Join everything by `well_id`.
- ✅ Check train horizontal/typewell/PNG ids.
- ✅ Check test horizontal/typewell ids.
- ✅ Check sample ids against hidden `TVT_input` rows.

💡 A mismatched horizontal/typewell pair breaks GR alignment before modeling even starts.


In [ ]:
# Build file lists, extract well ids, and verify horizontal/typewell/image matching.

def well_id_from_path(path: Path) -> str:
    name = path.name
    if '__' in name:
        return name.split('__')[0]
    return path.stem

train_horizontal_files = sorted(TRAIN_DIR.glob('*__horizontal_well.csv'))
train_typewell_files = sorted(TRAIN_DIR.glob('*__typewell.csv'))
train_png_files = sorted(TRAIN_DIR.glob('*.png'))
test_horizontal_files = sorted(TEST_DIR.glob('*__horizontal_well.csv')) if TEST_DIR.exists() else []
test_typewell_files = sorted(TEST_DIR.glob('*__typewell.csv')) if TEST_DIR.exists() else []

file_inventory = pd.DataFrame({
    'group': ['train_horizontal', 'train_typewell', 'train_png', 'test_horizontal', 'test_typewell'],
    'count': [len(train_horizontal_files), len(train_typewell_files), len(train_png_files), len(test_horizontal_files), len(test_typewell_files)],
})
display(file_inventory)

train_h_ids = {well_id_from_path(p) for p in train_horizontal_files}
train_t_ids = {well_id_from_path(p) for p in train_typewell_files}
train_png_ids = {p.stem for p in train_png_files}
test_h_ids = {well_id_from_path(p) for p in test_horizontal_files}
test_t_ids = {well_id_from_path(p) for p in test_typewell_files}

checks = {
    'train horizontal/typewell/png id sets equal': train_h_ids == train_t_ids == train_png_ids,
    'test horizontal/typewell id sets equal': test_h_ids == test_t_ids,
}
print(json.dumps(checks, indent=2))
print('test ids:', sorted(test_h_ids)[:10])


### 1.1 Task Description Signals

- 🧭 Main hint: horizontal GR can be aligned to typewell GR on the TVT axis.
- 🧵 This motivates beam search, DTW, and local typewell residual features.


In [ ]:
# Extract task-description text used to identify domain signals and constraints.

def extract_pptx_slide_text(pptx_path: Path) -> pd.DataFrame:
    if not pptx_path.exists():
        return pd.DataFrame(columns=['slide', 'text'])
    rows = []
    with zipfile.ZipFile(pptx_path) as zf:
        slide_names = sorted(
            [name for name in zf.namelist() if re.match(r'ppt/slides/slide\d+\.xml$', name)],
            key=lambda name: int(re.search(r'slide(\d+)\.xml$', name).group(1)),
        )
        for slide_name in slide_names:
            root = ET.fromstring(zf.read(slide_name))
            ns_text = '{http://schemas.openxmlformats.org/drawingml/2006/main}t'
            text_parts = [node.text.strip() for node in root.iter(ns_text) if node.text and node.text.strip()]
            rows.append({'slide': slide_name, 'text': ' | '.join(text_parts)})
    return pd.DataFrame(rows)

pptx_text = extract_pptx_slide_text(PPTX_PATH)
print('slide_count:', len(pptx_text))
display(pptx_text.head(14))


## 2. Column Check and Representative Well

- 🧪 Train has `TVT` labels and formation columns.
- 🧾 Hidden test rows only expose `MD`, `X/Y/Z`, `GR`, and prefix `TVT_input`.
- 🛡️ Final features must be reproducible from the hidden-test schema.


In [ ]:
# Inspect one representative well for schema, dtypes, and missingness.

representative_well_id = sorted(train_h_ids)[0]
print('representative_well_id:', representative_well_id)

representative_train_h = pd.read_csv(TRAIN_DIR / f'{representative_well_id}__horizontal_well.csv')
representative_typewell = pd.read_csv(TRAIN_DIR / f'{representative_well_id}__typewell.csv')
representative_test_h = pd.read_csv(TEST_DIR / f'{representative_well_id}__horizontal_well.csv') if (TEST_DIR / f'{representative_well_id}__horizontal_well.csv').exists() else None

print('train horizontal shape:', representative_train_h.shape)
print('train horizontal columns:', list(representative_train_h.columns))
print('typewell shape:', representative_typewell.shape)
print('typewell columns:', list(representative_typewell.columns))
if representative_test_h is not None:
    print('test horizontal shape:', representative_test_h.shape)
    print('test horizontal columns:', list(representative_test_h.columns))

print('\nTrain horizontal missing counts:')
display(representative_train_h.isna().sum().to_frame('missing'))
print('\nTypewell missing counts:')
display(representative_typewell.isna().sum().to_frame('missing'))

print('\nTrain horizontal head:')
display(representative_train_h.head())
print('\nTypewell head:')
display(representative_typewell.head())


## 3. Prediction Zone and Submission Mapping

- 🎯 Predict only rows where `TVT_input` is missing.
- 🧩 Each id is `{well_id}_{row_index}`.
- ✅ `submission.csv` must preserve sample order exactly.

💡 Each test well has one hidden tail, so this is prefix-conditioned forecasting rather than random interpolation.


In [ ]:
from pathlib import Path
from IPython.display import Image, display

figure_image_path = Path("/kaggle/input/datasets/pilkwang/pilkwang-public-dataset-for-notebooks-figures/ROGII_Graph_Fig2.png")
if figure_image_path.exists():
    display(Image(filename=str(figure_image_path)))


**Figure 7. Residual target around the last-known TVT anchor.**  
The model predicts $\Delta TVT_i = TVT_i - T_L$ rather than raw TVT. This reduces well-to-well offset variation and focuses the model on hidden-tail stratigraphic drift.


In [ ]:
# Define prediction tails and verify sample_submission ids match missing TVT_input rows.

def prediction_zone_info(df: pd.DataFrame) -> dict:
    mask = df['TVT_input'].isna().to_numpy()
    null_indices = np.flatnonzero(mask)
    groups = 0
    in_group = False
    for value in mask:
        if value and not in_group:
            groups += 1
            in_group = True
        elif not value:
            in_group = False
    return {
        'n_rows': len(df),
        'known_rows': int((~mask).sum()),
        'prediction_rows': int(mask.sum()),
        'first_prediction_index': int(null_indices[0]) if len(null_indices) else None,
        'last_prediction_index': int(null_indices[-1]) if len(null_indices) else None,
        'n_missing_groups': groups,
    }

print('Representative prediction zone:')
display(pd.Series(prediction_zone_info(representative_train_h)).to_frame('value'))

if SAMPLE_SUBMISSION.exists() and test_horizontal_files:
    sample = pd.read_csv(SAMPLE_SUBMISSION)
    expected_ids = []
    for path in test_horizontal_files:
        wid = well_id_from_path(path)
        df = pd.read_csv(path)
        pred_idx = np.flatnonzero(df['TVT_input'].isna().to_numpy())
        expected_ids.extend([f'{wid}_{i}' for i in pred_idx])
    print('sample_submission rows:', len(sample))
    print('test prediction ids:', len(expected_ids))
    print('exact id set match:', set(sample['id']) == set(expected_ids))
    display(sample.head())


## 📊 4. Horizontal Well Aggregate Summary

- 📏 Prefix length: how much anchor history is available.
- 🧵 Tail length: how much each well contributes to row-level loss.
- 🌋 GR missingness: how reliable alignment features can be.
- ⚓ Constant-anchor error: how hard the well is before modeling.

🛡️ Target-derived tail summaries stay in EDA/diagnostics; they are not strict features.


In [ ]:
# Build well-level summaries for prefix length, tail length, GR missingness, geometry, and baseline difficulty.

def summarize_horizontal_file(path: Path) -> dict:
    wid = well_id_from_path(path)
    df = pd.read_csv(path)
    mask = df['TVT_input'].isna().to_numpy()
    pred_idx = np.flatnonzero(mask)
    first_pred = int(pred_idx[0]) if len(pred_idx) else len(df)
    last_known_idx = max(first_pred - 1, 0)
    y = df['TVT'].to_numpy() if 'TVT' in df.columns else np.full(len(df), np.nan)
    tail_y = y[mask] if 'TVT' in df.columns else np.array([])
    tvt_d = pd.Series(y).diff().to_numpy() if 'TVT' in df.columns else np.array([])
    dx = float(df['X'].iloc[-1] - df['X'].iloc[0])
    dy = float(df['Y'].iloc[-1] - df['Y'].iloc[0])
    dz = float(df['Z'].iloc[-1] - df['Z'].iloc[0])
    azimuth = (np.degrees(np.arctan2(dx, dy)) + 360.0) % 360.0
    prefix = df.loc[~mask]
    tail = df.loc[mask]
    return {
        'well_id': wid,
        'n_rows': len(df),
        'md_start': float(df['MD'].iloc[0]),
        'md_end': float(df['MD'].iloc[-1]),
        'md_span': float(df['MD'].iloc[-1] - df['MD'].iloc[0]),
        'known_rows': int((~mask).sum()),
        'tail_rows': int(mask.sum()),
        'ps_index': first_pred,
        'missing_tvt_input_groups': prediction_zone_info(df)['n_missing_groups'],
        'last_known_tvt': float(df['TVT_input'].iloc[last_known_idx]) if len(prefix) else np.nan,
        'ps_x': float(df['X'].iloc[first_pred]) if len(pred_idx) else np.nan,
        'ps_y': float(df['Y'].iloc[first_pred]) if len(pred_idx) else np.nan,
        'ps_z': float(df['Z'].iloc[first_pred]) if len(pred_idx) else np.nan,
        'x_start': float(df['X'].iloc[0]),
        'y_start': float(df['Y'].iloc[0]),
        'z_start': float(df['Z'].iloc[0]),
        'x_end': float(df['X'].iloc[-1]),
        'y_end': float(df['Y'].iloc[-1]),
        'z_end': float(df['Z'].iloc[-1]),
        'xy_span': float(np.hypot(dx, dy)),
        'z_delta': dz,
        'azimuth_deg': float(azimuth),
        'gr_missing_rows': int(df['GR'].isna().sum()),
        'gr_missing_rate': float(df['GR'].isna().mean()),
        'gr_missing_prefix_rate': float(prefix['GR'].isna().mean()) if len(prefix) else np.nan,
        'gr_missing_tail_rate': float(tail['GR'].isna().mean()) if len(tail) else np.nan,
        'tvt_start': float(y[0]) if len(y) else np.nan,
        'tvt_end': float(y[-1]) if len(y) else np.nan,
        'tvt_end_minus_start': float(y[-1] - y[0]) if len(y) else np.nan,
        'tail_tvt_range': float(np.nanmax(tail_y) - np.nanmin(tail_y)) if len(tail_y) else np.nan,
        'tail_end_delta_from_last_known': float(tail_y[-1] - df['TVT_input'].iloc[last_known_idx]) if len(tail_y) else np.nan,
        'tail_median_abs_step': float(np.nanmedian(np.abs(np.diff(tail_y)))) if len(tail_y) > 1 else np.nan,
        'whole_median_abs_step': float(np.nanmedian(np.abs(tvt_d[1:]))) if len(tvt_d) > 1 else np.nan,
        'constant_tail_rmse': float(np.sqrt(np.nanmean((tail_y - df['TVT_input'].iloc[last_known_idx]) ** 2))) if len(tail_y) else np.nan,
    }

h_summary = pd.DataFrame([summarize_horizontal_file(path) for path in train_horizontal_files])
print('h_summary shape:', h_summary.shape)
display(h_summary.head())


In [ ]:
# Summarize well-level distributions that affect validation and model design.

summary_cols = [
    'n_rows', 'known_rows', 'tail_rows', 'ps_index',
    'gr_missing_rate', 'gr_missing_prefix_rate', 'gr_missing_tail_rate',
    'tail_tvt_range', 'tail_end_delta_from_last_known', 'tail_median_abs_step',
    'constant_tail_rmse', 'xy_span', 'z_delta', 'azimuth_deg',
]

display(h_summary[summary_cols].describe(percentiles=[0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).T)

print('Prediction zone missing group counts:')
print(h_summary['missing_tvt_input_groups'].value_counts().sort_index())

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
plot_specs = [
    ('n_rows', 'Rows per horizontal well'),
    ('known_rows', 'Known TVT_input prefix length'),
    ('tail_rows', 'Prediction tail length'),
    ('gr_missing_rate', 'GR missing rate'),
    ('tail_tvt_range', 'Tail TVT range'),
    ('constant_tail_rmse', 'Last-known constant RMSE by well'),
]
for ax, (col, title) in zip(axes.ravel(), plot_specs):
    sns.histplot(h_summary[col].dropna(), bins=40, ax=ax)
    ax.set_title(title)
polish_current_figure()
plt.tight_layout()
plt.show()


### 4.1 Interpretation: Horizontal Summary

| Finding | Modeling implication |
|---|---|
| Missing `TVT_input` forms one tail block | Treat the task as prefix-conditioned forecasting |
| Tail lengths are often thousands of rows | Small slope bias can accumulate |
| GR missingness is substantial | Avoid GR-only models |
| Tail TVT ranges are often tens of feet | `last_known_TVT` is a strong anchor |

Residual target:

$$
\Delta y_{w,i}=y_{w,i}-y_{w,\mathrm{PS}-1}.
$$

🚫 Diagnostic labels such as `tail_rows`, `tail_tvt_range`, `tail_end_delta_from_last_known`, and `constant_tail_rmse` are not strict features.


### 4.2 Geosteering Geometry Diagnostics

Horizontal wells are not just rows in a table. `MD`, `X`, `Y`, and `Z` describe how the well path moves through a stratigraphic volume. If formation surfaces are locally smooth, vertical movement and trajectory curvature should affect TVT drift.

Useful path proxies:

$$
\frac{dZ}{dMD}_i = \frac{Z_i-Z_{i-1}}{MD_i-MD_{i-1}},\qquad
 dXY_i=\sqrt{(X_i-X_{i-1})^2+(Y_i-Y_{i-1})^2}
$$

A simple curvature proxy is the local change in the normalized trajectory direction:

$$
\kappa_i \approx \sqrt{\Delta(dX/dMD)^2+\Delta(dY/dMD)^2+\Delta(dZ/dMD)^2}
$$


In [ ]:
# Geosteering trajectory diagnostics: path slope, curvature, hidden displacement, and baseline difficulty.

def _nan_stat(values, fn, default=np.nan):
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return default
    return float(fn(arr))


def longest_true_run(mask) -> int:
    best = cur = 0
    for flag in np.asarray(mask, dtype=bool):
        if flag:
            cur += 1
            best = max(best, cur)
        else:
            cur = 0
    return int(best)


def summarize_geosteering_path(path: Path, split: str) -> dict:
    wid = well_id_from_path(path)
    df = pd.read_csv(path, usecols=['MD', 'X', 'Y', 'Z', 'GR', 'TVT_input'])
    pred_mask = df['TVT_input'].isna().to_numpy()
    tail = df.loc[pred_mask]
    md = df['MD'].to_numpy(dtype=float)
    x = df['X'].to_numpy(dtype=float)
    y = df['Y'].to_numpy(dtype=float)
    z = df['Z'].to_numpy(dtype=float)
    dmd = np.diff(md)
    safe_dmd = np.where(np.abs(dmd) > 1e-9, dmd, np.nan)
    dx_dmd = np.diff(x) / safe_dmd
    dy_dmd = np.diff(y) / safe_dmd
    dz_dmd = np.diff(z) / safe_dmd
    if len(dx_dmd) >= 2:
        curvature = np.sqrt(np.diff(dx_dmd) ** 2 + np.diff(dy_dmd) ** 2 + np.diff(dz_dmd) ** 2)
    else:
        curvature = np.array([], dtype=float)
    if len(tail):
        hidden_xy_span = float(np.hypot(tail['X'].iloc[-1] - tail['X'].iloc[0], tail['Y'].iloc[-1] - tail['Y'].iloc[0]))
        hidden_z_span = float(tail['Z'].max() - tail['Z'].min())
        hidden_md_span = float(tail['MD'].iloc[-1] - tail['MD'].iloc[0])
    else:
        hidden_xy_span = hidden_z_span = hidden_md_span = np.nan
    return {
        'split': split,
        'well_id': wid,
        'rows': int(len(df)),
        'md_monotonic': bool(pd.Series(md).is_monotonic_increasing),
        'hidden_rows': int(pred_mask.sum()),
        'hidden_md_span': hidden_md_span,
        'hidden_xy_span': hidden_xy_span,
        'hidden_z_span': hidden_z_span,
        'dz_dmd_median': _nan_stat(dz_dmd, np.median),
        'dz_dmd_p95_abs': _nan_stat(np.abs(dz_dmd), lambda a: np.quantile(a, 0.95)),
        'curvature_median': _nan_stat(curvature, np.median),
        'curvature_p95': _nan_stat(curvature, lambda a: np.quantile(a, 0.95)),
        'gr_missing_rate': float(df['GR'].isna().mean()),
        'hidden_gr_missing_rate': float(tail['GR'].isna().mean()) if len(tail) else np.nan,
        'longest_gr_nan_run': longest_true_run(df['GR'].isna().to_numpy()),
        'hidden_longest_gr_nan_run': longest_true_run(tail['GR'].isna().to_numpy()) if len(tail) else 0,
    }

geo_summary = pd.DataFrame(
    [summarize_geosteering_path(path, 'train') for path in train_horizontal_files]
    + [summarize_geosteering_path(path, 'test') for path in test_horizontal_files]
)
geo_train = geo_summary.query("split == 'train'").merge(
    h_summary[['well_id', 'constant_tail_rmse', 'tail_tvt_range', 'tail_end_delta_from_last_known']],
    on='well_id',
    how='left',
)

print('Geosteering summary shape:', geo_summary.shape)
display(geo_summary.groupby('split')[['hidden_rows', 'hidden_md_span', 'hidden_xy_span', 'hidden_z_span', 'hidden_gr_missing_rate', 'hidden_longest_gr_nan_run']].describe().T)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.scatterplot(data=geo_train, x='hidden_z_span', y='constant_tail_rmse', hue='hidden_gr_missing_rate', palette='viridis', ax=axes[0])
axes[0].set_title('Hidden Z span vs constant-anchor RMSE')
sns.scatterplot(data=geo_train, x='hidden_xy_span', y='tail_tvt_range', hue='dz_dmd_p95_abs', palette='magma', ax=axes[1])
axes[1].set_title('Hidden XY displacement vs tail TVT range')
sns.histplot(data=geo_summary, x='hidden_longest_gr_nan_run', hue='split', bins=40, ax=axes[2], element='step')
axes[2].set_title('Longest hidden GR NaN run')
polish_current_figure()
plt.tight_layout()
plt.show()


### 4.3 GR Log Quality and Gap Diagnostics

GR is the main stratigraphic correlation log. Missing GR does not create target leakage by itself, because GR is an observed covariate. The risk is different: long NaN gaps weaken PF/beam likelihood and can make interpolation dominate the signal.

A useful rule of thumb:

| GR condition | Modeling effect |
|---|---|
| short isolated gaps | interpolation is usually harmless |
| long hidden gaps | PF/beam observation likelihood becomes weak |
| high prefix missingness | typewell calibration scale becomes unstable |
| high hidden missingness | selector should lean more on geometry/formation priors |


In [ ]:
# GR quality report for train/test prefix and hidden intervals.

def summarize_gr_quality(path: Path, split: str) -> dict:
    wid = well_id_from_path(path)
    df = pd.read_csv(path, usecols=['GR', 'TVT_input'])
    hidden = df['TVT_input'].isna()
    prefix = ~hidden
    return {
        'split': split,
        'well_id': wid,
        'rows': int(len(df)),
        'prefix_rows': int(prefix.sum()),
        'hidden_rows': int(hidden.sum()),
        'gr_missing_rate': float(df['GR'].isna().mean()),
        'prefix_gr_missing_rate': float(df.loc[prefix, 'GR'].isna().mean()) if prefix.any() else np.nan,
        'hidden_gr_missing_rate': float(df.loc[hidden, 'GR'].isna().mean()) if hidden.any() else np.nan,
        'longest_gr_nan_run': longest_true_run(df['GR'].isna().to_numpy()),
        'prefix_longest_gr_nan_run': longest_true_run(df.loc[prefix, 'GR'].isna().to_numpy()) if prefix.any() else 0,
        'hidden_longest_gr_nan_run': longest_true_run(df.loc[hidden, 'GR'].isna().to_numpy()) if hidden.any() else 0,
    }

gr_quality = pd.DataFrame(
    [summarize_gr_quality(path, 'train') for path in train_horizontal_files]
    + [summarize_gr_quality(path, 'test') for path in test_horizontal_files]
)
gr_train = gr_quality.query("split == 'train'").merge(
    h_summary[['well_id', 'constant_tail_rmse', 'tail_tvt_range']],
    on='well_id',
    how='left',
)

display(gr_quality.groupby('split')[['prefix_gr_missing_rate', 'hidden_gr_missing_rate', 'prefix_longest_gr_nan_run', 'hidden_longest_gr_nan_run']].describe().T)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.histplot(data=gr_quality, x='hidden_gr_missing_rate', hue='split', bins=40, element='step', ax=axes[0])
axes[0].set_title('Hidden GR missing rate')
sns.scatterplot(data=gr_train, x='hidden_gr_missing_rate', y='constant_tail_rmse', size='hidden_longest_gr_nan_run', sizes=(20, 180), ax=axes[1])
axes[1].set_title('GR missingness vs anchor difficulty')
sns.scatterplot(data=gr_train, x='hidden_longest_gr_nan_run', y='tail_tvt_range', hue='prefix_gr_missing_rate', palette='viridis', ax=axes[2])
axes[2].set_title('Hidden GR gap length vs TVT range')
polish_current_figure()
plt.tight_layout()
plt.show()


In [ ]:
from pathlib import Path
from IPython.display import Image, display

figure_image_path = Path("/kaggle/input/datasets/pilkwang/pilkwang-public-dataset-for-notebooks-figures/ROGII_Graph_Fig4.png")
if figure_image_path.exists():
    display(Image(filename=str(figure_image_path)))


**Figure 8. Gamma ray as a stratigraphic barcode.**  
Hidden horizontal GR is aligned against typewell GR to infer stratigraphic position. Missing GR gaps are interpolated before PF/beam tracking so that the observation likelihood remains continuous.


## 5. Leakage Boundary and Column Roles

### 🗂️ Column role map

| Role | Examples | How the notebook uses it |
|---|---|---|
| **Observable covariates** | `MD`, `X`, `Y`, `Z`, `GR`, typewell logs | direct row features |
| **Known-prefix target** | prefix `TVT_input` | anchor, prefix statistics, GR calibration pairs |
| **Hidden-tail target** | tail `TVT` | labels for train/CV only; never features |
| **Train-only formation surfaces** | `ANCC`, `ASTNU`, `ASTNL`, `EGFDU`, `EGFDL`, `BUDA` | auxiliary labels for fold-safe spatial imputers |

### ✅ Safe pattern

| Step | Action | Leakage status |
|---|---|---:|
| 1 | fit `(X,Y) -> formation top` from training wells | ✅ reference model |
| 2 | project `formation_hat(X,Y)` onto validation/test rows | ✅ reproducible feature |
| 3 | build `TVT ≈ -Z + formation_hat + prefix_bias` | ✅ target-free formula |

### 🚫 Unsafe pattern

| Pattern | Problem |
|---|---|
| use `ANCC` directly as a feature | hidden test horizontal files do not provide it |
| fit the imputer with validation wells inside GroupKFold | fold leakage |
| use tail `TVT` summaries or bfilled target values | direct answer-key leakage |

📌 **Clean distinction:** formation tops are not observed test features, but they can define a fold-safe spatial reference model.


In [ ]:
# Classify columns by hidden-test availability and leakage risk.

train_horizontal_columns = set(representative_train_h.columns)
test_horizontal_columns = set(representative_test_h.columns) if representative_test_h is not None else {'MD', 'X', 'Y', 'Z', 'GR', 'TVT_input'}

column_roles = []
for col in representative_train_h.columns:
    if col == 'TVT':
        role = 'target_train_only'
    elif col in {'ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA'}:
        role = 'train_only_surface_diagnostic'
    elif col in test_horizontal_columns:
        role = 'safe_hidden_test_feature'
    else:
        role = 'unknown_train_only'
    column_roles.append({'column': col, 'role': role})

display(pd.DataFrame(column_roles))


## 6. TVT_input Consistency Check

- ✅ `TVT_input` should exactly equal `TVT` in the known prefix.
- ⚓ If true, `last_known_TVT` is a clean anchor.
- 📈 Prefix slopes and ranges can be built from `TVT_input` only.


In [ ]:
# Verify that TVT_input equals TVT throughout the known prefix.

max_abs_prefix_errors = []
for path in train_horizontal_files:
    df = pd.read_csv(path, usecols=['TVT', 'TVT_input'])
    known = df['TVT_input'].notna()
    if known.any():
        max_abs_prefix_errors.append(float((df.loc[known, 'TVT'] - df.loc[known, 'TVT_input']).abs().max()))

print('max over wells of max_abs(TVT - TVT_input) in known prefix:', np.nanmax(max_abs_prefix_errors))
print('wells with non-zero prefix mismatch:', sum(err > 1e-9 for err in max_abs_prefix_errors))


## 7. Target Behavior, Smoothness, and Jumps

- 📉 TVT is usually curve-like, not independent row noise.
- 🧯 Large jumps are suspicious and should be guarded by post-processing.
- 🧵 Smoothness supports fade-in, slope clipping, and optional curve smoothing.


In [ ]:
# Measure TVT step smoothness, tail drift, and jump frequency.

# Collect dTVT step distribution and jump counts.
dtvt_values = []
jump_counts = Counter()
for path in train_horizontal_files:
    df = pd.read_csv(path, usecols=['TVT'])
    d = df['TVT'].diff().dropna().to_numpy()
    dtvt_values.append(d)
    for threshold in [0.1, 0.5, 1, 2, 5, 10, 25, 50, 100]:
        jump_counts[f'abs_dTVT_gt_{threshold}'] += int(np.sum(np.abs(d) > threshold))

dtvt_values = np.concatenate(dtvt_values)
print('dTVT step percentiles:')
display(pd.Series(dtvt_values).describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).to_frame('dTVT'))
print('\nLarge jump counts:')
display(pd.Series(jump_counts).to_frame('count'))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.histplot(dtvt_values[(dtvt_values > -2) & (dtvt_values < 2)], bins=120, ax=axes[0])
axes[0].set_title('dTVT per 1 ft MD, clipped to [-2, 2]')
sns.histplot(h_summary['tail_end_delta_from_last_known'], bins=50, ax=axes[1])
axes[1].set_title('Tail end delta from last known TVT')
sns.histplot(h_summary['tail_tvt_range'], bins=50, ax=axes[2])
axes[2].set_title('Tail TVT range by well')
polish_current_figure()
plt.tight_layout()
plt.show()


### 7.1 Interpretation: Target Behavior

### What we learned

Most row-level `dTVT` values are small, but wells are not always monotonic. TVT can increase, decrease, or remain nearly constant.

### Modeling implications

1. Blindly extrapolating the prefix slope is risky.
2. Row-wise predictions should use continuity or smoothing postprocessing.
3. A knot-level or curve-level target may be more stable than independent row-level predictions.


## 8. Typewell Data Inventory

- 🧭 Typewell GR is a reference curve indexed by TVT.
- 📏 Sampling density, TVT span, GR range, and geology labels define the reference space.
- 🔗 Alignment confidence and PF residual scale are analyzed in Section 9 after the typewell helper functions are available.


In [ ]:
display_rogii_figure(
    "ROGII_Fig5.png",
    "Figure 9 — Typewell Alignment and Sequence Signals",
    "Typewell GR matching, beam/candidate paths, and prefix self-correlation as physics-style residual signals.",
    width=1120,
)


In [ ]:
# Summarize typewell TVT sampling, GR ranges, and geology labels.

def summarize_typewell_file(path: Path) -> dict:
    wid = well_id_from_path(path)
    df = pd.read_csv(path)
    steps = df['TVT'].diff().dropna().round(6)
    labels = df['Geology'].fillna('<blank>').astype(str)
    return {
        'well_id': wid,
        'n_rows': len(df),
        'tvt_min': float(df['TVT'].min()),
        'tvt_max': float(df['TVT'].max()),
        'tvt_span': float(df['TVT'].max() - df['TVT'].min()),
        'gr_min': float(df['GR'].min()),
        'gr_median': float(df['GR'].median()),
        'gr_max': float(df['GR'].max()),
        'geology_unique_count': int(labels.nunique()),
        'mode_tvt_step': float(steps.mode().iloc[0]) if len(steps) else np.nan,
    }

t_summary = pd.DataFrame([summarize_typewell_file(path) for path in train_typewell_files])
print('t_summary shape:', t_summary.shape)
display(t_summary.head())
display(t_summary.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T)

geology_counter = Counter()
tvt_step_counter = Counter()
for path in train_typewell_files:
    df = pd.read_csv(path)
    geology_counter.update(df['Geology'].fillna('<blank>').astype(str).tolist())
    tvt_step_counter.update(df['TVT'].diff().dropna().round(6).tolist())

print('Top geology labels:')
display(pd.DataFrame(geology_counter.most_common(25), columns=['Geology', 'row_count']))
print('Most common TVT sampling steps:')
display(pd.DataFrame(tvt_step_counter.most_common(10), columns=['TVT_step', 'count']))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.histplot(t_summary['n_rows'], bins=50, ax=axes[0])
axes[0].set_title('Typewell rows per well')
sns.histplot(t_summary['tvt_span'], bins=50, ax=axes[1])
axes[1].set_title('Typewell TVT span')
sns.histplot(t_summary['gr_median'], bins=50, ax=axes[2])
axes[2].set_title('Typewell median GR')
polish_current_figure()
plt.tight_layout()
plt.show()


## 9. Stratigraphic Surface and Typewell Alignment Signals

This section connects two physical views of the same target coordinate.

- 🧭 `TVT + Z` behaves like a structural/formation-surface coordinate plus a well-specific offset.
- 🔗 Known-prefix GR can be compared with typewell GR at the same `TVT_input` to estimate alignment confidence.
- 🌋 GR residual scale gives a natural PF observation-noise proxy.
- ⚠️ Formation columns are train-only observations; use them as diagnostics or through reproducible spatial estimates, not as direct hidden-test inputs.

### 9.1 Known-Prefix GR Alignment Signal

- Compare known-prefix horizontal GR with typewell GR at the same `TVT_input`.
- Good prefix alignment makes typewell path features more trustworthy.
- Low correlation is not fatal: GR is noisy and often missing.


In [ ]:
# Interpolate typewell GR onto known-prefix TVT positions and compute horizontal/typewell GR correlation.

def typewell_gr_at_tvt(typewell_df: pd.DataFrame, tvt_values: np.ndarray) -> np.ndarray:
    tw = typewell_df[['TVT', 'GR']].dropna().sort_values('TVT')
    if len(tw) < 2:
        return np.full(len(tvt_values), np.nan)
    x = tw['TVT'].to_numpy()
    y = tw['GR'].to_numpy()
    pred = np.interp(tvt_values, x, y, left=np.nan, right=np.nan)
    return pred

def safe_corr(a: np.ndarray, b: np.ndarray) -> float:
    valid = np.isfinite(a) & np.isfinite(b)
    if valid.sum() < 30:
        return np.nan
    aa = a[valid]
    bb = b[valid]
    if np.nanstd(aa) < 1e-9 or np.nanstd(bb) < 1e-9:
        return np.nan
    return float(np.corrcoef(aa, bb)[0, 1])

alignment_rows = []
for path in train_horizontal_files:
    wid = well_id_from_path(path)
    h = pd.read_csv(path, usecols=['GR', 'TVT_input'])
    tw = pd.read_csv(TRAIN_DIR / f'{wid}__typewell.csv')
    known = h['TVT_input'].notna()
    hv = h.loc[known, 'GR'].to_numpy()
    tvt = h.loc[known, 'TVT_input'].to_numpy()
    tw_gr = typewell_gr_at_tvt(tw, tvt)
    valid = np.isfinite(hv) & np.isfinite(tw_gr)
    alignment_rows.append({
        'well_id': wid,
        'known_valid_gr_points': int(valid.sum()),
        'prefix_horizontal_vs_typewell_gr_corr': safe_corr(hv, tw_gr),
        'prefix_horizontal_gr_mean': float(np.nanmean(hv)) if np.isfinite(hv).any() else np.nan,
        'prefix_typewell_gr_mean_at_tvt': float(np.nanmean(tw_gr)) if np.isfinite(tw_gr).any() else np.nan,
    })

alignment = pd.DataFrame(alignment_rows).merge(h_summary[['well_id', 'constant_tail_rmse', 'gr_missing_prefix_rate']], on='well_id', how='left')
display(alignment.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(alignment['prefix_horizontal_vs_typewell_gr_corr'].dropna(), bins=50, ax=axes[0])
axes[0].set_title('Known-prefix horizontal GR vs typewell GR correlation')
sns.scatterplot(
    data=alignment,
    x='prefix_horizontal_vs_typewell_gr_corr',
    y='constant_tail_rmse',
    hue='gr_missing_prefix_rate',
    palette='viridis',
    ax=axes[1],
)
axes[1].set_title('Alignment correlation vs constant baseline difficulty')
polish_current_figure()
plt.tight_layout()
plt.show()


In [ ]:
from pathlib import Path
from IPython.display import Image, display

figure_image_path = Path("/kaggle/input/datasets/pilkwang/pilkwang-public-dataset-for-notebooks-figures/ROGII_Graph_Fig3.png")
if figure_image_path.exists():
    display(Image(filename=str(figure_image_path)))


**Figure 10. Formation-surface interpretation of TVT.**  
Formation tops can be treated as spatial surfaces $S(X,Y)$. If $S(X,Y)$ is inferred from neighboring wells, then TVT can be approximated by $-Z + S(X,Y) + b_w$, where $b_w$ is a well-specific offset estimated from the known prefix.


### 9.2 TVT + Z as a Formation-Surface Coordinate

A useful geological approximation is:

$$
TVT \approx -Z + S(X,Y) + b_w
$$

or equivalently:

$$
TVT + Z \approx S(X,Y)+b_w
$$

Here `S(X,Y)` is a formation-top or structural surface, and `b_w` is a well-specific offset. If `TVT + Z - F` is nearly constant inside a well for a formation column `F`, then that formation surface is a meaningful physical prior.

Formation columns are train-only observations. They should not be copied directly into hidden test rows; the safe use is through fold-safe or spatially reproducible surface estimates.


In [ ]:
# Within-well residual stability of TVT + Z - formation-top columns.

candidate_formation_cols = ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']
train_schema = pd.read_csv(train_horizontal_files[0], nrows=0).columns.tolist()
FORMATION_SURFACE_COLUMNS = [c for c in candidate_formation_cols if c in train_schema]
print('FORMATION_SURFACE_COLUMNS:', FORMATION_SURFACE_COLUMNS)

formation_residual_rows = []
for path in train_horizontal_files:
    wid = well_id_from_path(path)
    usecols = ['TVT', 'Z'] + FORMATION_SURFACE_COLUMNS
    df = pd.read_csv(path, usecols=usecols)
    strat_coord = df['TVT'].to_numpy(dtype=float) + df['Z'].to_numpy(dtype=float)
    for col in FORMATION_SURFACE_COLUMNS:
        f = df[col].to_numpy(dtype=float)
        valid = np.isfinite(strat_coord) & np.isfinite(f)
        resid = strat_coord[valid] - f[valid]
        formation_residual_rows.append({
            'well_id': wid,
            'formation': col,
            'rows': int(len(df)),
            'valid_rows': int(valid.sum()),
            'missing_rate': float(1.0 - valid.mean()) if len(valid) else np.nan,
            'resid_median': _nan_stat(resid, np.median),
            'resid_std': _nan_stat(resid, np.std),
            'resid_iqr': _nan_stat(resid, lambda a: np.quantile(a, 0.75) - np.quantile(a, 0.25)),
            'resid_p95_abs': _nan_stat(np.abs(resid - np.nanmedian(resid)) if len(resid) else [], lambda a: np.quantile(a, 0.95)),
        })

formation_residual = pd.DataFrame(formation_residual_rows)
formation_residual_summary = (
    formation_residual
    .groupby('formation', as_index=False)
    .agg(
        wells=('well_id', 'nunique'),
        median_within_well_std=('resid_std', 'median'),
        p90_within_well_std=('resid_std', lambda s: float(np.nanquantile(s, 0.90))),
        median_iqr=('resid_iqr', 'median'),
        median_missing_rate=('missing_rate', 'median'),
    )
    .sort_values('median_within_well_std')
)

display(formation_residual_summary)

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
sns.barplot(data=formation_residual_summary, x='median_within_well_std', y='formation', ax=axes[0])
axes[0].set_title('Within-well stability of TVT + Z - formation')
sns.boxplot(data=formation_residual, x='resid_std', y='formation', ax=axes[1], showfliers=False)
axes[1].set_title('Distribution of within-well residual std')
polish_current_figure()
plt.tight_layout()
plt.show()


### 9.3 Formation Surface Continuity Proxy

A formation surface is useful only if it is spatially smooth enough to estimate at unseen wells. This quick diagnostic uses one row per well and predicts each well's median formation value from nearby well centroids.

It is not the final imputer. It is a sanity check for whether `X/Y` locality contains formation-top information.


In [ ]:
# Leave-one-well-out nearest-centroid proxy for formation-top continuity.

centroid_rows = []
for path in train_horizontal_files:
    wid = well_id_from_path(path)
    usecols = ['X', 'Y'] + FORMATION_SURFACE_COLUMNS
    df = pd.read_csv(path, usecols=usecols)
    row = {
        'well_id': wid,
        'x_mean': float(df['X'].mean()),
        'y_mean': float(df['Y'].mean()),
    }
    for col in FORMATION_SURFACE_COLUMNS:
        row[col] = float(df[col].median()) if df[col].notna().any() else np.nan
    centroid_rows.append(row)

formation_centroids = pd.DataFrame(centroid_rows)


def loo_knn_centroid_surface(frame: pd.DataFrame, value_col: str, k: int = 8) -> pd.DataFrame:
    cols = ['well_id', 'x_mean', 'y_mean', value_col]
    work = frame[cols].dropna().reset_index(drop=True)
    if len(work) < 2:
        return pd.DataFrame(columns=['well_id', 'formation', 'actual', 'pred_loo_knn', 'error', 'abs_error', 'nearest_distance'])
    k_eff = min(int(k), len(work) - 1)
    coords = work[['x_mean', 'y_mean']].to_numpy(dtype=float)
    values = work[value_col].to_numpy(dtype=float)
    rows = []
    for i in range(len(work)):
        dist = np.sqrt(np.sum((coords - coords[i]) ** 2, axis=1))
        order = np.argsort(dist)
        nbr = [j for j in order if j != i][:k_eff]
        d = np.maximum(dist[nbr], 1e-6)
        w = 1.0 / d
        pred = float(np.sum(w * values[nbr]) / np.sum(w))
        rows.append({
            'well_id': work.loc[i, 'well_id'],
            'formation': value_col,
            'actual': float(values[i]),
            'pred_loo_knn': pred,
            'error': float(values[i] - pred),
            'abs_error': float(abs(values[i] - pred)),
            'nearest_distance': float(dist[nbr[0]]) if nbr else np.nan,
        })
    return pd.DataFrame(rows)

surface_loo = pd.concat(
    [loo_knn_centroid_surface(formation_centroids, col, k=8) for col in FORMATION_SURFACE_COLUMNS],
    ignore_index=True,
)
surface_loo_summary = (
    surface_loo
    .groupby('formation', as_index=False)
    .agg(
        wells=('well_id', 'nunique'),
        loo_rmse=('error', lambda s: float(np.sqrt(np.nanmean(np.square(s))))),
        loo_mae=('abs_error', 'mean'),
        p90_abs_error=('abs_error', lambda s: float(np.nanquantile(s, 0.90))),
        median_nearest_distance=('nearest_distance', 'median'),
    )
    .sort_values('loo_rmse')
)

display(surface_loo_summary)

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=surface_loo_summary, x='loo_rmse', y='formation', ax=ax)
ax.set_title('LOO nearest-centroid formation surface RMSE proxy')
polish_current_figure()
plt.tight_layout()
plt.show()


### 9.4 Prefix Typewell Residual and PF Noise Scale

Known-prefix rows let us check whether the horizontal GR curve is compatible with the typewell GR curve at known `TVT_input`.

A simple residual is:

$$
r_i = GR_i^{horizontal} - GR^{typewell}(TVT_i^{input})
$$

The residual scale is also the natural PF observation-noise scale:

$$
gs_w = \operatorname{std}(r_i)
$$

If prefix GR does not align with the typewell under known TVT, hidden typewell tracking should be down-weighted.


In [ ]:
# Enhanced prefix horizontal-vs-typewell GR diagnostics.

prefix_typewell_rows = []
for path in train_horizontal_files:
    wid = well_id_from_path(path)
    h = pd.read_csv(path, usecols=['GR', 'TVT_input'])
    tw = pd.read_csv(TRAIN_DIR / f'{wid}__typewell.csv')
    known = h['TVT_input'].notna()
    hv = h.loc[known, 'GR'].to_numpy(dtype=float)
    tvt = h.loc[known, 'TVT_input'].to_numpy(dtype=float)
    tw_gr = typewell_gr_at_tvt(tw, tvt)
    valid = np.isfinite(hv) & np.isfinite(tw_gr)
    resid = hv[valid] - tw_gr[valid]
    if valid.sum() >= 3 and np.nanstd(tw_gr[valid]) > 1e-9:
        slope, intercept = np.polyfit(tw_gr[valid], hv[valid], 1)
    else:
        slope, intercept = np.nan, np.nan
    prefix_typewell_rows.append({
        'well_id': wid,
        'valid_prefix_points': int(valid.sum()),
        'prefix_tw_gr_corr': safe_corr(hv, tw_gr),
        'prefix_tw_gr_rmse': float(np.sqrt(np.nanmean(resid ** 2))) if len(resid) else np.nan,
        'prefix_tw_gr_mae': float(np.nanmean(np.abs(resid))) if len(resid) else np.nan,
        'prefix_tw_gr_resid_std': _nan_stat(resid, np.std),
        'prefix_tw_affine_slope': float(slope),
        'prefix_tw_affine_intercept': float(intercept),
    })

prefix_typewell_diag = pd.DataFrame(prefix_typewell_rows).merge(
    h_summary[['well_id', 'constant_tail_rmse', 'gr_missing_prefix_rate', 'gr_missing_tail_rate']],
    on='well_id',
    how='left',
)

display(prefix_typewell_diag.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.histplot(prefix_typewell_diag['prefix_tw_gr_resid_std'].dropna(), bins=50, ax=axes[0])
axes[0].set_title('Prefix typewell residual std: PF gs proxy')
sns.scatterplot(data=prefix_typewell_diag, x='prefix_tw_gr_corr', y='prefix_tw_gr_rmse', hue='gr_missing_prefix_rate', palette='viridis', ax=axes[1])
axes[1].set_title('Prefix GR correlation vs residual RMSE')
sns.scatterplot(data=prefix_typewell_diag, x='prefix_tw_gr_resid_std', y='constant_tail_rmse', hue='gr_missing_tail_rate', palette='magma', ax=axes[2])
axes[2].set_title('PF noise scale vs anchor difficulty')
polish_current_figure()
plt.tight_layout()
plt.show()


## 10. Baseline Evaluation

- ⚓ Constant anchor: stay at `last_known_TVT`.
- 📈 Linear prefix extrapolation: extend the known-prefix trend.
- 🧯 If these are competitive, the learned model needs conservative residuals.


In [ ]:
# Evaluate constant and linear baselines on the supervised prediction tail.

def rmse_from_sse(sse: float, n: int) -> float:
    return float(np.sqrt(sse / n)) if n else np.nan

baseline_sse = Counter()
baseline_n = Counter()
per_well_baseline = []

for path in train_horizontal_files:
    wid = well_id_from_path(path)
    df = pd.read_csv(path, usecols=['MD', 'Z', 'TVT', 'TVT_input'])
    mask = df['TVT_input'].isna().to_numpy()
    if not mask.any():
        continue
    known_idx = np.flatnonzero(~mask)
    tail_idx = np.flatnonzero(mask)
    tvt = df['TVT'].to_numpy()
    tvt_input = df['TVT_input'].to_numpy()
    md = df['MD'].to_numpy()
    z = df['Z'].to_numpy()
    known_y = tvt_input[known_idx]
    last_known = float(known_y[-1])
    yy = tvt[tail_idx]
    predictions = {}
    predictions['last_known_constant'] = np.full(len(tail_idx), last_known)
    
    # Intentionally naive baselines: useful to prove that blind extrapolation is dangerous.
    for name, x, idx in [
        ('prefix_linear_md', md, known_idx),
        ('prefix_linear_z', z, known_idx),
    ]:
        coef = np.polyfit(x[idx], known_y, 1)
        predictions[name] = np.polyval(coef, x[tail_idx])
    
    last200_idx = known_idx[-min(200, len(known_idx)):]
    for name, x in [('last200_linear_md', md), ('last200_linear_z', z)]:
        last200_y = tvt_input[last200_idx]
        coef = np.polyfit(x[last200_idx], last200_y, 1)
        predictions[name] = np.polyval(coef, x[tail_idx])
    
    row = {'well_id': wid, 'tail_rows': len(tail_idx)}
    for name, pred in predictions.items():
        err = pred - yy
        sse = float(np.sum(err ** 2))
        baseline_sse[name] += sse
        baseline_n[name] += len(err)
        row[f'{name}_rmse'] = float(np.sqrt(np.mean(err ** 2)))
    per_well_baseline.append(row)

baseline_report = pd.DataFrame({
    'baseline': list(baseline_sse.keys()),
    'global_row_rmse': [rmse_from_sse(baseline_sse[k], baseline_n[k]) for k in baseline_sse.keys()],
    'n_rows': [baseline_n[k] for k in baseline_sse.keys()],
}).sort_values('global_row_rmse')
per_well_baseline = pd.DataFrame(per_well_baseline)

display(baseline_report)
display(per_well_baseline.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T)

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=baseline_report, x='global_row_rmse', y='baseline', ax=ax)
ax.set_title('Global row-level RMSE of simple baselines')
polish_current_figure()
plt.tight_layout()
plt.show()


### 10.1 Interpretation: Baseline Evaluation

### What the baseline means

`last_known_constant` is the null model: the tail stays at the final known TVT anchor.

### Modeling implication

To beat this baseline, a model must gain on drifting wells without creating unnecessary movement on flat wells. Residual prediction with shrinkage and clipping directly controls this trade-off.


### 10.2 Metric-Oriented Baseline EDA

The leaderboard metric is row-weighted RMSE. Long hidden tails therefore matter more than short tails.

$$
RMSE_{row}=\sqrt{\frac{1}{\sum_w n_w}\sum_w\sum_i e_{wi}^2}
$$

A complementary well-level view is:

$$
RMSE_{well}=\frac{1}{W}\sum_w \sqrt{\frac{1}{n_w}\sum_i e_{wi}^2}
$$

A model can improve row RMSE by helping long wells while hurting many short wells, so both views are useful.


In [ ]:
# Row-weighted vs well-level contribution of the anchor baseline.

if len(per_well_baseline) and 'last_known_constant_rmse' in per_well_baseline.columns:
    metric_baseline = per_well_baseline[['well_id', 'tail_rows', 'last_known_constant_rmse']].copy()
    metric_baseline['row_weight'] = metric_baseline['tail_rows'] / metric_baseline['tail_rows'].sum()
    metric_baseline['weighted_sse_contribution'] = (
        metric_baseline['tail_rows'] * metric_baseline['last_known_constant_rmse'] ** 2
    )
    metric_baseline['weighted_sse_share'] = metric_baseline['weighted_sse_contribution'] / metric_baseline['weighted_sse_contribution'].sum()
    display(metric_baseline.sort_values('weighted_sse_share', ascending=False).head(20))
    print('row-weighted anchor RMSE:', float(np.sqrt(metric_baseline['weighted_sse_contribution'].sum() / metric_baseline['tail_rows'].sum())))
    print('mean well anchor RMSE:', float(metric_baseline['last_known_constant_rmse'].mean()))
    print('median well anchor RMSE:', float(metric_baseline['last_known_constant_rmse'].median()))
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    sns.scatterplot(data=metric_baseline, x='tail_rows', y='last_known_constant_rmse', size='weighted_sse_share', sizes=(20, 220), ax=axes[0])
    axes[0].set_title('Tail length and anchor difficulty')
    sns.histplot(metric_baseline['weighted_sse_share'], bins=50, ax=axes[1])
    axes[1].set_title('Per-well share of row-weighted SSE')
    polish_current_figure()
    plt.tight_layout()
    plt.show()
else:
    print('per_well_baseline is not available yet.')


## 🧱 11. Row-Level Features

### Domain framing

The target is a hidden coordinate transform from measured depth to typewell TVT. The feature table combines anchor context, current-row geometry, GR texture, typewell matching, and optional spatial geology references.

### Strict feature groups

| Group | Examples | Leakage boundary |
|---|---|---|
| Prefix context | prefix length, GR stats, TVT range, prefix slopes | prefix only |
| Row position | `MD_i - MD_PS`, `X_i - X_PS`, `Y_i - Y_PS`, `Z_i - Z_PS` | current row |
| Current GR | raw GR, prefix-normalized GR, missing flag | current row |
| GR events | backward differences, MD-normalized slopes | rows up to `i` |
| Trailing GR context | rolling mean/std/range | rows up to `i` |
| Typewell alignment | typewell GR at prefix-derived TVT baselines | no true tail TVT |
| Local offset search | best GR match around prefix-derived baseline | target-free GR only |
| Calibrated typewell | affine-calibrated typewell residuals | prefix fit only |
| Typewell interval context | Geology interval phase at prefix-derived TVT positions | reference data + prefix baseline |

### Offline feature groups

| Feature | Guardrail |
|---|---|
| Tail length / tail fraction | uses the full prediction tail |
| Full-row fraction and MD tail fraction | uses full test-file geometry |
| Gap geometry and GR quantiles | summarizes the full hidden interval |
| Centered GR rolling and lead/lag GR | uses future GR covariates, not future TVT labels |
| Candidate-path typewell endpoints | uses tail fraction to compare plausible TVT drift paths |
| Formation-plane/KNN references | projects train formation-top geometry onto target-free coordinates |
| Beam-path typewell alignment | uses the full hidden GR sequence to form a path feature |

🚫 Excluded from all policies: `TVT_input_bfill`, true tail `TVT`, target-derived tail summaries, true future TVT knots, and direct train-only surface columns.


In [ ]:
# Construct leakage-safe row-level tail features and residual targets for one well.

def safe_slope(x: np.ndarray, y: np.ndarray) -> float:
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 2:
        return np.nan
    if np.nanstd(x[valid]) < 1e-9:
        return np.nan
    return float(np.polyfit(x[valid], y[valid], 1)[0])


def normalize_required_feature_columns(required_feature_columns) -> set[str] | None:
    if required_feature_columns is None:
        return None
    return {str(col) for col in required_feature_columns}


def needs_any(required_columns: set[str] | None, prefixes=(), exact=()) -> bool:
    if required_columns is None:
        return True
    prefixes = tuple(prefixes)
    exact = set(exact)
    for col in required_columns:
        if col in exact or any(col.startswith(prefix) for prefix in prefixes):
            return True
    return False




def typewell_gr_at_tvt(typewell_df: pd.DataFrame, tvt_values: np.ndarray) -> np.ndarray:
    if not {'TVT', 'GR'}.issubset(typewell_df.columns):
        return np.full(len(tvt_values), np.nan)
    tw = typewell_df[['TVT', 'GR']].dropna().sort_values('TVT')
    if len(tw) < 2:
        return np.full(len(tvt_values), np.nan)
    x = tw['TVT'].to_numpy(dtype=float)
    y = tw['GR'].to_numpy(dtype=float)
    return np.interp(np.asarray(tvt_values, dtype=float), x, y, left=np.nan, right=np.nan)


def safe_corr(a: np.ndarray, b: np.ndarray, min_points: int = 30) -> float:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    valid = np.isfinite(a) & np.isfinite(b)
    if valid.sum() < min_points:
        return np.nan
    aa = a[valid]
    bb = b[valid]
    if np.nanstd(aa) < 1e-9 or np.nanstd(bb) < 1e-9:
        return np.nan
    return float(np.corrcoef(aa, bb)[0, 1])


def recent_mean_diff(values: np.ndarray, window: int) -> float:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    values = values[-(window + 1):]
    if len(values) < 2:
        return np.nan
    return float(np.diff(values).mean())


def recent_slope_window(y_values: np.ndarray, x_values: np.ndarray, window: int) -> float:
    y_values = np.asarray(y_values, dtype=float)[-window:]
    x_values = np.asarray(x_values, dtype=float)[-window:]
    valid = np.isfinite(y_values) & np.isfinite(x_values)
    if valid.sum() < 2:
        return np.nan
    x = x_values[valid]
    y = y_values[valid]
    centered_x = x - x.mean()
    denominator = float(np.dot(centered_x, centered_x))
    if denominator <= 1e-12:
        return np.nan
    return float(np.dot(centered_x, y - y.mean()) / denominator)


def nearest_sorted_index(sorted_values: np.ndarray, target: float) -> int:
    values = np.asarray(sorted_values, dtype=float)
    if len(values) == 0:
        return 0
    idx = int(np.searchsorted(values, target, side='left'))
    if idx >= len(values):
        return len(values) - 1
    if idx > 0 and abs(values[idx - 1] - target) <= abs(values[idx] - target):
        return idx - 1
    return idx


def smooth_gr_for_beam(values: np.ndarray, fallback: float, radius: int) -> np.ndarray:
    series = pd.Series(values, dtype='float64').interpolate(limit_direction='both').fillna(fallback)
    if radius <= 0:
        return series.to_numpy(dtype=float)
    return series.rolling(radius * 2 + 1, center=True, min_periods=1).mean().to_numpy(dtype=float)


def beam_typewell_path(
    gr_values: np.ndarray,
    tw_tvt: np.ndarray,
    tw_gr: np.ndarray,
    start_tvt: float,
    beam_size: int = 10,
    move_cost: float = 20.0,
    emit_scale: float = 144.0,
    radius: int = 2,
) -> np.ndarray:
    tw_tvt = np.asarray(tw_tvt, dtype=float)
    tw_gr = np.asarray(tw_gr, dtype=float)
    valid_tw = np.isfinite(tw_tvt) & np.isfinite(tw_gr)
    tw_tvt = tw_tvt[valid_tw]
    tw_gr = tw_gr[valid_tw]
    order = np.argsort(tw_tvt)
    tw_tvt = tw_tvt[order]
    tw_gr = tw_gr[order]
    n = len(gr_values)
    if n == 0 or len(tw_tvt) < 2 or not np.isfinite(start_tvt):
        return np.full(n, np.nan)

    fallback = float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.0
    smoothed_gr = smooth_gr_for_beam(gr_values, fallback=fallback, radius=radius)
    start_idx = nearest_sorted_index(tw_tvt, start_tvt)
    states = {start_idx: 0.0}
    backpointers: list[dict[int, int]] = []

    for gr_value in smoothed_gr:
        candidates: dict[int, float] = {}
        parents: dict[int, int] = {}
        if not np.isfinite(gr_value):
            gr_value = fallback
        for idx, cost in states.items():
            for delta in (-1, 0, 1):
                next_idx = idx + delta
                if next_idx < 0 or next_idx >= len(tw_tvt):
                    continue
                emit_cost = ((gr_value - tw_gr[next_idx]) ** 2) / max(emit_scale, 1e-6)
                total_cost = cost + emit_cost + move_cost * abs(delta)
                if next_idx not in candidates or total_cost < candidates[next_idx]:
                    candidates[next_idx] = float(total_cost)
                    parents[next_idx] = idx
        kept = sorted(candidates.items(), key=lambda item: item[1])[:beam_size]
        if not kept:
            return np.full(n, np.nan)
        states = {idx: cost for idx, cost in kept}
        backpointers.append({idx: parents[idx] for idx, _ in kept})

    final_idx = min(states, key=states.get)
    path = [final_idx]
    for step in range(len(backpointers) - 1, 0, -1):
        path.append(backpointers[step][path[-1]])
    path.reverse()
    return tw_tvt[np.asarray(path, dtype=int)]


def offline_beam_feature_names(prefix: str = 'tw_beam') -> list[str]:
    names = [
        f'{prefix}_tight_delta',
        f'{prefix}_conservative_delta',
        f'{prefix}_loose_delta',
        f'{prefix}_vcons_delta',
        f'{prefix}_sm5_delta',
        f'{prefix}_vloose_delta',
        f'{prefix}_gap',
        f'{prefix}_spread',
        f'{prefix}_mean_delta',
        f'{prefix}_std_delta',
        f'{prefix}_tight_step',
        f'{prefix}_conservative_step',
        f'{prefix}_loose_step',
        f'{prefix}_vcons_step',
        f'{prefix}_sm5_step',
        f'{prefix}_vloose_step',
        f'{prefix}_gr_at_conservative',
        f'{prefix}_gr_at_loose',
        f'{prefix}_gr_minus_conservative',
        f'{prefix}_gr_minus_loose',
    ]
    if prefix == 'tw_beam':
        names += [
            'beam_tight_delta',
            'beam_cons_delta',
            'beam_loose_delta',
            'beam_vcons_delta',
            'beam_sm5_delta',
            'beam_vloose_delta',
            'beam_mean_delta',
            'beam_std_delta',
            'beam_spread',
            'beam_gap',
            'beam_median_delta',
            'beam_sm5_vs_cons',
            'beam_vcons_vs_loose',
            'gr_minus_tw_beam_cons',
            'gr_minus_tw_beam_loose',
        ]
    return names

def safe_stat(series, func) -> float:
    values = pd.to_numeric(series, errors='coerce').to_numpy(dtype=float)
    valid = np.isfinite(values)
    if not valid.any():
        return np.nan
    return float(func(values[valid]))


def safe_affine_fit(x: np.ndarray, y: np.ndarray, min_points: int = 30) -> tuple[float, float]:
    """Fit y ~= a*x + b using only finite prefix points."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < min_points or np.nanstd(x[valid]) < 1e-9:
        return np.nan, np.nan
    a, b = np.polyfit(x[valid], y[valid], 1)
    return float(a), float(b)


def apply_affine(x: np.ndarray | float, a: float, b: float):
    if np.isfinite(a) and np.isfinite(b):
        return a * x + b
    return np.full_like(np.asarray(x, dtype=float), np.nan, dtype=float)



def local_typewell_offset_features(
    typewell_df: pd.DataFrame,
    baseline_tvt: np.ndarray,
    horizontal_gr: np.ndarray,
    offsets: np.ndarray,
    affine_a: float | None = None,
    affine_b: float | None = None,
    score_scale: float = 20.0,
    prefix: str = 'typewell_local',
) -> dict[str, np.ndarray]:
    baseline_tvt = np.asarray(baseline_tvt, dtype=float)
    horizontal_gr = np.asarray(horizontal_gr, dtype=float)
    offsets = np.asarray(offsets, dtype=float)
    n = len(baseline_tvt)
    if n == 0 or len(offsets) == 0:
        return {
            f'{prefix}_best_delta': np.full(n, np.nan),
            f'{prefix}_best_abs_resid': np.full(n, np.nan),
            f'{prefix}_zero_abs_resid': np.full(n, np.nan),
            f'{prefix}_top2_gap': np.full(n, np.nan),
            f'{prefix}_soft_delta_mean': np.full(n, np.nan),
            f'{prefix}_best_gr': np.full(n, np.nan),
        }
    candidate_tvt = baseline_tvt[:, None] + offsets[None, :]
    candidate_gr = typewell_gr_at_tvt(typewell_df, candidate_tvt.reshape(-1)).reshape(n, len(offsets))
    if affine_a is not None and affine_b is not None and np.isfinite(affine_a) and np.isfinite(affine_b):
        candidate_gr = affine_a * candidate_gr + affine_b
    abs_resid = np.abs(horizontal_gr[:, None] - candidate_gr)
    valid = np.isfinite(abs_resid)
    masked = np.where(valid, abs_resid, np.inf)
    any_valid = valid.any(axis=1)
    best_idx = np.argmin(masked, axis=1)
    best_abs = np.where(any_valid, masked[np.arange(n), best_idx], np.nan)
    best_delta = np.where(any_valid, offsets[best_idx], np.nan)
    best_gr = np.where(any_valid, candidate_gr[np.arange(n), best_idx], np.nan)
    zero_idx = int(np.argmin(np.abs(offsets)))
    zero_abs = abs_resid[:, zero_idx] if len(offsets) else np.full(n, np.nan)
    if len(offsets) >= 2:
        sorted_abs = np.sort(masked, axis=1)
        top2_gap = np.full(n, np.nan)
        top2_ok = any_valid & np.isfinite(sorted_abs[:, 1])
        top2_gap[top2_ok] = sorted_abs[top2_ok, 1] - sorted_abs[top2_ok, 0]
    else:
        top2_gap = np.full(n, np.nan)
    weights = np.zeros_like(abs_resid, dtype=float)
    weights[valid] = np.exp(-abs_resid[valid] / score_scale)
    denom = weights.sum(axis=1)
    soft_delta = np.full(n, np.nan)
    denom_ok = denom > 0
    soft_delta[denom_ok] = (weights[denom_ok] * offsets[None, :]).sum(axis=1) / denom[denom_ok]
    return {
        f'{prefix}_best_delta': best_delta,
        f'{prefix}_best_abs_resid': best_abs,
        f'{prefix}_zero_abs_resid': zero_abs,
        f'{prefix}_top2_gap': top2_gap,
        f'{prefix}_soft_delta_mean': soft_delta,
        f'{prefix}_best_gr': best_gr,
    }

CANDIDATE_PATH_ENDPOINTS = np.array([-60.0, -40.0, -25.0, -15.0, -8.0, 0.0, 8.0, 15.0, 25.0, 40.0, 60.0])


def candidate_endpoint_label(endpoint: float) -> str:
    sign = 'm' if endpoint < 0 else 'p'
    value = str(int(abs(endpoint))) if float(endpoint).is_integer() else str(abs(endpoint)).replace('.', 'p')
    return f'{sign}{value}'


def candidate_path_feature_names(prefix: str = 'tw_path', endpoints: np.ndarray = CANDIDATE_PATH_ENDPOINTS) -> list[str]:
    names: list[str] = []
    for endpoint in endpoints:
        label = candidate_endpoint_label(float(endpoint))
        names.extend([
            f'{prefix}_gr_diff_{label}',
            f'{prefix}_gr_absdiff_{label}',
            f'{prefix}_boundary_count_{label}',
            f'{prefix}_boundary_nearest_{label}',
        ])
    names.extend([
        f'{prefix}_min_absdiff',
        f'{prefix}_best_endpoint',
        f'{prefix}_best_endpoint_centered',
        f'{prefix}_top2_absdiff_gap',
        f'{prefix}_soft_endpoint_mean',
        f'{prefix}_best_gr_resid',
    ])
    return names


def typewell_boundary_tvt(typewell_df: pd.DataFrame) -> np.ndarray:
    if 'Geology' not in typewell_df.columns or 'TVT' not in typewell_df.columns:
        return np.array([], dtype=float)
    tw = typewell_df[['TVT', 'Geology']].dropna().sort_values('TVT')
    if len(tw) < 2:
        return np.array([], dtype=float)
    tvt = pd.to_numeric(tw['TVT'], errors='coerce').to_numpy(dtype=float)
    geology = tw['Geology'].astype(str).to_numpy()
    valid = np.isfinite(tvt)
    tvt = tvt[valid]
    geology = geology[valid]
    if len(tvt) < 2:
        return np.array([], dtype=float)
    change_idx = np.flatnonzero(geology[1:] != geology[:-1]) + 1
    return tvt[change_idx]




def typewell_interval_boundaries(typewell_df: pd.DataFrame) -> np.ndarray:
    if 'TVT' not in typewell_df.columns:
        return np.array([], dtype=float)
    tvt = pd.to_numeric(typewell_df['TVT'], errors='coerce').dropna().to_numpy(dtype=float)
    if len(tvt) < 2:
        return np.array([], dtype=float)
    boundaries = typewell_boundary_tvt(typewell_df)
    bounds = np.unique(np.r_[np.nanmin(tvt), boundaries, np.nanmax(tvt)])
    return np.sort(bounds[np.isfinite(bounds)])


def typewell_interval_context_features(typewell_df: pd.DataFrame, target_tvt, prefix: str) -> dict[str, np.ndarray]:
    target = np.asarray(target_tvt, dtype=float)
    original_shape = target.shape
    target_flat = target.reshape(-1)
    bounds = typewell_interval_boundaries(typewell_df)
    out = {
        f'{prefix}_prev_boundary_dist': np.full(len(target_flat), np.nan),
        f'{prefix}_next_boundary_dist': np.full(len(target_flat), np.nan),
        f'{prefix}_interval_thickness': np.full(len(target_flat), np.nan),
        f'{prefix}_interval_phase': np.full(len(target_flat), np.nan),
        f'{prefix}_interval_phase_sin': np.full(len(target_flat), np.nan),
        f'{prefix}_interval_phase_cos': np.full(len(target_flat), np.nan),
        f'{prefix}_boundary_balance': np.full(len(target_flat), np.nan),
        f'{prefix}_boundary_proximity': np.full(len(target_flat), np.nan),
    }
    if len(bounds) < 2:
        return {k: v.reshape(original_shape) for k, v in out.items()}
    valid = np.isfinite(target_flat)
    if valid.any():
        idx = np.searchsorted(bounds, target_flat[valid], side='right') - 1
        idx = np.clip(idx, 0, len(bounds) - 2)
        top = bounds[idx]
        base = bounds[idx + 1]
        thickness = base - top
        ok = np.isfinite(thickness) & (np.abs(thickness) > 1e-9)
        valid_positions = np.flatnonzero(valid)
        rows = valid_positions[ok]
        phase = (target_flat[rows] - top[ok]) / thickness[ok]
        prev_dist = target_flat[rows] - top[ok]
        next_dist = base[ok] - target_flat[rows]
        out[f'{prefix}_prev_boundary_dist'][rows] = prev_dist
        out[f'{prefix}_next_boundary_dist'][rows] = next_dist
        out[f'{prefix}_interval_thickness'][rows] = thickness[ok]
        out[f'{prefix}_interval_phase'][rows] = phase
        out[f'{prefix}_interval_phase_sin'][rows] = np.sin(np.pi * phase)
        out[f'{prefix}_interval_phase_cos'][rows] = np.cos(np.pi * phase)
        out[f'{prefix}_boundary_balance'][rows] = (next_dist - prev_dist) / (thickness[ok] + 1e-6)
        out[f'{prefix}_boundary_proximity'][rows] = np.minimum(prev_dist, next_dist) / (thickness[ok] + 1e-6)
    return {k: v.reshape(original_shape) for k, v in out.items()}


def typewell_interval_context_feature_names(prefix: str) -> list[str]:
    return [
        f'{prefix}_prev_boundary_dist',
        f'{prefix}_next_boundary_dist',
        f'{prefix}_interval_thickness',
        f'{prefix}_interval_phase',
        f'{prefix}_interval_phase_sin',
        f'{prefix}_interval_phase_cos',
        f'{prefix}_boundary_balance',
        f'{prefix}_boundary_proximity',
    ]

def boundary_count_between(boundaries: np.ndarray, start_tvt: float, target_tvt: np.ndarray) -> np.ndarray:
    target_tvt = np.asarray(target_tvt, dtype=float)
    if len(boundaries) == 0 or not np.isfinite(start_tvt):
        return np.full(len(target_tvt), np.nan)
    lo = np.minimum(start_tvt, target_tvt)
    hi = np.maximum(start_tvt, target_tvt)
    valid = np.isfinite(lo) & np.isfinite(hi)
    counts = np.full(len(target_tvt), np.nan)
    if valid.any():
        counts[valid] = ((boundaries[None, :] > lo[valid, None]) & (boundaries[None, :] <= hi[valid, None])).sum(axis=1)
    return counts


def nearest_boundary_distance(boundaries: np.ndarray, target_tvt: np.ndarray) -> np.ndarray:
    target_tvt = np.asarray(target_tvt, dtype=float)
    if len(boundaries) == 0:
        return np.full(len(target_tvt), np.nan)
    dist = np.full(len(target_tvt), np.nan)
    valid = np.isfinite(target_tvt)
    if valid.any():
        dist[valid] = np.min(np.abs(target_tvt[valid, None] - boundaries[None, :]), axis=1)
    return dist



SELF_CORR_PREFIXES = ('selfcorr_', 'sc_', 'hyb_', 'tdsc_')
PF_LITE_PREFIXES = ('pf_lite_',)
TDBC_OFFSETS = np.array([-40, -20, -10, -5, -3, 0, 3, 5, 10, 20, 40], dtype=float)
TDSC_OFFSETS = np.array([-30, -15, -8, -4, -2, 0, 2, 4, 8, 15, 30], dtype=float)


def typewell_offset_family_feature_names(prefix: str, offsets: np.ndarray) -> list[str]:
    return [f'{prefix}_{candidate_endpoint_label(float(offset))}' for offset in offsets]


def typewell_offset_family_features(
    typewell_df: pd.DataFrame,
    anchor_tvt: np.ndarray,
    horizontal_gr: np.ndarray,
    offsets: np.ndarray,
    prefix: str,
) -> dict[str, np.ndarray]:
    anchor_tvt = np.asarray(anchor_tvt, dtype=float)
    horizontal_gr = np.asarray(horizontal_gr, dtype=float)
    out: dict[str, np.ndarray] = {}
    for offset in np.asarray(offsets, dtype=float):
        label = candidate_endpoint_label(float(offset))
        tw_gr = typewell_gr_at_tvt(typewell_df, anchor_tvt + float(offset))
        out[f'{prefix}_{label}'] = horizontal_gr - tw_gr
    return out


def selfcorr_feature_names() -> list[str]:
    return [
        'selfcorr_tvt',
        'selfcorr_delta',
        'selfcorr_score',
        'selfcorr_trust',
        'selfcorr_top2_gap',
        'hyb_delta',
        'beam_vs_sc',
        'dense_vs_sc',
        'plane_vs_sc',
    ]


def pf_lite_feature_names() -> list[str]:
    return [
        'pf_lite_tvt',
        'pf_lite_delta',
        'pf_lite_std',
        'pf_lite_weight_sum',
        'pf_lite_candidate_count',
        'pf_lite_vs_dense',
        'pf_lite_vs_plane',
        'pf_lite_vs_sc',
        'pf_lite_vs_beam_cons',
        'pf_lite_gr_abs_resid',
    ]


def gr_window_signature(values: np.ndarray, half_window: int = 15) -> np.ndarray:
    values = pd.Series(values, dtype='float64').interpolate(limit_direction='both')
    fallback = float(values.dropna().median()) if values.notna().any() else 0.0
    values = values.fillna(fallback)
    window = max(3, int(half_window) * 2 + 1)
    roll = values.rolling(window, center=True, min_periods=max(3, window // 3))
    mean = roll.mean()
    std = roll.std().fillna(0.0)
    rng = (roll.max() - roll.min()).fillna(0.0)
    grad = values.diff().rolling(window, center=True, min_periods=max(3, window // 3)).mean().fillna(0.0)
    center = values
    sig = np.column_stack([
        mean.to_numpy(dtype=float),
        std.to_numpy(dtype=float),
        rng.to_numpy(dtype=float),
        grad.to_numpy(dtype=float),
        center.to_numpy(dtype=float),
    ])
    return sig.astype(np.float32)


def selfcorr_prefix_tvt_features(
    prefix_gr: np.ndarray,
    prefix_tvt: np.ndarray,
    tail_gr: np.ndarray,
    last_known_tvt: float,
    half_window: int = 15,
    stride: int = 3,
) -> dict[str, np.ndarray]:
    n_tail = len(tail_gr)
    empty = {
        'selfcorr_tvt': np.full(n_tail, np.nan, dtype=np.float32),
        'selfcorr_delta': np.full(n_tail, np.nan, dtype=np.float32),
        'selfcorr_score': np.full(n_tail, np.nan, dtype=np.float32),
        'selfcorr_trust': np.full(n_tail, np.nan, dtype=np.float32),
        'selfcorr_top2_gap': np.full(n_tail, np.nan, dtype=np.float32),
    }
    prefix_gr = np.asarray(prefix_gr, dtype=float)
    prefix_tvt = np.asarray(prefix_tvt, dtype=float)
    tail_gr = np.asarray(tail_gr, dtype=float)
    valid_prefix = np.isfinite(prefix_gr) & np.isfinite(prefix_tvt)
    if n_tail == 0 or valid_prefix.sum() < max(30, half_window * 2):
        return empty

    prefix_sig_all = gr_window_signature(prefix_gr, half_window=half_window)
    tail_sig = gr_window_signature(tail_gr, half_window=half_window)
    candidates = np.flatnonzero(valid_prefix)
    candidates = candidates[::max(1, int(stride))]
    if len(candidates) < 5:
        return empty
    prefix_sig = prefix_sig_all[candidates]
    prefix_tvt_candidates = prefix_tvt[candidates]
    good = np.isfinite(prefix_sig).all(axis=1) & np.isfinite(prefix_tvt_candidates)
    if good.sum() < 5:
        return empty
    prefix_sig = prefix_sig[good]
    prefix_tvt_candidates = prefix_tvt_candidates[good]

    center = np.nanmedian(prefix_sig, axis=0)
    scale = np.nanstd(prefix_sig, axis=0)
    scale = np.where(~np.isfinite(scale) | (scale < 1e-6), 1.0, scale)
    prefix_z = (prefix_sig - center) / scale
    tail_z = (tail_sig - center) / scale
    finite_tail = np.isfinite(tail_z).all(axis=1)

    # This is a compact nearest-window proxy for prefix self-correlation. It is much cheaper
    # than row-by-row raw-window correlation and keeps the same modeling role.
    dist = np.full((n_tail, min(2, len(prefix_z))), np.nan, dtype=np.float32)
    nn_idx = np.full((n_tail, min(2, len(prefix_z))), -1, dtype=int)
    try:
        from sklearn.neighbors import NearestNeighbors
        nn = NearestNeighbors(n_neighbors=min(2, len(prefix_z)), algorithm='auto')
        nn.fit(prefix_z)
        d, idx = nn.kneighbors(tail_z[finite_tail])
        dist[finite_tail, :d.shape[1]] = d.astype(np.float32)
        nn_idx[finite_tail, :idx.shape[1]] = idx
    except Exception:
        diff = tail_z[finite_tail, None, :] - prefix_z[None, :, :]
        d_all = np.sqrt(np.nanmean(diff * diff, axis=2))
        order = np.argsort(d_all, axis=1)[:, :min(2, len(prefix_z))]
        d = np.take_along_axis(d_all, order, axis=1)
        dist[finite_tail, :d.shape[1]] = d.astype(np.float32)
        nn_idx[finite_tail, :order.shape[1]] = order

    best_valid = nn_idx[:, 0] >= 0
    sc_tvt = np.full(n_tail, np.nan, dtype=np.float32)
    sc_tvt[best_valid] = prefix_tvt_candidates[nn_idx[best_valid, 0]].astype(np.float32)
    best_dist = dist[:, 0]
    score = np.exp(-np.clip(best_dist, 0.0, 20.0) / 2.5).astype(np.float32)
    score[~best_valid] = np.nan
    if dist.shape[1] >= 2:
        top2_gap = (dist[:, 1] - dist[:, 0]).astype(np.float32)
        top2_gap[~np.isfinite(top2_gap)] = np.nan
    else:
        top2_gap = np.full(n_tail, np.nan, dtype=np.float32)
    prefix_len_conf = np.clip(valid_prefix.sum() / 250.0, 0.0, 1.0)
    trust = np.clip(score * prefix_len_conf, 0.0, 1.0).astype(np.float32)
    return {
        'selfcorr_tvt': sc_tvt,
        'selfcorr_delta': sc_tvt - float(last_known_tvt),
        'selfcorr_score': score,
        'selfcorr_trust': trust,
        'selfcorr_top2_gap': top2_gap,
    }


def weighted_candidate_tvt_features(
    typewell_df: pd.DataFrame,
    tail_gr: np.ndarray,
    tail_z: np.ndarray,
    candidate_tvt: dict[str, np.ndarray],
    last_known_tvt: float,
    dense_ancc: np.ndarray | None = None,
) -> dict[str, np.ndarray]:
    names = list(candidate_tvt)
    n = len(tail_gr)
    if not names:
        return {name: np.full(n, np.nan, dtype=np.float32) for name in pf_lite_feature_names()}
    cand = np.column_stack([np.asarray(candidate_tvt[name], dtype=float) for name in names])
    finite = np.isfinite(cand)
    tw_gr = typewell_gr_at_tvt(typewell_df, cand.reshape(-1)).reshape(cand.shape)
    gr_abs = np.abs(np.asarray(tail_gr, dtype=float)[:, None] - tw_gr)
    gr_penalty = np.clip(
        np.nan_to_num(gr_abs, nan=60.0, posinf=60.0, neginf=60.0) / 20.0,
        0.0,
        6.0,
    )
    weights = np.exp(-gr_penalty) * finite
    if dense_ancc is not None:
        dense_ancc = np.asarray(dense_ancc, dtype=float)
        dense_valid = np.isfinite(dense_ancc)
        if dense_valid.any():
            ancc_abs = np.abs((cand + np.asarray(tail_z, dtype=float)[:, None]) - dense_ancc[:, None])
            ancc_penalty = np.clip(
                np.nan_to_num(ancc_abs, nan=120.0, posinf=120.0, neginf=120.0) / 35.0,
                0.0,
                6.0,
            )
            dense_factor = 0.20 + 0.80 * np.exp(-ancc_penalty)
            dense_factor = np.where(dense_valid[:, None], dense_factor, 1.0)
            weights *= dense_factor
    denom = weights.sum(axis=1)
    safe = np.where(denom <= 1e-8, 1.0, denom)
    pf_tvt = np.sum(np.where(finite, cand, 0.0) * weights, axis=1) / safe
    cand_masked = np.where(finite, cand, np.nan)
    finite_count = finite.sum(axis=1)
    fallback_tvt = np.full(n, np.nan, dtype=float)
    fallback_std = np.full(n, np.nan, dtype=float)
    fallback_rows = finite_count > 0
    if fallback_rows.any():
        fallback_tvt[fallback_rows] = np.nanmedian(cand_masked[fallback_rows], axis=1)
        fallback_std[fallback_rows] = np.nanstd(cand_masked[fallback_rows], axis=1)
    low_weight = denom <= 1e-8
    pf_tvt[low_weight] = fallback_tvt[low_weight]
    var = np.sum(((cand - pf_tvt[:, None]) ** 2) * weights, axis=1) / safe
    pf_std = np.sqrt(np.maximum(var, 0.0))
    pf_std[low_weight] = fallback_std[low_weight]
    gr_masked = np.where(finite, gr_abs, np.inf)
    best_gr_abs = np.min(gr_masked, axis=1)
    best_gr_abs[~np.isfinite(best_gr_abs)] = np.nan
    out = {
        'pf_lite_tvt': pf_tvt.astype(np.float32),
        'pf_lite_delta': (pf_tvt - float(last_known_tvt)).astype(np.float32),
        'pf_lite_std': pf_std.astype(np.float32),
        'pf_lite_weight_sum': denom.astype(np.float32),
        'pf_lite_candidate_count': finite.sum(axis=1).astype(np.float32),
        'pf_lite_gr_abs_resid': best_gr_abs.astype(np.float32),
    }
    def delta_from(name: str):
        arr = candidate_tvt.get(name)
        if arr is None:
            return np.full(n, np.nan, dtype=np.float32)
        return (pf_tvt - np.asarray(arr, dtype=float)).astype(np.float32)
    out['pf_lite_vs_dense'] = delta_from('dense')
    out['pf_lite_vs_plane'] = delta_from('plane')
    out['pf_lite_vs_sc'] = delta_from('selfcorr')
    out['pf_lite_vs_beam_cons'] = delta_from('beam_cons')
    return out


def typewell_candidate_path_features(
    typewell_df: pd.DataFrame,
    last_known_tvt: float,
    tail_frac: np.ndarray,
    horizontal_gr: np.ndarray,
    endpoints: np.ndarray = CANDIDATE_PATH_ENDPOINTS,
    prefix: str = 'tw_path',
    score_scale: float = 20.0,
) -> dict[str, np.ndarray]:
    tail_frac = np.asarray(tail_frac, dtype=float)
    horizontal_gr = np.asarray(horizontal_gr, dtype=float)
    endpoints = np.asarray(endpoints, dtype=float)
    n = len(tail_frac)
    if n == 0 or len(endpoints) == 0 or not np.isfinite(last_known_tvt):
        return {name: np.full(n, np.nan) for name in candidate_path_feature_names(prefix, endpoints)}

    candidate_tvt = last_known_tvt + tail_frac[:, None] * endpoints[None, :]
    candidate_gr = typewell_gr_at_tvt(typewell_df, candidate_tvt.reshape(-1)).reshape(n, len(endpoints))
    gr_diff = horizontal_gr[:, None] - candidate_gr
    absdiff = np.abs(gr_diff)
    valid = np.isfinite(absdiff)
    masked = np.where(valid, absdiff, np.inf)
    any_valid = valid.any(axis=1)
    best_idx = np.argmin(masked, axis=1)
    best_abs = np.where(any_valid, masked[np.arange(n), best_idx], np.nan)
    best_endpoint = np.where(any_valid, endpoints[best_idx], np.nan)
    best_gr = np.where(any_valid, candidate_gr[np.arange(n), best_idx], np.nan)

    if len(endpoints) >= 2:
        sorted_abs = np.sort(masked, axis=1)
        top2_gap = np.full(n, np.nan)
        top2_ok = any_valid & np.isfinite(sorted_abs[:, 1])
        top2_gap[top2_ok] = sorted_abs[top2_ok, 1] - sorted_abs[top2_ok, 0]
    else:
        top2_gap = np.full(n, np.nan)

    weights = np.zeros_like(absdiff, dtype=float)
    weights[valid] = np.exp(-absdiff[valid] / score_scale)
    denom = weights.sum(axis=1)
    soft_endpoint = np.full(n, np.nan)
    denom_ok = denom > 0
    soft_endpoint[denom_ok] = (weights[denom_ok] * endpoints[None, :]).sum(axis=1) / denom[denom_ok]

    boundaries = typewell_boundary_tvt(typewell_df)
    max_endpoint = np.nanmax(np.abs(endpoints)) if len(endpoints) else np.nan
    features: dict[str, np.ndarray] = {}
    for j, endpoint in enumerate(endpoints):
        label = candidate_endpoint_label(float(endpoint))
        features[f'{prefix}_gr_diff_{label}'] = gr_diff[:, j]
        features[f'{prefix}_gr_absdiff_{label}'] = absdiff[:, j]
        features[f'{prefix}_boundary_count_{label}'] = boundary_count_between(boundaries, last_known_tvt, candidate_tvt[:, j])
        features[f'{prefix}_boundary_nearest_{label}'] = nearest_boundary_distance(boundaries, candidate_tvt[:, j])
    features[f'{prefix}_min_absdiff'] = best_abs
    features[f'{prefix}_best_endpoint'] = best_endpoint
    features[f'{prefix}_best_endpoint_centered'] = best_endpoint / max_endpoint if np.isfinite(max_endpoint) and max_endpoint > 0 else np.nan
    features[f'{prefix}_top2_absdiff_gap'] = top2_gap
    features[f'{prefix}_soft_endpoint_mean'] = soft_endpoint
    features[f'{prefix}_best_gr_resid'] = horizontal_gr - best_gr
    return features


FORMATION_TOP_COLUMNS = ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']
FORMATION_LABELS = [name.lower() for name in FORMATION_TOP_COLUMNS]


def formation_feature_names(include_row: bool = True) -> list[str]:
    plane_names = [f'formation_plane_{label}' for label in FORMATION_LABELS]
    per_formation_formula_names: list[str] = []
    for label in FORMATION_LABELS:
        per_formation_formula_names.extend([
            f'formation_plane_anchor_b_{label}',
            f'formation_plane_anchor_b50_{label}',
            f'formation_plane_prefix_rmse_{label}',
            f'formation_plane_prefix_mae_{label}',
            f'formation_plane_tvt_formula_{label}',
            f'formation_plane_delta_formula_{label}',
            f'formation_plane_delta_formula50_{label}',
        ])
    names = [
        *plane_names,
        'formation_plane_min_dist',
        'formation_plane_anchor_b',
        'formation_plane_anchor_b50',
        'formation_plane_prefix_rmse',
        'formation_plane_prefix_mae',
        'formation_plane_tvt_formula',
        'formation_plane_delta_formula',
        'formation_plane_delta_from_slope_last200',
        *per_formation_formula_names,
        'formation_plane_formula_mean_delta',
        'formation_plane_formula_std_delta',
        'formation_plane_formula_min_delta',
        'formation_plane_formula_max_delta',
    ]
    if include_row:
        names += [
            'formation_row_ancc',
            'formation_row_ancc_std',
            'formation_row_min_dist',
            'formation_row_anchor_b',
            'formation_row_anchor_b50',
            'formation_row_prefix_rmse',
            'formation_row_prefix_mae',
            'formation_row_prefix_bias',
            'formation_row_tvt_formula',
            'formation_row_delta_formula',
            'formation_row_delta_formula50',
            'formation_row_delta_from_plane',
            'formation_formula_mean_delta',
            'formation_formula_abs_gap',
            'dense_ancc',
            'dense_std',
            'dense_dist',
            'dense_rmse',
            'dense_bias',
            'dense_nb_std',
            'tvt_dense_delta',
            'tvt_dense50_delta',
            'spatial_vs_dense',
        ]
    return names


def _safe_weighted_plane_predict(xy_query: np.ndarray, xy_neighbors: np.ndarray, values_neighbors: np.ndarray, weights: np.ndarray) -> np.ndarray:
    xy_query = np.asarray(xy_query, dtype=float)
    n_query = len(xy_query)
    n_targets = values_neighbors.shape[2]
    out = np.full((n_query, n_targets), np.nan, dtype=np.float32)
    for r in range(n_query):
        valid = np.isfinite(weights[r]) & (weights[r] > 0)
        if valid.sum() < 3:
            if valid.any():
                sw = weights[r, valid].sum()
                out[r] = (values_neighbors[r, valid] * weights[r, valid, None]).sum(axis=0) / max(sw, 1e-12)
            continue
        x = xy_neighbors[r, valid, 0]
        y = xy_neighbors[r, valid, 1]
        w = weights[r, valid]
        X = np.column_stack([x, y, np.ones(len(x))])
        WX = X * w[:, None]
        ata = X.T @ WX
        ata.flat[::4] += 1e-8
        atb = X.T @ (values_neighbors[r, valid] * w[:, None])
        try:
            coef = np.linalg.solve(ata, atb)
        except np.linalg.LinAlgError:
            coef = np.linalg.pinv(ata) @ atb
        qx, qy = xy_query[r]
        out[r] = (qx * coef[0] + qy * coef[1] + coef[2]).astype(np.float32)
    return out


class FormationPlaneKNN:
    """Spatial plane-fit imputer for train-only formation top columns."""

    def __init__(self, well_ids, split_dir: Path, formations=FORMATION_TOP_COLUMNS, k: int = 10):
        self.formations = list(formations)
        self.k = int(k)
        rows = []
        for well_id in sorted(well_ids):
            path = split_dir / f'{well_id}__horizontal_well.csv'
            if not path.exists():
                continue
            try:
                df = pd.read_csv(path, usecols=lambda c: c in {'X', 'Y', *self.formations}).dropna(subset=['X', 'Y'])
            except Exception:
                continue
            if df.empty or not set(self.formations).issubset(df.columns):
                continue
            valid = df[['X', 'Y', *self.formations]].dropna()
            if valid.empty:
                continue
            row = {
                'well_id': well_id,
                'x': float(valid['X'].median()),
                'y': float(valid['Y'].median()),
            }
            for col in self.formations:
                row[col] = float(valid[col].median())
            rows.append(row)
        self.df = pd.DataFrame(rows)
        self.ready = cKDTree is not None and len(self.df) >= 3
        if not self.ready:
            self.well_to_idx = {}
            self.xy = np.empty((0, 2), dtype=float)
            self.values = np.empty((0, len(self.formations)), dtype=float)
            self.scale = np.ones(2, dtype=float)
            self.tree = None
            return
        self.well_to_idx = {well_id: i for i, well_id in enumerate(self.df['well_id'].to_numpy())}
        self.xy = self.df[['x', 'y']].to_numpy(dtype=float)
        self.values = self.df[self.formations].to_numpy(dtype=float)
        self.scale = np.nanstd(self.xy, axis=0)
        self.scale = np.where(self.scale < 1e-6, 1.0, self.scale)
        self.tree = cKDTree(self.xy / self.scale)

    def impute(self, xy_query, self_wid=None, k: int | None = None):
        xy_query = np.atleast_2d(np.asarray(xy_query, dtype=float))
        n = len(xy_query)
        if not self.ready:
            return np.full((n, len(self.formations)), np.nan, dtype=np.float32), np.full(n, np.nan, dtype=np.float32)
        k = int(k or self.k)
        n_fetch = min(max(k + 5, k), len(self.df))
        dist, idx = self.tree.query(xy_query / self.scale, k=n_fetch)
        dist = np.atleast_2d(dist)
        idx = np.atleast_2d(idx)
        if dist.shape[0] != n:
            dist = dist.reshape(n, -1)
            idx = idx.reshape(n, -1)
        if self_wid is not None and self_wid in self.well_to_idx:
            self_idx = self.well_to_idx[self_wid]
            dist = np.where(idx == self_idx, np.inf, dist)
        valid = np.isfinite(dist)
        safe_order_k = min(max(k - 1, 0), dist.shape[1] - 1)
        order = np.argpartition(np.where(valid, dist, np.inf), kth=safe_order_k, axis=1)[:, :k]
        d_k = np.take_along_axis(dist, order, axis=1)
        idx_k = np.take_along_axis(idx, order, axis=1)
        valid_k = np.isfinite(d_k)
        weights = np.where(valid_k, 1.0 / (d_k + 1e-3), 0.0)
        xy_neighbors = self.xy[idx_k]
        value_neighbors = self.values[idx_k]
        pred = _safe_weighted_plane_predict(xy_query, xy_neighbors, value_neighbors, weights)
        no_neighbor = weights.sum(axis=1) <= 1e-12
        if no_neighbor.any():
            pred[no_neighbor] = np.nanmean(self.values, axis=0).astype(np.float32)
        min_dist = np.where(valid_k, d_k, np.inf).min(axis=1).astype(np.float32)
        min_dist[~np.isfinite(min_dist)] = np.nan
        return pred.astype(np.float32), min_dist


class RowANCCKNN:
    """Sampled row-level KNN imputer for ANCC only."""

    def __init__(self, well_ids, split_dir: Path, samples_per_well: int = 400, seed: int = 42):
        xs, ys, vals, wids = [], [], [], []
        rng = np.random.default_rng(seed)
        for well_id in sorted(well_ids):
            path = split_dir / f'{well_id}__horizontal_well.csv'
            if not path.exists():
                continue
            try:
                df = pd.read_csv(path, usecols=lambda c: c in {'X', 'Y', 'ANCC'}).dropna()
            except Exception:
                continue
            if df.empty:
                continue
            if len(df) > samples_per_well:
                idx = np.sort(rng.choice(df.index.to_numpy(), size=samples_per_well, replace=False))
                df = df.loc[idx]
            xs.append(df['X'].to_numpy(dtype=float))
            ys.append(df['Y'].to_numpy(dtype=float))
            vals.append(df['ANCC'].to_numpy(dtype=float))
            wids.extend([well_id] * len(df))
        self.ready = cKDTree is not None and bool(xs)
        if not self.ready:
            self.xy = np.empty((0, 2), dtype=float)
            self.ancc = np.empty(0, dtype=float)
            self.wids = np.empty(0, dtype=object)
            self.scale = np.ones(2, dtype=float)
            self.tree = None
            return
        self.xy = np.column_stack([np.concatenate(xs), np.concatenate(ys)])
        self.ancc = np.concatenate(vals).astype(np.float32)
        self.wids = np.asarray(wids, dtype=object)
        self.scale = np.nanstd(self.xy, axis=0)
        self.scale = np.where(self.scale < 1e-6, 1.0, self.scale)
        self.tree = cKDTree(self.xy / self.scale)

    def impute(self, xy_query, self_wid=None, k: int = 20, extra_fetch: int = 450):
        xy_query = np.atleast_2d(np.asarray(xy_query, dtype=float))
        n = len(xy_query)
        if not self.ready:
            return (
                np.full(n, np.nan, dtype=np.float32),
                np.full(n, np.nan, dtype=np.float32),
                np.full(n, np.nan, dtype=np.float32),
            )
        n_fetch = min(max(k + extra_fetch, k), len(self.ancc))
        dist, idx = self.tree.query(xy_query / self.scale, k=n_fetch, workers=-1)
        dist = np.atleast_2d(dist)
        idx = np.atleast_2d(idx)
        if dist.shape[0] != n:
            dist = dist.reshape(n, -1)
            idx = idx.reshape(n, -1)
        if self_wid is not None:
            dist = np.where(self.wids[idx] == self_wid, np.inf, dist)
        valid = np.isfinite(dist)
        safe_order_k = min(max(k - 1, 0), dist.shape[1] - 1)
        order = np.argpartition(np.where(valid, dist, np.inf), kth=safe_order_k, axis=1)[:, :k]
        d_k = np.take_along_axis(dist, order, axis=1)
        idx_k = np.take_along_axis(idx, order, axis=1)
        valid_k = np.isfinite(d_k)
        weights = np.where(valid_k, 1.0 / (d_k + 1e-3), 0.0)
        sw = weights.sum(axis=1)
        safe = np.where(sw <= 1e-12, 1.0, sw)
        neighbor_vals = self.ancc[idx_k]
        pred = (neighbor_vals * weights).sum(axis=1) / safe
        no_neighbor = sw <= 1e-12
        if no_neighbor.any():
            pred[no_neighbor] = float(np.nanmean(self.ancc))
        var = (((neighbor_vals - pred[:, None]) ** 2) * weights).sum(axis=1) / safe
        std = np.sqrt(np.maximum(var, 0.0))
        min_dist = np.where(valid_k, d_k, np.inf).min(axis=1)
        min_dist[~np.isfinite(min_dist)] = np.nan
        return pred.astype(np.float32), std.astype(np.float32), min_dist.astype(np.float32)


def make_formation_imputers(well_ids, split_dir: Path, need_row_ancc: bool = False, seed: int = 42):
    plane = FormationPlaneKNN(well_ids, split_dir=split_dir, k=10)
    row = RowANCCKNN(well_ids, split_dir=split_dir, samples_per_well=400, seed=seed) if need_row_ancc else None
    return plane, row


def make_tail_features_for_well(
    well_id: str,
    split_dir: Path,
    include_target: bool = True,
    use_beam_features: bool | None = None,
    required_feature_columns=None,
    formation_plane_imputer=None,
    row_ancc_imputer=None,
    exclude_query_well_from_formation: bool = True,
) -> pd.DataFrame:
    if use_beam_features is None:
        use_beam_features = bool(globals().get('ENABLE_OFFLINE_BEAM_FEATURES', False))
    required_feature_columns = normalize_required_feature_columns(required_feature_columns)
    need_trailing_gr_roll = needs_any(required_feature_columns, prefixes=('gr_roll_',))
    need_offline_gr_context = needs_any(
        required_feature_columns,
        prefixes=('gr_center_',),
        exact=('gr_cumsum_since_ps',),
    )
    need_typewell_features = needs_any(
        required_feature_columns,
        prefixes=(
            'typewell_', 'gr_minus_typewell_', 'prefix_horizontal_vs_typewell_',
            'calibrated_', 'gr_minus_calibrated_', 'prefix_typewell_',
            'tw_path_', 'tw_path_ease_', 'tw_beam_', 'beam_',
            'selfcorr_', 'sc_', 'hyb_', 'tdbc_', 'tdsc_', 'pf_lite_',
        ),
    )
    need_typewell_slope_features = needs_any(
        required_feature_columns,
        prefixes=('gr_minus_typewell_slope_',),
        exact=('typewell_gr_at_slope_baseline_all', 'typewell_gr_at_slope_baseline_last200'),
    )
    need_local_typewell = needs_any(required_feature_columns, prefixes=('typewell_local_last200_',))
    need_calibrated_typewell = needs_any(
        required_feature_columns,
        prefixes=('calibrated_', 'gr_minus_calibrated_'),
        exact=(
            'prefix_typewell_gr_affine_a',
            'prefix_typewell_gr_affine_b',
            'prefix_horizontal_vs_calibrated_typewell_gr_mae',
            'prefix_horizontal_vs_calibrated_typewell_gr_rmse',
            'typewell_calibrated_gr_at_last_known_tvt',
            'typewell_calibrated_gr_at_slope_baseline_all',
            'typewell_calibrated_gr_at_slope_baseline_last200',
        ),
    )
    need_local_calibrated_typewell = needs_any(required_feature_columns, prefixes=('calibrated_typewell_local_last200_',))
    need_typewell_interval_context = needs_any(
        required_feature_columns,
        prefixes=('typewell_last_geo_', 'typewell_baseline_last200_geo_'),
    )
    need_typewell_anchor_offsets = needs_any(required_feature_columns, prefixes=('typewell_anchor_gr_diff_',))
    need_calibrated_anchor_offsets = needs_any(required_feature_columns, prefixes=('calibrated_typewell_anchor_gr_diff_',))
    need_candidate_path = needs_any(required_feature_columns, prefixes=('tw_path_', 'tw_path_ease_'))
    need_selfcorr = needs_any(required_feature_columns, prefixes=('selfcorr_', 'sc_', 'hyb_', 'tdsc_', 'pf_lite_'))
    need_pf_lite = needs_any(required_feature_columns, prefixes=('pf_lite_',))
    need_tdbc_offsets = needs_any(required_feature_columns, prefixes=('tdbc_',))
    need_tdsc_offsets = needs_any(required_feature_columns, prefixes=('tdsc_',))
    need_beam = bool(use_beam_features and needs_any(required_feature_columns, prefixes=('tw_beam_', 'beam_', 'tdbc_', 'pf_lite_')))
    need_formation_plane = needs_any(required_feature_columns, prefixes=('formation_plane_', 'spatial_vs_'))
    need_row_ancc = needs_any(
        required_feature_columns,
        prefixes=('formation_row_', 'formation_formula_', 'dense_', 'tvt_dense_', 'spatial_vs_'),
    )
    need_formation_features = need_formation_plane or need_row_ancc
    path = split_dir / f'{well_id}__horizontal_well.csv'
    df = pd.read_csv(path)
    mask = df['TVT_input'].isna().to_numpy()
    pred_idx = np.flatnonzero(mask)
    if len(pred_idx) == 0:
        return pd.DataFrame()
    ps = int(pred_idx[0])
    tail = df.iloc[pred_idx].copy()
    prefix = df.iloc[:ps].copy()
    last_known_tvt = float(prefix['TVT_input'].iloc[-1])
    last_known_md = float(prefix['MD'].iloc[-1])
    ps_md = float(df['MD'].iloc[ps])
    ps_x = float(df['X'].iloc[ps])
    ps_y = float(df['Y'].iloc[ps])
    ps_z = float(df['Z'].iloc[ps])
    prefix_tvt = pd.to_numeric(prefix['TVT_input'], errors='coerce')
    prefix_gr = pd.to_numeric(prefix['GR'], errors='coerce')
    tail_gr = pd.to_numeric(tail['GR'], errors='coerce').to_numpy(dtype=float)
    prefix_gr_mean = safe_stat(prefix_gr, np.mean)
    prefix_gr_std = safe_stat(prefix_gr, np.std)
    prefix_tvt_values = prefix_tvt.to_numpy(dtype=float)
    prefix_md_values = pd.to_numeric(prefix['MD'], errors='coerce').to_numpy(dtype=float)
    prefix_z_values = pd.to_numeric(prefix['Z'], errors='coerce').to_numpy(dtype=float)
    prefix_tvt_slope_all = safe_slope(prefix_md_values, prefix_tvt_values)
    prefix_tvt_slope_last200 = safe_slope(prefix['MD'].tail(200).to_numpy(), prefix_tvt.tail(200).to_numpy())
    prefix_tvt_step20 = recent_mean_diff(prefix_tvt_values, 20)
    prefix_tvt_step100 = recent_mean_diff(prefix_tvt_values, 100)
    prefix_tvt_md_slope100 = recent_slope_window(prefix_tvt_values, prefix_md_values, 100)
    prefix_tvt_z_slope100 = recent_slope_window(prefix_tvt_values, prefix_z_values, 100)
    
    # Use only the known prefix to estimate orientation. Full-well endpoint azimuth would use future tail information.
    if len(prefix) >= 2:
        dx_prefix = float(prefix['X'].iloc[-1] - prefix['X'].iloc[0])
        dy_prefix = float(prefix['Y'].iloc[-1] - prefix['Y'].iloc[0])
        prefix_azimuth = (np.degrees(np.arctan2(dx_prefix, dy_prefix)) + 360.0) % 360.0
    else:
        prefix_azimuth = np.nan
    
    md_since_ps = tail['MD'].to_numpy(dtype=float) - ps_md
    tail_len = len(tail)
    tail_row_number = np.arange(tail_len, dtype=float)
    tail_frac = tail_row_number / max(tail_len - 1, 1)
    n_rows = len(df)
    row_frac = pred_idx.astype(float) / max(n_rows - 1, 1)
    md_tail_span = float(tail['MD'].iloc[-1] - ps_md) if len(tail) else np.nan
    md_tail_frac = md_since_ps / md_tail_span if np.isfinite(md_tail_span) and abs(md_tail_span) > 1e-9 else np.full(tail_len, np.nan)
    tail_gr_missing_rate = float(pd.to_numeric(tail['GR'], errors='coerce').isna().mean())
    tail_frac2 = tail_frac ** 2
    tail_frac3 = tail_frac ** 3
    sqrt_tail_frac = np.sqrt(np.clip(tail_frac, 0.0, None))
    log1p_tail_row = np.log1p(tail_row_number)
    sin_tail_frac_pi = np.sin(np.pi * tail_frac)
    sin_tail_frac_2pi = np.sin(2.0 * np.pi * tail_frac)
    cos_tail_frac_3pi = np.cos(3.0 * np.pi * tail_frac)
    gap_x_delta = float(tail['X'].iloc[-1] - tail['X'].iloc[0]) if len(tail) else np.nan
    gap_y_delta = float(tail['Y'].iloc[-1] - tail['Y'].iloc[0]) if len(tail) else np.nan
    gap_z_delta = float(tail['Z'].iloc[-1] - tail['Z'].iloc[0]) if len(tail) else np.nan
    gap_xy_span = float(np.hypot(gap_x_delta, gap_y_delta)) if np.isfinite(gap_x_delta) and np.isfinite(gap_y_delta) else np.nan
    gap_z_over_xy = gap_z_delta / (gap_xy_span + 1.0) if np.isfinite(gap_z_delta) and np.isfinite(gap_xy_span) else np.nan
    finite_tail_gr = tail_gr[np.isfinite(tail_gr)]
    if len(finite_tail_gr):
        gap_gr_mean = float(np.mean(finite_tail_gr))
        gap_gr_std = float(np.std(finite_tail_gr))
        gap_gr_min = float(np.min(finite_tail_gr))
        gap_gr_max = float(np.max(finite_tail_gr))
        gap_gr_q = {q: float(np.quantile(finite_tail_gr, q)) for q in [0.05, 0.25, 0.50, 0.75, 0.95]}
    else:
        gap_gr_mean = gap_gr_std = gap_gr_min = gap_gr_max = np.nan
        gap_gr_q = {q: np.nan for q in [0.05, 0.25, 0.50, 0.75, 0.95]}
    x_delta_ps = tail['X'].to_numpy(dtype=float) - ps_x
    y_delta_ps = tail['Y'].to_numpy(dtype=float) - ps_y
    z_delta_ps = tail['Z'].to_numpy(dtype=float) - ps_z
    safe_md_since_ps = np.where(np.abs(md_since_ps) > 1e-9, md_since_ps, np.nan)
    dist_xyz_ps = np.sqrt(x_delta_ps ** 2 + y_delta_ps ** 2 + z_delta_ps ** 2)
    slope_delta_all = prefix_tvt_slope_all * md_since_ps if np.isfinite(prefix_tvt_slope_all) else np.full(len(tail), np.nan)
    slope_delta_last200 = prefix_tvt_slope_last200 * md_since_ps if np.isfinite(prefix_tvt_slope_last200) else np.full(len(tail), np.nan)
    slope_baseline_tvt_all = last_known_tvt + slope_delta_all
    slope_baseline_tvt_last200 = last_known_tvt + slope_delta_last200
    prefix_valid_gr = prefix_gr.dropna()
    if len(prefix_valid_gr):
        prefix_last_valid_gr = float(prefix_valid_gr.iloc[-1])
        prefix_last_valid_gr_index = int(prefix_valid_gr.index[-1])
        prefix_last_valid_gr_md = float(df['MD'].iloc[prefix_last_valid_gr_index])
    else:
        prefix_last_valid_gr = np.nan
        prefix_last_valid_gr_index = -1
        prefix_last_valid_gr_md = np.nan
    gr_prefix_z = (tail_gr - prefix_gr_mean) / (prefix_gr_std + 1e-6) if np.isfinite(prefix_gr_std) else np.full(len(tail), np.nan)
    
    md_full = pd.to_numeric(df['MD'], errors='coerce')
    x_full = pd.to_numeric(df['X'], errors='coerce')
    y_full = pd.to_numeric(df['Y'], errors='coerce')
    z_full = pd.to_numeric(df['Z'], errors='coerce')
    gr_full_numeric = pd.to_numeric(df['GR'], errors='coerce')
    md_step_1 = md_full.diff(1).iloc[pred_idx].to_numpy(dtype=float)
    x_step_1 = x_full.diff(1).iloc[pred_idx].to_numpy(dtype=float)
    y_step_1 = y_full.diff(1).iloc[pred_idx].to_numpy(dtype=float)
    z_step_1 = z_full.diff(1).iloc[pred_idx].to_numpy(dtype=float)
    gr_diff_1 = gr_full_numeric.diff(1).iloc[pred_idx].to_numpy(dtype=float)
    gr_diff_5 = gr_full_numeric.diff(5).iloc[pred_idx].to_numpy(dtype=float)
    safe_md_step_1 = np.where(np.abs(md_step_1) > 1e-9, md_step_1, np.nan)
    
    out = pd.DataFrame({
        'well_id': well_id,
        'row_index': pred_idx,
        'id': [f'{well_id}_{i}' for i in pred_idx],
        'MD': tail['MD'].to_numpy(),
        'X': tail['X'].to_numpy(),
        'Y': tail['Y'].to_numpy(),
        'Z': tail['Z'].to_numpy(),
        'GR': tail_gr,
        'GR_isna': tail['GR'].isna().astype(int).to_numpy(),
        'GR_prefix_z': gr_prefix_z,
        'gr_diff_1': gr_diff_1,
        'gr_diff_5': gr_diff_5,
        'gr_slope_md_1': gr_diff_1 / safe_md_step_1,
        'md_step_1': md_step_1,
        'x_step_1': x_step_1,
        'y_step_1': y_step_1,
        'z_step_1': z_step_1,
        'trajectory_step_1': np.sqrt(x_step_1 ** 2 + y_step_1 ** 2 + z_step_1 ** 2),
        'z_slope_md_1': z_step_1 / safe_md_step_1,
        'last_known_TVT': last_known_tvt,
        'last_known_MD': last_known_md,
        'tail_len': tail_len,
        'tail_row_number': tail_row_number,
        'tail_frac': tail_frac,
        'tail_frac2': tail_frac2,
        'tail_frac3': tail_frac3,
        'sqrt_tail_frac': sqrt_tail_frac,
        'log1p_tail_row': log1p_tail_row,
        'sin_tail_frac_pi': sin_tail_frac_pi,
        'sin_tail_frac_2pi': sin_tail_frac_2pi,
        'cos_tail_frac_3pi': cos_tail_frac_3pi,
        'n_rows': n_rows,
        'row_frac': row_frac,
        'md_tail_span': md_tail_span,
        'md_tail_frac': md_tail_frac,
        'gap_md_span': md_tail_span,
        'gap_x_delta': gap_x_delta,
        'gap_y_delta': gap_y_delta,
        'gap_z_delta': gap_z_delta,
        'gap_xy_span': gap_xy_span,
        'gap_z_over_xy': gap_z_over_xy,
        'gap_gr_mean': gap_gr_mean,
        'gap_gr_std': gap_gr_std,
        'gap_gr_min': gap_gr_min,
        'gap_gr_p05': gap_gr_q[0.05],
        'gap_gr_p25': gap_gr_q[0.25],
        'gap_gr_p50': gap_gr_q[0.50],
        'gap_gr_p75': gap_gr_q[0.75],
        'gap_gr_p95': gap_gr_q[0.95],
        'gap_gr_max': gap_gr_max,
        'tail_gr_missing_rate': tail_gr_missing_rate,
        'md_since_ps': md_since_ps,
        'x_delta_ps': x_delta_ps,
        'y_delta_ps': y_delta_ps,
        'z_delta_ps': z_delta_ps,
        'xy_dist_ps': np.hypot(x_delta_ps, y_delta_ps),
        'dist_xyz_ps': dist_xyz_ps,
        'dx_per_md_since_ps': x_delta_ps / safe_md_since_ps,
        'dy_per_md_since_ps': y_delta_ps / safe_md_since_ps,
        'dz_per_md_since_ps': z_delta_ps / safe_md_since_ps,
        'prefix_len': len(prefix),
        'prefix_azimuth_deg': prefix_azimuth,
        'prefix_gr_missing_rate': float(prefix_gr.isna().mean()),
        'prefix_gr_mean': prefix_gr_mean,
        'prefix_gr_std': prefix_gr_std,
        'prefix_gr_min': safe_stat(prefix_gr, np.min),
        'prefix_gr_max': safe_stat(prefix_gr, np.max),
        'prefix_last_valid_gr': prefix_last_valid_gr,
        'rows_since_prefix_last_valid_gr': pred_idx - prefix_last_valid_gr_index if prefix_last_valid_gr_index >= 0 else np.nan,
        'md_since_prefix_last_valid_gr': tail['MD'].to_numpy(dtype=float) - prefix_last_valid_gr_md,
        'gr_minus_prefix_last_valid_gr': tail_gr - prefix_last_valid_gr,
        'gr_minus_prefix_gr_mean': tail_gr - prefix_gr_mean,
        'prefix_tvt_min': safe_stat(prefix_tvt, np.min),
        'prefix_tvt_max': safe_stat(prefix_tvt, np.max),
        'prefix_tvt_range': safe_stat(prefix_tvt, np.max) - safe_stat(prefix_tvt, np.min),
        'prefix_tvt_mean': safe_stat(prefix_tvt, np.mean),
        'prefix_tvt_std': safe_stat(prefix_tvt, np.std),
        'prefix_tvt_slope_md_all': prefix_tvt_slope_all,
        'prefix_tvt_slope_md_last200': prefix_tvt_slope_last200,
        'prefix_tvt_step20': prefix_tvt_step20,
        'prefix_tvt_step100': prefix_tvt_step100,
        'prefix_tvt_md_slope100': prefix_tvt_md_slope100,
        'prefix_tvt_z_slope100': prefix_tvt_z_slope100,
        'slope_baseline_delta_all': slope_delta_all,
        'slope_baseline_delta_last200': slope_delta_last200,
        'slope_baseline_tvt_all': slope_baseline_tvt_all,
        'slope_baseline_tvt_last200': slope_baseline_tvt_last200,
    })
    
    # Rolling GR features are trailing only for strict mode.
    gr_full = df['GR'].copy()
    if need_trailing_gr_roll:
        for window in [25, 100, 300]:
            roll = gr_full.rolling(window, min_periods=max(5, window // 5))
            out[f'gr_roll_mean_{window}'] = roll.mean().iloc[pred_idx].to_numpy()
            out[f'gr_roll_std_{window}'] = roll.std().iloc[pred_idx].to_numpy()
            out[f'gr_roll_min_{window}'] = roll.min().iloc[pred_idx].to_numpy()
            out[f'gr_roll_max_{window}'] = roll.max().iloc[pred_idx].to_numpy()
            out[f'gr_roll_range_{window}'] = (roll.max() - roll.min()).iloc[pred_idx].to_numpy()
    
    # Offline features use target-free covariates from the full provided horizontal file.
    gr_full_filled = None
    if need_offline_gr_context:
        for window in [5, 21, 51, 151, 301]:
            center_roll = gr_full.rolling(window, center=True, min_periods=max(1, min(5, window // 2)))
            out[f'gr_center_roll_mean_{window}'] = center_roll.mean().iloc[pred_idx].to_numpy()
            out[f'gr_center_roll_std_{window}'] = center_roll.std().iloc[pred_idx].to_numpy()
            out[f'gr_center_roll_range_{window}'] = (center_roll.max() - center_roll.min()).iloc[pred_idx].to_numpy()
        fallback_gr = prefix_gr_mean if np.isfinite(prefix_gr_mean) else 0.0
        gr_full_filled = gr_full_numeric.interpolate(limit_direction='both').fillna(fallback_gr)
        gr_cumsum = gr_full_filled.cumsum()
        cumsum_anchor = float(gr_cumsum.iloc[ps - 1]) if ps > 0 else 0.0
        out['gr_center_grad_1'] = gr_full_filled.diff().fillna(0.0).iloc[pred_idx].to_numpy()
        out['gr_center_lag1'] = gr_full_filled.shift(1).bfill().iloc[pred_idx].to_numpy()
        out['gr_center_lead1'] = gr_full_filled.shift(-1).ffill().iloc[pred_idx].to_numpy()
        out['gr_center_lag5'] = gr_full_filled.shift(5).bfill().iloc[pred_idx].to_numpy()
        out['gr_center_lead5'] = gr_full_filled.shift(-5).ffill().iloc[pred_idx].to_numpy()
        out['gr_center_lag15'] = gr_full_filled.shift(15).bfill().iloc[pred_idx].to_numpy()
        out['gr_center_lead15'] = gr_full_filled.shift(-15).ffill().iloc[pred_idx].to_numpy()
        out['gr_center_lag30'] = gr_full_filled.shift(30).bfill().iloc[pred_idx].to_numpy()
        out['gr_center_lead30'] = gr_full_filled.shift(-30).ffill().iloc[pred_idx].to_numpy()
        out['gr_center_grad_2'] = gr_full_filled.diff(2).fillna(0.0).iloc[pred_idx].to_numpy()
        out['gr_cumsum_since_ps'] = (gr_cumsum.iloc[pred_idx] - cumsum_anchor).to_numpy()

    selfcorr_tvt = np.full(len(tail), np.nan, dtype=np.float32)
    selfcorr_delta = np.full(len(tail), np.nan, dtype=np.float32)
    if need_selfcorr:
        sc_features = selfcorr_prefix_tvt_features(
            prefix_gr.to_numpy(dtype=float),
            prefix_tvt_values,
            tail_gr,
            last_known_tvt,
            half_window=15,
            stride=3,
        )
        for col, values in sc_features.items():
            out[col] = values
        selfcorr_tvt = np.asarray(sc_features['selfcorr_tvt'], dtype=float)
        selfcorr_delta = np.asarray(sc_features['selfcorr_delta'], dtype=float)
        out['hyb_delta'] = selfcorr_delta

    plane_delta = np.full(len(tail), np.nan, dtype=float)
    row_delta = np.full(len(tail), np.nan, dtype=float)

    # Formation top features use train-only surfaces only as auxiliary labels for a spatial imputer.
    if need_formation_features:
        self_wid = well_id if exclude_query_well_from_formation else None
        prefix_xy = prefix[['X', 'Y']].to_numpy(dtype=float)
        tail_xy = tail[['X', 'Y']].to_numpy(dtype=float)
        prefix_z = prefix['Z'].to_numpy(dtype=float)
        tail_z = tail['Z'].to_numpy(dtype=float)
        if need_formation_plane and formation_plane_imputer is not None and getattr(formation_plane_imputer, 'ready', False):
            plane_prefix, _ = formation_plane_imputer.impute(prefix_xy, self_wid=self_wid)
            plane_tail, plane_dist = formation_plane_imputer.impute(tail_xy, self_wid=self_wid)
            out['formation_plane_min_dist'] = plane_dist
            plane_delta_stack = []
            for label_idx, label in enumerate(FORMATION_LABELS):
                plane_prefix_surface = plane_prefix[:, label_idx]
                plane_tail_surface = plane_tail[:, label_idx]
                out[f'formation_plane_{label}'] = plane_tail_surface
                valid_b = np.isfinite(prefix_tvt_values) & np.isfinite(prefix_z) & np.isfinite(plane_prefix_surface)
                anchor_series = prefix_tvt_values + prefix_z - plane_prefix_surface
                anchor_b = float(np.nanmedian(anchor_series[valid_b])) if valid_b.any() else np.nan
                if valid_b.any():
                    valid_idx = np.flatnonzero(valid_b)
                    tail_idx = valid_idx[-min(50, len(valid_idx)):]
                    anchor_b50 = float(np.nanmedian(anchor_series[tail_idx]))
                else:
                    anchor_b50 = np.nan
                tvt_formula = -tail_z + plane_tail_surface + anchor_b
                tvt_formula50 = -tail_z + plane_tail_surface + anchor_b50
                prefix_formula = -prefix_z + plane_prefix_surface + anchor_b
                prefix_valid_formula = valid_b & np.isfinite(prefix_formula)
                prefix_resid = prefix_tvt_values[prefix_valid_formula] - prefix_formula[prefix_valid_formula]
                delta_formula = tvt_formula - last_known_tvt
                out[f'formation_plane_anchor_b_{label}'] = anchor_b
                out[f'formation_plane_anchor_b50_{label}'] = anchor_b50
                out[f'formation_plane_prefix_rmse_{label}'] = float(np.sqrt(np.mean(prefix_resid ** 2))) if len(prefix_resid) else np.nan
                out[f'formation_plane_prefix_mae_{label}'] = float(np.mean(np.abs(prefix_resid))) if len(prefix_resid) else np.nan
                out[f'formation_plane_tvt_formula_{label}'] = tvt_formula
                out[f'formation_plane_delta_formula_{label}'] = delta_formula
                out[f'formation_plane_delta_formula50_{label}'] = tvt_formula50 - last_known_tvt
                plane_delta_stack.append(delta_formula)
                if label == 'ancc':
                    out['formation_plane_anchor_b'] = anchor_b
                    out['formation_plane_anchor_b50'] = anchor_b50
                    out['formation_plane_prefix_rmse'] = out[f'formation_plane_prefix_rmse_{label}']
                    out['formation_plane_prefix_mae'] = out[f'formation_plane_prefix_mae_{label}']
                    out['formation_plane_tvt_formula'] = tvt_formula
                    out['formation_plane_delta_formula'] = delta_formula
                    out['formation_plane_delta_from_slope_last200'] = (tvt_formula - slope_baseline_tvt_last200)
            if plane_delta_stack:
                plane_delta_matrix = np.vstack(plane_delta_stack)
                out['formation_plane_formula_mean_delta'] = np.nanmean(plane_delta_matrix, axis=0)
                out['formation_plane_formula_std_delta'] = np.nanstd(plane_delta_matrix, axis=0)
                out['formation_plane_formula_min_delta'] = np.nanmin(plane_delta_matrix, axis=0)
                out['formation_plane_formula_max_delta'] = np.nanmax(plane_delta_matrix, axis=0)
            plane_delta = np.asarray(out.get('formation_plane_delta_formula', np.full(len(tail), np.nan)), dtype=float)
        if need_row_ancc and row_ancc_imputer is not None and getattr(row_ancc_imputer, 'ready', False):
            row_prefix_ancc, row_prefix_std, _ = row_ancc_imputer.impute(prefix_xy, self_wid=self_wid)
            row_tail_ancc, row_tail_std, row_dist = row_ancc_imputer.impute(tail_xy, self_wid=self_wid)
            valid_b = np.isfinite(prefix_tvt_values) & np.isfinite(prefix_z) & np.isfinite(row_prefix_ancc)
            row_anchor_series = prefix_tvt_values + prefix_z - row_prefix_ancc
            row_anchor_b = float(np.nanmedian(row_anchor_series[valid_b])) if valid_b.any() else np.nan
            if valid_b.any():
                valid_idx = np.flatnonzero(valid_b)
                tail_idx = valid_idx[-min(50, len(valid_idx)):]
                row_anchor_b50 = float(np.nanmedian(row_anchor_series[tail_idx]))
            else:
                row_anchor_b50 = np.nan
            row_tvt_formula = -tail_z + row_tail_ancc + row_anchor_b
            row_tvt_formula50 = -tail_z + row_tail_ancc + row_anchor_b50
            row_prefix_formula = -prefix_z + row_prefix_ancc + row_anchor_b
            prefix_valid_formula = valid_b & np.isfinite(row_prefix_formula)
            row_prefix_resid = prefix_tvt_values[prefix_valid_formula] - row_prefix_formula[prefix_valid_formula]
            out['formation_row_ancc'] = row_tail_ancc
            out['formation_row_ancc_std'] = row_tail_std
            out['formation_row_min_dist'] = row_dist
            out['formation_row_anchor_b'] = row_anchor_b
            out['formation_row_anchor_b50'] = row_anchor_b50
            out['formation_row_prefix_rmse'] = float(np.sqrt(np.mean(row_prefix_resid ** 2))) if len(row_prefix_resid) else np.nan
            out['formation_row_prefix_mae'] = float(np.mean(np.abs(row_prefix_resid))) if len(row_prefix_resid) else np.nan
            out['formation_row_prefix_bias'] = float(np.mean(row_prefix_resid)) if len(row_prefix_resid) else np.nan
            out['formation_row_tvt_formula'] = row_tvt_formula
            out['formation_row_delta_formula'] = row_tvt_formula - last_known_tvt
            out['formation_row_delta_formula50'] = row_tvt_formula50 - last_known_tvt
            row_delta = np.asarray(out['formation_row_delta_formula'], dtype=float)
            out['formation_row_delta_from_plane'] = row_delta - plane_delta
            out['dense_ancc'] = row_tail_ancc
            out['dense_std'] = row_tail_std
            out['dense_dist'] = row_dist
            out['dense_rmse'] = out['formation_row_prefix_rmse']
            out['dense_bias'] = out['formation_row_prefix_bias']
            out['dense_nb_std'] = row_tail_std
            out['tvt_dense_delta'] = row_delta
            out['tvt_dense50_delta'] = out['formation_row_delta_formula50']
            out['spatial_vs_dense'] = plane_delta - row_delta
            stacked_delta = np.vstack([plane_delta, row_delta])
            finite_count = np.isfinite(stacked_delta).sum(axis=0)
            summed_delta = np.nansum(stacked_delta, axis=0)
            out['formation_formula_mean_delta'] = np.divide(
                summed_delta,
                finite_count,
                out=np.full(len(tail), np.nan, dtype=float),
                where=finite_count > 0,
            )
            out['formation_formula_abs_gap'] = np.abs(row_delta - plane_delta)
    
    # Typewell features are available reference-well features. They are evaluated only at prefix-derived TVT baselines.
    tw_path = split_dir / f'{well_id}__typewell.csv'
    if need_typewell_features and tw_path.exists():
        tw = pd.read_csv(tw_path)
    else:
        tw = pd.DataFrame(columns=['TVT', 'GR'])
    if need_typewell_features and {'TVT', 'GR'}.issubset(tw.columns) and tw[['TVT', 'GR']].dropna().shape[0] >= 2:
        tw_tvt = pd.to_numeric(tw['TVT'], errors='coerce')
        tw_gr = pd.to_numeric(tw['GR'], errors='coerce')
        tw_last_known_gr = typewell_gr_at_tvt(tw, np.array([last_known_tvt]))[0]
        tw_slope_all_gr = typewell_gr_at_tvt(tw, slope_baseline_tvt_all) if need_typewell_slope_features else np.full(len(tail), np.nan)
        tw_slope_last200_gr = typewell_gr_at_tvt(tw, slope_baseline_tvt_last200) if (need_typewell_slope_features or need_local_typewell or need_local_calibrated_typewell or need_calibrated_typewell) else np.full(len(tail), np.nan)
        prefix_tw_gr = typewell_gr_at_tvt(tw, prefix_tvt.to_numpy(dtype=float))
        prefix_gr_values = prefix_gr.to_numpy(dtype=float)
        valid_prefix_align = np.isfinite(prefix_gr_values) & np.isfinite(prefix_tw_gr)
        prefix_align_residual = prefix_gr_values - prefix_tw_gr
        prefix_align_absdiff = (
            float(np.mean(np.abs(prefix_align_residual[valid_prefix_align])))
            if valid_prefix_align.any()
            else np.nan
        )
        prefix_align_rmse = (
            float(np.sqrt(np.mean(prefix_align_residual[valid_prefix_align] ** 2)))
            if valid_prefix_align.any()
            else np.nan
        )
        calib_a = calib_b = np.nan
        prefix_calib_mae = prefix_calib_rmse = np.nan
        tw_last_known_gr_calibrated = np.nan
        tw_slope_all_gr_calibrated = np.full(len(tail), np.nan)
        tw_slope_last200_gr_calibrated = np.full(len(tail), np.nan)
        if need_calibrated_typewell or need_local_calibrated_typewell or need_calibrated_anchor_offsets:
            calib_a, calib_b = safe_affine_fit(prefix_tw_gr, prefix_gr_values)
            prefix_tw_gr_calibrated = apply_affine(prefix_tw_gr, calib_a, calib_b)
            tw_last_known_gr_calibrated = apply_affine(np.array([tw_last_known_gr]), calib_a, calib_b)[0]
            tw_slope_all_gr_calibrated = apply_affine(tw_slope_all_gr, calib_a, calib_b)
            tw_slope_last200_gr_calibrated = apply_affine(tw_slope_last200_gr, calib_a, calib_b)
            valid_prefix_calib = np.isfinite(prefix_gr_values) & np.isfinite(prefix_tw_gr_calibrated)
            prefix_calib_residual = prefix_gr_values - prefix_tw_gr_calibrated
            prefix_calib_mae = (
                float(np.mean(np.abs(prefix_calib_residual[valid_prefix_calib])))
                if valid_prefix_calib.any()
                else np.nan
            )
            prefix_calib_rmse = (
                float(np.sqrt(np.mean(prefix_calib_residual[valid_prefix_calib] ** 2)))
                if valid_prefix_calib.any()
                else np.nan
            )
        local_offsets = np.arange(-60.0, 60.1, 5.0)
        local_raw = {}
        if need_local_typewell:
            local_raw = local_typewell_offset_features(
                tw,
                slope_baseline_tvt_last200,
                tail_gr,
                local_offsets,
                prefix='typewell_local_last200',
            )
        local_calibrated = {}
        if need_local_calibrated_typewell:
            local_calibrated = local_typewell_offset_features(
                tw,
                slope_baseline_tvt_last200,
                tail_gr,
                local_offsets,
                affine_a=calib_a,
                affine_b=calib_b,
                prefix='calibrated_typewell_local_last200',
            )
        out['typewell_tvt_min'] = safe_stat(tw_tvt, np.min)
        out['typewell_tvt_max'] = safe_stat(tw_tvt, np.max)
        out['typewell_tvt_range'] = safe_stat(tw_tvt, np.max) - safe_stat(tw_tvt, np.min)
        out['typewell_gr_mean'] = safe_stat(tw_gr, np.mean)
        out['typewell_gr_std'] = safe_stat(tw_gr, np.std)
        out['typewell_gr_at_last_known_tvt'] = tw_last_known_gr
        out['typewell_gr_at_slope_baseline_all'] = tw_slope_all_gr
        out['typewell_gr_at_slope_baseline_last200'] = tw_slope_last200_gr
        out['gr_minus_typewell_last_known_tvt'] = tail_gr - tw_last_known_gr
        out['gr_minus_typewell_slope_baseline_all'] = tail_gr - tw_slope_all_gr
        out['gr_minus_typewell_slope_baseline_last200'] = tail_gr - tw_slope_last200_gr
        out['prefix_horizontal_vs_typewell_gr_corr'] = safe_corr(prefix_gr_values, prefix_tw_gr)
        out['prefix_horizontal_vs_typewell_gr_mae'] = prefix_align_absdiff
        out['prefix_horizontal_vs_typewell_gr_rmse'] = prefix_align_rmse
        if need_typewell_interval_context:
            last_geo_context = typewell_interval_context_features(tw, np.array([last_known_tvt]), prefix='typewell_last_geo')
            for col, values in last_geo_context.items():
                out[col] = values[0]
            baseline_geo_context = typewell_interval_context_features(tw, slope_baseline_tvt_last200, prefix='typewell_baseline_last200_geo')
            for col, values in baseline_geo_context.items():
                out[col] = values
        anchor_offsets = np.array([-80, -40, -20, -10, -5, 0, 5, 10, 20, 40, 80], dtype=float)
        if need_typewell_anchor_offsets:
            for anchor_offset in anchor_offsets:
                label = candidate_endpoint_label(float(anchor_offset))
                anchor_gr = typewell_gr_at_tvt(tw, np.array([last_known_tvt + float(anchor_offset)]))[0]
                out[f'typewell_anchor_gr_diff_{label}'] = tail_gr - anchor_gr
        if need_local_typewell:
            for col, values in local_raw.items():
                out[col] = values
            out['typewell_local_last200_gr_resid_best'] = tail_gr - out['typewell_local_last200_best_gr']
            out['typewell_local_last200_best_vs_zero_gain'] = out['typewell_local_last200_zero_abs_resid'] - out['typewell_local_last200_best_abs_resid']
        out['prefix_typewell_gr_affine_a'] = calib_a
        out['prefix_typewell_gr_affine_b'] = calib_b
        out['prefix_horizontal_vs_calibrated_typewell_gr_mae'] = prefix_calib_mae
        out['prefix_horizontal_vs_calibrated_typewell_gr_rmse'] = prefix_calib_rmse
        if need_calibrated_anchor_offsets:
            for anchor_offset in anchor_offsets:
                label = candidate_endpoint_label(float(anchor_offset))
                anchor_gr = typewell_gr_at_tvt(tw, np.array([last_known_tvt + float(anchor_offset)]))[0]
                anchor_gr_calibrated = apply_affine(np.array([anchor_gr]), calib_a, calib_b)[0]
                out[f'calibrated_typewell_anchor_gr_diff_{label}'] = tail_gr - anchor_gr_calibrated
        out['typewell_calibrated_gr_at_last_known_tvt'] = tw_last_known_gr_calibrated
        out['typewell_calibrated_gr_at_slope_baseline_all'] = tw_slope_all_gr_calibrated
        out['typewell_calibrated_gr_at_slope_baseline_last200'] = tw_slope_last200_gr_calibrated
        out['gr_minus_calibrated_typewell_last_known_tvt'] = tail_gr - tw_last_known_gr_calibrated
        out['gr_minus_calibrated_typewell_slope_baseline_all'] = tail_gr - tw_slope_all_gr_calibrated
        out['gr_minus_calibrated_typewell_slope_baseline_last200'] = tail_gr - tw_slope_last200_gr_calibrated
        out['calibrated_typewell_slope_last200_gr_prefix_z'] = (tw_slope_last200_gr_calibrated - prefix_gr_mean) / (prefix_gr_std + 1e-6) if np.isfinite(prefix_gr_std) else np.nan
        if need_local_calibrated_typewell:
            for col, values in local_calibrated.items():
                out[col] = values
            out['calibrated_typewell_local_last200_gr_resid_best'] = tail_gr - out['calibrated_typewell_local_last200_best_gr']
            out['calibrated_typewell_local_last200_best_vs_zero_gain'] = out['calibrated_typewell_local_last200_zero_abs_resid'] - out['calibrated_typewell_local_last200_best_abs_resid']
        if need_candidate_path:
            candidate_path_features = typewell_candidate_path_features(
                tw,
                last_known_tvt,
                tail_frac,
                tail_gr,
                endpoints=CANDIDATE_PATH_ENDPOINTS,
                prefix='tw_path',
            )
            candidate_path_ease_features = typewell_candidate_path_features(
                tw,
                last_known_tvt,
                np.power(np.clip(tail_frac, 0.0, None), 1.45),
                tail_gr,
                endpoints=CANDIDATE_PATH_ENDPOINTS,
                prefix='tw_path_ease',
            )
            for col, values in candidate_path_features.items():
                out[col] = values
            for col, values in candidate_path_ease_features.items():
                out[col] = values
        if need_beam:
            if gr_full_filled is None:
                fallback_gr = prefix_gr_mean if np.isfinite(prefix_gr_mean) else 0.0
                gr_full_filled = gr_full_numeric.interpolate(limit_direction='both').fillna(fallback_gr)
            hidden_gr_for_beam = gr_full_filled.iloc[pred_idx].to_numpy(dtype=float)
            tw_tvt_values = tw_tvt.to_numpy(dtype=float)
            tw_gr_values = tw_gr.to_numpy(dtype=float)
            beam_configs = {
                'tight': dict(beam_size=5, move_cost=50.0, emit_scale=200.0, radius=1),
                'conservative': dict(beam_size=10, move_cost=20.0, emit_scale=144.0, radius=2),
                'loose': dict(beam_size=15, move_cost=8.0, emit_scale=64.0, radius=2),
                'vcons': dict(beam_size=8, move_cost=35.0, emit_scale=220.0, radius=1),
                'sm5': dict(beam_size=10, move_cost=14.0, emit_scale=90.0, radius=5),
                'vloose': dict(beam_size=20, move_cost=3.0, emit_scale=25.0, radius=3),
            }
            beam_paths = {
                name: beam_typewell_path(
                    hidden_gr_for_beam,
                    tw_tvt_values,
                    tw_gr_values,
                    last_known_tvt,
                    **config,
                )
                for name, config in beam_configs.items()
            }
            beam_tight = beam_paths['tight']
            beam_cons = beam_paths['conservative']
            beam_loose = beam_paths['loose']
            beam_vcons = beam_paths['vcons']
            beam_sm5 = beam_paths['sm5']
            beam_vloose = beam_paths['vloose']
            beam_deltas = np.vstack([
                beam_tight - last_known_tvt,
                beam_cons - last_known_tvt,
                beam_loose - last_known_tvt,
                beam_vcons - last_known_tvt,
                beam_sm5 - last_known_tvt,
                beam_vloose - last_known_tvt,
            ])
            out['tw_beam_tight_delta'] = beam_tight - last_known_tvt
            out['tw_beam_conservative_delta'] = beam_cons - last_known_tvt
            out['tw_beam_loose_delta'] = beam_loose - last_known_tvt
            out['tw_beam_vcons_delta'] = beam_vcons - last_known_tvt
            out['tw_beam_sm5_delta'] = beam_sm5 - last_known_tvt
            out['tw_beam_vloose_delta'] = beam_vloose - last_known_tvt
            out['tw_beam_gap'] = beam_loose - beam_cons
            out['tw_beam_spread'] = np.nanmax(beam_deltas, axis=0) - np.nanmin(beam_deltas, axis=0)
            out['tw_beam_mean_delta'] = np.nanmean(beam_deltas, axis=0)
            out['tw_beam_std_delta'] = np.nanstd(beam_deltas, axis=0)
            out['tw_beam_tight_step'] = np.r_[np.nan, np.diff(beam_tight)]
            out['tw_beam_conservative_step'] = np.r_[np.nan, np.diff(beam_cons)]
            out['tw_beam_loose_step'] = np.r_[np.nan, np.diff(beam_loose)]
            out['tw_beam_vcons_step'] = np.r_[np.nan, np.diff(beam_vcons)]
            out['tw_beam_sm5_step'] = np.r_[np.nan, np.diff(beam_sm5)]
            out['tw_beam_vloose_step'] = np.r_[np.nan, np.diff(beam_vloose)]
            out['tw_beam_gr_at_conservative'] = np.interp(beam_cons, tw_tvt_values, tw_gr_values, left=np.nan, right=np.nan)
            out['tw_beam_gr_at_loose'] = np.interp(beam_loose, tw_tvt_values, tw_gr_values, left=np.nan, right=np.nan)
            out['tw_beam_gr_minus_conservative'] = tail_gr - out['tw_beam_gr_at_conservative']
            out['tw_beam_gr_minus_loose'] = tail_gr - out['tw_beam_gr_at_loose']
            out['beam_tight_delta'] = out['tw_beam_tight_delta']
            out['beam_cons_delta'] = out['tw_beam_conservative_delta']
            out['beam_loose_delta'] = out['tw_beam_loose_delta']
            out['beam_vcons_delta'] = out['tw_beam_vcons_delta']
            out['beam_sm5_delta'] = out['tw_beam_sm5_delta']
            out['beam_vloose_delta'] = out['tw_beam_vloose_delta']
            out['beam_mean_delta'] = out['tw_beam_mean_delta']
            out['beam_std_delta'] = out['tw_beam_std_delta']
            out['beam_spread'] = out['tw_beam_spread']
            out['beam_gap'] = out['tw_beam_gap']
            out['beam_median_delta'] = np.nanmedian(beam_deltas, axis=0)
            out['beam_sm5_vs_cons'] = out['beam_sm5_delta'] - out['beam_cons_delta']
            out['beam_vcons_vs_loose'] = out['beam_vcons_delta'] - out['beam_loose_delta']
            out['gr_minus_tw_beam_cons'] = out['tw_beam_gr_minus_conservative']
            out['gr_minus_tw_beam_loose'] = out['tw_beam_gr_minus_loose']
            if need_selfcorr and np.isfinite(selfcorr_delta).any():
                sc_weight = np.clip(np.nan_to_num(out.get('selfcorr_trust', np.zeros(len(tail))), nan=0.0), 0.0, 1.0) * 0.60
                out['hyb_delta'] = (1.0 - sc_weight) * out['beam_cons_delta'] + sc_weight * selfcorr_delta
                out['beam_vs_sc'] = out['beam_cons_delta'] - selfcorr_delta
                out['dense_vs_sc'] = row_delta - selfcorr_delta
                out['plane_vs_sc'] = plane_delta - selfcorr_delta
            if need_tdbc_offsets:
                for col, values in typewell_offset_family_features(tw, beam_cons, tail_gr, TDBC_OFFSETS, prefix='tdbc').items():
                    out[col] = values
            if need_tdsc_offsets:
                for col, values in typewell_offset_family_features(tw, selfcorr_tvt, tail_gr, TDSC_OFFSETS, prefix='tdsc').items():
                    out[col] = values
            if need_pf_lite:
                candidate_tvt = {
                    'last_known': np.full(len(tail), last_known_tvt, dtype=float),
                    'beam_cons': beam_cons,
                    'beam_loose': beam_loose,
                    'beam_sm5': beam_sm5,
                    'selfcorr': selfcorr_tvt,
                    'plane': last_known_tvt + plane_delta,
                    'dense': last_known_tvt + row_delta,
                }
                if 'formation_formula_mean_delta' in out:
                    candidate_tvt['formation_mean'] = last_known_tvt + np.asarray(out['formation_formula_mean_delta'], dtype=float)
                for col, values in weighted_candidate_tvt_features(
                    tw,
                    tail_gr,
                    tail['Z'].to_numpy(dtype=float),
                    candidate_tvt,
                    last_known_tvt,
                    dense_ancc=np.asarray(out['dense_ancc'], dtype=float) if 'dense_ancc' in out else None,
                ).items():
                    out[col] = values
    else:
        for col in [
            'typewell_tvt_min',
            'typewell_tvt_max',
            'typewell_tvt_range',
            'typewell_gr_mean',
            'typewell_gr_std',
            'typewell_gr_at_last_known_tvt',
            'typewell_gr_at_slope_baseline_all',
            'typewell_gr_at_slope_baseline_last200',
            'gr_minus_typewell_last_known_tvt',
            'gr_minus_typewell_slope_baseline_all',
            'gr_minus_typewell_slope_baseline_last200',
            'prefix_horizontal_vs_typewell_gr_corr',
            'prefix_horizontal_vs_typewell_gr_mae',
            'prefix_horizontal_vs_typewell_gr_rmse',
            'typewell_local_last200_best_delta',
            'typewell_local_last200_best_abs_resid',
            'typewell_local_last200_zero_abs_resid',
            'typewell_local_last200_top2_gap',
            'typewell_local_last200_soft_delta_mean',
            'typewell_local_last200_best_gr',
            'typewell_local_last200_gr_resid_best',
            'typewell_local_last200_best_vs_zero_gain',
            'prefix_typewell_gr_affine_a',
            'prefix_typewell_gr_affine_b',
            'prefix_horizontal_vs_calibrated_typewell_gr_mae',
    'prefix_horizontal_vs_typewell_gr_rmse',
    'prefix_tvt_step20',
    'typewell_last_geo_interval_phase',
    'typewell_baseline_last200_geo_boundary_proximity',
            'prefix_horizontal_vs_calibrated_typewell_gr_rmse',
            'typewell_calibrated_gr_at_last_known_tvt',
            'typewell_calibrated_gr_at_slope_baseline_all',
            'typewell_calibrated_gr_at_slope_baseline_last200',
            'gr_minus_calibrated_typewell_last_known_tvt',
            'gr_minus_calibrated_typewell_slope_baseline_all',
            'gr_minus_calibrated_typewell_slope_baseline_last200',
            'calibrated_typewell_slope_last200_gr_prefix_z',
            'calibrated_typewell_local_last200_best_delta',
            'calibrated_typewell_local_last200_best_abs_resid',
            'calibrated_typewell_local_last200_zero_abs_resid',
            'calibrated_typewell_local_last200_top2_gap',
            'calibrated_typewell_local_last200_soft_delta_mean',
            'calibrated_typewell_local_last200_best_gr',
            'calibrated_typewell_local_last200_gr_resid_best',
            'calibrated_typewell_local_last200_best_vs_zero_gain',
        ]:
            out[col] = np.nan
        for col in candidate_path_feature_names(prefix='tw_path'):
            out[col] = np.nan
        for col in candidate_path_feature_names(prefix='tw_path_ease'):
            out[col] = np.nan
        for col in typewell_interval_context_feature_names(prefix='typewell_last_geo'):
            out[col] = np.nan
        for col in typewell_interval_context_feature_names(prefix='typewell_baseline_last200_geo'):
            out[col] = np.nan
        for offset in [-80, -40, -20, -10, -5, 0, 5, 10, 20, 40, 80]:
            label = candidate_endpoint_label(float(offset))
            out[f'typewell_anchor_gr_diff_{label}'] = np.nan
            out[f'calibrated_typewell_anchor_gr_diff_{label}'] = np.nan
        if need_beam:
            for col in offline_beam_feature_names(prefix='tw_beam'):
                out[col] = np.nan
        if need_selfcorr:
            for col in selfcorr_feature_names():
                if col not in out:
                    out[col] = np.nan
        if need_tdbc_offsets:
            for col in typewell_offset_family_feature_names('tdbc', TDBC_OFFSETS):
                out[col] = np.nan
        if need_tdsc_offsets:
            for col in typewell_offset_family_feature_names('tdsc', TDSC_OFFSETS):
                out[col] = np.nan
        if need_pf_lite:
            for col in pf_lite_feature_names():
                out[col] = np.nan
    
    if include_target and 'TVT' in df.columns:
        out['target_tvt'] = tail['TVT'].to_numpy()
        out['target_delta_from_last_known'] = out['target_tvt'] - last_known_tvt
    return out

preview_well_id = globals().get('representative_well_id')
if preview_well_id is None and globals().get('train_h_ids'):
    preview_well_id = sorted(train_h_ids)[0]

if preview_well_id is not None:
    representative_feature_frame = make_tail_features_for_well(preview_well_id, TRAIN_DIR, include_target=True, use_beam_features=False)
    print('representative_feature_frame shape:', representative_feature_frame.shape)
    display(representative_feature_frame.head())
    preview_columns = [
        'target_delta_from_last_known',
        'md_since_ps',
        'prefix_azimuth_deg',
        'prefix_horizontal_vs_typewell_gr_corr',
        'prefix_horizontal_vs_calibrated_typewell_gr_mae',
        'gr_minus_calibrated_typewell_slope_baseline_last200',
        'typewell_local_last200_best_delta',
        'typewell_local_last200_best_vs_zero_gain',
        'tw_path_min_absdiff',
        'tw_path_best_endpoint',
        'tw_path_top2_absdiff_gap',
        'tw_path_ease_min_absdiff',
        'gap_gr_std',
        'sin_tail_frac_pi',
        'gr_center_roll_mean_21',
        'gr_center_lead1',
        'gr_cumsum_since_ps',
        'calibrated_typewell_local_last200_best_delta',
        'calibrated_typewell_local_last200_best_vs_zero_gain',
        'GR_prefix_z',
        'gr_diff_1',
        'gr_slope_md_1',
        'gr_roll_range_25',
        'rows_since_prefix_last_valid_gr',
        'md_since_prefix_last_valid_gr',
        'trajectory_step_1',
        'z_slope_md_1',
        'GR_isna',
    ]
    preview_columns = [col for col in preview_columns if col in representative_feature_frame.columns]
    display(representative_feature_frame[preview_columns].describe().T)
else:
    print('Representative feature preview skipped because no training wells are available.')


## 12. Curve-Level Target Diagnostics

- 🧵 Tail curves can be summarized by a few knots.
- 📉 Smooth tails suggest row predictions should be smoothed or clipped.
- 🛡️ Knot labels are diagnostics/training targets, not inference features.


In [ ]:
# Convert each tail TVT curve into fixed-fraction knot delta targets.

def make_knot_targets(path: Path, knots=np.linspace(0, 1, 21)) -> dict:
    wid = well_id_from_path(path)
    df = pd.read_csv(path, usecols=['TVT', 'TVT_input'])
    mask = df['TVT_input'].isna().to_numpy()
    pred_idx = np.flatnonzero(mask)
    if len(pred_idx) == 0:
        return {'well_id': wid}
    last_known = float(df['TVT_input'].iloc[pred_idx[0] - 1])
    tail_y = df['TVT'].iloc[pred_idx].to_numpy()
    x = np.linspace(0, 1, len(tail_y))
    knot_delta = np.interp(knots, x, tail_y - last_known)
    row = {'well_id': wid, 'tail_len': len(tail_y), 'last_known_TVT': last_known}
    for k, value in zip(knots, knot_delta):
        row[f'delta_knot_{k:.2f}'] = float(value)
    row['tail_min_delta'] = float(np.min(tail_y - last_known))
    row['tail_max_delta'] = float(np.max(tail_y - last_known))
    row['tail_end_delta'] = float(tail_y[-1] - last_known)
    return row

knot_targets = pd.DataFrame([make_knot_targets(path) for path in train_horizontal_files])
print('knot_targets shape:', knot_targets.shape)
display(knot_targets.head())

selected_knot_cols = ['delta_knot_0.00', 'delta_knot_0.25', 'delta_knot_0.50', 'delta_knot_0.75', 'delta_knot_1.00', 'tail_min_delta', 'tail_max_delta']
display(knot_targets[selected_knot_cols].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T)

fig, ax = plt.subplots(figsize=(10, 5))
for _, row in knot_targets.sample(min(80, len(knot_targets)), random_state=42).iterrows():
    values = [row[f'delta_knot_{k:.2f}'] for k in np.linspace(0, 1, 21)]
    ax.plot(np.linspace(0, 1, 21), values, alpha=0.18, color='tab:blue')
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Sample tail delta curves from last known TVT')
ax.set_xlabel('Tail fraction')
ax.set_ylabel('TVT delta from last known')
polish_current_figure()
plt.tight_layout()
plt.show()


## 13. Nearby-Well Spatial Signal

- 🌐 Nearby wells can share structural drift.
- 🧲 Spatial priors are useful when GR alignment is ambiguous.
- 🛡️ Validation-fold targets must not enter the neighbor reference table.


In [ ]:
display_rogii_figure(
    "ROGII_Fig4.png",
    "Figure 11 — Formation Plane and Dense ANCC Features",
    "Formation-top surfaces, local KNN estimates, and well-specific offsets used to form spatial TVT priors.",
    width=1120,
)


In [ ]:
# Estimate leave-one-out nearest-neighbor spatial drift signal.

spatial_frame = h_summary[['well_id', 'ps_x', 'ps_y', 'azimuth_deg', 'tail_end_delta_from_last_known', 'tail_tvt_range', 'constant_tail_rmse']].dropna().reset_index(drop=True)
coords = spatial_frame[['ps_x', 'ps_y']].to_numpy()
# Pairwise distance is small enough: 773 x 773.
dx = coords[:, None, 0] - coords[None, :, 0]
dy = coords[:, None, 1] - coords[None, :, 1]
dist = np.sqrt(dx ** 2 + dy ** 2)
np.fill_diagonal(dist, np.inf)

for k in [3, 5, 10, 20]:
    nn = np.argsort(dist, axis=1)[:, :k]
    neighbor_mean = spatial_frame['tail_end_delta_from_last_known'].to_numpy()[nn].mean(axis=1)
    spatial_frame[f'nn{k}_tail_end_delta_mean'] = neighbor_mean
    corr = np.corrcoef(spatial_frame['tail_end_delta_from_last_known'], neighbor_mean)[0, 1]
    print(f'leave-one-out nearest {k} mean vs own tail_end_delta correlation: {corr:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=spatial_frame, x='nn5_tail_end_delta_mean', y='tail_end_delta_from_last_known', hue='constant_tail_rmse', palette='viridis', ax=axes[0])
axes[0].axline((0, 0), slope=1, color='black', linestyle='--', linewidth=1)
axes[0].set_title('Nearest-neighbor tail-end drift signal')
sns.scatterplot(data=spatial_frame, x='ps_x', y='ps_y', hue='tail_end_delta_from_last_known', palette='coolwarm', ax=axes[1], s=35)
axes[1].set_title('Spatial distribution of tail-end drift')
axes[1].set_aspect('equal', adjustable='box')
polish_current_figure()
plt.tight_layout()
plt.show()


## 14. Representative Well Plot

Look for patterns that aggregate tables hide:

- 🧵 TVT continuity around Prediction Start
- 🌋 GR missingness and local GR events
- 🧭 typewell-vs-horizontal GR similarity
- ⚓ flat tail vs drifting tail
- 🧯 jumps that would need clipping or smoothing


In [ ]:
# Plot a representative well across TVT, GR, map trajectory, and typewell-aligned GR axes.

def plot_well_overview(well_id: str):
    h = pd.read_csv(TRAIN_DIR / f'{well_id}__horizontal_well.csv')
    tw = pd.read_csv(TRAIN_DIR / f'{well_id}__typewell.csv')
    mask = h['TVT_input'].isna().to_numpy()
    ps = int(np.flatnonzero(mask)[0]) if mask.any() else len(h)
    last_known_tvt = h['TVT_input'].iloc[ps - 1]
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    ax = axes[0, 0]
    ax.plot(h.index, h['TVT'], label='TVT target', color='tab:blue')
    ax.plot(h.index, h['TVT_input'], label='TVT_input known prefix', color='tab:orange')
    ax.axvline(ps, color='red', linestyle='--', label='Prediction Start')
    ax.set_title(f'{well_id}: TVT and Prediction Start')
    ax.set_xlabel('row index')
    ax.set_ylabel('TVT')
    ax.legend()
    
    ax = axes[0, 1]
    ax.plot(h.index, h['GR'], color='black', linewidth=1)
    ax.axvline(ps, color='red', linestyle='--')
    ax.set_title('Horizontal GR along row index')
    ax.set_xlabel('row index')
    ax.set_ylabel('GR')
    
    ax = axes[1, 0]
    ax.plot(h['X'], h['Y'], color='tab:green')
    ax.scatter(h['X'].iloc[ps], h['Y'].iloc[ps], color='red', s=80, label='PS')
    ax.set_title('Map view of horizontal trajectory')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_aspect('equal', adjustable='box')
    ax.legend()
    
    ax = axes[1, 1]
    known = h['TVT_input'].notna() & h['GR'].notna()
    ax.plot(tw['GR'], tw['TVT'], color='red', alpha=0.8, label='Typewell GR')
    ax.scatter(h.loc[known, 'GR'], h.loc[known, 'TVT_input'], s=5, alpha=0.35, color='black', label='Horizontal known prefix GR')
    ax.axhline(last_known_tvt, color='blue', linestyle='--', label='last known TVT')
    ax.invert_yaxis()
    ax.set_title('GR on TVT axis')
    ax.set_xlabel('GR')
    ax.set_ylabel('TVT')
    ax.legend()
    
    polish_current_figure()
    plt.tight_layout()
    plt.show()

plot_well_overview(representative_well_id)


In [ ]:
from pathlib import Path
from IPython.display import Image, display

figure_image_path = Path("/kaggle/input/datasets/pilkwang/pilkwang-public-dataset-for-notebooks-figures/ROGII_Graph_Fig9.png")
if figure_image_path.exists():
    display(Image(filename=str(figure_image_path)))


**Figure 12. EDA-driven feature engineering pipeline.**  
The final feature set is derived from geosteering observations: anchor persistence, GR correlation, formation-surface geometry, state-space continuity, selector regimes, and estimator disagreement.


## 🧠 15. Model Logic from EDA

### 🧩 Prediction shape

```text
last_known_TVT
  + residual from geology / GR / trajectory signals
  + light calibration
  = row-level TVT
```

### 📏 Validation guardrails

| Rule | Use |
|---|---|
| `GroupKFold(well_id)` | avoid same-well leakage in diagnostics |
| hidden-tail rows only | match submission rows |
| row-level loss | long tails should matter |
| one postprocess policy | avoid fold-specific oracle behavior |

### 🧬 Signal mix

| Signal | Use |
|---|---|
| ⚓ residual target | predict movement from anchor |
| 🧭 typewell alignment | locate GR events on TVT axis |
| 🌐 formation / ANCC | add spatial geology priors |
| 🧵 beam / DTW / self-corr | produce target-free pseudo-TVT paths |
| 🚀 same-matrix stack | let models compare all signals together |
| 🧯 fade-in / clipping | protect the first tail rows and flat wells |


### 15.1 EDA-Driven Feature Engineering Plan

The feature design follows the same chain throughout the notebook:

$$
\text{EDA observation}\rightarrow\text{geologic interpretation}\rightarrow\text{target-free estimator}\rightarrow\text{validation / ablation}
$$

| EDA conclusion | Interpretation | Feature / estimator block | Policy |
|---|---|---|---|
| Last known TVT is a strong baseline | hidden TVT often starts near the anchor | residual target, `last_known_TVT`, fade/hold features | strict |
| GR is stratigraphic but has NaN gaps | PF/beam likelihood needs continuous observations | interpolated GR, missing-rate, longest-gap features | target-free |
| Prefix GR vs typewell mismatch varies by well | typewell tracking confidence is well-specific | prefix GR residual RMSE/corr/std, PF `gs` scale | strict |
| `TVT + Z` relates to formation surfaces | structural coordinate can be tracked as state | formation formula, PF-ANCC state | spatial/imputed only |
| Hidden length and Z span define regimes | long/high-span wells need different drift priors | `n_eval`, `z_span`, selector code | target-free |
| PF/beam/formation estimators disagree on hard wells | disagreement is uncertainty | estimator range, pairwise gaps, gated blend features | target-free |
| Same-well train/test overlap can be powerful | public-visible wells may have a physical shortcut | same-well physical/contact estimator | public-aggressive |

The important separation is:

$$
\text{PF / beam / GR interpolation} = \text{target-free feature engineering}
$$

$$
\text{same-well physical contact} = \text{public-aggressive overlap estimator}
$$

This section keeps the blocks explicit so that a public-overlap run and a more private-safe diagnostic run can be compared without changing the rest of the notebook.


In [ ]:
from pathlib import Path
from IPython.display import Image, display

figure_image_path = Path("/kaggle/input/datasets/pilkwang/pilkwang-public-dataset-for-notebooks-figures/ROGII_Graph_Fig7.png")
if figure_image_path.exists():
    display(Image(filename=str(figure_image_path)))


**Figure 13. Selector regime map.**  
Each well is assigned to a selector regime using hidden interval length $n_{\mathrm{eval}}$ and vertical span $z_{\mathrm{span}}$. Each regime selects a PF scale, optional beam mixture, and anchor hold weight.


In [ ]:
# EDA-driven feature registry and well-level feature-policy audit.

FEATURE_POLICY = 'public_aggressive' if bool(globals().get('PF_SELECTOR_USE_SAME_WELL_PHYSICAL', False)) else 'private_safe'

EDA_DRIVEN_FEATURE_BLOCKS = {
    'anchor_residual': {
        'eda_basis': 'last_known_TVT baseline is strong',
        'policy': 'strict',
        'enabled': True,
        'examples': ['last_known_TVT', 'md_since_last_known', 'tail_frac'],
    },
    'gr_quality': {
        'eda_basis': 'GR missing gaps affect PF/beam observation likelihood',
        'policy': 'target_free',
        'enabled': True,
        'examples': ['gr_interp', 'hidden_gr_missing_rate', 'hidden_longest_gr_nan_run'],
    },
    'prefix_typewell_calibration': {
        'eda_basis': 'prefix GR vs typewell mismatch varies by well',
        'policy': 'strict',
        'enabled': True,
        'examples': ['prefix_tw_gr_rmse', 'prefix_tw_gr_corr', 'prefix_tw_gr_resid_std'],
    },
    'pf_state_space': {
        'eda_basis': 'TVT + Z behaves like a formation-relative state coordinate',
        'policy': 'target_free',
        'enabled': True,
        'examples': ['pf_scale_3_tvt', 'pf_scale_8_tvt', 'pf_likelihood_spread'],
    },
    'beam_alignment': {
        'eda_basis': 'typewell GR provides a stratigraphic reference curve',
        'policy': 'target_free',
        'enabled': True,
        'examples': ['beam14_tvt', 'beam_spread', 'beam_vs_pf'],
    },
    'selector_regime': {
        'eda_basis': 'hidden length and Z-span define drift regimes',
        'policy': 'target_free',
        'enabled': True,
        'examples': ['selector_n_eval', 'selector_z_span', 'selector_code'],
    },
    'same_well_physical': {
        'eda_basis': 'same-well train/test overlap can expose a strong physical contact path',
        'policy': 'public_aggressive',
        'enabled': FEATURE_POLICY == 'public_aggressive',
        'examples': ['same_well_physical_available', 'same_well_contact_tvt'],
    },
}

feature_block_report = pd.DataFrame([
    {
        'block': block,
        'policy': cfg['policy'],
        'enabled': bool(cfg['enabled']),
        'eda_basis': cfg['eda_basis'],
        'examples': ', '.join(cfg['examples']),
    }
    for block, cfg in EDA_DRIVEN_FEATURE_BLOCKS.items()
])
display(feature_block_report)
print('FEATURE_POLICY:', FEATURE_POLICY)

# Selector-regime audit mirrors the high-score PF selector thresholds used later in the final engine.
SELECTOR_N_EVAL_THRESHOLD_AUDIT = 4840.0
SELECTOR_Z_SPAN_THRESHOLDS_AUDIT = (136.73000000000016, 185.5133333333342)
SELECTOR_BIN_VARIANT_AUDIT = {
    0: 'pf_scale_5_hold_0.2',
    1: 'pf_scale_3_hold_0.15',
    2: 'pf_scale_12_beam_0.2_hold_0.15',
    3: 'pf_scale_5_hold_0.15',
    4: 'pf_scale_5_beam_0.05_hold_0.05',
    5: 'pf_scale_12_beam_0.2_hold_0.05',
}


def compact_selector_variant(name: str) -> str:
    match = re.search(r'pf_scale_([0-9.]+).*?beam_([0-9.]+).*?hold_([0-9.]+)', str(name))
    if match:
        scale, beam, hold = match.groups()
        return f'PF s={scale} b={beam} h={hold}'
    match = re.search(r'pf_scale_([0-9.]+).*?hold_([0-9.]+)', str(name))
    if match:
        scale, hold = match.groups()
        return f'PF s={scale} h={hold}'
    return str(name).replace('_', ' ')


def selector_regime_code(n_eval: float, z_span: float) -> tuple[int, str]:
    n_bin = int(float(n_eval) > SELECTOR_N_EVAL_THRESHOLD_AUDIT)
    z_bin = int(np.searchsorted(SELECTOR_Z_SPAN_THRESHOLDS_AUDIT, float(z_span), side='right'))
    code = int(n_bin + 2 * z_bin)
    return code, SELECTOR_BIN_VARIANT_AUDIT.get(code, 'pf_scale_8_hold_0.2')


def prefix_typewell_calibration_for_path(path: Path, split: str) -> dict:
    wid = well_id_from_path(path)
    root = TRAIN_DIR if split == 'train' else TEST_DIR
    tw_path = root / f'{wid}__typewell.csv'
    h = pd.read_csv(path, usecols=['GR', 'TVT_input'])
    if not tw_path.exists():
        return {'split': split, 'well_id': wid, 'prefix_tw_valid_points': 0}
    tw = pd.read_csv(tw_path)
    known = h['TVT_input'].notna()
    hv = h.loc[known, 'GR'].to_numpy(dtype=float)
    tvt = h.loc[known, 'TVT_input'].to_numpy(dtype=float)
    tw_gr = typewell_gr_at_tvt(tw, tvt)
    valid = np.isfinite(hv) & np.isfinite(tw_gr)
    resid = hv[valid] - tw_gr[valid]
    return {
        'split': split,
        'well_id': wid,
        'prefix_tw_valid_points': int(valid.sum()),
        'prefix_tw_gr_corr': safe_corr(hv, tw_gr),
        'prefix_tw_gr_rmse': float(np.sqrt(np.nanmean(resid ** 2))) if len(resid) else np.nan,
        'prefix_tw_gr_resid_std': _nan_stat(resid, np.std),
    }

prefix_feature_audit = pd.DataFrame(
    [prefix_typewell_calibration_for_path(path, 'train') for path in train_horizontal_files]
    + [prefix_typewell_calibration_for_path(path, 'test') for path in test_horizontal_files]
)

selector_feature_audit = geo_summary[['split', 'well_id', 'hidden_rows', 'hidden_z_span', 'hidden_gr_missing_rate', 'hidden_longest_gr_nan_run']].copy()
selector_codes = selector_feature_audit.apply(
    lambda row: selector_regime_code(row['hidden_rows'], row['hidden_z_span']),
    axis=1,
)
selector_feature_audit['selector_code'] = [code for code, _ in selector_codes]
selector_feature_audit['selector_variant'] = [variant for _, variant in selector_codes]
selector_feature_audit['selector_label'] = selector_feature_audit['selector_variant'].map(compact_selector_variant)
selector_feature_audit['same_well_physical_available'] = (
    (selector_feature_audit['split'] == 'test')
    & selector_feature_audit['well_id'].isin(train_h_ids)
)

feature_policy_audit = selector_feature_audit.merge(
    prefix_feature_audit,
    on=['split', 'well_id'],
    how='left',
)

cols = [
    'split', 'well_id', 'hidden_rows', 'hidden_z_span', 'selector_code', 'selector_variant',
    'hidden_gr_missing_rate', 'hidden_longest_gr_nan_run',
    'prefix_tw_gr_corr', 'prefix_tw_gr_rmse', 'prefix_tw_gr_resid_std',
    'same_well_physical_available',
]
display(feature_policy_audit[cols].sort_values(['split', 'well_id']).head(20))

print('Selector-code counts by split:')
display(feature_policy_audit.groupby(['split', 'selector_code', 'selector_variant']).size().reset_index(name='well_count'))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.scatterplot(data=feature_policy_audit, x='hidden_rows', y='hidden_z_span', hue='selector_label', style='split', ax=axes[0], s=70)
axes[0].axvline(SELECTOR_N_EVAL_THRESHOLD_AUDIT, color='black', linestyle='--', linewidth=1)
for z_thr in SELECTOR_Z_SPAN_THRESHOLDS_AUDIT:
    axes[0].axhline(z_thr, color='gray', linestyle=':', linewidth=1)
axes[0].set_title('Selector regimes from hidden length and Z span')
sns.scatterplot(data=feature_policy_audit, x='prefix_tw_gr_resid_std', y='hidden_gr_missing_rate', hue='split', ax=axes[1])
axes[1].set_title('PF noise scale proxy vs hidden GR missingness')
sns.histplot(data=feature_policy_audit, x='prefix_tw_gr_rmse', hue='split', bins=40, element='step', ax=axes[2])
axes[2].set_title('Prefix typewell residual RMSE')
polish_current_figure()
plt.tight_layout()
plt.show()


## 🧮 16. Residual Prediction Model

### 🔁 Residual pipeline

| Stage | Result | Use |
|---|---|---|
| ⚓ Anchor | flat TVT path | strong null prior |
| 🌲 Model | raw residual | learn drift |
| 🧯 Clip / shrink | bounded residual | avoid extreme corrections |
| 🌅 Fade-in | guarded early tail | keep first rows near anchor |
| 🧵 Slope limiter | smoother TVT curve | suppress row jumps |

### 🧠 Feature intuition

- 📚 Prefix context: local TVT history
- 🧭 Typewell: GR events on the TVT axis
- 🌋 GR texture: local stratigraphic changes
- 🛤️ Trajectory: geometry and vertical movement
- 🌐 Formation / ANCC: spatial geology
- 🧯 Postprocess: anchor-aware curve control


### 16.0 Feature Leakage Review

### Strict policy inputs

| Feature source | Why it is allowed |
|---|---|
| `MD`, `X`, `Y`, `Z`, `GR` at row $i$ | observed at the row |
| Prefix aggregates | known before tail prediction |
| `last_known_TVT` and prefix TVT statistics | known from prefix `TVT_input` |
| Backward GR differences | previous/current rows only |
| Trailing GR rolling features | rows up to $i$ only |
| Typewell GR at prefix-derived baselines | reference log + prefix baseline |
| Prefix-only affine GR calibration | fit only where `TVT_input` is present |
| Anchor-aware slope clipping | anchor, train-fold slope bound, previous clipped prediction |

### Offline policy inputs

| Feature source | Offline reason |
|---|---|
| Tail length and tail fraction | prediction rows are known before writing predictions |
| Full-row fraction and MD tail fraction | test trajectory is provided |
| Gap geometry and full-tail GR summaries | target-free covariates from the provided file |
| Centered GR rolling and lead/lag GR | future GR covariates, not future TVT labels |
| Candidate-path typewell features | tail fraction defines candidate paths, but no true tail TVT is used |
| Spatial formation-plane features | train-only surfaces are auxiliary labels for a reference imputer; validation folds use only training wells |
| Beam-path typewell features | full hidden GR sequence is used; true tail TVT is not used |

### Excluded from all feature policies

| Excluded item | Reason |
|---|---|
| `TVT` in the prediction tail | direct target leakage |
| Train-only surfaces (`ANCC` ... `BUDA`) | hidden-test feature mismatch |
| `TVT_input_bfill` | reads rows after the prediction row |
| Target-derived tail summaries | answer-key information |
| Nearby target drift from validation wells | fold leakage |


In [ ]:
# Configure the residual model, feature sets, validation, and causal post-processing.

from sklearn.model_selection import GroupKFold
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error

# Execution controls.
NOTEBOOK_RUN_VERSION = 'ROGII_EDA_v7_dtw_hill_super_stack_2026_05_17'
# Local research runs keep grouped CV enabled. Kaggle submission runs skip CV by default
# and reserve the final Kaggle path for the same-matrix super stack.
QUICK_CHECK_MODE = False
KAGGLE_MEMORY_SAFE_MODE = False if KAGGLE_NOTEBOOK_RUN else True
ENABLE_KAGGLE_BEAM_FEATURES = bool(KAGGLE_NOTEBOOK_RUN)
RUN_HGB_DIAGNOSTIC_SUBMISSIONS = False if KAGGLE_NOTEBOOK_RUN else not KAGGLE_NOTEBOOK_RUN
RUN_GROUPED_CV = not KAGGLE_NOTEBOOK_RUN
RUN_SUPER_MONOLITH_STACK = bool(KAGGLE_NOTEBOOK_RUN)

MODEL_CONFIG = {
    # Resource-controlled settings for grouped CV and final training.
    'random_state': 42,
    'n_splits': 5,
    'cv_folds_to_run': 5,
    'train_rows_per_well': 350,
    'final_train_rows_per_well': 700,
    'feature_policy_for_selection': 'strict',
    'residual_shrinkage': 0.80,
    'alpha_bounds': (0.0, 1.5),
    'delta_clip_quantiles': (0.005, 0.995),
    'fade_in_tau_md_to_compare': [None, 25.0, 50.0, 100.0, 200.0],
    'slope_clip_quantiles_to_compare': [0.90, 0.95, 0.975, 0.99, 0.995],
    'postprocess_slope_quantile': 0.995,
    'feature_sets_to_compare': None,
    'select_best_feature_set_from_cv': True,
    'apply_slope_clip_if_cv_improves': True,
    'write_submission': True,
    'write_overlap_excluded_diagnostic': False,
}

if QUICK_CHECK_MODE:
    MODEL_CONFIG.update({
        'cv_folds_to_run': 1,
        'train_rows_per_well': 150,
        'final_train_rows_per_well': 150,
        'feature_sets_to_compare': [
            'calibrated_typewell_alignment',
            'offline_candidate_path_alignment',
        ],
    })

BASE_FEATURE_COLUMNS = [
    # Row position and trajectory after Prediction Start.
    'md_since_ps',
    'x_delta_ps',
    'y_delta_ps',
    'z_delta_ps',
    'xy_dist_ps',
    'MD',
    'X',
    'Y',
    'Z',
    # Prefix-only well context.
    'prefix_len',
    'prefix_azimuth_deg',
    'prefix_gr_missing_rate',
    # Row-level GR signal and trailing GR context.
    'GR',
    'GR_isna',
    'GR_prefix_z',
    'gr_diff_1',
    'gr_diff_5',
    'gr_slope_md_1',
    'md_step_1',
    'x_step_1',
    'y_step_1',
    'z_step_1',
    'trajectory_step_1',
    'z_slope_md_1',
    'prefix_gr_mean',
    'prefix_gr_std',
    'gr_roll_mean_25',
    'gr_roll_std_25',
    'gr_roll_min_25',
    'gr_roll_max_25',
    'gr_roll_range_25',
    'gr_roll_mean_100',
    'gr_roll_std_100',
    'gr_roll_range_100',
    'gr_roll_mean_300',
    'gr_roll_std_300',
    'gr_roll_range_300',
    # Known-prefix TVT trend. This can help, but shrinkage/clipping protects against blind extrapolation.
    'prefix_tvt_slope_md_all',
    'prefix_tvt_slope_md_last200',
]

PREFIX_CONTEXT_COLUMNS = [
    'last_known_TVT',
    'prefix_gr_min',
    'prefix_gr_max',
    'prefix_last_valid_gr',
    'rows_since_prefix_last_valid_gr',
    'md_since_prefix_last_valid_gr',
    'gr_minus_prefix_last_valid_gr',
    'gr_minus_prefix_gr_mean',
    'prefix_tvt_min',
    'prefix_tvt_max',
    'prefix_tvt_range',
    'prefix_tvt_mean',
    'prefix_tvt_std',
    'prefix_tvt_step20',
    'prefix_tvt_step100',
    'prefix_tvt_md_slope100',
    'prefix_tvt_z_slope100',
    'slope_baseline_delta_all',
    'slope_baseline_delta_last200',
    'slope_baseline_tvt_all',
    'slope_baseline_tvt_last200',
]

TYPEWELL_INTERVAL_CONTEXT_COLUMNS = (
    typewell_interval_context_feature_names(prefix='typewell_last_geo')
    + typewell_interval_context_feature_names(prefix='typewell_baseline_last200_geo')
)

TYPEWELL_ANCHOR_OFFSET_COLUMNS = [
    f'typewell_anchor_gr_diff_{candidate_endpoint_label(float(offset))}'
    for offset in [-80, -40, -20, -10, -5, 0, 5, 10, 20, 40, 80]
]

CALIBRATED_TYPEWELL_ANCHOR_OFFSET_COLUMNS = [
    f'calibrated_typewell_anchor_gr_diff_{candidate_endpoint_label(float(offset))}'
    for offset in [-80, -40, -20, -10, -5, 0, 5, 10, 20, 40, 80]
]

TYPEWELL_ALIGNMENT_COLUMNS = [
    'typewell_tvt_min',
    'typewell_tvt_max',
    'typewell_tvt_range',
    'typewell_gr_mean',
    'typewell_gr_std',
    'typewell_gr_at_last_known_tvt',
    'typewell_gr_at_slope_baseline_all',
    'typewell_gr_at_slope_baseline_last200',
    'gr_minus_typewell_last_known_tvt',
    'gr_minus_typewell_slope_baseline_all',
    'gr_minus_typewell_slope_baseline_last200',
    'prefix_horizontal_vs_typewell_gr_corr',
    'prefix_horizontal_vs_typewell_gr_mae',
    'prefix_horizontal_vs_typewell_gr_rmse',
    *TYPEWELL_INTERVAL_CONTEXT_COLUMNS,
    *TYPEWELL_ANCHOR_OFFSET_COLUMNS,
    'typewell_local_last200_best_delta',
    'typewell_local_last200_best_abs_resid',
    'typewell_local_last200_zero_abs_resid',
    'typewell_local_last200_top2_gap',
    'typewell_local_last200_soft_delta_mean',
    'typewell_local_last200_gr_resid_best',
    'typewell_local_last200_best_vs_zero_gain',
]

CALIBRATED_TYPEWELL_ALIGNMENT_COLUMNS = [
    'prefix_typewell_gr_affine_a',
    'prefix_typewell_gr_affine_b',
    'prefix_horizontal_vs_calibrated_typewell_gr_mae',
    'prefix_horizontal_vs_calibrated_typewell_gr_rmse',
    *CALIBRATED_TYPEWELL_ANCHOR_OFFSET_COLUMNS,
    'typewell_calibrated_gr_at_last_known_tvt',
    'typewell_calibrated_gr_at_slope_baseline_all',
    'typewell_calibrated_gr_at_slope_baseline_last200',
    'gr_minus_calibrated_typewell_last_known_tvt',
    'gr_minus_calibrated_typewell_slope_baseline_all',
    'gr_minus_calibrated_typewell_slope_baseline_last200',
    'calibrated_typewell_slope_last200_gr_prefix_z',
    'calibrated_typewell_local_last200_best_delta',
    'calibrated_typewell_local_last200_best_abs_resid',
    'calibrated_typewell_local_last200_zero_abs_resid',
    'calibrated_typewell_local_last200_top2_gap',
    'calibrated_typewell_local_last200_soft_delta_mean',
    'calibrated_typewell_local_last200_gr_resid_best',
    'calibrated_typewell_local_last200_best_vs_zero_gain',
]

FORMATION_PLANE_COLUMNS = formation_feature_names(include_row=False)
FORMATION_ROW_ANCC_COLUMNS = [
    col
    for col in formation_feature_names(include_row=True)
    if col not in set(FORMATION_PLANE_COLUMNS)
]

# Beam alignment is expensive during repeated local CV. Keep it off locally by default,
# and enable it only for explicit Kaggle all-row experiments.
ENABLE_OFFLINE_BEAM_FEATURES = bool(ENABLE_KAGGLE_BEAM_FEATURES and KAGGLE_NOTEBOOK_RUN and not KAGGLE_MEMORY_SAFE_MODE)
OFFLINE_CANDIDATE_PATH_COLUMNS = candidate_path_feature_names(prefix='tw_path')
OFFLINE_CANDIDATE_PATH_EASE_COLUMNS = candidate_path_feature_names(prefix='tw_path_ease')
OFFLINE_BEAM_FEATURE_COLUMNS = offline_beam_feature_names(prefix='tw_beam')
SELF_CORR_FEATURE_COLUMNS = selfcorr_feature_names()
TDBC_OFFSET_COLUMNS = typewell_offset_family_feature_names('tdbc', TDBC_OFFSETS)
TDSC_OFFSET_COLUMNS = typewell_offset_family_feature_names('tdsc', TDSC_OFFSETS)
PF_LITE_FEATURE_COLUMNS = pf_lite_feature_names()

OFFLINE_EXTRA_FEATURE_COLUMNS = [
    'tail_len',
    'tail_row_number',
    'tail_frac',
    'tail_frac2',
    'tail_frac3',
    'sqrt_tail_frac',
    'log1p_tail_row',
    'sin_tail_frac_pi',
    'sin_tail_frac_2pi',
    'cos_tail_frac_3pi',
    'n_rows',
    'row_frac',
    'md_tail_span',
    'md_tail_frac',
    'tail_gr_missing_rate',
    'gap_md_span',
    'gap_x_delta',
    'gap_y_delta',
    'gap_z_delta',
    'gap_xy_span',
    'gap_z_over_xy',
    'gap_gr_mean',
    'gap_gr_std',
    'gap_gr_min',
    'gap_gr_p05',
    'gap_gr_p25',
    'gap_gr_p50',
    'gap_gr_p75',
    'gap_gr_p95',
    'gap_gr_max',
    'dist_xyz_ps',
    'dx_per_md_since_ps',
    'dy_per_md_since_ps',
    'dz_per_md_since_ps',
    'gr_center_roll_mean_5',
    'gr_center_roll_std_5',
    'gr_center_roll_range_5',
    'gr_center_roll_mean_21',
    'gr_center_roll_std_21',
    'gr_center_roll_range_21',
    'gr_center_grad_1',
    'gr_center_lag1',
    'gr_center_lead1',
    'gr_center_lag5',
    'gr_center_lead5',
    'gr_cumsum_since_ps',
    'gr_center_roll_mean_51',
    'gr_center_roll_std_51',
    'gr_center_roll_range_51',
    'gr_center_roll_mean_151',
    'gr_center_roll_std_151',
    'gr_center_roll_range_151',
    'gr_center_roll_mean_301',
    'gr_center_roll_std_301',
    'gr_center_roll_range_301',
]

STRICT_FEATURE_SETS = {
    'causal_base': BASE_FEATURE_COLUMNS,
    'prefix_context': BASE_FEATURE_COLUMNS + PREFIX_CONTEXT_COLUMNS,
    'typewell_alignment': BASE_FEATURE_COLUMNS + PREFIX_CONTEXT_COLUMNS + TYPEWELL_ALIGNMENT_COLUMNS,
    'calibrated_typewell_alignment': (
        BASE_FEATURE_COLUMNS
        + PREFIX_CONTEXT_COLUMNS
        + TYPEWELL_ALIGNMENT_COLUMNS
        + CALIBRATED_TYPEWELL_ALIGNMENT_COLUMNS
    ),
}

OFFLINE_FEATURE_SETS = {
    'offline_prefix_context': STRICT_FEATURE_SETS['prefix_context'] + OFFLINE_EXTRA_FEATURE_COLUMNS,
    'offline_typewell_alignment': STRICT_FEATURE_SETS['typewell_alignment'] + OFFLINE_EXTRA_FEATURE_COLUMNS,
    'offline_calibrated_typewell_alignment': STRICT_FEATURE_SETS['calibrated_typewell_alignment'] + OFFLINE_EXTRA_FEATURE_COLUMNS,
    'offline_candidate_path_alignment': STRICT_FEATURE_SETS['typewell_alignment'] + OFFLINE_EXTRA_FEATURE_COLUMNS + OFFLINE_CANDIDATE_PATH_COLUMNS + OFFLINE_CANDIDATE_PATH_EASE_COLUMNS,
    'offline_candidate_path_calibrated_alignment': STRICT_FEATURE_SETS['calibrated_typewell_alignment'] + OFFLINE_EXTRA_FEATURE_COLUMNS + OFFLINE_CANDIDATE_PATH_COLUMNS + OFFLINE_CANDIDATE_PATH_EASE_COLUMNS,
    'offline_formation_plane_alignment': STRICT_FEATURE_SETS['typewell_alignment'] + OFFLINE_EXTRA_FEATURE_COLUMNS + FORMATION_PLANE_COLUMNS,
    'offline_formation_top_alignment': STRICT_FEATURE_SETS['typewell_alignment'] + OFFLINE_EXTRA_FEATURE_COLUMNS + FORMATION_PLANE_COLUMNS + FORMATION_ROW_ANCC_COLUMNS,
}
if ENABLE_OFFLINE_BEAM_FEATURES:
    OFFLINE_FEATURE_SETS['offline_beam_candidate_path_alignment'] = (
        STRICT_FEATURE_SETS['typewell_alignment']
        + OFFLINE_EXTRA_FEATURE_COLUMNS
        + OFFLINE_CANDIDATE_PATH_COLUMNS
        + OFFLINE_CANDIDATE_PATH_EASE_COLUMNS
        + OFFLINE_BEAM_FEATURE_COLUMNS
    )

FEATURE_SETS = {**STRICT_FEATURE_SETS, **OFFLINE_FEATURE_SETS}
FEATURE_SET_POLICY = {name: 'strict' for name in STRICT_FEATURE_SETS}
FEATURE_SET_POLICY.update({name: 'offline' for name in OFFLINE_FEATURE_SETS})

SUPER_MONOLITH_FEATURE_SET = 'offline_super220_alignment'
SUPER_BASE_COLUMNS = [
    'last_known_TVT', 'prefix_len', 'MD', 'X', 'Y', 'Z', 'GR', 'GR_isna', 'GR_prefix_z',
    'md_since_ps', 'x_delta_ps', 'y_delta_ps', 'z_delta_ps', 'xy_dist_ps', 'dist_xyz_ps',
    'tail_len', 'tail_row_number', 'tail_frac', 'tail_frac2', 'sqrt_tail_frac', 'log1p_tail_row',
    'md_tail_span', 'md_tail_frac',
    'dx_per_md_since_ps', 'dy_per_md_since_ps', 'dz_per_md_since_ps',
    'prefix_tvt_slope_md_last200', 'prefix_tvt_step20', 'prefix_tvt_step100',
    'prefix_tvt_md_slope100', 'prefix_tvt_z_slope100',
]
SUPER_GR_COLUMNS = [
    'prefix_gr_mean', 'prefix_gr_std', 'gr_minus_prefix_last_valid_gr', 'gr_minus_prefix_gr_mean',
    'gr_diff_1', 'gr_diff_5', 'gr_slope_md_1',
    'gr_roll_mean_25', 'gr_roll_std_25', 'gr_roll_range_25',
    'tail_gr_missing_rate', 'gap_gr_mean', 'gap_gr_std', 'gap_gr_min', 'gap_gr_p50', 'gap_gr_p95', 'gap_gr_max',
    'gr_center_roll_mean_5', 'gr_center_roll_std_5', 'gr_center_roll_range_5',
    'gr_center_roll_mean_21', 'gr_center_roll_std_21', 'gr_center_roll_range_21',
    'gr_center_roll_mean_51', 'gr_center_roll_std_51', 'gr_center_roll_range_51',
    'gr_center_roll_mean_151', 'gr_center_roll_std_151', 'gr_center_roll_range_151',
    'gr_center_lag1', 'gr_center_lead1', 'gr_center_lag5', 'gr_center_lead5',
    'gr_center_lag15', 'gr_center_lead15', 'gr_center_lag30', 'gr_center_lead30',
    'gr_center_grad_1', 'gr_center_grad_2', 'gr_cumsum_since_ps',
]
SUPER_TYPEWELL_COLUMNS = [
    'typewell_tvt_range', 'typewell_gr_mean', 'typewell_gr_std',
    'typewell_gr_at_last_known_tvt', 'gr_minus_typewell_last_known_tvt',
    'prefix_horizontal_vs_typewell_gr_corr', 'prefix_horizontal_vs_typewell_gr_mae', 'prefix_horizontal_vs_typewell_gr_rmse',
    'typewell_local_last200_best_delta', 'typewell_local_last200_best_abs_resid',
    'typewell_local_last200_top2_gap', 'typewell_local_last200_soft_delta_mean',
    'typewell_local_last200_best_vs_zero_gain',
    'typewell_anchor_gr_diff_m80', 'typewell_anchor_gr_diff_m40', 'typewell_anchor_gr_diff_m20',
    'typewell_anchor_gr_diff_m10', 'typewell_anchor_gr_diff_m5', 'typewell_anchor_gr_diff_p0',
    'typewell_anchor_gr_diff_p5', 'typewell_anchor_gr_diff_p10', 'typewell_anchor_gr_diff_p20',
    'typewell_anchor_gr_diff_p40', 'typewell_anchor_gr_diff_p80',
]
SUPER_PATH_COLUMNS = [
    'tw_path_min_absdiff', 'tw_path_best_endpoint', 'tw_path_best_endpoint_centered',
    'tw_path_top2_absdiff_gap', 'tw_path_soft_endpoint_mean', 'tw_path_best_gr_resid',
    'tw_path_ease_min_absdiff', 'tw_path_ease_best_endpoint', 'tw_path_ease_best_endpoint_centered',
    'tw_path_ease_top2_absdiff_gap', 'tw_path_ease_soft_endpoint_mean', 'tw_path_ease_best_gr_resid',
]
SUPER_BEAM_COLUMNS = [
    'tw_beam_tight_delta', 'tw_beam_conservative_delta', 'tw_beam_loose_delta',
    'tw_beam_vcons_delta', 'tw_beam_sm5_delta', 'tw_beam_vloose_delta',
    'tw_beam_gap', 'tw_beam_spread', 'tw_beam_mean_delta', 'tw_beam_std_delta',
    'tw_beam_tight_step', 'tw_beam_conservative_step', 'tw_beam_loose_step',
    'tw_beam_vcons_step', 'tw_beam_sm5_step', 'tw_beam_vloose_step',
    'tw_beam_gr_at_conservative', 'tw_beam_gr_at_loose', 'tw_beam_gr_minus_conservative', 'tw_beam_gr_minus_loose',
    'beam_vcons_delta', 'beam_sm5_delta', 'beam_median_delta', 'beam_sm5_vs_cons', 'beam_vcons_vs_loose',
]
SUPER_FORMATION_COLUMNS = [
    'formation_plane_min_dist', 'formation_plane_anchor_b', 'formation_plane_anchor_b50',
    'formation_plane_prefix_rmse', 'formation_plane_prefix_mae',
    'formation_plane_tvt_formula', 'formation_plane_delta_formula', 'formation_plane_delta_from_slope_last200',
    'formation_plane_formula_mean_delta', 'formation_plane_formula_std_delta',
    'formation_plane_formula_min_delta', 'formation_plane_formula_max_delta',
]
for label in FORMATION_LABELS:
    SUPER_FORMATION_COLUMNS += [
        f'formation_plane_prefix_rmse_{label}',
        f'formation_plane_delta_formula_{label}',
        f'formation_plane_delta_formula50_{label}',
    ]
SUPER_ROW_ANCC_COLUMNS = [
    'formation_row_ancc', 'formation_row_ancc_std', 'formation_row_min_dist',
    'formation_row_prefix_rmse', 'formation_row_prefix_bias',
    'formation_row_tvt_formula', 'formation_row_delta_formula', 'formation_row_delta_formula50',
    'formation_row_delta_from_plane', 'formation_formula_mean_delta', 'formation_formula_abs_gap',
    'dense_ancc', 'dense_std', 'dense_dist', 'dense_rmse', 'dense_bias',
    'tvt_dense_delta', 'tvt_dense50_delta', 'spatial_vs_dense',
]
SUPER_SELF_CORR_COLUMNS = [
    'selfcorr_delta', 'selfcorr_score', 'selfcorr_trust', 'selfcorr_top2_gap',
    'hyb_delta', 'beam_vs_sc', 'dense_vs_sc', 'plane_vs_sc',
]
SUPER_OFFSET_COLUMNS = TDBC_OFFSET_COLUMNS + TDSC_OFFSET_COLUMNS
SUPER_PF_LITE_COLUMNS = [
    'pf_lite_delta', 'pf_lite_std', 'pf_lite_weight_sum', 'pf_lite_candidate_count',
    'pf_lite_vs_dense', 'pf_lite_vs_plane', 'pf_lite_vs_sc', 'pf_lite_vs_beam_cons',
    'pf_lite_gr_abs_resid',
]
SUPER_MONOLITH_CANDIDATE_COLUMNS = list(dict.fromkeys(
    SUPER_BASE_COLUMNS + SUPER_GR_COLUMNS + SUPER_TYPEWELL_COLUMNS + SUPER_PATH_COLUMNS
    + (SUPER_BEAM_COLUMNS if ENABLE_OFFLINE_BEAM_FEATURES else [])
    + SUPER_FORMATION_COLUMNS + SUPER_ROW_ANCC_COLUMNS
    + SUPER_SELF_CORR_COLUMNS + SUPER_OFFSET_COLUMNS + SUPER_PF_LITE_COLUMNS
))
AVAILABLE_FEATURE_COLUMNS = set(
    FEATURE_SETS['offline_formation_top_alignment']
    + OFFLINE_CANDIDATE_PATH_COLUMNS
    + OFFLINE_CANDIDATE_PATH_EASE_COLUMNS
    + OFFLINE_BEAM_FEATURE_COLUMNS
    + SUPER_GR_COLUMNS
    + SELF_CORR_FEATURE_COLUMNS
    + TDBC_OFFSET_COLUMNS
    + TDSC_OFFSET_COLUMNS
    + PF_LITE_FEATURE_COLUMNS
    + ['row_index']
)
SUPER_MONOLITH_FEATURE_COLUMNS = [
    col for col in SUPER_MONOLITH_CANDIDATE_COLUMNS if col in AVAILABLE_FEATURE_COLUMNS
]
FEATURE_SETS[SUPER_MONOLITH_FEATURE_SET] = SUPER_MONOLITH_FEATURE_COLUMNS
FEATURE_SET_POLICY[SUPER_MONOLITH_FEATURE_SET] = 'offline'

DEFAULT_FEATURE_SETS_TO_COMPARE = [
    name
    for name in FEATURE_SETS
    if name not in {
        # Row-level ANCC KNN is useful as a targeted experiment, but it is heavier than the
        # plane-fit formation features and is not needed for the memory-safe submission path.
        'offline_formation_top_alignment',
        'offline_super220_alignment',
    }
]
if MODEL_CONFIG['feature_sets_to_compare'] is None:
    MODEL_CONFIG['feature_sets_to_compare'] = DEFAULT_FEATURE_SETS_TO_COMPARE

# Latest full grouped-CV selection saved in this notebook.
# These defaults make submission-only runs reproducible without rerunning CV.
SELECTED_FEATURE_SET = 'calibrated_typewell_alignment'
FEATURE_COLUMNS = FEATURE_SETS[SELECTED_FEATURE_SET]
SELECTED_POLICY_METRIC = 'global_alpha_0.812_fade_200_0_slope_q_0_9'
SELECTED_SHRINKAGE_ALPHA = 0.811832
SELECTED_FADE_IN_TAU_MD = 200.0
SELECTED_SLOPE_QUANTILE = 0.90
APPLY_SELECTED_SLOPE_CLIP = True
BEST_OVERALL_FEATURE_SET = 'offline_candidate_path_alignment'
BEST_OVERALL_SHRINKAGE_ALPHA = 0.941149
BEST_OVERALL_FADE_IN_TAU_MD = 200.0
BEST_OVERALL_SLOPE_QUANTILE = 0.90
APPLY_BEST_OVERALL_SLOPE_CLIP = True

COMPACT_LGBM_STYLE_COLUMNS = [
    'row_index', 'last_known_TVT', 'prefix_len', 'tail_len', 'tail_row_number', 'tail_frac',
    'n_rows', 'row_frac', 'md_tail_frac', 'md_tail_span',
    'MD', 'Z', 'X', 'Y', 'GR', 'GR_isna',
    'gr_center_roll_mean_5', 'gr_center_roll_mean_21',
    'gr_center_grad_1', 'gr_center_roll_std_5', 'gr_center_roll_std_21',
    'gr_center_lag1', 'gr_center_lead1', 'gr_center_lag5', 'gr_center_lead5', 'gr_cumsum_since_ps',
    'md_since_ps', 'z_delta_ps', 'x_delta_ps', 'y_delta_ps',
    'dx_per_md_since_ps', 'dy_per_md_since_ps', 'dz_per_md_since_ps', 'xy_dist_ps', 'dist_xyz_ps',
    'prefix_tvt_step20', 'prefix_tvt_step100', 'prefix_tvt_md_slope100', 'prefix_tvt_z_slope100',
    'prefix_horizontal_vs_typewell_gr_rmse', 'prefix_horizontal_vs_typewell_gr_mae',
    'typewell_anchor_gr_diff_m80', 'typewell_anchor_gr_diff_m40', 'typewell_anchor_gr_diff_m20',
    'typewell_anchor_gr_diff_m10', 'typewell_anchor_gr_diff_m5', 'typewell_anchor_gr_diff_p0',
    'typewell_anchor_gr_diff_p5', 'typewell_anchor_gr_diff_p10', 'typewell_anchor_gr_diff_p20',
    'typewell_anchor_gr_diff_p40', 'typewell_anchor_gr_diff_p80',
]
COMPACT_LGBM_ALWAYS_AVAILABLE_COLUMNS = ['row_index']
COMPACT_LGBM_COMPACT_EXTRA_COLUMNS = [
    'tail_frac2',
    'tail_frac3',
    'sqrt_tail_frac',
    'log1p_tail_row',
    'sin_tail_frac_pi',
    'sin_tail_frac_2pi',
    'cos_tail_frac_3pi',
    'gap_md_span',
    'gap_x_delta',
    'gap_y_delta',
    'gap_z_delta',
    'gap_xy_span',
    'gap_z_over_xy',
    'gap_gr_mean',
    'gap_gr_std',
    'gap_gr_min',
    'gap_gr_p05',
    'gap_gr_p25',
    'gap_gr_p50',
    'gap_gr_p75',
    'gap_gr_p95',
    'gap_gr_max',
    'tail_gr_missing_rate',
    'gr_center_roll_range_5',
    'gr_center_roll_range_21',
    'gr_center_roll_mean_51',
    'gr_center_roll_std_51',
    'gr_center_roll_range_51',
    'gr_center_roll_mean_151',
    'gr_center_roll_std_151',
    'gr_center_roll_range_151',
]
COMPACT_LGBM_STYLE_COLUMNS = list(dict.fromkeys(COMPACT_LGBM_STYLE_COLUMNS + COMPACT_LGBM_COMPACT_EXTRA_COLUMNS))
COMPACT_LGBM_STYLE_COLUMNS = [
    col
    for col in COMPACT_LGBM_STYLE_COLUMNS
    if col in FEATURE_SETS['offline_typewell_alignment'] + OFFLINE_EXTRA_FEATURE_COLUMNS + COMPACT_LGBM_ALWAYS_AVAILABLE_COLUMNS
]
FEATURE_SETS['offline_compact_lgbm_style'] = list(dict.fromkeys(COMPACT_LGBM_STYLE_COLUMNS))
FEATURE_SET_POLICY['offline_compact_lgbm_style'] = 'offline'
COMPACT_LGBM_FORMATION_STYLE_COLUMNS = list(dict.fromkeys(COMPACT_LGBM_STYLE_COLUMNS + FORMATION_PLANE_COLUMNS))
FEATURE_SETS['offline_compact_lgbm_formation_style'] = COMPACT_LGBM_FORMATION_STYLE_COLUMNS
FEATURE_SET_POLICY['offline_compact_lgbm_formation_style'] = 'offline'
COMPACT_LGBM_FORMATION_TOP_STYLE_COLUMNS = list(dict.fromkeys(
    COMPACT_LGBM_STYLE_COLUMNS + FORMATION_PLANE_COLUMNS + FORMATION_ROW_ANCC_COLUMNS
))
FEATURE_SETS['offline_compact_lgbm_formation_top_style'] = COMPACT_LGBM_FORMATION_TOP_STYLE_COLUMNS
FEATURE_SET_POLICY['offline_compact_lgbm_formation_top_style'] = 'offline'
# The compact feature set is available for targeted experiments but excluded from default CV
# unless explicitly listed in MODEL_CONFIG['feature_sets_to_compare'].

feature_set_report = pd.DataFrame({
    'feature_set': list(FEATURE_SETS.keys()),
    'feature_policy': [FEATURE_SET_POLICY[name] for name in FEATURE_SETS],
    'feature_count': [len(cols) for cols in FEATURE_SETS.values()],
})
display(feature_set_report)
print('NOTEBOOK_RUN_VERSION:', NOTEBOOK_RUN_VERSION)
print('RUN_GROUPED_CV:', RUN_GROUPED_CV)
print('KAGGLE_MEMORY_SAFE_MODE:', KAGGLE_MEMORY_SAFE_MODE)
print('ENABLE_OFFLINE_BEAM_FEATURES:', ENABLE_OFFLINE_BEAM_FEATURES)
print('RUN_HGB_DIAGNOSTIC_SUBMISSIONS:', RUN_HGB_DIAGNOSTIC_SUBMISSIONS)
print('RUN_SUPER_MONOLITH_STACK:', RUN_SUPER_MONOLITH_STACK)
print('SUPER_MONOLITH_FEATURE_SET:', SUPER_MONOLITH_FEATURE_SET)
print('QUICK_CHECK_MODE:', QUICK_CHECK_MODE)
print('selection_feature_policy:', MODEL_CONFIG['feature_policy_for_selection'])
print('configured_feature_set_before_cv:', SELECTED_FEATURE_SET)


### 16.1 Feature Table Builder

- 🧱 Build one row per eligible hidden-tail row.
- 🧮 Cap rows per well in local diagnostics to keep runtime sane.
- 🛡️ Sample only inside the training fold; never sample validation labels into features.

| Effect | Use |
|---|---|
| fewer rows | faster local experiments |
| per-well cap | very long wells do not dominate diagnostics |
| fold-local sampling | keeps validation clean |


In [ ]:
# Build tail feature matrices and define model/post-processing helpers.

def feature_columns_require_beam(feature_columns) -> bool:
    if feature_columns is None:
        return bool(globals().get('ENABLE_OFFLINE_BEAM_FEATURES', False))
    return any(str(col).startswith(('tw_beam_', 'beam_', 'tdbc_', 'pf_lite_')) for col in feature_columns)


def feature_columns_require_formation(feature_columns) -> bool:
    if feature_columns is None:
        return False
    return any(str(col).startswith(('formation_', 'dense_', 'tvt_dense_', 'spatial_vs_')) for col in feature_columns)


def feature_columns_require_row_ancc(feature_columns) -> bool:
    if feature_columns is None:
        return False
    return any(str(col).startswith(('formation_row_', 'formation_formula_', 'dense_', 'tvt_dense_', 'spatial_vs_')) for col in feature_columns)


def compact_tail_feature_frame(
    frame: pd.DataFrame,
    keep_columns=None,
    include_target: bool = True,
    keep_metadata: bool = True,
) -> pd.DataFrame:
    if keep_columns is not None:
        required_columns = []
        if keep_metadata:
            required_columns += [
                'well_id',
                'row_index',
                'id',
                'MD',
                'last_known_TVT',
                'last_known_MD',
                'md_since_ps',
            ]
        if include_target:
            required_columns += ['target_tvt', 'target_delta_from_last_known']
        frame = frame.copy()
        for col in keep_columns:
            if col not in frame.columns:
                frame[col] = np.nan
        wanted = list(dict.fromkeys(required_columns + list(keep_columns)))
        frame = frame[[col for col in wanted if col in frame.columns]]
    else:
        frame = frame.copy()

    for col in frame.columns:
        if pd.api.types.is_float_dtype(frame[col]):
            frame[col] = pd.to_numeric(frame[col], downcast='float')
        elif pd.api.types.is_integer_dtype(frame[col]) and col not in {'row_index'}:
            frame[col] = pd.to_numeric(frame[col], downcast='integer')
    return frame


def build_tail_feature_frame(
    well_ids,
    split_dir: Path,
    include_target: bool,
    rows_per_well: int | None = None,
    random_state: int = 42,
    use_beam_features: bool | None = None,
    keep_columns=None,
    formation_plane_imputer=None,
    row_ancc_imputer=None,
    imputer_well_ids=None,
    keep_metadata: bool = True,
    exclude_query_well_from_formation: bool = True,
) -> pd.DataFrame:
    if use_beam_features is None:
        use_beam_features = bool(globals().get('ENABLE_OFFLINE_BEAM_FEATURES', False))
    need_formation = feature_columns_require_formation(keep_columns)
    need_row_ancc = feature_columns_require_row_ancc(keep_columns)
    if need_formation and formation_plane_imputer is None:
        source_wells = imputer_well_ids if imputer_well_ids is not None else globals().get('all_train_wells', globals().get('train_h_ids', []))
        formation_plane_imputer, maybe_row = make_formation_imputers(
            source_wells,
            TRAIN_DIR,
            need_row_ancc=need_row_ancc and row_ancc_imputer is None,
            seed=random_state,
        )
        if row_ancc_imputer is None:
            row_ancc_imputer = maybe_row
    rng = np.random.default_rng(random_state)
    frames = []
    for well_id in sorted(well_ids):
        frame = make_tail_features_for_well(
            well_id,
            split_dir,
            include_target=include_target,
            use_beam_features=use_beam_features,
            required_feature_columns=keep_columns,
            formation_plane_imputer=formation_plane_imputer,
            row_ancc_imputer=row_ancc_imputer,
            exclude_query_well_from_formation=exclude_query_well_from_formation,
        )
        if frame.empty:
            continue
        if rows_per_well is not None and len(frame) > rows_per_well:
            sampled_idx = np.sort(rng.choice(frame.index.to_numpy(), size=rows_per_well, replace=False))
            frame = frame.loc[sampled_idx]
        frame = compact_tail_feature_frame(
            frame,
            keep_columns=keep_columns,
            include_target=include_target,
            keep_metadata=keep_metadata,
        )
        frames.append(frame)
        gc.collect()
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True, copy=False)


def model_ready_xy(frame: pd.DataFrame, feature_columns=None):
    if feature_columns is None:
        feature_columns = FEATURE_COLUMNS
    X = frame[feature_columns].copy()
    y_delta = frame['target_delta_from_last_known'].to_numpy()
    y_tvt = frame['target_tvt'].to_numpy()
    return X, y_delta, y_tvt


def make_residual_model(random_state: int = 42):
    # Fixed-iteration HGB avoids row-random internal early stopping inside a grouped-CV fold.
    return HistGradientBoostingRegressor(
        loss='squared_error',
        learning_rate=0.06,
        max_iter=220,
        max_leaf_nodes=31,
        l2_regularization=0.05,
        early_stopping=False,
        random_state=random_state,
    )


def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def clip_delta_by_train_quantiles(delta_pred: np.ndarray, train_delta: np.ndarray, config=MODEL_CONFIG) -> np.ndarray:
    lo_q, hi_q = config['delta_clip_quantiles']
    lo, hi = np.nanquantile(train_delta, [lo_q, hi_q])
    return np.clip(delta_pred, lo, hi)


def shrink_delta(delta: np.ndarray, alpha: float | None = None, config=MODEL_CONFIG) -> np.ndarray:
    if alpha is None:
        alpha = config['residual_shrinkage']
    return float(alpha) * np.asarray(delta, dtype=float)


def fade_in_delta(frame: pd.DataFrame, delta: np.ndarray, tau_md: float | None) -> np.ndarray:
    """Dampen residuals close to Prediction Start; tau=None leaves them unchanged."""
    values = np.asarray(delta, dtype=float)
    if tau_md is None or not np.isfinite(tau_md) or tau_md <= 0:
        return values
    md_since = frame['md_since_ps'].to_numpy(dtype=float)
    rho = 1.0 - np.exp(-np.maximum(md_since, 0.0) / float(tau_md))
    return values * rho


def fade_in_tau_label(tau_md: float | None) -> str:
    if tau_md is None or (isinstance(tau_md, float) and not np.isfinite(tau_md)):
        return 'none'
    return str(float(tau_md)).replace('.', '_')


def fit_global_alpha_from_fold_parts(fold_parts, tau_md: float | None, config=MODEL_CONFIG) -> tuple[float, float]:
    """Fit one alpha over all validation folds for a fixed fade-in setting."""
    numerator = 0.0
    denominator = 0.0
    for part in fold_parts:
        frame = part['frame']
        base_delta = fade_in_delta(frame, part['clipped_delta'], tau_md)
        y_delta = frame['target_delta_from_last_known'].to_numpy(dtype=float)
        valid = np.isfinite(base_delta) & np.isfinite(y_delta)
        numerator += float(np.dot(y_delta[valid], base_delta[valid]))
        denominator += float(np.dot(base_delta[valid], base_delta[valid]))
    if denominator <= 1e-12:
        alpha = 0.0
    else:
        alpha = numerator / denominator
    lo, hi = config['alpha_bounds']
    alpha = float(np.clip(alpha, lo, hi))
    score = score_policy_from_fold_parts(fold_parts, alpha=alpha, tau_md=tau_md, slope_quantile=None)['rmse']
    return alpha, score


def score_policy_from_fold_parts(fold_parts, alpha: float, tau_md: float | None, slope_quantile: float | None) -> dict[str, float]:
    sse = 0.0
    n = 0
    for part in fold_parts:
        frame = part['frame']
        delta = shrink_delta(fade_in_delta(frame, part['clipped_delta'], tau_md), alpha=alpha)
        pred = frame['last_known_TVT'].to_numpy(dtype=float) + delta
        if slope_quantile is not None:
            pred = causal_slope_clip_by_well(frame, pred, part['slope_bounds'][float(slope_quantile)])
        y = frame['target_tvt'].to_numpy(dtype=float)
        valid = np.isfinite(y) & np.isfinite(pred)
        err = y[valid] - pred[valid]
        sse += float(np.dot(err, err))
        n += int(valid.sum())
    return {'rmse': float(np.sqrt(sse / n)), 'sse': sse, 'n': n}


def policy_metric_name(alpha: float, tau_md: float | None, slope_quantile: float | None) -> str:
    tau_label = fade_in_tau_label(tau_md)
    if slope_quantile is None:
        slope_label = 'none'
    else:
        slope_label = f'q_{slope_quantile_label(float(slope_quantile))}'
    return f'global_alpha_{alpha:.3f}_fade_{tau_label}_slope_{slope_label}'


def best_alpha_for_clipped_delta(
    last_known_tvt: np.ndarray,
    clipped_delta: np.ndarray,
    y_true: np.ndarray,
    alphas=None,
) -> tuple[float, float]:
    if alphas is None:
        alphas = np.linspace(0.0, 1.5, 61)
    best_score = np.inf
    best_alpha = np.nan
    for alpha in alphas:
        score = rmse(y_true, last_known_tvt + alpha * clipped_delta)
        if score < best_score:
            best_score = score
            best_alpha = float(alpha)
    return best_alpha, float(best_score)


def estimate_abs_tvt_slope_quantiles(paths, quantiles, max_wells=None) -> dict[float, float]:
    slopes = []
    selected_paths = list(paths)[:max_wells] if max_wells is not None else list(paths)
    for path in selected_paths:
        df = pd.read_csv(path, usecols=['MD', 'TVT'])
        md = df['MD'].to_numpy(dtype=float)
        tvt = df['TVT'].to_numpy(dtype=float)
        dmd = np.diff(md)
        dtvt = np.diff(tvt)
        valid = np.isfinite(dmd) & np.isfinite(dtvt) & (np.abs(dmd) > 1e-9)
        if valid.any():
            slopes.append(np.abs(dtvt[valid] / dmd[valid]))
    if not slopes:
        return {float(q): np.nan for q in quantiles}
    values = np.concatenate(slopes)
    return {float(q): float(np.nanquantile(values, q)) for q in quantiles}


def estimate_abs_tvt_slope_quantile(paths, quantile=0.995, max_wells=None) -> float:
    return estimate_abs_tvt_slope_quantiles(paths, [quantile], max_wells=max_wells)[float(quantile)]


def slope_quantile_label(q: float) -> str:
    return str(float(q)).replace('.', '_')


def causal_slope_clip_by_well(frame: pd.DataFrame, pred_tvt: np.ndarray, max_abs_slope: float) -> np.ndarray:
    """Anchor-aware forward slope limiter. It never averages with future rows."""
    clipped = np.asarray(pred_tvt, dtype=float).copy()
    if not np.isfinite(max_abs_slope) or max_abs_slope <= 0:
        return clipped
    required = ['well_id', 'row_index', 'MD', 'last_known_TVT', 'last_known_MD']
    frame_ordered = frame[required].copy()
    frame_ordered['_pos'] = np.arange(len(frame_ordered))
    for _, g in frame_ordered.sort_values(['well_id', 'row_index']).groupby('well_id', sort=False):
        pos = g['_pos'].to_numpy(dtype=int)
        md = g['MD'].to_numpy(dtype=float)
        prev_tvt = float(g['last_known_TVT'].iloc[0])
        prev_md = float(g['last_known_MD'].iloc[0])
        for k in range(len(pos)):
            step_md = abs(md[k] - prev_md)
            limit = max_abs_slope * max(step_md, 1e-6)
            clipped[pos[k]] = np.clip(clipped[pos[k]], prev_tvt - limit, prev_tvt + limit)
            prev_tvt = clipped[pos[k]]
            prev_md = md[k]
    return clipped


MAX_ABS_TVT_SLOPE_BY_QUANTILE = estimate_abs_tvt_slope_quantiles(
    train_horizontal_files,
    quantiles=MODEL_CONFIG['slope_clip_quantiles_to_compare'],
)
MAX_ABS_TVT_SLOPE = MAX_ABS_TVT_SLOPE_BY_QUANTILE[float(MODEL_CONFIG['postprocess_slope_quantile'])]
print('feature builder configured')
print('final_train_max_abs_tvt_slope_by_quantile:', MAX_ABS_TVT_SLOPE_BY_QUANTILE)


### 16.2 GroupKFold Validation

This is a local diagnostic scaffold. Kaggle submission runs skip the expensive grouped validation cells and use the final Section 18 engine.

| Variant | Use |
|---|---|
| Raw | inspect the residual model |
| Delta-clipped | bound extreme corrections |
| Global-shrunk | calibrate residual scale |
| Fade-in | protect early tail rows |
| Slope-limited | suppress implausible jumps |

⚠️ `GroupKFold` splits wells, not geology. Fold results are diagnostics, not guarantees.


In [ ]:
# Run well-level GroupKFold validation and choose one global post-processing policy per feature set.

all_train_wells = np.array(sorted(train_h_ids))
gkf = GroupKFold(n_splits=MODEL_CONFIG['n_splits'])
fold_splits = list(gkf.split(all_train_wells, groups=all_train_wells))[:MODEL_CONFIG['cv_folds_to_run']]
slope_quantiles = [float(q) for q in MODEL_CONFIG['slope_clip_quantiles_to_compare']]
tau_candidates = MODEL_CONFIG['fade_in_tau_md_to_compare']
feature_names_to_compare = list(MODEL_CONFIG['feature_sets_to_compare'])
cv_required_feature_columns = list(dict.fromkeys(
    col
    for name in feature_names_to_compare
    if name in FEATURE_SETS
    for col in FEATURE_SETS[name]
))
cv_requires_beam_features = any(
    feature_columns_require_beam(FEATURE_SETS[name])
    for name in feature_names_to_compare
    if name in FEATURE_SETS
)
cv_requires_formation_features = feature_columns_require_formation(cv_required_feature_columns)
cv_requires_row_ancc_features = feature_columns_require_row_ancc(cv_required_feature_columns)

if not RUN_GROUPED_CV:
    cv_report = pd.DataFrame()
    policy_grid = pd.DataFrame()
    fold_overview = pd.DataFrame()
    feature_fold_parts = {}
    cv_summary = pd.DataFrame([
        {
            'feature_set': SELECTED_FEATURE_SET,
            'feature_policy': FEATURE_SET_POLICY[SELECTED_FEATURE_SET],
            'feature_count': len(FEATURE_SETS[SELECTED_FEATURE_SET]),
            'row_weighted_policy_rmse': np.nan,
            'selected_policy_metric': SELECTED_POLICY_METRIC,
            'policy_alpha': SELECTED_SHRINKAGE_ALPHA,
            'policy_fade_tau_md': SELECTED_FADE_IN_TAU_MD,
            'policy_slope_quantile': SELECTED_SLOPE_QUANTILE,
            'policy_apply_slope_clip': APPLY_SELECTED_SLOPE_CLIP,
        },
        {
            'feature_set': BEST_OVERALL_FEATURE_SET,
            'feature_policy': FEATURE_SET_POLICY[BEST_OVERALL_FEATURE_SET],
            'feature_count': len(FEATURE_SETS[BEST_OVERALL_FEATURE_SET]),
            'row_weighted_policy_rmse': np.nan,
            'selected_policy_metric': 'stored_best_overall',
            'policy_alpha': BEST_OVERALL_SHRINKAGE_ALPHA,
            'policy_fade_tau_md': BEST_OVERALL_FADE_IN_TAU_MD,
            'policy_slope_quantile': BEST_OVERALL_SLOPE_QUANTILE,
            'policy_apply_slope_clip': APPLY_BEST_OVERALL_SLOPE_CLIP,
        },
    ])
    print('Grouped CV skipped. Using stored CV selections from the latest full validation run.')
    display(cv_summary)
else:
    cv_rows = []
    policy_rows = []
    fold_overview_rows = []
    feature_fold_parts = {name: [] for name in feature_names_to_compare}

    for fold, (tr_idx, va_idx) in enumerate(fold_splits, start=1):
        train_wells = all_train_wells[tr_idx]
        valid_wells = all_train_wells[va_idx]
        fold_train_paths = [TRAIN_DIR / f'{well_id}__horizontal_well.csv' for well_id in train_wells]
        fold_slope_bounds = estimate_abs_tvt_slope_quantiles(
            fold_train_paths,
            quantiles=slope_quantiles,
        )
        configured_fold_slope = fold_slope_bounds[float(MODEL_CONFIG['postprocess_slope_quantile'])]
        fold_plane_imputer = fold_row_imputer = None
        if cv_requires_formation_features:
            fold_plane_imputer, fold_row_imputer = make_formation_imputers(
                train_wells,
                TRAIN_DIR,
                need_row_ancc=cv_requires_row_ancc_features,
                seed=MODEL_CONFIG['random_state'] + fold,
            )
    
        train_frame = build_tail_feature_frame(
            train_wells,
            TRAIN_DIR,
            include_target=True,
            rows_per_well=MODEL_CONFIG['train_rows_per_well'],
            random_state=MODEL_CONFIG['random_state'] + fold,
            use_beam_features=cv_requires_beam_features,
            keep_columns=cv_required_feature_columns,
            formation_plane_imputer=fold_plane_imputer,
            row_ancc_imputer=fold_row_imputer,
            imputer_well_ids=train_wells,
        )
        valid_frame = build_tail_feature_frame(
            valid_wells,
            TRAIN_DIR,
            include_target=True,
            rows_per_well=None,
            random_state=MODEL_CONFIG['random_state'] + 1000 + fold,
            use_beam_features=cv_requires_beam_features,
            keep_columns=cv_required_feature_columns,
            formation_plane_imputer=fold_plane_imputer,
            row_ancc_imputer=fold_row_imputer,
            imputer_well_ids=train_wells,
        )
        last_known_valid = valid_frame['last_known_TVT'].to_numpy(dtype=float)
        y_valid_tvt = valid_frame['target_tvt'].to_numpy(dtype=float)
        y_valid_delta = valid_frame['target_delta_from_last_known'].to_numpy(dtype=float)
        fold_baseline_rmse = rmse(y_valid_tvt, last_known_valid)
    
        fold_overview_rows.append({
            'fold': fold,
            'valid_wells': len(valid_wells),
            'valid_rows': len(valid_frame),
            'constant_baseline_rmse': fold_baseline_rmse,
        })
        compact_valid = valid_frame[[
            'well_id',
            'row_index',
            'MD',
            'last_known_MD',
            'last_known_TVT',
            'target_tvt',
            'target_delta_from_last_known',
            'md_since_ps',
            'tail_row_number',
        ]].copy()
    
        for feature_set_name in feature_names_to_compare:
            feature_columns = FEATURE_SETS[feature_set_name]
            X_train, y_train_delta, _ = model_ready_xy(train_frame, feature_columns)
            X_valid, _, _ = model_ready_xy(valid_frame, feature_columns)
            model = make_residual_model(random_state=MODEL_CONFIG['random_state'] + fold)
            model.fit(X_train, y_train_delta)
            raw_delta = model.predict(X_valid)
            clipped_delta = clip_delta_by_train_quantiles(raw_delta, y_train_delta)
            fixed_shrunk_delta = shrink_delta(clipped_delta)
            pred_raw = last_known_valid + raw_delta
            pred_clipped = last_known_valid + clipped_delta
            pred_fixed_shrunk = last_known_valid + fixed_shrunk_delta
            fold_best_alpha, fold_best_alpha_rmse = best_alpha_for_clipped_delta(last_known_valid, clipped_delta, y_valid_tvt)
            pred_configured_slope = causal_slope_clip_by_well(valid_frame, pred_fixed_shrunk, configured_fold_slope)
        
            cv_rows.append({
                'fold': fold,
                'feature_set': feature_set_name,
                'feature_policy': FEATURE_SET_POLICY[feature_set_name],
                'feature_count': len(feature_columns),
                'train_wells': len(train_wells),
                'valid_wells': len(valid_wells),
                'train_rows_sampled': len(train_frame),
                'valid_rows_full': len(valid_frame),
                'configured_fold_max_abs_tvt_slope': configured_fold_slope,
                'baseline_constant_rmse': fold_baseline_rmse,
                'rmse_raw_model': rmse(y_valid_tvt, pred_raw),
                'rmse_delta_clipped': rmse(y_valid_tvt, pred_clipped),
                'rmse_fixed_shrunk_delta': rmse(y_valid_tvt, pred_fixed_shrunk),
                'rmse_anchor_slope_limited': rmse(y_valid_tvt, pred_configured_slope),
                'fixed_shrinkage_alpha': MODEL_CONFIG['residual_shrinkage'],
                'fold_oracle_alpha_from_delta_clip': fold_best_alpha,
                'rmse_fold_oracle_alpha_delta_clip': fold_best_alpha_rmse,
                'delta_target_std_valid': float(np.nanstd(y_valid_delta)),
                'delta_pred_std_raw': float(np.nanstd(raw_delta)),
                'delta_pred_std_clipped': float(np.nanstd(clipped_delta)),
                'delta_pred_std_fixed_shrunk': float(np.nanstd(fixed_shrunk_delta)),
            })
            feature_fold_parts[feature_set_name].append({
                'fold': fold,
                'frame': compact_valid,
                'clipped_delta': clipped_delta,
                'slope_bounds': fold_slope_bounds,
            })

    for feature_set_name in feature_names_to_compare:
        feature_columns = FEATURE_SETS[feature_set_name]
        fold_parts = feature_fold_parts[feature_set_name]
        fade_candidates = []
        for tau_md in tau_candidates:
            alpha, no_slope_rmse = fit_global_alpha_from_fold_parts(fold_parts, tau_md=tau_md)
            fade_candidates.append({
                'tau_md': tau_md,
                'alpha': alpha,
                'no_slope_rmse': no_slope_rmse,
            })
        best_fade = min(fade_candidates, key=lambda row: row['no_slope_rmse'])
    
        slope_options = [None] + slope_quantiles
        for slope_q in slope_options:
            score = score_policy_from_fold_parts(
                fold_parts,
                alpha=best_fade['alpha'],
                tau_md=best_fade['tau_md'],
                slope_quantile=slope_q,
            )
            policy_rows.append({
                'feature_set': feature_set_name,
                'feature_policy': FEATURE_SET_POLICY[feature_set_name],
                'feature_count': len(feature_columns),
                'policy_metric': policy_metric_name(best_fade['alpha'], best_fade['tau_md'], slope_q),
                'policy_alpha': best_fade['alpha'],
                'policy_fade_tau_md': best_fade['tau_md'],
                'policy_slope_quantile': slope_q,
                'policy_apply_slope_clip': slope_q is not None,
                'policy_rmse': score['rmse'],
                'policy_rows': score['n'],
                'global_alpha_no_slope_rmse': best_fade['no_slope_rmse'],
            })

    cv_report = pd.DataFrame(cv_rows)
    policy_grid = pd.DataFrame(policy_rows)
    fold_overview = pd.DataFrame(fold_overview_rows)

    if len(cv_report) and len(policy_grid):
        print('CV folds evaluated:', cv_report['fold'].nunique())
        print('Feature sets compared:', cv_report['feature_set'].nunique())
        print('Postprocess policies compared:', len(policy_grid))
    
        agg_spec = {
            'feature_policy': ('feature_policy', 'first'),
            'feature_count': ('feature_count', 'first'),
            'mean_baseline_rmse': ('baseline_constant_rmse', 'mean'),
            'mean_rmse_raw_model': ('rmse_raw_model', 'mean'),
            'mean_rmse_delta_clipped': ('rmse_delta_clipped', 'mean'),
            'mean_rmse_fixed_shrunk_delta': ('rmse_fixed_shrunk_delta', 'mean'),
            'mean_rmse_anchor_slope_limited': ('rmse_anchor_slope_limited', 'mean'),
            'mean_fold_oracle_alpha': ('fold_oracle_alpha_from_delta_clip', 'mean'),
            'mean_rmse_fold_oracle_alpha': ('rmse_fold_oracle_alpha_delta_clip', 'mean'),
            'std_rmse_fixed_shrunk_delta': ('rmse_fixed_shrunk_delta', 'std'),
            'std_rmse_anchor_slope_limited': ('rmse_anchor_slope_limited', 'std'),
        }
        cv_summary = cv_report.groupby('feature_set', as_index=False).agg(**agg_spec)
    
        row_weighted_metric_columns = [
            'baseline_constant_rmse',
            'rmse_raw_model',
            'rmse_delta_clipped',
            'rmse_fixed_shrunk_delta',
            'rmse_anchor_slope_limited',
            'rmse_fold_oracle_alpha_delta_clip',
        ]
        weighted_rows = []
        for feature_set_name, g in cv_report.groupby('feature_set'):
            weights = g['valid_rows_full'].to_numpy(dtype=float)
            row = {'feature_set': feature_set_name}
            for metric_col in row_weighted_metric_columns:
                values = g[metric_col].to_numpy(dtype=float)
                row[f'row_weighted_{metric_col}'] = float(np.sqrt(np.sum(weights * values ** 2) / np.sum(weights)))
            weighted_rows.append(row)
        cv_summary = cv_summary.merge(pd.DataFrame(weighted_rows), on='feature_set', how='left')
    
        best_policy_by_feature = (
            policy_grid.sort_values('policy_rmse')
            .drop_duplicates('feature_set')
            .rename(columns={
                'policy_rmse': 'row_weighted_policy_rmse',
                'policy_metric': 'selected_policy_metric',
            })
        )
        cv_summary = cv_summary.merge(
            best_policy_by_feature[[
                'feature_set',
                'selected_policy_metric',
                'policy_alpha',
                'policy_fade_tau_md',
                'policy_slope_quantile',
                'policy_apply_slope_clip',
                'row_weighted_policy_rmse',
                'global_alpha_no_slope_rmse',
            ]],
            on='feature_set',
            how='left',
        )
        cv_summary['row_weighted_policy_improvement'] = (
            cv_summary['row_weighted_baseline_constant_rmse'] - cv_summary['row_weighted_policy_rmse']
        )
        cv_summary = cv_summary.sort_values(['feature_policy', 'row_weighted_policy_rmse'])
    
        display(fold_overview.style.format({
            'valid_rows': '{:,.0f}',
            'constant_baseline_rmse': '{:.4f}',
        }))
    
        cv_summary_display = cv_summary[[
            'feature_set',
            'feature_policy',
            'feature_count',
            'row_weighted_baseline_constant_rmse',
            'row_weighted_rmse_raw_model',
            'global_alpha_no_slope_rmse',
            'row_weighted_policy_rmse',
            'policy_alpha',
            'policy_fade_tau_md',
            'policy_slope_quantile',
            'selected_policy_metric',
        ]].rename(columns={
            'row_weighted_baseline_constant_rmse': 'baseline_rmse',
            'row_weighted_rmse_raw_model': 'raw_rmse',
            'global_alpha_no_slope_rmse': 'global_alpha_rmse',
            'row_weighted_policy_rmse': 'policy_rmse',
            'policy_alpha': 'alpha',
            'policy_fade_tau_md': 'fade_tau_md',
            'policy_slope_quantile': 'slope_q',
        })
        display(cv_summary_display.style.format({
            'baseline_rmse': '{:.4f}',
            'raw_rmse': '{:.4f}',
            'global_alpha_rmse': '{:.4f}',
            'policy_rmse': '{:.4f}',
            'alpha': '{:.3f}',
            'fade_tau_md': lambda x: 'none' if pd.isna(x) else f'{x:.0f}',
            'slope_q': lambda x: 'none' if pd.isna(x) else f'{x:.3f}',
        }))
    
        best_overall = cv_summary.sort_values('row_weighted_policy_rmse').iloc[0]
        BEST_OVERALL_FEATURE_SET = str(best_overall['feature_set'])
        BEST_OVERALL_SHRINKAGE_ALPHA = float(best_overall['policy_alpha'])
        BEST_OVERALL_FADE_IN_TAU_MD = None if pd.isna(best_overall['policy_fade_tau_md']) else float(best_overall['policy_fade_tau_md'])
        BEST_OVERALL_SLOPE_QUANTILE = None if pd.isna(best_overall['policy_slope_quantile']) else float(best_overall['policy_slope_quantile'])
        APPLY_BEST_OVERALL_SLOPE_CLIP = bool(best_overall['policy_apply_slope_clip'])
    
        if MODEL_CONFIG['select_best_feature_set_from_cv']:
            selection_policy = MODEL_CONFIG['feature_policy_for_selection']
            selection_summary = cv_summary[cv_summary['feature_policy'].eq(selection_policy)].copy()
            if selection_summary.empty:
                raise ValueError(f'No feature sets available for selection policy: {selection_policy}')
            selected = selection_summary.sort_values('row_weighted_policy_rmse').iloc[0]
            SELECTED_FEATURE_SET = str(selected['feature_set'])
            FEATURE_COLUMNS = FEATURE_SETS[SELECTED_FEATURE_SET]
            SELECTED_POLICY_METRIC = str(selected['selected_policy_metric'])
            SELECTED_SHRINKAGE_ALPHA = float(selected['policy_alpha'])
            SELECTED_FADE_IN_TAU_MD = None if pd.isna(selected['policy_fade_tau_md']) else float(selected['policy_fade_tau_md'])
            SELECTED_SLOPE_QUANTILE = None if pd.isna(selected['policy_slope_quantile']) else float(selected['policy_slope_quantile'])
            APPLY_SELECTED_SLOPE_CLIP = bool(selected['policy_apply_slope_clip'])
            selected_summary = pd.Series({
                'selection_policy': selection_policy,
                'selected_feature_set': SELECTED_FEATURE_SET,
                'selected_feature_count': len(FEATURE_COLUMNS),
                'selected_policy_metric': SELECTED_POLICY_METRIC,
                'selected_shrinkage_alpha': SELECTED_SHRINKAGE_ALPHA,
                'selected_fade_in_tau_md': SELECTED_FADE_IN_TAU_MD,
                'selected_slope_quantile': SELECTED_SLOPE_QUANTILE,
                'apply_slope_clip': APPLY_SELECTED_SLOPE_CLIP,
                'selected_policy_rmse': float(selected['row_weighted_policy_rmse']),
                'best_overall_feature_set': BEST_OVERALL_FEATURE_SET,
                'best_overall_policy': best_overall['feature_policy'],
                'best_overall_policy_rmse': float(best_overall['row_weighted_policy_rmse']),
            })
            display(selected_summary.to_frame('value'))
        
        report_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else DATA_DIR
        cv_report.to_csv(report_dir / 'v7_cv_report.csv', index=False)
        policy_grid.to_csv(report_dir / 'v7_policy_grid.csv', index=False)
        cv_summary.to_csv(report_dir / 'v7_cv_summary.csv', index=False)
        fold_overview.to_csv(report_dir / 'v7_cv_fold_overview.csv', index=False)
        print('CV reports written to:', report_dir)


### 16.2.1 Policy Diagnostics

Optional checks for the lighter residual pipeline:

- 🧯 Which postprocess step changes the curve most?
- 🌅 Does behavior differ near Prediction Start?
- 🧵 Are gains broad across wells or concentrated in a few drifting wells?

No test labels are used here.


In [ ]:
if not RUN_GROUPED_CV:
    selected_policy_stage_report = pd.DataFrame()
    selected_policy_tail_md_diagnostics = pd.DataFrame()
    selected_policy_tail_row_diagnostics = pd.DataFrame()
    selected_policy_well_gain = pd.DataFrame()
    print('OOF diagnostics skipped because grouped CV was skipped. Run with RUN_GROUPED_CV=True to refresh diagnostics.')
else:
    # Diagnose the selected OOF policy by stage, tail position, and well-level gain.

    required_diagnostic_vars = [
        'feature_fold_parts',
        'SELECTED_FEATURE_SET',
        'SELECTED_SHRINKAGE_ALPHA',
        'SELECTED_FADE_IN_TAU_MD',
        'SELECTED_SLOPE_QUANTILE',
        'APPLY_SELECTED_SLOPE_CLIP',
    ]
    missing_diagnostic_vars = [name for name in required_diagnostic_vars if name not in globals()]
    if missing_diagnostic_vars:
        raise RuntimeError(f'Selected-policy diagnostic variables are missing: {missing_diagnostic_vars}. Run the validation cell first.')

    if SELECTED_FEATURE_SET not in feature_fold_parts:
        raise KeyError(f'Selected feature set is not available in OOF fold parts: {SELECTED_FEATURE_SET}')

    selected_fold_parts = feature_fold_parts[SELECTED_FEATURE_SET]
    stage_sse = {
        'constant_anchor': 0.0,
        'delta_clipped': 0.0,
        'global_alpha': 0.0,
        'fade_in': 0.0,
        'selected_policy': 0.0,
    }
    stage_n = {name: 0 for name in stage_sse}
    tail_md_bin_rows = []
    tail_row_bin_rows = []
    well_gain_rows = []

    md_bins = [0, 10, 50, 100, 250, 500, 1000, 2000, 5000, np.inf]
    row_bins = [-0.5, 0.5, 4.5, 9.5, 24.5, 49.5, 99.5, 249.5, 499.5, 999.5, np.inf]
    row_labels = ['0', '1-4', '5-9', '10-24', '25-49', '50-99', '100-249', '250-499', '500-999', '1000+']

    for part in selected_fold_parts:
        frame = part['frame']
        y = frame['target_tvt'].to_numpy(dtype=float)
        anchor = frame['last_known_TVT'].to_numpy(dtype=float)
        clipped_delta = np.asarray(part['clipped_delta'], dtype=float)
        pred_delta_clipped = anchor + clipped_delta
        pred_global_alpha = anchor + shrink_delta(clipped_delta, alpha=SELECTED_SHRINKAGE_ALPHA)
        faded_delta = shrink_delta(
            fade_in_delta(frame, clipped_delta, SELECTED_FADE_IN_TAU_MD),
            alpha=SELECTED_SHRINKAGE_ALPHA,
        )
        pred_fade_in = anchor + faded_delta
        pred_selected = pred_fade_in.copy()
        if APPLY_SELECTED_SLOPE_CLIP and SELECTED_SLOPE_QUANTILE is not None:
            slope_bound = part['slope_bounds'][float(SELECTED_SLOPE_QUANTILE)]
            pred_selected = causal_slope_clip_by_well(frame, pred_selected, slope_bound)
        stage_predictions = {
            'constant_anchor': anchor,
            'delta_clipped': pred_delta_clipped,
            'global_alpha': pred_global_alpha,
            'fade_in': pred_fade_in,
            'selected_policy': pred_selected,
        }
        for stage_name, pred in stage_predictions.items():
            valid = np.isfinite(y) & np.isfinite(pred)
            err = y[valid] - pred[valid]
            stage_sse[stage_name] += float(np.dot(err, err))
            stage_n[stage_name] += int(valid.sum())
        diag = pd.DataFrame({
            'well_id': frame['well_id'].to_numpy(),
            'md_since_ps': frame['md_since_ps'].to_numpy(dtype=float),
            'tail_row_number': frame['tail_row_number'].to_numpy(dtype=float),
            'target_delta': frame['target_delta_from_last_known'].to_numpy(dtype=float),
            'constant_error_sq': (y - anchor) ** 2,
            'selected_error_sq': (y - pred_selected) ** 2,
        })
        diag['md_bin'] = pd.cut(diag['md_since_ps'], bins=md_bins, include_lowest=True)
        md_group = diag.groupby('md_bin', observed=False).agg(
            rows=('selected_error_sq', 'size'),
            constant_sse=('constant_error_sq', 'sum'),
            selected_sse=('selected_error_sq', 'sum'),
            target_delta_std=('target_delta', 'std'),
        ).reset_index()
        tail_md_bin_rows.append(md_group)
        diag['tail_row_bin'] = pd.cut(
            diag['tail_row_number'],
            bins=row_bins,
            labels=row_labels,
            include_lowest=True,
        )
        row_group = diag.groupby('tail_row_bin', observed=False).agg(
            rows=('selected_error_sq', 'size'),
            constant_sse=('constant_error_sq', 'sum'),
            selected_sse=('selected_error_sq', 'sum'),
            target_delta_std=('target_delta', 'std'),
        ).reset_index()
        tail_row_bin_rows.append(row_group)
        well_group = diag.groupby('well_id', as_index=False).agg(
            rows=('selected_error_sq', 'size'),
            constant_sse=('constant_error_sq', 'sum'),
            selected_sse=('selected_error_sq', 'sum'),
            target_delta_min=('target_delta', 'min'),
            target_delta_max=('target_delta', 'max'),
            target_delta_std=('target_delta', 'std'),
            max_md_since_ps=('md_since_ps', 'max'),
        )
        well_gain_rows.append(well_group)

    selected_policy_stage_report = pd.DataFrame([
        {
            'stage': stage_name,
            'rows': stage_n[stage_name],
            'rmse': float(np.sqrt(stage_sse[stage_name] / stage_n[stage_name])),
        }
        for stage_name in stage_sse
    ])
    constant_rmse = float(selected_policy_stage_report.loc[
        selected_policy_stage_report['stage'].eq('constant_anchor'),
        'rmse',
    ].iloc[0])
    selected_policy_stage_report['gain_vs_constant'] = constant_rmse - selected_policy_stage_report['rmse']


    def combine_bin_diagnostics(rows, bin_col):
        out = pd.concat(rows, ignore_index=True).groupby(bin_col, observed=False).agg(
            rows=('rows', 'sum'),
            constant_sse=('constant_sse', 'sum'),
            selected_sse=('selected_sse', 'sum'),
            target_delta_std=('target_delta_std', 'mean'),
        ).reset_index()
        out['constant_rmse'] = np.sqrt(out['constant_sse'] / out['rows'])
        out['selected_rmse'] = np.sqrt(out['selected_sse'] / out['rows'])
        out['gain_vs_constant'] = out['constant_rmse'] - out['selected_rmse']
        out[bin_col] = out[bin_col].astype(str)
        return out[[bin_col, 'rows', 'constant_rmse', 'selected_rmse', 'gain_vs_constant', 'target_delta_std']]

    selected_policy_tail_md_diagnostics = combine_bin_diagnostics(tail_md_bin_rows, 'md_bin')
    selected_policy_tail_row_diagnostics = combine_bin_diagnostics(tail_row_bin_rows, 'tail_row_bin')

    selected_policy_well_gain = pd.concat(well_gain_rows, ignore_index=True)
    selected_policy_well_gain['constant_rmse'] = np.sqrt(
        selected_policy_well_gain['constant_sse'] / selected_policy_well_gain['rows']
    )
    selected_policy_well_gain['selected_rmse'] = np.sqrt(
        selected_policy_well_gain['selected_sse'] / selected_policy_well_gain['rows']
    )
    selected_policy_well_gain['gain_vs_constant'] = (
        selected_policy_well_gain['constant_rmse'] - selected_policy_well_gain['selected_rmse']
    )
    selected_policy_well_gain['target_delta_range'] = (
        selected_policy_well_gain['target_delta_max'] - selected_policy_well_gain['target_delta_min']
    )

    well_gain_distribution = selected_policy_well_gain['gain_vs_constant'].describe(
        percentiles=[0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
    ).to_frame('gain_vs_constant')
    well_gain_distribution.loc['improved_well_count', 'gain_vs_constant'] = float((selected_policy_well_gain['gain_vs_constant'] > 0).sum())
    well_gain_distribution.loc['hurt_well_count', 'gain_vs_constant'] = float((selected_policy_well_gain['gain_vs_constant'] < 0).sum())

    report_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else DATA_DIR
    selected_policy_stage_report.to_csv(report_dir / 'v7_selected_policy_stage_report.csv', index=False)
    selected_policy_tail_md_diagnostics.to_csv(report_dir / 'v7_selected_policy_tail_md_diagnostics.csv', index=False)
    selected_policy_tail_row_diagnostics.to_csv(report_dir / 'v7_selected_policy_tail_row_diagnostics.csv', index=False)
    selected_policy_well_gain.to_csv(report_dir / 'v7_selected_policy_well_gain.csv', index=False)

    print('Selected diagnostic feature set:', SELECTED_FEATURE_SET)
    print('Selected diagnostic reports written to:', report_dir)
    display(selected_policy_stage_report.style.format({
        'rmse': '{:.4f}',
        'gain_vs_constant': '{:.4f}',
    }))
    display(selected_policy_tail_md_diagnostics.style.format({
        'constant_rmse': '{:.4f}',
        'selected_rmse': '{:.4f}',
        'gain_vs_constant': '{:.4f}',
        'target_delta_std': '{:.4f}',
    }))
    display(well_gain_distribution.style.format({'gain_vs_constant': '{:.4f}'}))
    display(selected_policy_well_gain.sort_values('gain_vs_constant').head(8)[[
        'well_id',
        'rows',
        'constant_rmse',
        'selected_rmse',
        'gain_vs_constant',
        'target_delta_range',
    ]].style.format({
        'constant_rmse': '{:.4f}',
        'selected_rmse': '{:.4f}',
        'gain_vs_constant': '{:.4f}',
        'target_delta_range': '{:.4f}',
    }))


### 16.3 Optional Residual-Pipeline Prediction Tables

This section belongs to the lighter HGB residual pipeline used for feature-policy debugging. It is kept as a reproducible diagnostic path, while the final scoring path is the DTW super-stack engine in Section 18.

### Prediction rule

For each test row, the diagnostic prediction starts from `last_known_TVT` and adds a bounded residual correction. The correction can be clipped, globally shrunk, faded in near Prediction Start, and passed through an anchor-aware slope limiter.

### Diagnostic outputs

| Prediction table | Scope | Meaning |
|---|---:|---|
| strict residual diagnostic | all train wells | conservative feature-policy sanity check |
| overlap-excluded diagnostic | optional | checks behavior when visible overlap wells are excluded |
| offline residual diagnostic | all train wells | batch-feature sanity check before the final super-stack path |


In [ ]:
# Fit residual models and write id/tvt prediction files.

required_selection_vars = [
    'SELECTED_FEATURE_SET',
    'SELECTED_POLICY_METRIC',
    'SELECTED_SHRINKAGE_ALPHA',
    'SELECTED_FADE_IN_TAU_MD',
    'SELECTED_SLOPE_QUANTILE',
    'APPLY_SELECTED_SLOPE_CLIP',
    'BEST_OVERALL_FEATURE_SET',
    'BEST_OVERALL_SHRINKAGE_ALPHA',
    'BEST_OVERALL_FADE_IN_TAU_MD',
    'BEST_OVERALL_SLOPE_QUANTILE',
    'APPLY_BEST_OVERALL_SLOPE_CLIP',
]
missing_selection_vars = [name for name in required_selection_vars if name not in globals()]
if missing_selection_vars:
    raise RuntimeError(f'CV selection variables are missing: {missing_selection_vars}. Run the validation cell before submission construction.')

all_train_wells = np.array(sorted(train_h_ids))

final_model_settings = pd.Series({
    'selected_feature_set': SELECTED_FEATURE_SET,
    'selected_feature_policy': FEATURE_SET_POLICY[SELECTED_FEATURE_SET],
    'selected_feature_count': len(FEATURE_SETS[SELECTED_FEATURE_SET]),
    'selected_policy_metric': SELECTED_POLICY_METRIC,
    'selected_shrinkage_alpha': SELECTED_SHRINKAGE_ALPHA,
    'selected_fade_in_tau_md': SELECTED_FADE_IN_TAU_MD,
    'apply_causal_slope_clip': APPLY_SELECTED_SLOPE_CLIP,
    'selected_slope_quantile': SELECTED_SLOPE_QUANTILE,
    'best_overall_feature_set': BEST_OVERALL_FEATURE_SET,
    'best_overall_feature_policy': FEATURE_SET_POLICY[BEST_OVERALL_FEATURE_SET],
    'best_overall_shrinkage_alpha': BEST_OVERALL_SHRINKAGE_ALPHA,
    'best_overall_fade_in_tau_md': BEST_OVERALL_FADE_IN_TAU_MD,
    'best_overall_apply_slope_clip': APPLY_BEST_OVERALL_SLOPE_CLIP,
    'best_overall_slope_quantile': BEST_OVERALL_SLOPE_QUANTILE,
}).to_frame('value')
display(final_model_settings)

test_overlap_wells = np.array(sorted(set(train_h_ids) & set(test_h_ids)))
overlap_excluded_train_wells = np.array(sorted(set(all_train_wells) - set(test_overlap_wells)))
prediction_file_summaries = []


def train_and_predict_submission(
    training_wells,
    output_path: Path,
    label: str,
    feature_set_name: str,
    alpha: float,
    fade_tau_md: float | None,
    slope_quantile: float | None,
    apply_slope_clip: bool,
):
    feature_columns = FEATURE_SETS[feature_set_name]
    final_plane_imputer = final_row_imputer = None
    if feature_columns_require_formation(feature_columns):
        final_plane_imputer, final_row_imputer = make_formation_imputers(
            training_wells,
            TRAIN_DIR,
            need_row_ancc=feature_columns_require_row_ancc(feature_columns),
            seed=MODEL_CONFIG['random_state'] + 999,
        )
    train_frame = build_tail_feature_frame(
        training_wells,
        TRAIN_DIR,
        include_target=True,
        rows_per_well=MODEL_CONFIG['final_train_rows_per_well'],
        random_state=MODEL_CONFIG['random_state'] + 999,
        use_beam_features=feature_columns_require_beam(feature_columns),
        keep_columns=feature_columns,
        formation_plane_imputer=final_plane_imputer,
        row_ancc_imputer=final_row_imputer,
        imputer_well_ids=training_wells,
    )
    X_train = train_frame[feature_columns].copy()
    y_train_delta = train_frame['target_delta_from_last_known'].to_numpy(dtype=float)
    model = make_residual_model(random_state=MODEL_CONFIG['random_state'] + 999)
    model.fit(X_train, y_train_delta)
    
    if not test_horizontal_files:
        prediction_file_summaries.append({
            'label': label,
            'feature_set': feature_set_name,
            'train_wells': len(training_wells),
            'train_rows': len(train_frame),
            'prediction_rows': 0,
            'output_file': None,
            'missing_predictions': np.nan,
        })
        return None
    
    test_wells = sorted(test_h_ids)
    test_frame = build_tail_feature_frame(
        test_wells,
        TEST_DIR,
        include_target=False,
        rows_per_well=None,
        random_state=MODEL_CONFIG['random_state'] + 2026,
        use_beam_features=feature_columns_require_beam(feature_columns),
        keep_columns=feature_columns,
        formation_plane_imputer=final_plane_imputer,
        row_ancc_imputer=final_row_imputer,
        imputer_well_ids=training_wells,
    )
    test_delta_raw = model.predict(test_frame[feature_columns])
    test_delta_clipped = clip_delta_by_train_quantiles(test_delta_raw, y_train_delta)
    test_delta_final = shrink_delta(
        fade_in_delta(test_frame, test_delta_clipped, fade_tau_md),
        alpha=alpha,
    )
    test_pred = test_frame['last_known_TVT'].to_numpy(dtype=float) + test_delta_final
    max_abs_slope = np.nan
    if apply_slope_clip and slope_quantile is not None:
        training_slope_paths = [TRAIN_DIR / f'{well_id}__horizontal_well.csv' for well_id in training_wells]
        training_slope_bounds = estimate_abs_tvt_slope_quantiles(
            training_slope_paths,
            quantiles=[float(slope_quantile)],
        )
        max_abs_slope = training_slope_bounds[float(slope_quantile)]
        test_pred = causal_slope_clip_by_well(test_frame, test_pred, max_abs_slope)
    submission = pd.DataFrame({'id': test_frame['id'].to_numpy(), 'tvt': test_pred})
    
    if SAMPLE_SUBMISSION.exists():
        sample = pd.read_csv(SAMPLE_SUBMISSION)
        submission = sample[['id']].merge(submission, on='id', how='left')
        missing = int(submission['tvt'].isna().sum())
        if missing:
            missing_ids = submission.loc[submission['tvt'].isna(), 'id'].head(10).tolist()
            raise ValueError(f'Missing predictions after sample alignment: {missing}. First missing ids: {missing_ids}')
    else:
        missing = int(submission['tvt'].isna().sum())
    
    if MODEL_CONFIG['write_submission']:
        submission.to_csv(output_path, index=False)
    prediction_file_summaries.append({
        'label': label,
        'feature_set': feature_set_name,
        'feature_policy': FEATURE_SET_POLICY[feature_set_name],
        'alpha': alpha,
        'fade_tau_md': fade_tau_md,
        'slope_quantile': slope_quantile,
        'apply_slope_clip': apply_slope_clip,
        'max_abs_slope_used': max_abs_slope,
        'train_wells': len(training_wells),
        'train_rows': len(train_frame),
        'target_delta_mean': float(np.mean(y_train_delta)),
        'target_delta_std': float(np.std(y_train_delta)),
        'prediction_rows': len(submission),
        'missing_predictions': missing,
        'tvt_mean': float(submission['tvt'].mean()),
        'tvt_std': float(submission['tvt'].std()),
        'tvt_min': float(submission['tvt'].min()),
        'tvt_max': float(submission['tvt'].max()),
        'output_file': output_path.name,
    })
    return submission

KAGGLE_WORKING_DIR = Path('/kaggle/working')
OUTPUT_DIR = KAGGLE_WORKING_DIR if KAGGLE_WORKING_DIR.exists() else DATA_DIR
selected_main_file = (
    'submission_hgb_strict_v7.csv'
    if KAGGLE_WORKING_DIR.exists()
    else 'submission_simple_residual_v7.csv'
)
selected_overlap_excluded_file = (
    'submission_hgb_overlap_excluded_v7.csv'
    if KAGGLE_WORKING_DIR.exists()
    else 'submission_simple_residual_overlap_excluded_v7.csv'
)
best_overall_file = (
    'submission_hgb_offline_candidate_path_v7.csv'
    if KAGGLE_WORKING_DIR.exists()
    else 'submission_simple_residual_best_overall_v7.csv'
)

if RUN_HGB_DIAGNOSTIC_SUBMISSIONS:
    submission_all_train = train_and_predict_submission(
        all_train_wells,
        OUTPUT_DIR / selected_main_file,
        'selected_all_train',
        SELECTED_FEATURE_SET,
        SELECTED_SHRINKAGE_ALPHA,
        SELECTED_FADE_IN_TAU_MD,
        SELECTED_SLOPE_QUANTILE,
        APPLY_SELECTED_SLOPE_CLIP,
    )

    if MODEL_CONFIG['write_overlap_excluded_diagnostic'] and len(overlap_excluded_train_wells):
        submission_overlap_excluded = train_and_predict_submission(
            overlap_excluded_train_wells,
            OUTPUT_DIR / selected_overlap_excluded_file,
            'selected_overlap_excluded',
            SELECTED_FEATURE_SET,
            SELECTED_SHRINKAGE_ALPHA,
            SELECTED_FADE_IN_TAU_MD,
            SELECTED_SLOPE_QUANTILE,
            APPLY_SELECTED_SLOPE_CLIP,
        )

    if BEST_OVERALL_FEATURE_SET != SELECTED_FEATURE_SET:
        submission_best_overall = train_and_predict_submission(
            all_train_wells,
            OUTPUT_DIR / best_overall_file,
            'best_overall_all_train',
            BEST_OVERALL_FEATURE_SET,
            BEST_OVERALL_SHRINKAGE_ALPHA,
            BEST_OVERALL_FADE_IN_TAU_MD,
            BEST_OVERALL_SLOPE_QUANTILE,
            APPLY_BEST_OVERALL_SLOPE_CLIP,
        )

else:
    print('HGB diagnostic submissions skipped in memory-safe Kaggle mode.')
    submission_all_train = None

prediction_file_summary = pd.DataFrame(prediction_file_summaries)
prediction_file_summary.to_csv(OUTPUT_DIR / 'v7_prediction_file_summary.csv', index=False)
display(prediction_file_summary)
if submission_all_train is not None:
    display(submission_all_train.head())


### 16.4 Extra Signals

### 🧩 Pipeline pieces

| Piece | Use |
|---|---|
| Well files | trajectory, GR, prefix TVT, typewell reference |
| Feature policy | strict vs offline inputs |
| Feature table | leakage-checked row covariates |
| Residual model | movement away from `last_known_TVT` |
| Model stack | combine learners on the same matrix |
| Postprocess | shrink, fade, smooth, guard |
| Prediction file | aligned `id,tvt` table |

### 🧬 Candidate families

| Family | Use | Guardrail |
|---|---|---|
| 🧭 Candidate paths | plausible tail-end TVT shifts | no true tail TVT |
| 🪨 Formation plane/KNN | spatial structure | imputer-derived only |
| 🌐 Row ANCC / dense surface | local surface correction | train-derived reference |
| 📡 Beam alignment | sequence-constrained GR matching | hidden GR only |
| 🪞 Self-correlation | same-well GR motifs | hidden GR only |
| 🧵 DTW | stretched/compressed GR motifs | hidden GR only |
| 🌲 LGB/CatBoost | nonlinear residual interactions | same feature policy |
| 🧯 Smoothing | curve regularity | after prediction |


## 17. Offline Candidate-Path Feature Check

This section checks batch-only feature families before the final super-stack engine. The competition-facing prediction is still produced in Section 18.

### 🌐 Offline signal map

| Addition | Inputs | Use |
|---|---:|---|
| Recent prefix step/slope | ✅ strict | short-horizon drift prior |
| Typewell anchor residuals | ✅ strict | current GR vs nearby TVT anchors |
| Position / GR distribution | 🌐 offline | hidden interval shape |
| Centered / lead-lag GR | 🌐 offline | local GR context |
| Candidate paths | 🌐 offline | plausible tail endpoint shifts |
| Formation plane | 🌐 reference | projected geology surface |
| Beam alignment | 🌐 offline | constrained GR path |
| Strong tabular models | ✅/🌐 same policy | residual learner diversity |

### 🛡️ Guardrails

| Feature family | Future GR? | Future TVT? | Direct test formations? |
|---|---:|---:|---:|
| Candidate paths | ✅ | 🚫 | 🚫 |
| Beam paths | ✅ | 🚫 | 🚫 |
| Formation outputs | target-free coordinates | 🚫 | 🚫 |

⚙️ Beam and strong-model checks are controlled separately because repeated grouped construction is expensive.


In [ ]:
# Optional comparison: offline candidate-path features with a stronger residual model.

RUN_V7_STRONG_MODEL_EXPERIMENT = False

V7_STRONG_MODEL_CONFIG = {
    'feature_sets': ['offline_candidate_path_alignment'],
    'models': ['lightgbm'],
    'folds_to_run': min(2, MODEL_CONFIG['cv_folds_to_run']),
    'train_rows_per_well': 350,
    'random_state': MODEL_CONFIG['random_state'] + 2606,
}


def available_strong_models() -> dict[str, object]:
    models = {}
    try:
        from lightgbm import LGBMRegressor
        models['lightgbm'] = LGBMRegressor
    except Exception as exc:
        print(f'LightGBM unavailable: {exc}')
    try:
        from xgboost import XGBRegressor
        models['xgboost'] = XGBRegressor
    except Exception as exc:
        print(f'XGBoost unavailable: {exc}')
    try:
        from catboost import CatBoostRegressor
        models['catboost'] = CatBoostRegressor
    except Exception as exc:
        print(f'CatBoost unavailable: {exc}')
    return models


def make_strong_residual_model(model_name: str, random_state: int):
    registry = available_strong_models()
    if model_name == 'lightgbm' and 'lightgbm' in registry:
        LGBMRegressor = registry['lightgbm']
        params = dict(
            objective='regression_l2',
            metric='rmse',
            n_estimators=500,
            learning_rate=0.035,
            num_leaves=63,
            min_child_samples=80,
            subsample=0.85,
            subsample_freq=1,
            colsample_bytree=0.80,
            reg_alpha=0.05,
            reg_lambda=1.0,
            random_state=random_state,
            n_jobs=-1,
            verbose=-1,
        )
        params.update(lightgbm_accelerator_params())
        return LGBMRegressor(**params)
    if model_name == 'xgboost' and 'xgboost' in registry:
        XGBRegressor = registry['xgboost']
        params = dict(
            objective='reg:squarederror',
            eval_metric='rmse',
            n_estimators=700,
            learning_rate=0.035,
            max_depth=7,
            min_child_weight=20,
            subsample=0.80,
            colsample_bytree=0.80,
            reg_alpha=1.0,
            reg_lambda=20.0,
            tree_method='hist',
            random_state=random_state,
            n_jobs=-1,
        )
        if LIGHTGBM_DEVICE_TYPE == 'gpu':
            params['device'] = 'cuda'
        return XGBRegressor(**params)
    if model_name == 'catboost' and 'catboost' in registry:
        CatBoostRegressor = registry['catboost']
        return CatBoostRegressor(
            loss_function='RMSE',
            iterations=900,
            learning_rate=0.035,
            depth=8,
            l2_leaf_reg=8.0,
            random_seed=random_state,
            task_type='GPU' if LIGHTGBM_DEVICE_TYPE == 'gpu' else 'CPU',
            verbose=False,
            allow_writing_files=False,
        )
    raise ValueError(f'Model is not available or not configured: {model_name}')


def run_v7_strong_model_cv() -> pd.DataFrame:
    available = available_strong_models()
    rows = []
    strong_fold_parts = []
    selected_feature_sets = [name for name in V7_STRONG_MODEL_CONFIG['feature_sets'] if name in FEATURE_SETS]
    selected_models = [name for name in V7_STRONG_MODEL_CONFIG['models'] if name in available]
    if not selected_feature_sets or not selected_models:
        return pd.DataFrame(columns=['model', 'feature_set', 'rmse', 'alpha', 'fade_tau_md', 'slope_quantile', 'n_rows'])

    for feature_name in selected_feature_sets:
        feature_cols = FEATURE_SETS[feature_name]
        for model_name in selected_models:
            fold_parts = []
            for fold_idx, (train_idx, valid_idx) in enumerate(fold_splits[:V7_STRONG_MODEL_CONFIG['folds_to_run']], start=1):
                train_wells = all_train_wells[train_idx]
                valid_wells = all_train_wells[valid_idx]
                fold_plane_imputer = fold_row_imputer = None
                if feature_columns_require_formation(feature_cols):
                    fold_plane_imputer, fold_row_imputer = make_formation_imputers(
                        train_wells,
                        TRAIN_DIR,
                        need_row_ancc=feature_columns_require_row_ancc(feature_cols),
                        seed=V7_STRONG_MODEL_CONFIG['random_state'] + fold_idx,
                    )
                train_frame = build_tail_feature_frame(
                    train_wells,
                    TRAIN_DIR,
                    include_target=True,
                    rows_per_well=V7_STRONG_MODEL_CONFIG['train_rows_per_well'],
                    random_state=V7_STRONG_MODEL_CONFIG['random_state'] + fold_idx,
                    use_beam_features=feature_columns_require_beam(feature_cols),
                    keep_columns=feature_cols,
                    formation_plane_imputer=fold_plane_imputer,
                    row_ancc_imputer=fold_row_imputer,
                    imputer_well_ids=train_wells,
                )
                valid_frame = build_tail_feature_frame(
                    valid_wells,
                    TRAIN_DIR,
                    include_target=True,
                    rows_per_well=None,
                    random_state=V7_STRONG_MODEL_CONFIG['random_state'] + fold_idx,
                    use_beam_features=feature_columns_require_beam(feature_cols),
                    keep_columns=feature_cols,
                    formation_plane_imputer=fold_plane_imputer,
                    row_ancc_imputer=fold_row_imputer,
                    imputer_well_ids=train_wells,
                )
                model = make_strong_residual_model(model_name, V7_STRONG_MODEL_CONFIG['random_state'] + fold_idx)
                X_train, y_train_delta, _ = model_ready_xy(train_frame, feature_cols)
                X_valid, _, _ = model_ready_xy(valid_frame, feature_cols)
                model.fit(X_train, y_train_delta)
                raw_delta = model.predict(X_valid)
                clipped_delta = clip_delta_by_train_quantiles(raw_delta, y_train_delta)
                fold_train_paths = [TRAIN_DIR / f'{well_id}__horizontal_well.csv' for well_id in train_wells]
                slope_bounds = estimate_abs_tvt_slope_quantiles(fold_train_paths, quantiles=slope_quantiles)
                fold_parts.append({
                    'fold': fold_idx,
                    'frame': valid_frame,
                    'raw_delta': np.asarray(raw_delta, dtype=float),
                    'clipped_delta': np.asarray(clipped_delta, dtype=float),
                    'slope_bounds': slope_bounds,
                })
                rows.append({
                    'model': model_name,
                    'feature_set': feature_name,
                    'fold': fold_idx,
                    'n_train_rows': len(train_frame),
                    'n_valid_rows': len(valid_frame),
                    'raw_rmse': rmse(
                        valid_frame['target_tvt'].to_numpy(dtype=float),
                        valid_frame['last_known_TVT'].to_numpy(dtype=float) + raw_delta,
                    ),
                    'clipped_rmse': rmse(
                        valid_frame['target_tvt'].to_numpy(dtype=float),
                        valid_frame['last_known_TVT'].to_numpy(dtype=float) + clipped_delta,
                    ),
                })

            policy_rows = []
            for tau in MODEL_CONFIG['fade_in_tau_md_to_compare']:
                alpha, _ = fit_global_alpha_from_fold_parts(fold_parts, tau_md=tau)
                score = score_policy_from_fold_parts(
                    fold_parts,
                    alpha=alpha,
                    tau_md=tau,
                    slope_quantile=None,
                )
                score.update({'alpha': alpha, 'tau_md': tau, 'slope_quantile': np.nan})
                policy_rows.append(score)
                for slope_q in MODEL_CONFIG['slope_clip_quantiles_to_compare']:
                    score = score_policy_from_fold_parts(
                        fold_parts,
                        alpha=alpha,
                        tau_md=tau,
                        slope_quantile=float(slope_q),
                    )
                    score.update({'alpha': alpha, 'tau_md': tau, 'slope_quantile': float(slope_q)})
                    policy_rows.append(score)
            policy = pd.DataFrame(policy_rows).sort_values('rmse').reset_index(drop=True)
            best = policy.iloc[0].to_dict()
            strong_fold_parts.append({
                'model': model_name,
                'feature_set': feature_name,
                'fold_parts': fold_parts,
                'best_policy': best,
            })
            rows.append({
                'model': model_name,
                'feature_set': feature_name,
                'fold': 'global_policy',
                'n_train_rows': np.nan,
                'n_valid_rows': int(sum(len(part['frame']) for part in fold_parts)),
                'raw_rmse': np.nan,
                'clipped_rmse': np.nan,
                'best_policy_rmse': best['rmse'],
                'best_policy_alpha': best['alpha'],
                'best_policy_fade_tau_md': best['tau_md'],
                'best_policy_slope_quantile': best['slope_quantile'],
            })

    result = pd.DataFrame(rows)
    globals()['v7_strong_fold_parts'] = strong_fold_parts
    return result


if RUN_V7_STRONG_MODEL_EXPERIMENT:
    v7_strong_model_report = run_v7_strong_model_cv()
else:
    v7_strong_model_report = pd.DataFrame()

if len(v7_strong_model_report):
    v7_strong_model_report.to_csv(OUTPUT_DIR / 'v7_strong_model_cv_report.csv', index=False)
    display(v7_strong_model_report.tail(10))
else:
    print('No v7 strong-model experiment was run. Check availability and configuration.')


In [ ]:
from pathlib import Path
from IPython.display import Image, display

figure_image_path = Path("/kaggle/input/datasets/pilkwang/pilkwang-public-dataset-for-notebooks-figures/ROGII_Graph_Fig5.png")
if figure_image_path.exists():
    display(Image(filename=str(figure_image_path)))


**Figure 14. Particle-filter tracking of hidden TVT.**  
Each particle represents a possible hidden TVT path. Particles evolve under a smooth transition prior and are weighted by GR likelihood against the typewell. The final PF estimate is a likelihood-weighted ensemble across many fixed seeds.


In [ ]:
from pathlib import Path
from IPython.display import Image, display

figure_image_path = Path("/kaggle/input/datasets/pilkwang/pilkwang-public-dataset-for-notebooks-figures/ROGII_Graph_Fig6.png")
if figure_image_path.exists():
    display(Image(filename=str(figure_image_path)))


**Figure 15. Multi-beam typewell alignment.**  
Beam search finds smooth candidate paths through the typewell GR curve. Multiple beam configurations encode different assumptions about GR mismatch tolerance, smoothness, and allowed stratigraphic movement.


## 18. Super Stack Submission Engine

### 🧭 Core idea

- ⚓ Predict residuals: `TVT - last_known_TVT`.
- 🧬 Build several target-free pseudo-TVT paths.
- 🌲 Train LGB seeds + CatBoost on one shared feature matrix.
- 🏁 Compare single, average, ridge, and sparse hill-climb stacks.
- 🧯 Apply light residual calibration, optional PF/DTW mixing, and smoothing.

### 🧩 Signal map

| Signal | Intuition | Confidence clue |
|---|---|---|
| 📡 Beam | smooth GR-to-typewell path | beam spread |
| 🧵 DTW | stretched/compressed GR motifs | DTW cost + stochastic spread |
| 🪞 Self-corr | hidden GR matched to prefix GR | correlation score |
| 🪨 Formation planes | spatial geology surfaces | plane distance + six-surface agreement |
| 🧲 Dense ANCC | local row-level surface | neighbor distance/std |
| 🧭 PF | smooth TVT / ANCC state | posterior spread |
| 🛤️ Trajectory | `MD`, `Z`, local movement | slope consistency |

### ⚓ Residual target

$$
\hat{TVT}_{w,i}=T_{w,L}+\hat{\Delta}_{w,i}
$$

- `T_{w,L}` keeps flat wells stable.
- `Δ` lets the model correct drifting wells.

### 🪨 Formation formula

$$
TVT_i \approx -Z_i + \widehat{S}_{F}(X_i,Y_i) + b_{w,F}
$$

- 🧭 `S_F`: local plane for each formation surface.
- 📍 `b_w,F`: prefix offset estimated from known rows.
- 🌐 Six formations give competing geology hypotheses.

### 🧵 DTW block

$$
D(i,j)=(GR_i^h-GR_j^{tw})^2+
\min\{D(i-1,j-1),D(i-1,j),D(i,j-1)\}
$$

- multi-radius Sakoe-Chiba bands
- cost-weighted DTW ensemble
- stochastic DTW spread as uncertainty
- typewell residual offsets around the DTW path
- disagreement: `dtw_vs_beam`, `dtw_vs_pf`, `dtw_vs_sc`

### 🏁 Stack and guard

| Step | Action |
|---|---|
| 🌲 LGB/CatBoost | learn nonlinear signal reliability |
| ➕ Ridge | positive-weight blend |
| 🪜 Hill-climb | sparse blend that can ignore weak learners |
| 🧯 Postprocess | shrink, fade, PF/DTW mix, smooth |
| ✅ Contract guard | enforce `id,tvt`, row count, order, finite values |


In [ ]:
# Super-stack final submission engine.
RUN_SUPER_STACK_SOLUTION = (
    bool(KAGGLE_NOTEBOOK_RUN)
    and not bool(globals().get('RUN_RIDGE_ARTIFACT_PROFILE', False))
    and not bool(globals().get('RUN_PF_RESIDUAL_GBDT_ONLY', False))
)
SUPER_STACK_SUBMISSION_OUTPUT = FINAL_SUBMISSION_OUTPUT

if not RUN_SUPER_STACK_SOLUTION:
    if bool(globals().get('RUN_RIDGE_ARTIFACT_PROFILE', False)):
        print(f"Super-stack final solution is skipped for {globals().get('SUBMISSION_PROFILE', 'ridge_artifact_exact')} profile.")
    elif bool(globals().get('RUN_PF_RESIDUAL_GBDT_ONLY', False)):
        print(f"Super-stack final solution is skipped for {globals().get('SUBMISSION_PROFILE', 'pf_residual_gbdt_exact')} profile.")
    else:
        print('Super-stack final solution is skipped outside Kaggle submission runs.')
else:
    # ─ Imports & Config ──────────────────────────────────────────────
    from pathlib import Path
    from scipy.interpolate import interp1d
    from scipy.spatial import cKDTree
    from scipy.signal import savgol_filter
    from sklearn.model_selection import GroupKFold
    from sklearn.linear_model import Ridge
    from sklearn.metrics import root_mean_squared_error
    try:
        from numba import njit
    except Exception:
        def njit(*args, **kwargs):
            def _decorator(func):
                return func
            return _decorator
    if not bool(globals().get('RUN_FAST_PF_SELECTOR_ONLY', False)):
        from catboost import CatBoostRegressor, Pool
        import lightgbm as lgb
    else:
        CatBoostRegressor = Pool = None
        lgb = None
    from joblib import Parallel, delayed
    import numpy as np, pandas as pd
    import glob, gc, time, multiprocessing, warnings, json
    warnings.filterwarnings("ignore")

    SEED = 42; np.random.seed(SEED)
    NCPU = 1  # DTW/PF feature building is RAM-bound; serial builds are safer and deterministic.

    def stable_seed(wid, salt=SEED):
        return int((sum((i + 1) * ord(ch) for i, ch in enumerate(str(wid))) + int(salt)) % (2**32 - 1))

    def _gpu_names():
        import subprocess
        try:
            out = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True, check=False).stdout.strip()
        except Exception:
            out = ""
        return [line.strip() for line in out.splitlines() if line.strip()]

    REFERENCE_GPU_NAMES = _gpu_names()
    if KAGGLE_NOTEBOOK_RUN and not REFERENCE_GPU_NAMES and not bool(globals().get('RUN_FAST_PF_SELECTOR_ONLY', False)):
        raise RuntimeError("Super-stack final solution requires a Kaggle GPU accelerator.")

    def _find():
        if 'DATA_DIR' in globals() and (Path(DATA_DIR) / "train").exists():
            return Path(DATA_DIR)
        for p in [Path("/kaggle/input/rogii-wellbore-geology-prediction"),
                  Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")]:
            if (p / "train").exists():
                return p
        kaggle_input = Path("/kaggle/input")
        if kaggle_input.exists():
            for p in kaggle_input.glob("*/sample_submission.csv"):
                return p.parent
        local = Path(".")
        if (local / "train").exists() and (local / "sample_submission.csv").exists():
            return local
        raise FileNotFoundError("Data not found")

    DATA      = _find()
    TRAIN_DIR = DATA / "train"
    TEST_DIR  = DATA / "test"
    SAMPLE    = DATA / "sample_submission.csv"
    SUPER_STACK_SUBMISSION_OUTPUT = FINAL_SUBMISSION_OUTPUT
    OUT       = FINAL_SUBMISSION_OUTPUT

    FORMATIONS = ["ANCC","ASTNU","ASTNL","EGFDU","EGFDL","BUDA"]
    PLANE_K    = 10          # centroid plane-fit neighbors
    DENSE_SPW  = 60          # dense samples per well (raised from 40)
    DENSE_K    = 20
    N_SPLITS   = 5

    # Beam configs: (beam_size, move_cost, emit_scale, smooth_r, tag)
    BEAMS = [
        (10, 20.0, 144.0, 2, "cons"),
        (10,  8.0,  64.0, 2, "loose"),
        ( 8, 35.0, 220.0, 1, "vcons"),
        (10, 14.0,  90.0, 5, "sm5"),
        (20,  4.0,  36.0, 3, "vloose"),
    ]

    # Particle filter (reduced particles for speed)
    PF_N  = 300;   ANCC_N = 300
    PF_MOM = 0.993; PF_VN  = 0.005; PF_PN  = 0.01
    PF_GR_SIG_MIN=10.; PF_GR_SIG_MAX=60.; PF_GR_SIG_DEF=30.
    PF_INIT_V_STD=0.02; PF_INIT_SPR=0.5; PF_RESAMP=0.5
    PF_ROUGH_P=0.2; PF_ROUGH_V=0.003; PF_GR_WIN=5; PF_GR_WT=0.3
    ANCC_ALPHA=0.998; ANCC_RN=0.002; ANCC_PN=0.005
    ANCC_IR=0.01; ANCC_IS=0.3; ANCC_RP=0.1; ANCC_RR=0.001

    # Constrained / stochastic DTW. These are the main v7 additions.
    DTW_RADII = (20, 50, 100)
    DTW_STRIDE = 3
    DTW_STOCH_K = 6
    DTW_STOCH_TEMP = 3.0

    # Model params
    LGB_P = dict(boosting_type="gbdt",learning_rate=0.04,num_leaves=127,
                 min_child_samples=20,subsample=0.8,colsample_bytree=0.8,
                 reg_lambda=5.,reg_alpha=0.1,objective="regression",
                 verbose=-1,n_jobs=-1,
                 device_type="gpu",gpu_use_dp=False,max_bin=255)
    LGB_SEEDS = [42, 7, 123]

    CB_P = dict(iterations=5000,learning_rate=0.04,depth=8,l2_leaf_reg=3.,
                min_data_in_leaf=20,loss_function="RMSE",
                random_seed=42,task_type="GPU",devices=("0:1" if len(REFERENCE_GPU_NAMES) >= 2 else "0"),
                od_type="Iter",od_wait=150,verbose=0)

    print("GPUs:", " | ".join(REFERENCE_GPU_NAMES) if REFERENCE_GPU_NAMES else "none")
    print(f"CPUs={NCPU}  train={len(list(TRAIN_DIR.glob('*__horizontal_well.csv')))} wells")


    # ─ Helpers + Beam Search ─────────────────────────────────────────
    def nn_idx(arr, v):
        i=int(np.searchsorted(arr,v,'left'))
        if i>=len(arr): return len(arr)-1
        if i>0 and abs(arr[i-1]-v)<=abs(arr[i]-v): return i-1
        return i

    def robust_slope(x, y, w=None):
        x=np.asarray(x,float); y=np.asarray(y,float)
        m=np.isfinite(x)&np.isfinite(y)
        if m.sum()<2: return 0.
        if np.std(x[m])<1e-6: return 0.
        return float(np.polyfit(x[m],y[m],1)[0])

    def affine_cal(kgr, tw_at_k, min_pts=20):
        v=np.isfinite(kgr)&np.isfinite(tw_at_k)
        if v.sum()<min_pts or np.std(tw_at_k[v])<1e-6:
            return 1., float(np.nanmean(kgr)-np.nanmean(tw_at_k)) if v.any() else 0.
        a,b=np.polyfit(tw_at_k[v],kgr[v],1)
        return float(a),float(b)

    def self_corr_tvt(kgr, ktvt, hgr, hw=15, stride=3):
        win=2*hw+1; nk=len(kgr); nh=len(hgr)
        if nk<win+1 or nh==0:
            return np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32)
        kg=pd.Series(kgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
        hg=pd.Series(hgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
        sts=np.arange(0,nk-win+1,stride,dtype=np.int32); M=len(sts)
        if M==0: return np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32)
        C=kg[sts[:,None]+np.arange(win,dtype=np.int32)[None,:]].astype(np.float32)
        Cn=(C-C.mean(1,keepdims=True))/(C.std(1,keepdims=True)+1e-6)
        hp=np.pad(hg,hw,mode='edge')
        H=hp[np.arange(nh)[:,None]+np.arange(win)[None,:]].astype(np.float32)
        Hn=(H-H.mean(1,keepdims=True))/(H.std(1,keepdims=True)+1e-6)
        ncc=Hn@Cn.T/win
        best=ncc.argmax(1); score=ncc.max(1).astype(np.float32)
        ctrs=np.clip(sts[best]+hw,0,nk-1)
        return ktvt[ctrs].astype(np.float32),score

    def beam_search(gr_h, tw_tvt, tw_gr, start_tvt, bs=10, mc=20., es=144., r=2):
        tw_tvt=np.asarray(tw_tvt,np.float32); tw_gr=np.asarray(tw_gr,np.float32)
        T=len(tw_tvt); fb=float(np.nanmean(tw_gr))
        sg=pd.Series(gr_h,dtype='float32').interpolate(limit_direction='both').fillna(fb)
        if r>0: sg=sg.rolling(r*2+1,center=True,min_periods=1).mean()
        sg=sg.to_numpy(np.float32)
        si=nn_idx(tw_tvt,start_tvt)
        bi=np.full(bs,si,np.int32); bc=np.zeros(bs,np.float64)
        ns=len(sg); bps=np.empty((ns,bs),np.int32); bpb=np.empty((ns,bs),np.int32)
        for s,gv in enumerate(sg):
            ci=np.clip(bi[:,None]+np.array([-1,0,1]),0,T-1)
            em=(gv-tw_gr[ci])**2/es; mv=mc*np.array([1,0,1])[None,:]
            cc=bc[:,None]+em+mv
            fi=ci.ravel(); fc=cc.ravel(); fp=np.repeat(np.arange(bs),3)
            ord=np.argsort(fc, kind='stable'); kept=[]; seen=set()
            for o in ord:
                t=int(fi[o])
                if t not in seen: seen.add(t); kept.append(o)
                if len(kept)==bs: break
            while len(kept)<bs: kept.append(kept[-1])
            kept=np.array(kept,np.int32)
            bps[s]=fp[kept]; bpb[s]=fi[kept]
            bi=fi[kept].astype(np.int32); bc=fc[kept]
        path=np.empty(ns,np.int32); cb=int(np.argmin(bc))
        for s in range(ns-1,-1,-1): path[s]=bpb[s,cb]; cb=bps[s,cb]
        return tw_tvt[path]


    @njit(cache=False)
    def _dtw_sakoe_chiba(query, ref, radius):
        N = len(query); M = len(ref)
        INF = 1e18
        D = np.full((N, M), INF)
        slope = (M - 1.0) / max(N - 1.0, 1.0)
        for i in range(N):
            j_center = int(round(i * slope))
            j_lo = max(0, j_center - radius)
            j_hi = min(M - 1, j_center + radius)
            for j in range(j_lo, j_hi + 1):
                cost = (query[i] - ref[j]) ** 2
                if i == 0 and j == 0:
                    D[i, j] = cost
                elif i == 0:
                    prev = D[i, j - 1]
                    D[i, j] = cost + (prev if prev < INF else INF)
                elif j == 0:
                    prev = D[i - 1, j]
                    D[i, j] = cost + (prev if prev < INF else INF)
                else:
                    a = D[i - 1, j - 1]
                    b = D[i - 1, j]
                    c = D[i, j - 1]
                    mn = a if a < b else b
                    mn = mn if mn < c else c
                    D[i, j] = cost + (mn if mn < INF else INF)
        i = N - 1; j = M - 1
        pi = np.zeros(N + M, np.int64)
        pj = np.zeros(N + M, np.int64)
        k = 0
        while i > 0 or j > 0:
            pi[k] = i; pj[k] = j; k += 1
            if i == 0:
                j -= 1
            elif j == 0:
                i -= 1
            else:
                a = D[i - 1, j - 1]
                b = D[i - 1, j]
                c = D[i, j - 1]
                if a <= b and a <= c:
                    i -= 1; j -= 1
                elif b <= c:
                    i -= 1
                else:
                    j -= 1
        pi[k] = 0; pj[k] = 0; k += 1
        return D, pi[:k], pj[:k]

    @njit(cache=False)
    def _dtw_path_to_tvt(pi, pj, tw_tvt, N):
        j_for_i = np.zeros(N, np.int64)
        for k in range(len(pi)):
            j_for_i[pi[k]] = pj[k]
        result = np.empty(N, np.float32)
        for i in range(N):
            result[i] = tw_tvt[j_for_i[i]]
        return result

    @njit(cache=False)
    def _dtw_path_slope(pi, pj, N, smooth_win=5):
        j_for_i = np.zeros(N, np.float64)
        for k in range(len(pi)):
            j_for_i[pi[k]] = float(pj[k])
        slope = np.zeros(N, np.float32)
        hw = smooth_win // 2
        for i in range(N):
            i0 = max(0, i - hw); i1 = min(N - 1, i + hw)
            if i1 > i0:
                slope[i] = float((j_for_i[i1] - j_for_i[i0]) / (i1 - i0))
            else:
                slope[i] = 1.0
        return slope

    @njit(cache=False)
    def _dtw_stochastic_realizations(query, ref, radius, K, temperature, seed):
        N = len(query); M = len(ref)
        INF = 1e18
        slope = (M - 1.0) / max(N - 1.0, 1.0)
        D_base = np.full((N, M), INF)
        for i in range(N):
            j_center = int(round(i * slope))
            j_lo = max(0, j_center - radius)
            j_hi = min(M - 1, j_center + radius)
            for j in range(j_lo, j_hi + 1):
                D_base[i, j] = (query[i] - ref[j]) ** 2
        np.random.seed(seed)
        paths = np.zeros((K, N), np.int64)
        for k in range(K):
            D = np.full((N, M), INF)
            for i in range(N):
                j_center = int(round(i * slope))
                j_lo = max(0, j_center - radius)
                j_hi = min(M - 1, j_center + radius)
                for j in range(j_lo, j_hi + 1):
                    u = np.random.random()
                    if u < 1e-12:
                        u = 1e-12
                    if u > 1.0 - 1e-12:
                        u = 1.0 - 1e-12
                    gumbel = -np.log(-np.log(u)) * temperature
                    cost = D_base[i, j] + gumbel
                    if i == 0 and j == 0:
                        D[i, j] = cost
                    elif i == 0:
                        prev = D[i, j - 1]
                        D[i, j] = cost + (prev if prev < INF else INF)
                    elif j == 0:
                        prev = D[i - 1, j]
                        D[i, j] = cost + (prev if prev < INF else INF)
                    else:
                        a = D[i - 1, j - 1]
                        b = D[i - 1, j]
                        c = D[i, j - 1]
                        mn = a if a < b else b
                        mn = mn if mn < c else c
                        D[i, j] = cost + (mn if mn < INF else INF)
            i = N - 1; j = M - 1
            j_for_i = np.zeros(N, np.int64)
            while i > 0 or j > 0:
                j_for_i[i] = j
                if i == 0:
                    j -= 1
                elif j == 0:
                    i -= 1
                else:
                    a = D[i - 1, j - 1]
                    b = D[i - 1, j]
                    c = D[i, j - 1]
                    if a <= b and a <= c:
                        i -= 1; j -= 1
                    elif b <= c:
                        i -= 1
                    else:
                        j -= 1
            j_for_i[0] = 0
            paths[k, :] = j_for_i
        return paths

    def _downsample_for_dtw(values, stride=DTW_STRIDE):
        n = len(values)
        if n == 0:
            return np.array([], dtype=np.int64), np.array([], dtype=np.float32)
        step = max(1, int(stride))
        idx = np.arange(0, n, step, dtype=np.int64)
        if idx[-1] != n - 1:
            idx = np.r_[idx, n - 1].astype(np.int64)
        return idx, np.asarray(values, dtype=np.float32)[idx]

    def _upsample_from_dtw(idx, values, n):
        if n == 0:
            return np.array([], dtype=np.float32)
        if len(idx) == 0 or len(values) == 0:
            return np.full(n, np.nan, dtype=np.float32)
        return np.interp(np.arange(n, dtype=np.float32), idx.astype(np.float32), np.asarray(values, dtype=np.float32)).astype(np.float32)

    def run_dtw_multiscale(query_gr, tw_tvt, tw_gr, last_tvt, radii=DTW_RADII):
        full_n = len(query_gr)
        idx, q = _downsample_for_dtw(query_gr, DTW_STRIDE)
        tw_idx, tw_gr_ds = _downsample_for_dtw(tw_gr, DTW_STRIDE)
        tw_tvt_ds = np.asarray(tw_tvt, dtype=np.float32)[tw_idx] if len(tw_idx) else np.array([], dtype=np.float32)
        N = len(q)
        if full_n == 0 or N == 0 or len(tw_gr_ds) == 0:
            empty = np.array([], dtype=np.float32)
            return {r: empty for r in radii}, {r: empty for r in radii}, {r: np.inf for r in radii}, empty
        qn = (q - np.nanmean(q)) / (np.nanstd(q) + 1e-6)
        rn = (tw_gr_ds - np.nanmean(tw_gr_ds)) / (np.nanstd(tw_gr_ds) + 1e-6)
        qn_f = np.nan_to_num(qn, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float64)
        rn_f = np.nan_to_num(rn, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float64)
        dtw_tvts = {}; dtw_slopes = {}; dtw_costs = {}
        inv_cost_sum = 0.0; tvt_stack = []
        length_gap = abs(len(qn_f) - len(rn_f))
        for r in radii:
            band = int(length_gap + int(r))
            D, pi, pj = _dtw_sakoe_chiba(qn_f, rn_f, band)
            cost = float(D[len(qn_f) - 1, len(rn_f) - 1]) / max(len(qn_f) + len(rn_f), 1)
            tvt_pred_ds = _dtw_path_to_tvt(pi[::-1], pj[::-1], tw_tvt_ds, N)
            slope_ds = _dtw_path_slope(pi[::-1], pj[::-1], N)
            tvt_pred = _upsample_from_dtw(idx, tvt_pred_ds, full_n)
            slope = _upsample_from_dtw(idx, slope_ds, full_n)
            if not np.isfinite(cost):
                cost = 1e9
            dtw_tvts[r] = tvt_pred
            dtw_slopes[r] = slope
            dtw_costs[r] = cost
            ic = 1.0 / (cost + 1e-6)
            inv_cost_sum += ic
            tvt_stack.append((tvt_pred, ic))
        weights = np.array([ic / max(inv_cost_sum, 1e-9) for _, ic in tvt_stack], dtype=np.float32)
        tvts_mat = np.stack([t for t, _ in tvt_stack], axis=1)
        dtw_ens = (tvts_mat * weights[None, :]).sum(axis=1).astype(np.float32)
        return dtw_tvts, dtw_slopes, dtw_costs, dtw_ens

    def run_dtw_stochastic(query_gr, tw_tvt, tw_gr, last_tvt, radius=50, K=DTW_STOCH_K, temperature=DTW_STOCH_TEMP, seed=SEED):
        full_n = len(query_gr)
        idx, q = _downsample_for_dtw(query_gr, DTW_STRIDE)
        tw_idx, tw_gr_ds = _downsample_for_dtw(tw_gr, DTW_STRIDE)
        tw_tvt_ds = np.asarray(tw_tvt, dtype=np.float32)[tw_idx] if len(tw_idx) else np.array([], dtype=np.float32)
        N = len(q)
        if full_n == 0 or N == 0 or len(tw_gr_ds) == 0:
            empty = np.array([], dtype=np.float32)
            return empty, empty, empty
        qn = ((q - np.nanmean(q)) / (np.nanstd(q) + 1e-6))
        rn = ((tw_gr_ds - np.nanmean(tw_gr_ds)) / (np.nanstd(tw_gr_ds) + 1e-6))
        qn = np.nan_to_num(qn, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float64)
        rn = np.nan_to_num(rn, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float64)
        band = int(abs(len(qn) - len(rn)) + int(radius))
        paths = _dtw_stochastic_realizations(qn, rn, band, int(K), float(temperature), int(seed))
        tvt_realiz = np.empty((K, N), dtype=np.float32)
        for k in range(K):
            tvt_realiz[k, :] = tw_tvt_ds[paths[k, :]].astype(np.float32)
        mean_tvt = _upsample_from_dtw(idx, tvt_realiz.mean(axis=0).astype(np.float32), full_n)
        std_tvt = _upsample_from_dtw(idx, tvt_realiz.std(axis=0).astype(np.float32), full_n)
        cv_tvt = (std_tvt / (np.abs(mean_tvt) + 1e-6)).astype(np.float32)
        return mean_tvt, std_tvt, cv_tvt

    print("Helpers OK ✓")


    # ─ Particle Filters (TVT Z-velocity + ANCC) ──────────────────────

    def _cal_gr_sigma(hw, tw_tvt, tw_gr):
        kn=hw[hw['TVT_input'].notna() & hw['GR'].notna()]
        if len(kn)<20: return PF_GR_SIG_DEF
        ex=np.interp(kn['TVT_input'].values,tw_tvt,tw_gr)
        return np.clip(np.std(kn['GR'].values-ex),PF_GR_SIG_MIN,PF_GR_SIG_MAX)

    def _z_beta(hw):
        kn=hw[hw['TVT_input'].notna()]
        if len(kn)<30: return -1.,0.,0.1
        dz=np.diff(kn['Z'].values); dtvt=np.diff(kn['TVT_input'].values)
        dmd=np.diff(kn['MD'].values); m=dmd>0
        if m.sum()<10: return -1.,0.,0.1
        vz=dz[m]/dmd[m]; vt=dtvt[m]/dmd[m]
        A=np.column_stack([vz,np.ones_like(vz)])
        c,_,_,_=np.linalg.lstsq(A,vt,rcond=None)
        return c[0],c[1],max(np.std(vt-(c[0]*vz+c[1])),0.001)

    def _init_v(hw):
        kn=hw[hw['TVT_input'].notna()]
        if len(kn)<10: return 0.
        tail=kn.tail(20); dtvt=np.diff(tail['TVT_input'].values)
        dmd=np.diff(tail['MD'].values); m=dmd>0
        return 0. if m.sum()<3 else float(np.median(dtvt[m]/dmd[m]))

    def run_pf_z(hw, tw_tvt, tw_gr, N=PF_N):
        tw_s=pd.Series(tw_gr).rolling(PF_GR_WIN,center=True,min_periods=1).mean().values
        tf_p=interp1d(tw_tvt,tw_gr,bounds_error=False,fill_value=(tw_gr[0],tw_gr[-1]))
        tf_s=interp1d(tw_tvt,tw_s, bounds_error=False,fill_value=(tw_s[0], tw_s[-1]))
        tmin,tmax=tw_tvt.min(),tw_tvt.max()
        gs=_cal_gr_sigma(hw,tw_tvt,tw_gr); beta,icpt,zsig=_z_beta(hw)
        kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
        if len(ev)==0: return np.array([]),np.array([])
        gr_sm=hw['GR'].rolling(PF_GR_WIN,center=True,min_periods=1).mean()
        pos=float(kn['TVT_input'].iloc[-1])+np.random.normal(0,PF_INIT_SPR,N)
        vel=_init_v(hw)+np.random.normal(0,PF_INIT_V_STD,N)
        w=np.ones(N)/N
        md_v=ev['MD'].values; gr_v=ev['GR'].values; z_v=ev['Z'].values
        pm=float(kn['MD'].iloc[-1]); pz=float(kn['Z'].iloc[-1])
        pts=np.empty(len(ev)); std=np.empty(len(ev))
        for i,idx in enumerate(ev.index):
            dm=max(md_v[i]-pm,1.); dzd=(z_v[i]-pz)/dm
            ve=beta*dzd+icpt
            vel=PF_MOM*vel+np.random.normal(0,PF_VN,N)
            pos=pos+vel*dm+np.random.normal(0,PF_PN,N)
            pos=np.clip(pos,tmin-50,tmax+50)
            if not np.isnan(gr_v[i]):
                ep=tf_p(pos); lp=np.exp(-0.5*((gr_v[i]-ep)/gs)**2)
                gs_sm=gr_sm.iloc[hw.index.get_loc(idx)]
                if not np.isnan(gs_sm):
                    es=tf_s(pos); ls=np.exp(-0.5*((gs_sm-es)/(gs*1.5))**2)
                    lk=(1-PF_GR_WT)*lp+PF_GR_WT*ls
                else: lk=lp
                lk=np.maximum(lk,1e-300); w*=lk; ws=w.sum()
                w=(w/ws) if ws>0 else np.full(N,1./N)
            zs=max(zsig*2.,0.005); lz=np.exp(-0.5*((vel-ve)/zs)**2)
            lz=np.maximum(lz,1e-300); w*=lz; ws=w.sum()
            w=(w/ws) if ws>0 else np.full(N,1./N)
            ne=1./np.sum(w**2)
            if ne<PF_RESAMP*N:
                cum=np.cumsum(w); u=(np.arange(N)+np.random.uniform())/N
                ix=np.searchsorted(cum,u); pos=pos[ix]; vel=vel[ix]; w[:]=1./N
                pos+=np.random.normal(0,PF_ROUGH_P,N); vel+=np.random.normal(0,PF_ROUGH_V,N)
            pts[i]=np.average(pos,weights=w)
            std[i]=np.sqrt(np.average((pos-pts[i])**2,weights=w))
            pm=md_v[i]; pz=z_v[i]
        return pts,std

    def run_pf_ancc(hw, tw_tvt, tw_gr, N=ANCC_N):
        tmin,tmax=tw_tvt.min(),tw_tvt.max()
        gs=_cal_gr_sigma(hw,tw_tvt,tw_gr)
        kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
        if len(ev)==0: return np.array([]),np.array([])
        ls=float(kn['TVT_input'].iloc[-1]+kn['Z'].iloc[-1])
        tail=kn.tail(30); dt=np.diff(tail['TVT_input'].values); dz=np.diff(tail['Z'].values)
        dm=np.diff(tail['MD'].values); m=dm>0
        ir=float(np.median((dt+dz)[m]/dm[m])) if m.sum()>=3 else 0.
        pos=ls+np.random.normal(0,ANCC_IS,N); rate=ir+np.random.normal(0,ANCC_IR,N)
        w=np.ones(N)/N
        md_v=ev['MD'].values; z_v=ev['Z'].values; gr_v=ev['GR'].values; pm=float(kn['MD'].iloc[-1])
        pts=np.empty(len(ev)); std=np.empty(len(ev))
        for i in range(len(ev)):
            dm=max(md_v[i]-pm,1.)
            rate=ANCC_ALPHA*rate+np.random.normal(0,ANCC_RN,N)
            pos=pos+rate*dm+np.random.normal(0,ANCC_PN,N)
            tvt_e=np.clip(pos-z_v[i],tmin-50,tmax+50); pos=tvt_e+z_v[i]
            if not np.isnan(gr_v[i]):
                eg=np.interp(tvt_e,tw_tvt,tw_gr); lk=np.exp(-0.5*((gr_v[i]-eg)/gs)**2)
                lk=np.maximum(lk,1e-300); w*=lk; ws=w.sum()
                w=(w/ws) if ws>0 else np.full(N,1./N)
            ne=1./np.sum(w**2)
            if ne<PF_RESAMP*N:
                cum=np.cumsum(w); u=(np.arange(N)+np.random.uniform())/N
                ix=np.searchsorted(cum,u); pos=pos[ix]; rate=rate[ix]; w[:]=1./N
                pos+=np.random.normal(0,ANCC_RP,N); rate+=np.random.normal(0,ANCC_RR,N)
            tv=float(np.average(pos-z_v[i],weights=w)); pts[i]=tv
            std[i]=np.sqrt(np.average((pos-z_v[i]-tv)**2,weights=w))
            pm=md_v[i]
        return pts,std

    print("Particle Filters OK ✓")


    # ─ Spatial Imputers ──────────────────────────────────────────────
    # FormationPlaneKNN: full 6-formation plane-fit (ANCC + 5 others)
    # DenseANCCImputer: 60 pts/well IDW for fine spatial resolution

    class FormationPlaneKNN:
        def __init__(self, well_ids, data_dir):
            rows=[]
            for wid in well_ids:
                p=data_dir/f'{wid}__horizontal_well.csv'
                try: df=pd.read_csv(p,usecols=['X','Y']+FORMATIONS).dropna()
                except: continue
                if len(df)==0: continue
                row={'wid':wid,'x':float(df['X'].median()),'y':float(df['Y'].median())}
                for c in FORMATIONS: row[f'{c}_m']=float(df[c].median())
                rows.append(row)
            self.df=pd.DataFrame(rows)
            self.wmap={w:i for i,w in enumerate(self.df['wid'])}
            xy=self.df[['x','y']].to_numpy()
            self.scale=np.where(xy.std(0)<1e-3,1.,xy.std(0))
            self.tree=cKDTree(xy/self.scale)
            self.xa=self.df['x'].to_numpy(); self.ya=self.df['y'].to_numpy()
            self.fa=self.df[[f'{c}_m' for c in FORMATIONS]].to_numpy(np.float64)

        def impute(self, xy_q, self_wid=None, k=PLANE_K):
            q=xy_q/self.scale; nf=min(k+5,len(self.df))
            dist,idx=self.tree.query(q,k=nf,workers=-1)
            if self_wid in self.wmap: dist=np.where(idx==self.wmap[self_wid],np.inf,dist)
            ord=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
            dk=np.take_along_axis(dist,ord,1); ik=np.take_along_axis(idx,ord,1)
            vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.).astype(np.float64)
            xn=self.xa[ik]; yn=self.ya[ik]
            wx=w*xn; wy=w*yn
            A=np.zeros((len(q),3,3))
            A[:,0,0]=(wx*xn).sum(1); A[:,0,1]=(wx*yn).sum(1); A[:,0,2]=wx.sum(1)
            A[:,1,0]=A[:,0,1]; A[:,1,1]=(wy*yn).sum(1); A[:,1,2]=wy.sum(1)
            A[:,2,0]=A[:,0,2]; A[:,2,1]=A[:,1,2]; A[:,2,2]=w.sum(1)
            A[:,0,0]+=1e-9; A[:,1,1]+=1e-9; A[:,2,2]+=1e-9
            fn=self.fa[ik]   # (N,K,6)
            rhs=np.stack([(wx[:,:,None]*fn).sum(1),(wy[:,:,None]*fn).sum(1),(w[:,:,None]*fn).sum(1)],1)
            try: coef=np.linalg.solve(A,rhs)
            except:
                coef=np.zeros((len(q),3,6))
                for r in range(len(q)):
                    try: coef[r]=np.linalg.pinv(A[r])@rhs[r]
                    except: pass
            Xq=xy_q[:,0]; Yq=xy_q[:,1]
            pred=(Xq[:,None]*coef[:,0,:]+Yq[:,None]*coef[:,1,:]+coef[:,2,:]).astype(np.float32)
            pred[~vk.any(1)]=self.fa.mean(0)
            return pred, np.where(vk,dk,np.inf).min(1).astype(np.float32)

    class DenseANCCImputer:
        def __init__(self, well_ids, data_dir, spw=DENSE_SPW):
            xs,ys,anccs,wids=[],[],[],[]
            for wid in well_ids:
                p=data_dir/f'{wid}__horizontal_well.csv'
                try: df=pd.read_csv(p,usecols=['X','Y','ANCC']).dropna()
                except: continue
                if len(df)==0: continue
                ix=np.linspace(0,len(df)-1,min(spw,len(df)),dtype=int)
                s=df.iloc[ix]
                xs.append(s['X'].values); ys.append(s['Y'].values)
                anccs.append(s['ANCC'].values); wids.extend([wid]*len(s))
            self.xy=np.column_stack([np.concatenate(xs),np.concatenate(ys)])
            self.ancc=np.concatenate(anccs).astype(np.float32)
            self.wids=np.array(wids)
            self.scale=np.where(self.xy.std(0)<1e-3,1.,self.xy.std(0))
            self.tree=cKDTree(self.xy/self.scale)

        def impute(self, xy_q, self_wid=None, k=DENSE_K, nfetch=3000):
            xy_q=np.atleast_2d(xy_q); q=xy_q/self.scale
            nf=min(nfetch,len(self.ancc))
            dist,idx=self.tree.query(q,k=nf,workers=-1)
            if self_wid: dist=np.where(self.wids[idx]==self_wid,np.inf,dist)
            ord=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
            dk=np.take_along_axis(dist,ord,1); ik=np.take_along_axis(idx,ord,1)
            vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.)
            sw=w.sum(1); safe=np.where(sw<1e-9,1.,sw)
            an=self.ancc[ik]
            ap=(an*w).sum(1)/safe; ap=np.where(sw<1e-9,float(self.ancc.mean()),ap)
            var=((an-ap[:,None])**2*w).sum(1)/safe
            return (ap.astype(np.float32),
                    np.sqrt(np.maximum(var,0.)).astype(np.float32),
                    np.where(vk,dk,np.inf).min(1).astype(np.float32))

    if not bool(globals().get('RUN_FAST_PF_SELECTOR_ONLY', False)):
    # Build imputers
        hw_paths=sorted(TRAIN_DIR.glob('*__horizontal_well.csv'))
        train_wids=[p.stem.replace('__horizontal_well','') for p in hw_paths]
        print(f"Building imputers from {len(train_wids)} wells...")
        t0=time.time()
        FI=FormationPlaneKNN(train_wids,TRAIN_DIR)
        DI=DenseANCCImputer(train_wids,TRAIN_DIR)
        print(f"  FormationPF: {len(FI.df)} centroids | DenseANCC: {len(DI.ancc):,} pts  ({time.time()-t0:.0f}s)")


        # ─ Feature Builder (per well, global FI/DI, thread-safe) ──────────
        _FI=FI; _DI=DI

        ANCH_OFFS = np.array([-80,-40,-20,-10,-5,0,5,10,20,40,80],dtype=np.float32)
        BEAM_OFFS = np.array([-40,-20,-10,-5,-3,0,3,5,10,20,40],  dtype=np.float32)
        SC_OFFS   = np.array([-30,-15,-8,-4,-2,0,2,4,8,15,30],    dtype=np.float32)
        DTW_OFFS  = np.array([-20,-10,-5,-2,0,2,5,10,20],          dtype=np.float32)

        def build_well(hw_path, tw_path, is_train):
            global _FI,_DI
            wid=Path(hw_path).stem.replace('__horizontal_well','')
            well_seed = stable_seed(wid, SEED)
            np.random.seed(well_seed)
            try:
                hw=pd.read_csv(hw_path); tw=pd.read_csv(tw_path).sort_values('TVT')
            except: return None
            if is_train and 'TVT' not in hw.columns: return None
            kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
            if len(ev)==0 or len(kn)<10: return None
            if is_train and hw['TVT'].isna().all(): return None

            tw_tvt=tw['TVT'].to_numpy(np.float32); tw_gr=tw['GR'].to_numpy(np.float32)
            if len(tw_tvt)<3: return None

            # PF signals (use ANCC PF as primary)
            pf_a,std_a=run_pf_ancc(hw,tw_tvt,tw_gr)
            if len(pf_a)==0: return None
            pf_z,std_z=run_pf_z(hw,tw_tvt,tw_gr)
            pf_use=pf_a.astype(np.float32); std_use=std_a.astype(np.float32)
            has_z=len(pf_z)==len(pf_a) and not np.any(np.isnan(pf_z))

            # Beam search (5 configs)
            lk=kn.iloc[-1]; last_tvt=float(lk['TVT_input'])
            gr_full=hw['GR'].astype(float).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
            hgr=gr_full.iloc[ev.index[0]:].to_numpy(np.float32)
            kgr=gr_full.iloc[:len(kn)].to_numpy(np.float32)
            bpaths={}
            for (bs,mc,es,r,tag) in BEAMS:
                bpaths[tag]=beam_search(hgr,tw_tvt,tw_gr,last_tvt,bs,mc,es,r)
            beam_ref=(bpaths['cons']+bpaths['sm5'])/2.

            # Self-correlation
            ktvt=kn['TVT_input'].to_numpy(np.float32)
            sc_raw,sc_sc=self_corr_tvt(kgr,ktvt,hgr,hw=15,stride=3)
            sc_trust=float(np.clip(len(kn)/200.,0.,0.6))
            hyb_ref=(1-sc_trust)*beam_ref+sc_trust*sc_raw

            # Constrained / stochastic DTW over the full horizontal GR sequence.
            full_gr = gr_full.values.astype(np.float32)
            dtw_tvts_ms, dtw_slopes_ms, dtw_costs_ms, dtw_ens_ms = run_dtw_multiscale(
                full_gr, tw_tvt, tw_gr, last_tvt, radii=DTW_RADII
            )
            stoch_seed = stable_seed(wid, SEED + 2607)
            dtw_mean_stoch, dtw_std_stoch, dtw_cv_stoch = run_dtw_stochastic(
                full_gr, tw_tvt, tw_gr, last_tvt, radius=50, K=DTW_STOCH_K, temperature=DTW_STOCH_TEMP, seed=stoch_seed
            )
            nh=len(ev); ev_start=int(ev.index[0])
            dtw_ens_raw_ms = dtw_ens_ms.copy()
            dtw_anchor_error = np.float32(dtw_ens_raw_ms[ev_start] - np.float32(last_tvt)) if len(dtw_ens_raw_ms) > ev_start else np.float32(0.0)
            dtw_ens_ms = (dtw_ens_raw_ms - dtw_anchor_error).astype(np.float32)
            for r in DTW_RADII:
                if len(dtw_tvts_ms[r]) > ev_start:
                    shift_r = np.float32(dtw_tvts_ms[r][ev_start] - np.float32(last_tvt))
                    dtw_tvts_ms[r] = (dtw_tvts_ms[r] - shift_r).astype(np.float32)
            dtw_stoch_anchor_error = np.float32(dtw_mean_stoch[ev_start] - np.float32(last_tvt)) if len(dtw_mean_stoch) > ev_start else np.float32(0.0)
            dtw_mean_stoch = (dtw_mean_stoch - dtw_stoch_anchor_error).astype(np.float32)
            def _ev_slice(arr):
                return np.asarray(arr[ev_start:ev_start+nh], dtype=np.float32)
            dtw_ens_raw_ev = _ev_slice(dtw_ens_raw_ms)
            dtw_ens_ev = _ev_slice(dtw_ens_ms)
            dtw_mean_ev = _ev_slice(dtw_mean_stoch)
            dtw_std_ev = _ev_slice(dtw_std_stoch)
            dtw_cv_ev = _ev_slice(dtw_cv_stoch)
            dtw_per_radius_ev = {r: _ev_slice(dtw_tvts_ms[r]) for r in DTW_RADII}
            dtw_slope_ev = {r: _ev_slice(dtw_slopes_ms[r]) for r in DTW_RADII}
            dtw_slope_mean_ev = np.stack([dtw_slope_ev[r] for r in DTW_RADII], 1).mean(1).astype(np.float32)
            dtw_cost_arr = np.array([dtw_costs_ms[r] for r in DTW_RADII], dtype=np.float32)
            dtw_cost_min = float(np.nanmin(dtw_cost_arr))
            dtw_cost_range = float(np.nanmax(dtw_cost_arr) - np.nanmin(dtw_cost_arr))

            # Affine calibration
            tw_at_k=np.interp(ktvt,tw_tvt,tw_gr).astype(np.float32)
            a_cal,b_cal=affine_cal(kgr,tw_at_k)

            # Prefix stats
            kmd=kn['MD'].to_numpy(np.float32); kz=kn['Z'].to_numpy(np.float32)
            pfx_rmse=float(np.sqrt(np.mean((kgr-tw_at_k)**2)))
            slp_all=robust_slope(kmd,ktvt); slp_50=robust_slope(kmd[-50:],ktvt[-50:])
            slp_z=robust_slope(kz,ktvt)

            # Spatial ANCC (centroid plane-fit)
            swid=wid if is_train else None
            xy_ev=ev[['X','Y']].to_numpy(np.float64)
            xy_kn=kn[['X','Y']].to_numpy(np.float64)
            form_ev, knn_d=_FI.impute(xy_ev,self_wid=swid)   # (nh,6)
            form_kn,_     =_FI.impute(xy_kn,self_wid=swid)

            # b_well per formation + TVT formula
            z_kn=kn['Z'].to_numpy(np.float32); z_ev=ev['Z'].to_numpy(np.float32)
            tvt_formulas={}
            for fi2,fn in enumerate(FORMATIONS):
                b_v=ktvt+z_kn-form_kn[:,fi2]
                b_all=float(np.median(b_v)); b_50=float(np.median(b_v[-50:])) if len(b_v)>=5 else b_all
                tvt_formulas[f'tvtF_{fn}']=(-z_ev+form_ev[:,fi2]+b_all).astype(np.float32)
                tvt_formulas[f'tvtF50_{fn}']=(-z_ev+form_ev[:,fi2]+b_50).astype(np.float32)
                tvt_formulas[f'bw_{fn}']=np.float32(b_all)
                tvt_formulas[f'bw50_{fn}']=np.float32(b_50)

            # Dense ANCC
            d_ancc,d_std,d_dist=_DI.impute(xy_ev,self_wid=swid)
            d_kn,d_std_kn,_=_DI.impute(xy_kn,self_wid=swid)
            b_vd=ktvt+z_kn-d_kn
            b_d=float(np.median(b_vd)); b_d50=float(np.median(b_vd[-50:])) if len(b_vd)>=5 else b_d
            tvt_dense=(-z_ev+d_ancc+b_d).astype(np.float32)
            tvt_dense50=(-z_ev+d_ancc+b_d50).astype(np.float32)
            # Dense reliability in the known prefix should measure residual spread
            # around the fitted well offset, not the absolute offset magnitude.
            dense_offset_resid=(b_vd-b_d).astype(np.float32)
            d_rmse=float(np.sqrt(np.mean(dense_offset_resid**2)))
            d_bias=float(np.mean(dense_offset_resid)); d_nb_std=float(np.mean(d_std_kn))
            last_form_ancc=float(form_kn[-1,0]) if len(form_kn) else float(np.nanmean(form_ev[:,0]))

            # GR rolling features (multiple scales)
            gr_s=pd.Series(gr_full.values)
            rolls={}
            for w in [5,21,51,101]:
                r=gr_s.rolling(w,center=True,min_periods=1)
                rolls[f'grm{w}']=r.mean().iloc[ev.index].values.astype(np.float32)
                rolls[f'grs{w}']=r.std().fillna(0).iloc[ev.index].values.astype(np.float32)
            for lag in [1,5,15,30]:
                rolls[f'glag{lag}']=gr_s.shift(lag).bfill().iloc[ev.index].values.astype(np.float32)
                rolls[f'glead{lag}']=gr_s.shift(-lag).ffill().iloc[ev.index].values.astype(np.float32)
            gr_d1=gr_s.diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
            gr_d2=gr_s.diff().diff().fillna(0.).iloc[ev.index].values.astype(np.float32)

            # Slope baselines
            hmd=ev['MD'].to_numpy(np.float32); md_since=hmd-float(lk['MD'])
            slp_base_all=(last_tvt+slp_all*md_since).astype(np.float32)
            slp_base_50 =(last_tvt+slp_50 *md_since).astype(np.float32)

            # Trajectory
            mdd=hw['MD'].diff().replace(0,np.nan)
            dzdmd=(hw['Z'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
            dxdmd=(hw['X'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
            dydmd=(hw['Y'].diff()/mdd).iloc[ev.index].values.astype(np.float32)

            frac=(np.arange(nh)/max(nh-1,1)).astype(np.float32)
            def sc(v): return np.full(nh,np.float32(v),np.float32)

            feats={
                'well':wid,'id':[f'{wid}_{i}' for i in ev.index],
                'last_known_tvt':sc(last_tvt),
                # PF signals
                'pf_ancc':pf_use,'pf_ancc_std':std_use,
                'pf_ancc_delta':(pf_use-last_tvt).astype(np.float32),
                'pf_z':pf_z.astype(np.float32) if has_z else sc(last_tvt),
                'pf_z_delta':((pf_z-last_tvt).astype(np.float32) if has_z else sc(0.)),
                'pf_vs_z':((pf_use-pf_z.astype(np.float32)) if has_z else sc(0.)),
                # Beam paths (5)
                **{f'beam_{t}_d':(p-np.float32(last_tvt)).astype(np.float32) for t,p in bpaths.items()},
                'beam_mean_d':np.stack([(p-last_tvt) for p in bpaths.values()],1).mean(1).astype(np.float32),
                'beam_std_d': np.stack([(p-last_tvt) for p in bpaths.values()],1).std(1).astype(np.float32),
                'beam_med_d': np.median(np.stack([(p-last_tvt) for p in bpaths.values()],1),1).astype(np.float32),
                # Self-corr
                'sc_d':(sc_raw-np.float32(last_tvt)).astype(np.float32),'sc_score':sc_sc,'sc_trust':sc(sc_trust),
                'hyb_d':(hyb_ref-np.float32(last_tvt)).astype(np.float32),
                # DTW sequence alignment
                'dtw_ens_d_raw':(dtw_ens_raw_ev-np.float32(last_tvt)).astype(np.float32),
                'dtw_ens_d':(dtw_ens_ev-np.float32(last_tvt)).astype(np.float32),
                'dtw_anchor_error':sc(dtw_anchor_error),
                'dtw_anchor_abs_error':sc(abs(float(dtw_anchor_error))),
                'dtw_stoch_anchor_error':sc(dtw_stoch_anchor_error),
                'dtw_stoch_mean_d':(dtw_mean_ev-np.float32(last_tvt)).astype(np.float32),
                'dtw_stoch_std':dtw_std_ev,
                'dtw_stoch_cv':dtw_cv_ev,
                'dtw_slope_mean':dtw_slope_mean_ev,
                **{f'dtw_r{int(r)}_d':(dtw_per_radius_ev[r]-np.float32(last_tvt)).astype(np.float32) for r in DTW_RADII},
                **{f'dtw_slope_r{int(r)}':dtw_slope_ev[r] for r in DTW_RADII},
                'dtw_cost_min':sc(dtw_cost_min),
                'dtw_cost_range':sc(dtw_cost_range),
                'dtw_vs_beam':(dtw_ens_ev-beam_ref).astype(np.float32),
                'dtw_vs_pf':(dtw_ens_ev-pf_use).astype(np.float32),
                'dtw_vs_sc':(dtw_ens_ev-sc_raw).astype(np.float32),
                # Spatial / formula
                **tvt_formulas,
                'spatial_ancc_d':(form_ev[:,0]-np.float32(last_form_ancc)).astype(np.float32),
                'spatial_knn_dist':knn_d,
                # Dense ANCC
                'dense_ancc':d_ancc,'dense_std':d_std,'dense_dist':d_dist,
                'tvt_dense_d':(tvt_dense-last_tvt).astype(np.float32),
                'tvt_dense50_d':(tvt_dense50-last_tvt).astype(np.float32),
                'dense_rmse':sc(d_rmse),'dense_bias':sc(d_bias),'dense_nb_std':sc(d_nb_std),
                # PF vs spatial/dense
                'pf_vs_spatial':(pf_use-tvt_formulas['tvtF_ANCC']).astype(np.float32),
                'pf_vs_dense':(pf_use-tvt_dense).astype(np.float32),
                'spatial_vs_dense':(tvt_formulas['tvtF_ANCC']-tvt_dense).astype(np.float32),
                'beam_vs_spatial':(bpaths['cons']-tvt_formulas['tvtF_ANCC']).astype(np.float32),
                'dtw_vs_dense':(dtw_ens_ev-tvt_dense).astype(np.float32),
                'dtw_vs_form':(dtw_ens_ev-tvt_formulas['tvtF_ANCC']).astype(np.float32),
                # Affine cal
                'cal_a':sc(a_cal),'cal_b':sc(b_cal),
                # Prefix stats
                'pfx_rmse':sc(pfx_rmse),'known_len':sc(len(kn)),'eval_len':sc(nh),
                'slp_all':sc(slp_all),'slp_50':sc(slp_50),'slp_z':sc(slp_z),
                'slp_base_d_all':(slp_base_all-last_tvt).astype(np.float32),
                'slp_base_d_50': (slp_base_50 -last_tvt).astype(np.float32),
                'ktvt_range':sc(float(np.ptp(ktvt))),'ktvt_std':sc(float(ktvt.std())),
                # Position
                'md_since':md_since,'frac':frac,'frac2':frac**2,'sqrt_frac':np.sqrt(frac),
                'z':z_ev,
                'dx':(ev['X']-float(lk['X'])).to_numpy(np.float32),
                'dy':(ev['Y']-float(lk['Y'])).to_numpy(np.float32),
                'dz':(z_ev-float(lk['Z'])).astype(np.float32),
                'dxy':np.sqrt((ev['X']-float(lk['X']))**2+(ev['Y']-float(lk['Y']))**2).to_numpy(np.float32),
                'dzdmd':dzdmd,'dxdmd':dxdmd,'dydmd':dydmd,
                # GR row
                'gr':hgr,'gr_d1':gr_d1,'gr_d2':gr_d2,
                'gr_vs_tw_anc':hgr-np.float32(np.interp(last_tvt,tw_tvt,tw_gr)),
                'gr_vs_slp_all':hgr-np.interp(slp_base_all,tw_tvt,tw_gr).astype(np.float32),
                # tw_diff 3 families
                **{f'tda{int(o)}':hgr-np.float32(np.interp(last_tvt+o,tw_tvt,tw_gr)) for o in ANCH_OFFS},
                **{f'tdbc{int(o)}':hgr-np.interp(beam_ref+o,tw_tvt,tw_gr).astype(np.float32) for o in BEAM_OFFS},
                **{f'tdsc{int(o)}':hgr-np.interp(sc_raw+o,tw_tvt,tw_gr).astype(np.float32) for o in SC_OFFS},
                **{f'tddtw{int(o)}':hgr-np.interp(dtw_ens_ev+o,tw_tvt,tw_gr).astype(np.float32) for o in DTW_OFFS},
                # Typewell stats
                'tw_range':sc(float(np.ptp(tw_tvt))),'tw_gr_mean':sc(float(tw_gr.mean())),
            }
            for k,v in rolls.items(): feats[k]=v

            result=pd.DataFrame(feats)
            if is_train:
                if 'TVT' not in ev.columns or ev['TVT'].isna().all(): return None
                result['target']=(ev['TVT'].to_numpy(np.float32)-np.float32(last_tvt))
            return result


        def build_dataset(paths, is_train, label):
            args=[(str(p), str(p.parent/f'{p.stem.replace("__horizontal_well","")}__typewell.csv'), is_train)
                  for p in paths
                  if (p.parent/f'{p.stem.replace("__horizontal_well","")}__typewell.csv').exists()]
            print(f"  {label}: {len(args)} wells | {NCPU} threads")
            res=Parallel(n_jobs=NCPU,prefer='threads',verbose=3)(
                delayed(build_well)(hp,tp,it) for hp,tp,it in args)
            parts=[r for r in res if r is not None]
            print(f"  {label}: OK={len(parts)} skipped={len(args)-len(parts)}")
            return pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()

        print("Feature builder OK ✓")


    

# Target-free PF/beam selector candidate. This block mirrors the physical-model PF selector reference.
    PF_SELECTOR_BIN_VARIANTS = {
        0: "pf_scale_5_hold_0.2",
        1: "pf_scale_3_hold_0.15",
        2: "pf_scale_12_beam_0.2_hold_0.15",
        3: "pf_scale_5_hold_0.15",
        4: "pf_scale_5_beam_0.05_hold_0.05",
        5: "pf_scale_12_beam_0.2_hold_0.05",
    }
    PF_SELECTOR_GLOBAL_VARIANT = "pf_scale_8_hold_0.2"
    PF_SELECTOR_N_EVAL_THRESHOLD = 4840.0
    PF_SELECTOR_Z_SPAN_THRESHOLDS = (136.73000000000016, 185.5133333333342)
    PF_SELECTOR_BEAM_CONFIGS = [
        (10, 20.0, 144.0, 2),
        (10,  8.0,  64.0, 2),
        ( 8, 35.0, 220.0, 1),
        (10, 14.0,  90.0, 5),
        (20,  4.0,  36.0, 3),
        (12, 12.0, 100.0, 3),
        (15, 25.0, 180.0, 2),
        (20, 30.0, 200.0, 2),
        (15, 10.0,  80.0, 4),
        (25,  6.0,  50.0, 3),
        (10, 40.0, 300.0, 1),
        (12, 18.0, 120.0, 5),
        (30,  8.0,  70.0, 2),
        (10, 50.0, 400.0, 0),
    ]

    def _selector_tvt_from_contacts(hw_tr, tw_tr, ref_col='EGFDU'):
        tw_g = tw_tr.dropna(subset=['Geology'])
        ref_tvt = tw_g[tw_g['Geology'] == ref_col]['TVT'].min()
        if np.isnan(ref_tvt):
            ref_col = tw_g['Geology'].iloc[0]
            ref_tvt = tw_g[tw_g['Geology'] == ref_col]['TVT'].min()
        offset = (hw_tr['TVT'] - (ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]))).mean()
        return ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]) + offset

    def _selector_particle_filter(hw, tw, n_particles=500, seed=42):
        tw_s = tw.sort_values('TVT')
        tw_tvt = tw_s['TVT'].values.astype(float)
        tw_gr = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

        kn = hw[hw['TVT_input'].notna()]
        ev = hw[hw['TVT_input'].isna()]
        if len(ev) == 0:
            return hw['TVT_input'].values.astype(float).copy(), 0.0

        last = kn.iloc[-1]
        last_tvt = float(last['TVT_input'])
        last_Z = float(last['Z'])
        last_MD = float(last['MD'])

        tw_at_k = np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)
        gs = float(np.clip(np.nanstd(kn['GR'].fillna(0).values - tw_at_k), 10.0, 60.0))

        tail = kn.tail(30)
        dt = np.diff(tail['TVT_input'].values)
        dz = np.diff(tail['Z'].values)
        dm = np.diff(tail['MD'].values)
        m = dm > 0
        ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.0

        N = int(n_particles)
        rng = np.random.default_rng(seed)
        ls = last_tvt + last_Z
        pos = ls + 2.0 * rng.standard_normal(N)
        rate = ir + 0.01 * rng.standard_normal(N)
        w = np.ones(N) / N

        MOM = 0.998
        VN = 0.002
        PN = 0.005
        RP = 0.1
        RR = 0.001
        RESAMP = 0.5

        md_v = ev['MD'].values.astype(float)
        z_v = ev['Z'].values.astype(float)
        gr_interp = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean())
        gr_v = gr_interp.values.astype(float)[ev.index]

        out_vals = hw['TVT_input'].values.astype(float).copy()
        res = np.empty(len(ev))
        prev_MD = last_MD
        log_lik = 0.0

        for i in range(len(ev)):
            dm_step = max(md_v[i] - prev_MD, 1.0)
            rate = MOM * rate + VN * rng.standard_normal(N)
            pos = pos + rate * dm_step + PN * rng.standard_normal(N)
            tvt_p = pos - z_v[i]
            tvt_p = np.clip(tvt_p, tw_tvt[0] - 100, tw_tvt[-1] + 100)
            pos = tvt_p + z_v[i]

            eg = np.interp(tvt_p, tw_tvt, tw_gr)
            d = (gr_v[i] - eg) / gs
            lk = np.exp(-0.5 * np.minimum(d**2, 600.0))
            lk = np.maximum(lk, 1e-300)
            avg_lk = float((w * lk).sum())
            log_lik += np.log(max(avg_lk, 1e-300))
            w = w * lk
            ws = w.sum()
            w = w / ws if ws > 0 else np.ones(N) / N

            n_eff = 1.0 / (w**2).sum()
            if n_eff < RESAMP * N:
                cum = np.cumsum(w)
                u0 = rng.uniform(0, 1.0 / N)
                idx = np.clip(np.searchsorted(cum, u0 + np.arange(N) / N), 0, N - 1)
                pos = pos[idx] + RP * rng.standard_normal(N)
                rate = rate[idx] + RR * rng.standard_normal(N)
                w = np.ones(N) / N

            res[i] = float(np.dot(w, pos - z_v[i]))
            prev_MD = md_v[i]

        out_vals[list(ev.index)] = res
        return out_vals, log_lik

    def _selector_pf_scales(hw, tw, scales, n_particles=500, n_seeds=64):
        preds = []
        liks = []
        for seed in range(int(n_seeds)):
            pred, ll = _selector_particle_filter(hw, tw, n_particles=n_particles, seed=seed)
            preds.append(pred)
            liks.append(ll)
        pred_arr = np.stack(preds, 0)
        liks = np.array(liks)
        liks_n = liks - liks.max()
        out = {}
        for scale in scales:
            weights = np.exp(liks_n / float(scale))
            weights /= weights.sum()
            out[f"pf_scale_{scale:g}"] = (weights[:, None] * pred_arr).sum(0)
        out["pf_mean"] = pred_arr.mean(0)
        return out

    def _selector_beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs=10, mc=20.0, es=144.0, r=2):
        n = len(hgr)
        nt = len(tw_tvt)
        if n == 0:
            return np.array([last_tvt])

        if r > 0 and n > max(3, 2 * r + 1):
            win = min(2 * r + 1, n if n % 2 == 1 else n - 1)
            sgr = savgol_filter(hgr, win, min(2, win - 1))
        else:
            sgr = hgr.copy()

        si = int(np.argmin(np.abs(tw_tvt - last_tvt)))
        MOVES = np.array([-2, -1, 0, 1, 2], dtype=np.int64)
        MC = mc * np.array([2.0, 1.0, 0.0, 1.0, 2.0])

        bidx = np.full(bs, si, dtype=np.int64)
        bcost = np.full(bs, np.inf)
        bcost[0] = 0.0
        bn = 1
        result = np.zeros(n)

        for step in range(n):
            gv = sgr[step]
            ni = bidx[:bn, None] + MOVES[None, :]
            ci = np.clip(ni, 0, nt - 1)
            valid = (ni >= 0) & (ni < nt)

            gr_e = (gv - tw_gr[ci])**2 / es
            tot = bcost[:bn, None] + gr_e + MC[None, :]
            tot = np.where(valid, tot, np.inf)

            ni_f = ni.flatten()
            tot_f = tot.flatten()
            vf = valid.flatten()
            ni_f = ni_f[vf]
            tot_f = tot_f[vf]

            order = np.argsort(tot_f)
            ni_s = ni_f[order]
            tot_s = tot_f[order]

            _, first = np.unique(ni_s, return_index=True)
            ni_u = ni_s[first]
            tot_u = tot_s[first]

            kept = min(bs, len(ni_u))
            top = np.argpartition(tot_u, min(kept - 1, len(tot_u) - 1))[:kept]
            top = top[np.argsort(tot_u[top])]

            bidx[:kept] = ni_u[top]
            bcost[:kept] = tot_u[top]
            if kept < bs:
                bidx[kept:] = bidx[kept - 1]
                bcost[kept:] = np.inf
            bn = kept

            result[step] = tw_tvt[bidx[0]]

        return result

    def _selector_beam_ensemble(hw, tw):
        kn = hw[hw['TVT_input'].notna()]
        ev = hw[hw['TVT_input'].isna()]
        if len(ev) == 0:
            return hw['TVT_input'].values.astype(float).copy()

        last_tvt = float(kn.iloc[-1]['TVT_input'])
        tw_s = tw.sort_values('TVT')
        tw_tvt = tw_s['TVT'].values.astype(float)
        tw_gr = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

        gr_all = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean()).values.astype(float)
        hgr = gr_all[ev.index]

        beam_results = [
            _selector_beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r)
            for (bs, mc, es, r) in PF_SELECTOR_BEAM_CONFIGS
        ]
        beam_mean = np.stack(beam_results, 0).mean(0)

        out = hw['TVT_input'].values.astype(float).copy()
        out[list(ev.index)] = beam_mean
        return out

    def _selector_well_code(hw):
        eval_mask = hw['TVT_input'].isna().to_numpy()
        n_eval = float(eval_mask.sum())
        z_eval = hw.loc[eval_mask, 'Z'].values.astype(float)
        z_span = float(np.nanmax(z_eval) - np.nanmin(z_eval)) if len(z_eval) else 0.0
        n_bin = int(n_eval > PF_SELECTOR_N_EVAL_THRESHOLD)
        z_bin = int(np.searchsorted(PF_SELECTOR_Z_SPAN_THRESHOLDS, z_span, side='right'))
        code = n_bin + 2 * z_bin
        return code, PF_SELECTOR_BIN_VARIANTS.get(code, PF_SELECTOR_GLOBAL_VARIANT), n_eval, z_span

    def _selector_parse_variant(name):
        parts = name.split('_')
        scale = float(parts[2])
        beam_weight = 0.0
        hold_weight = 0.0
        if 'beam' in parts:
            beam_weight = float(parts[parts.index('beam') + 1])
        if 'hold' in parts:
            hold_weight = float(parts[parts.index('hold') + 1])
        return scale, beam_weight, hold_weight

    def _selector_apply_variant(name, pf_by_scale, tvt_beam, last_known_tvt):
        scale, beam_weight, hold_weight = _selector_parse_variant(name)
        base = pf_by_scale.get(f"pf_scale_{scale:g}")
        if base is None:
            base = pf_by_scale[PF_SELECTOR_GLOBAL_VARIANT.split('_beam_')[0].split('_hold_')[0]]
        pred = (1.0 - beam_weight) * base + beam_weight * tvt_beam
        pred = (1.0 - hold_weight) * pred + hold_weight * last_known_tvt
        return pred

    def _build_target_free_selector_submission(sample_df):
        scales = tuple(float(s) for s in globals().get('PF_SELECTOR_SCALES', (3.0, 5.0, 8.0, 12.0)))
        sample_work = sample_df[['id']].copy()
        sample_work['_well'] = sample_work['id'].astype(str).str[:8]
        sample_work['_row_idx'] = sample_work['id'].astype(str).str[9:].astype(int)
        train_wids = {path.stem.replace('__horizontal_well', '') for path in TRAIN_DIR.glob('*__horizontal_well.csv')}
        rows, report_rows = [], []
        for wid in sorted(sample_work['_well'].unique()):
            hw_path = TEST_DIR / f'{wid}__horizontal_well.csv'
            tw_path = TEST_DIR / f'{wid}__typewell.csv'
            if not hw_path.exists() or not tw_path.exists():
                raise FileNotFoundError(f'Missing selector input files for well {wid}')
            hw = pd.read_csv(hw_path)
            tw = pd.read_csv(tw_path)
            tvt_phys = None
            tw_ref = tw
            if bool(globals().get('PF_SELECTOR_USE_SAME_WELL_PHYSICAL', False)) and wid in train_wids:
                try:
                    hw_tr = pd.read_csv(TRAIN_DIR / f'{wid}__horizontal_well.csv')
                    tw_tr = pd.read_csv(TRAIN_DIR / f'{wid}__typewell.csv')
                    hw['TVT_input'] = hw_tr['TVT_input'].values
                    tvt_phys = _selector_tvt_from_contacts(hw_tr, tw_tr)
                    tw_ref = tw_tr
                except Exception as exc:
                    report_rows.append({'well_id': wid, 'stage': 'physical_fallback', 'message': str(exc)[:200]})
                    tvt_phys = None
                    tw_ref = tw
            code, variant, n_eval, z_span = _selector_well_code(hw)
            try:
                pf_by_scale = _selector_pf_scales(
                    hw, tw_ref, scales,
                    n_particles=int(globals().get('PF_SELECTOR_N_PARTICLES', 500)),
                    n_seeds=int(globals().get('PF_SELECTOR_N_SEEDS', 64)),
                )
            except Exception as exc:
                last_known = hw['TVT_input'].dropna()
                last_val = float(last_known.iloc[-1]) if len(last_known) > 0 else 0.0
                tvt_pf = hw['TVT_input'].fillna(last_val).values.astype(float)
                pf_by_scale = {f"pf_scale_{scale:g}": tvt_pf.copy() for scale in scales}
                report_rows.append({'well_id': wid, 'stage': 'pf_fallback', 'message': str(exc)[:200]})
            try:
                tvt_beam = _selector_beam_ensemble(hw, tw_ref)
            except Exception as exc:
                tvt_beam = pf_by_scale.get('pf_scale_8', next(iter(pf_by_scale.values()))).copy()
                report_rows.append({'well_id': wid, 'stage': 'beam_fallback', 'message': str(exc)[:200]})
            last_known = hw['TVT_input'].dropna()
            last_known_tvt = float(last_known.iloc[-1]) if len(last_known) > 0 else float(np.nanmean(pf_by_scale.get('pf_scale_8', next(iter(pf_by_scale.values())))))
            tvt_selector = _selector_apply_variant(variant, pf_by_scale, tvt_beam, last_known_tvt)
            ws = sample_work[sample_work['_well'] == wid]
            for _, row in ws.iterrows():
                ridx = int(row['_row_idx'])
                if tvt_phys is not None:
                    tvt_val = float(tvt_phys.iloc[ridx])
                else:
                    tvt_val = float(tvt_selector[ridx])
                rows.append({'id': row['id'], 'tvt': tvt_val})
            report_rows.append({
                'well_id': wid,
                'stage': 'selector',
                'selector_code': int(code),
                'selector_variant': variant,
                'n_eval': float(n_eval),
                'z_span': float(z_span),
                'rows': int(len(ws)),
                'used_same_well_physical': bool(tvt_phys is not None),
            })
        out = sample_df[['id']].merge(pd.DataFrame(rows), on='id', how='left')
        if out['tvt'].isna().any():
            bad = out.loc[out['tvt'].isna(), 'id'].head(10).tolist()
            raise RuntimeError(f'Target-free selector missing ids: {bad}')
        if not np.isfinite(out['tvt'].to_numpy(dtype=float)).all():
            raise RuntimeError('Target-free selector produced non-finite TVT values.')
        pd.DataFrame(report_rows).to_csv(OUTPUT_DIR / 'target_free_selector_report.csv', index=False)
        return out[['id', 'tvt']]

    def _run_fast_target_free_selector_submission():
        sample = pd.read_csv(SAMPLE)
        selector_sub = _build_target_free_selector_submission(sample)
        selector_sub = sample[['id']].merge(selector_sub, on='id', how='left')
        if selector_sub['tvt'].isna().any():
            bad = selector_sub.loc[selector_sub['tvt'].isna(), 'id'].head(10).tolist()
            raise RuntimeError(f'Fast selector missing sample ids: {bad}')
        selector_sub['tvt'] = pd.to_numeric(selector_sub['tvt'], errors='coerce')
        if not np.isfinite(selector_sub['tvt'].to_numpy(dtype=float)).all():
            raise RuntimeError('Fast selector produced non-finite TVT values.')
        pf_selector_output = OUTPUT_DIR / 'submission_pf_selector.csv'
        selector_sub[['id', 'tvt']].to_csv(pf_selector_output, index=False)
        selector_sub[['id', 'tvt']].to_csv(OUT, index=False)
        globals()['FINAL_SELECTED_BASE_SOURCE'] = pf_selector_output
        globals()['FINAL_BASE_SOURCE_LABEL'] = str(globals().get('SUBMISSION_PROFILE', 'fast_pf_selector'))

        candidate_selection_summary = pd.DataFrame([{
            'candidate': 'pf_selector',
            'selected': True,
            'oof_rmse_used_for_selection': np.nan,
            'tvt_mean': float(selector_sub['tvt'].mean()),
            'tvt_std': float(selector_sub['tvt'].std()),
            'tvt_min': float(selector_sub['tvt'].min()),
            'tvt_max': float(selector_sub['tvt'].max()),
        }])
        candidate_selection_summary.to_csv(OUTPUT_DIR / 'v7_candidate_selection_summary.csv', index=False)
        pd.DataFrame([{
            'final_source': str(globals().get('SUBMISSION_PROFILE', 'fast_pf_selector')),
            'final_output': str(OUT),
            'selector_output': str(pf_selector_output),
            'final_candidate_requested': str(globals().get('FINAL_V7_CANDIDATE', 'pf_selector')),
            'final_candidate_selected': 'pf_selector',
            'submission_rows': int(len(selector_sub)),
            'submission_tvt_mean': float(selector_sub['tvt'].mean()),
            'submission_tvt_std': float(selector_sub['tvt'].std()),
            'submission_tvt_min': float(selector_sub['tvt'].min()),
            'submission_tvt_max': float(selector_sub['tvt'].max()),
            'pf_selector_n_particles': int(globals().get('PF_SELECTOR_N_PARTICLES', 500)),
            'pf_selector_n_seeds': int(globals().get('PF_SELECTOR_N_SEEDS', 64)),
            'pf_selector_scales_json': json.dumps([float(s) for s in globals().get('PF_SELECTOR_SCALES', (3.0, 5.0, 8.0, 12.0))]),
            'pf_selector_use_same_well_physical': bool(globals().get('PF_SELECTOR_USE_SAME_WELL_PHYSICAL', False)),
        }]).to_csv(OUTPUT_DIR / 'submission_contract_guard_summary_v7.csv', index=False)
        print(f"\n✅  {OUT}  {len(selector_sub)} rows")
        print(f"Final candidate: pf_selector ({globals().get('SUBMISSION_PROFILE', 'fast_pf_selector')})")
        display(candidate_selection_summary)
        display(selector_sub.head(8))
        return selector_sub[['id', 'tvt']]

    if bool(globals().get('RUN_FAST_PF_SELECTOR_ONLY', False)):
        sub = _run_fast_target_free_selector_submission()
    else:
        # ─ Load Data ──────────────────────────────────────────────────────
        print("Building train..."); t0=time.time()
        train_df=build_dataset(hw_paths,is_train=True,label="train")
        print(f"  train: {train_df.shape}  ({time.time()-t0:.0f}s)")

        test_paths=sorted(TEST_DIR.glob('*__horizontal_well.csv'))
        print("Building test..."); t0=time.time()
        test_df=build_dataset(test_paths,is_train=False,label="test")
        print(f"  test: {test_df.shape}  ({time.time()-t0:.0f}s)")

        SKIP={'well','id','target'}
        feature_cols=[c for c in train_df.columns if c not in SKIP]
        print(f"#features: {len(feature_cols)}")

        X=train_df[feature_cols].replace([np.inf, -np.inf], np.nan).astype(np.float32)
        y=train_df['target'].astype(np.float32)
        g=train_df['well']
        Xt=test_df[feature_cols].replace([np.inf, -np.inf], np.nan).astype(np.float32)
        train_matrix_mb = X.memory_usage(deep=True).sum() / 1e6
        test_matrix_mb = Xt.memory_usage(deep=True).sum() / 1e6
        print(f"Train matrix memory MB: {train_matrix_mb:.1f}")
        print(f"Test matrix memory MB: {test_matrix_mb:.1f}")
        gc.collect()


        # ─ Training: LGB×3 seeds + CatBoost, GroupKFold(5), Ridge + hill stacks ──
        cv=GroupKFold(n_splits=N_SPLITS)
        splits=list(cv.split(X,y,g))

        fold_rows=[]

        def run_lgb(seed):
            p=dict(LGB_P,n_estimators=5000,seed=seed)
            oof=np.zeros(len(train_df),np.float32); tp=np.zeros(len(test_df),np.float32)
            for fold,(tr,va) in enumerate(splits):
                dtr=lgb.Dataset(X.iloc[tr],label=y.iloc[tr])
                dva=lgb.Dataset(X.iloc[va],label=y.iloc[va],reference=dtr)
                m=lgb.train(p,dtr,valid_sets=[dva],num_boost_round=p['n_estimators'],
                            callbacks=[lgb.early_stopping(150,verbose=False),lgb.log_evaluation(500)])
                oof[va]=m.predict(X.iloc[va],num_iteration=m.best_iteration).astype(np.float32)
                tp+=m.predict(Xt,num_iteration=m.best_iteration).astype(np.float32)/N_SPLITS
                fold_rmse = root_mean_squared_error(y.iloc[va], oof[va])
                fold_rows.append({'model': f'lgb{seed}', 'fold': int(fold + 1), 'rmse': float(fold_rmse), 'best_iteration': int(m.best_iteration)})
                print(f"   LGB{seed} fold{fold}: rmse={fold_rmse:.4f} iter={m.best_iteration}")
            r=root_mean_squared_error(y,oof); print(f"   LGB{seed} OOF={r:.4f}"); return oof,tp,r

        def run_cb():
            p=dict(CB_P)
            oof=np.zeros(len(train_df),np.float32); tp=np.zeros(len(test_df),np.float32)
            for fold,(tr,va) in enumerate(splits):
                m=CatBoostRegressor(**p)
                m.fit(Pool(X.iloc[tr].values,label=y.iloc[tr].values),
                      eval_set=Pool(X.iloc[va].values,label=y.iloc[va].values),use_best_model=True)
                oof[va]=m.predict(X.iloc[va].values).astype(np.float32)
                tp+=m.predict(Xt.values).astype(np.float32)/N_SPLITS
                fold_rmse = root_mean_squared_error(y.iloc[va], oof[va])
                best_iter = getattr(m, 'best_iteration_', None)
                fold_rows.append({'model': 'cb', 'fold': int(fold + 1), 'rmse': float(fold_rmse), 'best_iteration': int(best_iter) if best_iter is not None else np.nan})
                print(f"   CB fold{fold}: rmse={fold_rmse:.4f}")
            r=root_mean_squared_error(y,oof); print(f"   CB OOF={r:.4f}"); return oof,tp,r

        results={}
        for s in LGB_SEEDS:
            oof,tp,r=run_lgb(s); results[f'lgb{s}']={'oof':oof,'test':tp,'rmse':r}
        oof,tp,r=run_cb(); results['cb']={'oof':oof,'test':tp,'rmse':r}

        # Stack candidates: best single, simple average, positive ridge, and sparse hill-climb.
        stack_names=list(results.keys())
        Sx=np.column_stack([results[k]['oof'] for k in stack_names])
        St=np.column_stack([results[k]['test'] for k in stack_names])
        y_arr=y.values.astype(np.float32)

        ridge=Ridge(alpha=1.,fit_intercept=False,positive=True)
        ridge.fit(Sx,y_arr)
        oof_s=ridge.predict(Sx).astype(np.float32); test_s=ridge.predict(St).astype(np.float32)
        r_avg=root_mean_squared_error(y_arr,Sx.mean(1))
        r_stk=root_mean_squared_error(y_arr,oof_s)
        wts=ridge.coef_/max(ridge.coef_.sum(),1e-9)

        def _rmse_np(yv, pv):
            diff=yv.astype(np.float32)-pv.astype(np.float32)
            return float(np.sqrt(np.mean(diff*diff)))

        def hill_climb_stack(result_dict, yv, max_rounds=6):
            names=list(result_dict.keys())
            scores={name:_rmse_np(yv,result_dict[name]['oof']) for name in names}
            best_name=min(scores,key=scores.get)
            cur_oof=result_dict[best_name]['oof'].astype(np.float32).copy()
            cur_test=result_dict[best_name]['test'].astype(np.float32).copy()
            weights={best_name:1.0}
            best_score=scores[best_name]
            grid=np.array([0.01,0.02,0.03,0.05,0.08,0.10,0.15,0.20,0.25,0.30,0.35,0.40],dtype=np.float32)
            trace=[{'round':0,'added_model':best_name,'weight':1.0,'rmse':best_score}]
            for rd in range(1,max_rounds+1):
                step=None
                for name in names:
                    cand_oof=result_dict[name]['oof'].astype(np.float32)
                    for w in grid:
                        trial=(1.0-float(w))*cur_oof+float(w)*cand_oof
                        score=_rmse_np(yv,trial)
                        if score+1e-7<best_score:
                            step=(name,float(w),score,trial)
                            best_score=score
                if step is None:
                    break
                name,w,score,trial_oof=step
                cur_oof=trial_oof.astype(np.float32)
                cur_test=((1.0-w)*cur_test+w*result_dict[name]['test'].astype(np.float32)).astype(np.float32)
                for k in list(weights):
                    weights[k]*=(1.0-w)
                weights[name]=weights.get(name,0.0)+w
                trace.append({'round':rd,'added_model':name,'weight':w,'rmse':score})
            return cur_oof,cur_test,best_score,weights,trace

        hill_oof,hill_test,r_hill,hill_weights,hill_trace=hill_climb_stack(results,y_arr)
        best_single_name=min(results,key=lambda k: results[k]['rmse'])
        best_single_oof=results[best_single_name]['oof']
        best_single_test=results[best_single_name]['test']
        r_best_single=results[best_single_name]['rmse']

        stack_candidates={
            'best_single':(r_best_single,best_single_oof,best_single_test),
            'simple_avg':(r_avg,Sx.mean(1).astype(np.float32),St.mean(1).astype(np.float32)),
            'ridge_stack':(r_stk,oof_s,test_s),
            'hill_stack':(r_hill,hill_oof,hill_test),
        }
        selected_stack_name=min(stack_candidates,key=lambda k: stack_candidates[k][0])
        selected_stack_rmse,final_oof,final_test=stack_candidates[selected_stack_name]

        print(f"\nBest single OOF: {r_best_single:.4f} ({best_single_name})")
        print(f"Simple avg OOF: {r_avg:.4f}")
        print(f"Ridge stk OOF: {r_stk:.4f}  wts={dict(zip(stack_names,wts.round(4)))}")
        print(f"Hill stk OOF: {r_hill:.4f}  wts={ {k:round(v,4) for k,v in hill_weights.items()} }")
        print(f"Selected stack: {selected_stack_name}  OOF={selected_stack_rmse:.4f}")
        # ─ Post-Processing + Submission ───────────────────────────────────
        base=train_df['last_known_tvt'].values.astype(np.float32)
        ytrue=y.values.astype(np.float32)+base
        pf_train=train_df['pf_ancc_delta'].values.astype(np.float32)
        pf_test=test_df['pf_ancc_delta'].values.astype(np.float32)
        dtw_train=train_df['dtw_ens_d'].values.astype(np.float32) if 'dtw_ens_d' in train_df else np.zeros(len(train_df),np.float32)
        dtw_test=test_df['dtw_ens_d'].values.astype(np.float32) if 'dtw_ens_d' in test_df else np.zeros(len(test_df),np.float32)

        def _residual_postprocess(df, model_delta, pf_delta, dtw_delta, alpha, tau, w_pf, w_dtw):
            w_model=max(0.0,1.0-float(w_pf)-float(w_dtw))
            d=(w_model*model_delta.astype(np.float32)+float(w_pf)*pf_delta.astype(np.float32)+float(w_dtw)*dtw_delta.astype(np.float32))
            if tau is not None:
                d=d*(1.-np.exp(-np.maximum(df['md_since'].values.astype(np.float32),0.)/float(tau)))
            return (d*float(alpha)).astype(np.float32)

        def _smooth_values_by_well(df, values, sg_w=0, sg_p=3):
            if not sg_w or sg_w <= 0:
                return values.astype(np.float32)
            out=values.astype(np.float32).copy()
            for well,gp in df.groupby('well',sort=False):
                idx=gp.index.to_numpy()
                v=out[idx]
                n=len(v); wl=min(int(sg_w),n)
                if wl%2==0: wl-=1
                if wl>=int(sg_p)+2:
                    out[idx]=savgol_filter(v,wl,int(sg_p)).astype(np.float32)
            return out

        # Stage 1: choose residual shrinkage, fade-in, and small PF/DTW reference mixing.
        best_cfg=None; best_delta=None; best_r=np.inf
        alpha_grid=np.round(np.arange(0.84,1.061,0.02),2)
        tau_grid=[None,30.,50.,80.,120.,200.,300.]
        w_pf_grid=[0.0,0.03,0.06,0.10]
        w_dtw_grid=[0.0,0.03,0.06,0.10]
        for alpha in alpha_grid:
            for tau in tau_grid:
                for w_pf in w_pf_grid:
                    for w_dtw in w_dtw_grid:
                        if w_pf+w_dtw>0.18:
                            continue
                        d=_residual_postprocess(train_df,final_oof,pf_train,dtw_train,alpha,tau,w_pf,w_dtw)
                        pred=base+d
                        r=root_mean_squared_error(ytrue,pred)
                        if r<best_r:
                            best_r=float(r)
                            best_cfg={'alpha':float(alpha),'tau':tau,'w_pf':float(w_pf),'w_dtw':float(w_dtw),'sg_w':0,'sg_p':0}
                            best_delta=d

        no_smooth_r=float(best_r)

        # Stage 2: tune optional Savitzky-Golay smoothing on absolute OOF predictions.
        best_abs=base+best_delta
        for sg_w in [0,9,13,17,25,35]:
            for sg_p in [2,3]:
                if sg_w and sg_w<=sg_p+1:
                    continue
                cand=_smooth_values_by_well(train_df,best_abs,sg_w,sg_p)
                r=root_mean_squared_error(ytrue,cand)
                if r<best_r:
                    best_r=float(r)
                    best_cfg=dict(best_cfg,sg_w=int(sg_w),sg_p=int(sg_p))
        print(f"Best post-proc: {best_cfg}  abs TVT RMSE={best_r:.4f}")
        ALPHA=best_cfg['alpha']; TAU=best_cfg['tau']; W_PF=best_cfg['w_pf']; W_DTW=best_cfg['w_dtw']; SG_W=best_cfg['sg_w']; SG_P=best_cfg['sg_p']

        sample=pd.read_csv(SAMPLE)
        fb=float(train_df['last_known_tvt'].mean()+train_df['target'].mean())
        test_base=test_df['last_known_tvt'].values.astype(np.float32)

        test_delta_pp=_residual_postprocess(test_df,final_test,pf_test,dtw_test,ALPHA,TAU,W_PF,W_DTW)
        test_pred_abs=test_base+test_delta_pp
        test_pred_smooth=_smooth_values_by_well(test_df,test_pred_abs,SG_W,SG_P)




        candidate_predictions_abs={
            'best_single': test_base + best_single_test,
            'ridge': test_base + test_s,
            'hill': test_base + hill_test,
            'selected_raw': test_base + final_test,
            'no_smooth': test_pred_abs,
            'postproc': test_pred_smooth,
        }
        candidate_oof_rmse={
            'best_single': float(r_best_single),
            'ridge': float(r_stk),
            'hill': float(r_hill),
            'selected_raw': float(selected_stack_rmse),
            'no_smooth': float(no_smooth_r),
            'postproc': float(best_r),
        }

        pf_selector_abs = None
        if bool(globals().get('RUN_TARGET_FREE_SELECTOR_CANDIDATE', True)):
            try:
                selector_sub = _build_target_free_selector_submission(sample)
                pf_selector_abs = selector_sub['tvt'].to_numpy(dtype=np.float32)
                selector_lookup = selector_sub.rename(columns={'tvt': 'pf_selector_tvt'})
                selector_aligned = test_df[['id']].merge(selector_lookup, on='id', how='left')['pf_selector_tvt'].to_numpy(dtype=np.float32)
                if np.isnan(selector_aligned).any():
                    raise RuntimeError('Target-free selector could not align to test feature rows.')
                pf_selector_abs = selector_aligned
                diff_selector = np.abs(test_pred_smooth.astype(float) - pf_selector_abs.astype(float))
                aux_gate = float(globals().get('PF_SELECTOR_AS_AUX_GATED_MAX_WEIGHT', 0.015)) / (
                    1.0 + (diff_selector / float(globals().get('PF_SELECTOR_AS_AUX_GATED_SCALE', 4.0))) ** 2
                )
                postproc_sel15_gated_abs = (1.0 - aux_gate) * test_pred_smooth + aux_gate * pf_selector_abs
                no_smooth_diff_selector = np.abs(test_pred_abs.astype(float) - pf_selector_abs.astype(float))
                no_smooth_aux_gate = float(globals().get('PF_SELECTOR_AS_AUX_GATED_MAX_WEIGHT', 0.015)) / (
                    1.0 + (no_smooth_diff_selector / float(globals().get('PF_SELECTOR_AS_AUX_GATED_SCALE', 4.0))) ** 2
                )
                no_smooth_sel15_gated_abs = (1.0 - no_smooth_aux_gate) * test_pred_abs + no_smooth_aux_gate * pf_selector_abs
                candidate_predictions_abs['pf_selector'] = pf_selector_abs
                candidate_predictions_abs['postproc_sel15_gated'] = postproc_sel15_gated_abs
                candidate_predictions_abs['no_smooth_sel15_gated'] = no_smooth_sel15_gated_abs
                candidate_oof_rmse['pf_selector'] = np.nan
                candidate_oof_rmse['postproc_sel15_gated'] = np.nan
                candidate_oof_rmse['no_smooth_sel15_gated'] = np.nan
                pd.Series({
                    'rows': int(len(selector_sub)),
                    'test_rows_aligned': int(len(pf_selector_abs)),
                    'selector_as_aux_gate_mean': float(np.mean(aux_gate)),
                    'selector_as_aux_gate_p95': float(np.quantile(aux_gate, 0.95)),
                    'selector_as_aux_gate_max': float(np.max(aux_gate)),
                    'mean_abs_stack_diff': float(np.mean(diff_selector)),
                    'p95_abs_stack_diff': float(np.quantile(diff_selector, 0.95)),
                    'mean_abs_no_smooth_diff': float(np.mean(no_smooth_diff_selector)),
                    'p95_abs_no_smooth_diff': float(np.quantile(no_smooth_diff_selector, 0.95)),
                }).to_csv(OUTPUT_DIR / 'target_free_selector_summary.csv')
            except Exception as exc:
                print(f'Target-free PF/beam selector candidate skipped: {exc}')
        aliases={
            'auto':'postproc',
            'auto_oof':'postproc',
            'smooth':'postproc',
            'postprocessed':'postproc',
            'raw':'selected_raw',
            'selected':'selected_raw',
            'hill_stack':'hill',
            'ridge_stack':'ridge',
            'pf':'pf_selector',
            'public_selector':'pf_selector',
            'selector':'pf_selector',
            'sel15_gated':'postproc_sel15_gated',
            'postproc_sel15':'postproc_sel15_gated',
            'no_smooth_sel15':'no_smooth_sel15_gated',
        }
        requested_candidate=str(globals().get('FINAL_V7_CANDIDATE','postproc')).strip().lower()
        selected_candidate=aliases.get(requested_candidate, requested_candidate)
        if selected_candidate not in candidate_predictions_abs:
            raise ValueError(
                f"Unknown FINAL_V7_CANDIDATE={requested_candidate!r}. "
                f"Choose one of {sorted(candidate_predictions_abs)}."
            )

        def _submission_from_prediction(pred_abs):
            frame=pd.DataFrame({'id':test_df['id'].values,'pred':np.asarray(pred_abs,dtype=np.float32)})
            pred_lookup=(frame.groupby('id', as_index=False)['pred'].mean().rename(columns={'pred':'tvt'}))
            cand=sample[['id']].merge(pred_lookup,on='id',how='left')
            missing=int(cand['tvt'].isna().sum())
            cand['tvt']=cand['tvt'].fillna(fb).astype(float)
            if len(cand) != len(sample) or not cand['id'].equals(sample['id']):
                raise RuntimeError('Submission alignment failed for selected v7 candidate.')
            if not np.isfinite(cand['tvt']).all():
                raise RuntimeError('Non-finite TVT values found for selected v7 candidate.')
            return cand[['id','tvt']], missing

        candidate_selection_summary=pd.DataFrame([
            {
                'candidate': name,
                'selected': bool(name == selected_candidate),
                'oof_rmse_used_for_selection': float(candidate_oof_rmse.get(name, np.nan)),
                'tvt_mean': float(np.nanmean(pred)),
                'tvt_std': float(np.nanstd(pred)),
                'tvt_min': float(np.nanmin(pred)),
                'tvt_max': float(np.nanmax(pred)),
            }
            for name, pred in candidate_predictions_abs.items()
        ]).sort_values(['selected','oof_rmse_used_for_selection'], ascending=[False, True])

        sub, missing_predictions = _submission_from_prediction(candidate_predictions_abs[selected_candidate])
        sub.to_csv(OUT,index=False)
        globals()['FINAL_BASE_SOURCE_LABEL'] = selected_candidate

        print(f"\n✅  {OUT}  {len(sub)} rows")
        print("\n─── Final Summary ───────────────────────────")
        for k,v in results.items(): print(f"  {k}: OOF residual RMSE = {v['rmse']:.4f}")
        print(f"  Ridge stk: {r_stk:.4f}  |  Hill stk: {r_hill:.4f}  |  Selected: {selected_stack_name}  |  PostProc: {best_r:.4f}")
        print(f"  Final candidate: {selected_candidate}  (requested: {requested_candidate})  OOF proxy={candidate_oof_rmse[selected_candidate]:.4f}")
        print(sub.head(8).to_string(index=False))

        # Reports for prediction diagnostics and submission contract tracking.
        model_summary = pd.DataFrame(
            [{'model': k, 'metric_space': 'residual_delta', 'oof_rmse': float(v['rmse']), 'selected_stack': selected_stack_name} for k, v in results.items()]
            + [
                {'model': 'best_single', 'metric_space': 'residual_delta', 'oof_rmse': float(r_best_single), 'selected_stack': selected_stack_name},
                {'model': 'simple_avg', 'metric_space': 'residual_delta', 'oof_rmse': float(r_avg), 'selected_stack': selected_stack_name},
                {'model': 'ridge_stack', 'metric_space': 'residual_delta', 'oof_rmse': float(r_stk), 'selected_stack': selected_stack_name},
                {'model': 'hill_stack', 'metric_space': 'residual_delta', 'oof_rmse': float(r_hill), 'selected_stack': selected_stack_name},
                {'model': 'postprocessed_abs_tvt', 'metric_space': 'absolute_tvt', 'oof_rmse': float(best_r), 'selected_stack': selected_stack_name},
            ]
        )
        model_summary.to_csv(OUTPUT_DIR / 'v7_dtw_super_stack_model_summary.csv', index=False)
        pd.DataFrame([{'model': k, 'ridge_weight': float(w)} for k, w in zip(stack_names, wts)]).to_csv(OUTPUT_DIR / 'v7_dtw_super_stack_ridge_weights.csv', index=False)
        pd.DataFrame([{'model': k, 'hill_weight': float(v)} for k, v in hill_weights.items()]).to_csv(OUTPUT_DIR / 'v7_dtw_super_stack_hill_weights.csv', index=False)
        pd.DataFrame(hill_trace).to_csv(OUTPUT_DIR / 'v7_dtw_super_stack_hill_trace.csv', index=False)
        pd.DataFrame(fold_rows).to_csv(OUTPUT_DIR / 'v7_dtw_super_stack_fold_report.csv', index=False)
        candidate_selection_summary.to_csv(OUTPUT_DIR / 'v7_candidate_selection_summary.csv', index=False)
        display(candidate_selection_summary)
        contract_guard = pd.DataFrame([{
            'final_source': str(SUPER_STACK_SUBMISSION_OUTPUT),
            'final_output': str(OUT),
            'feature_count': int(len(feature_cols)),
            'train_rows': int(len(train_df)),
            'test_rows': int(len(test_df)),
            'best_single_oof_rmse': float(r_best_single),
            'simple_avg_oof_rmse': float(r_avg),
            'ridge_stack_oof_rmse': float(r_stk),
            'hill_stack_oof_rmse': float(r_hill),
            'selected_stack_oof_rmse': float(selected_stack_rmse),
            'postprocessed_abs_tvt_oof_rmse': float(best_r),
            'postprocess_alpha': float(ALPHA),
            'postprocess_tau': np.nan if TAU is None else float(TAU),
            'postprocess_w_pf': float(W_PF),
            'postprocess_w_dtw': float(W_DTW),
            'postprocess_sg_window': int(SG_W),
            'postprocess_sg_poly': int(SG_P),
            'model_count': int(len(results)),
            'ridge_weights_json': json.dumps({k: float(w) for k, w in zip(stack_names, wts)}, sort_keys=True),
            'ridge_weights_raw_json': json.dumps({k: float(w) for k, w in zip(stack_names, ridge.coef_)}, sort_keys=True),
            'hill_weights_json': json.dumps({k: float(v) for k, v in hill_weights.items()}, sort_keys=True),
            'selected_stack': selected_stack_name,
            'final_candidate_requested': requested_candidate,
            'final_candidate_selected': selected_candidate,
            'final_candidate_oof_rmse': float(candidate_oof_rmse[selected_candidate]),
            'train_matrix_memory_mb': float(train_matrix_mb),
            'test_matrix_memory_mb': float(test_matrix_mb),
            'formation_count': int(len(FORMATIONS)),
            'beam_count': int(len(BEAMS)),
            'dtw_enabled': True,
            'dtw_radii_json': json.dumps([int(r) for r in DTW_RADII]),
            'dtw_stoch_k': int(DTW_STOCH_K),
            'dtw_stride': int(DTW_STRIDE),
            'feature_build_ncpu': int(NCPU),
            'dtw_anchor_abs_error_train_median': float(train_df['dtw_anchor_abs_error'].median()) if 'dtw_anchor_abs_error' in train_df else np.nan,
            'dtw_anchor_abs_error_test_median': float(test_df['dtw_anchor_abs_error'].median()) if 'dtw_anchor_abs_error' in test_df else np.nan,
            'selfcorr_enabled': True,
            'pf_tvt_z_enabled': True,
            'pf_ancc_enabled': True,
            'dense_ancc_enabled': True,
            'formation_train_exclude_self': True,
            'formation_test_exclude_self': False,
            'missing_predictions_filled': int(missing_predictions),
            'submission_rows': int(len(sub)),
            'submission_tvt_mean': float(sub['tvt'].mean()),
            'submission_tvt_std': float(sub['tvt'].std()),
            'submission_tvt_min': float(sub['tvt'].min()),
            'submission_tvt_max': float(sub['tvt'].max()),
        }])
        contract_guard.to_csv(OUTPUT_DIR / 'submission_contract_guard_summary_v7.csv', index=False)


### 18.1 PF-Residual GBDT Profile

This profile uses the particle-filter path as the absolute TVT baseline and trains the tree ensemble only on the remaining PF error:

$$
R_i = TVT_i - T_i^{\mathrm{pf}}
$$

$$
T_i^{\mathrm{final}} = T_i^{\mathrm{pf}} + 0.40 f_{\mathrm{lgb}}(x_i) + 0.40 f_{\mathrm{xgb}}(x_i) + 0.20 f_{\mathrm{cat}}(x_i)
$$

The training rows are restricted to the blind-zone rows, so the correction model sees the same kind of rows that appear in `sample_submission.csv`.

The PF path is the primary estimate in this profile. The tree models are deliberately small correction models: they should learn systematic PF bias, not replace the physical trajectory. This is different from `fast_pf_selector`, which submits the selector trajectory directly without fitting GBDT models.


In [ ]:
# PF-residual GBDT submission profile.
if not bool(globals().get('RUN_PF_RESIDUAL_GBDT_ONLY', False)):
    print('PF-residual GBDT profile skipped.')
else:
    import os
    import glob
    import gc
    import json
    import time
    import warnings
    from concurrent.futures import ProcessPoolExecutor
    from pathlib import Path

    import numpy as np
    import pandas as pd
    from scipy.spatial import cKDTree
    from sklearn.model_selection import GroupKFold
    from sklearn.metrics import mean_squared_error

    try:
        from tqdm.auto import tqdm
    except Exception:
        def tqdm(iterable=None, **kwargs):
            return iterable if iterable is not None else []

    import lightgbm as lgb
    import xgboost as xgb
    from catboost import CatBoostRegressor
    from IPython.display import display

    warnings.filterwarnings('ignore')

    PFG_DATA = Path(DATA_DIR)
    PFG_TRAIN_DIR = Path(TRAIN_DIR)
    PFG_TEST_DIR = Path(TEST_DIR)
    PFG_SAMPLE = Path(SAMPLE_SUBMISSION)
    PFG_OUT = Path(FINAL_SUBMISSION_OUTPUT)
    PFG_PUBLIC_EXACT = bool(globals().get('RUN_PF_RESIDUAL_GBDT_EXACT', False))
    PFG_PROFILE = 'pf_residual_gbdt_exact' if PFG_PUBLIC_EXACT else 'pf_residual_gbdt'
    PFG_FILL_MODE = str(globals().get('PF_RESIDUAL_GBDT_FILL_MODE', 'median_guarded')).strip().lower()
    if PFG_PUBLIC_EXACT:
        PFG_FILL_MODE = 'public_native'

    PFG_FEATURES = [
        'MD', 'Z', 'delta_Z', 'delta_MD', 'delta_X', 'delta_Y', 'dip_proxy', 'dip_acceleration',
        'GR', 'GR_rolling_mean_5', 'GR_rolling_mean_10', 'GR_rolling_mean_20',
        'GR_rolling_min_20', 'GR_rolling_max_20',
        'baseline_tvt_all_slope', 'baseline_tvt_recent_slope',
        'Distance_From_Known_Z', 'Spatial_Neighbor_Dip',
        'tort_ac_25', 'tort_ac_75', 'dls_roll', 'tort_ac_25_slope',
        'pf_residual', 'dip_pf',
    ]

    def _pfg_has_gpu() -> bool:
        try:
            import subprocess
            out = subprocess.run(
                ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                capture_output=True, text=True, check=False,
            ).stdout.strip()
            return bool(out)
        except Exception:
            return False

    PFG_USE_GPU = bool(globals().get('PF_RESIDUAL_GBDT_USE_GPU', True)) and _pfg_has_gpu()
    PFG_WEIGHTS = {k: float(v) for k, v in dict(globals().get('PF_RESIDUAL_GBDT_WEIGHTS', {'lgb': 0.40, 'xgb': 0.40, 'catboost': 0.20})).items()}
    weight_sum = sum(PFG_WEIGHTS.values())
    if not np.isfinite(weight_sum) or weight_sum <= 0:
        raise RuntimeError('PF_RESIDUAL_GBDT_WEIGHTS must have a positive finite sum.')
    PFG_WEIGHTS = {k: v / weight_sum for k, v in PFG_WEIGHTS.items()}

    PFG_BEST_LGB = {
        'learning_rate': 0.00360119978754709,
        'num_leaves': 95,
        'min_data_in_leaf': 105,
        'feature_fraction': 0.7422468666035476,
        'bagging_fraction': 0.7512866236868891,
        'lambda_l1': 1.4459168566815875e-07,
        'lambda_l2': 5.883338084153789e-08,
        'objective': 'regression',
        'metric': 'rmse',
        'device': 'cpu',
        'n_jobs': -1,
        'random_state': 42,
        'feature_pre_filter': False,
        'verbose': -1,
        'max_depth': 7,
        'bagging_freq': 4,
    }
    PFG_BEST_XGB = {
        'learning_rate': 0.0036030022158995366,
        'max_depth': 4,
        'min_child_weight': 85,
        'subsample': 0.7460321728237946,
        'colsample_bytree': 0.7389856386147577,
        'reg_alpha': 0.0005081845905730973,
        'reg_lambda': 0.0003194917844992341,
        'objective': 'reg:squarederror',
        'tree_method': 'hist',
        'device': 'cuda' if PFG_USE_GPU else 'cpu',
        'random_state': 42,
    }
    PFG_BEST_CAT = {
        'learning_rate': 0.004013658318996944,
        'depth': 8,
        'l2_leaf_reg': 1.613987020948932,
        'bagging_temperature': 0.809303990063996,
        'iterations': int(globals().get('PF_RESIDUAL_GBDT_NUM_BOOST_ROUND', 1000)),
        'task_type': 'GPU' if PFG_USE_GPU else 'CPU',
        'random_seed': 42,
        'verbose': False,
        'min_data_in_leaf': 99,
    }

    def pfg_compute_tortuosity(df, windows=(25, 75), span=10, dls_win=31):
        n = len(df)
        out = {}
        if n < 5 or not all(c in df.columns for c in ['X', 'Y', 'Z', 'MD']):
            for window in windows:
                out[f'tort_ac_{window}'] = np.zeros(n, dtype=float)
            out['dls_roll'] = np.zeros(n, dtype=float)
            out['tort_ac_25_slope'] = np.zeros(n, dtype=float)
            return out
        points = np.column_stack([df['X'].values, df['Y'].values, df['Z'].values]).astype(float)
        idx = np.arange(n)
        seg = np.diff(points, axis=0)
        seglen = np.linalg.norm(seg, axis=1)
        cum = np.concatenate([[0.0], np.cumsum(seglen)])
        for window in windows:
            lo = np.clip(idx - int(window), 0, None)
            arc = cum[idx] - cum[lo]
            chord = np.linalg.norm(points[idx] - points[lo], axis=1)
            out[f'tort_ac_{window}'] = np.clip(arc / (chord + 1e-6) - 1.0, 0, None)
        hi = np.clip(idx + int(span), 0, n - 1)
        lo = np.clip(idx - int(span), 0, n - 1)
        dirs = points[hi] - points[lo]
        d0, d1 = dirs[:-1], dirs[1:]
        dot = np.sum(d0 * d1, axis=1)
        crs = np.linalg.norm(np.cross(d0, d1), axis=1)
        angle = np.concatenate([[0.0], np.arctan2(crs, dot)])
        dmd = np.gradient(df['MD'].values.astype(float))
        dls = angle / (np.abs(dmd) + 1e-6)
        out['dls_roll'] = pd.Series(dls).rolling(dls_win, center=True, min_periods=1).mean().values
        out['tort_ac_25_slope'] = pd.Series(out['tort_ac_25']).diff().rolling(
            dls_win, center=True, min_periods=1,
        ).mean().fillna(0).values
        return out

    def pfg_run_particle_filter(hw, tw, n_particles=500, seed=42):
        tw_s = tw.sort_values('TVT')
        tw_tvt = tw_s['TVT'].values.astype(float)
        tw_gr = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)
        known = hw[hw['TVT_input'].notna()]
        eval_rows = hw[hw['TVT_input'].isna()]
        if len(eval_rows) == 0:
            return hw['TVT_input'].values.astype(float).copy(), 0.0
        if len(known) == 0:
            baseline = float(np.nanmedian(tw_tvt))
            out = np.full(len(hw), baseline, dtype=float)
            return out, 0.0
        last = known.iloc[-1]
        last_tvt = float(last['TVT_input'])
        last_z = float(last['Z'])
        last_md = float(last['MD'])
        tw_at_known = np.interp(known['TVT_input'].values, tw_tvt, tw_gr)
        gr_sigma = float(np.clip(np.nanstd(known['GR'].fillna(0).values - tw_at_known), 10.0, 60.0))
        tail = known.tail(30)
        dt = np.diff(tail['TVT_input'].values)
        dz = np.diff(tail['Z'].values)
        dm = np.diff(tail['MD'].values)
        mask = dm > 0
        init_rate = float(np.median((dt + dz)[mask] / dm[mask])) if mask.sum() >= 3 else 0.0
        rng = np.random.default_rng(int(seed))
        n_particles = int(n_particles)
        pos = last_tvt + last_z + 2.0 * rng.standard_normal(n_particles)
        rate = init_rate + 0.01 * rng.standard_normal(n_particles)
        weights = np.ones(n_particles, dtype=float) / n_particles
        mom, vn, pn, rp, rr, resamp = 0.998, 0.002, 0.005, 0.1, 0.001, 0.5
        md_values = eval_rows['MD'].values.astype(float)
        z_values = eval_rows['Z'].values.astype(float)
        gr_interp = hw['GR'].interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
        gr_values = gr_interp.values.astype(float)[eval_rows.index]
        out_values = hw['TVT_input'].values.astype(float).copy()
        pred = np.empty(len(eval_rows), dtype=float)
        prev_md = last_md
        log_lik = 0.0
        for i in range(len(eval_rows)):
            dm_step = max(md_values[i] - prev_md, 1.0)
            rate = mom * rate + vn * rng.standard_normal(n_particles)
            pos = pos + rate * dm_step + pn * rng.standard_normal(n_particles)
            tvt_particle = np.clip(pos - z_values[i], tw_tvt[0] - 100.0, tw_tvt[-1] + 100.0)
            pos = tvt_particle + z_values[i]
            expected_gr = np.interp(tvt_particle, tw_tvt, tw_gr)
            scaled = (gr_values[i] - expected_gr) / gr_sigma
            likelihood = np.maximum(np.exp(-0.5 * np.minimum(scaled ** 2, 600.0)), 1e-300)
            log_lik += np.log(max(float((weights * likelihood).sum()), 1e-300))
            weights = weights * likelihood
            total_weight = weights.sum()
            weights = weights / total_weight if total_weight > 0 else np.ones(n_particles) / n_particles
            if 1.0 / np.sum(weights ** 2) < resamp * n_particles:
                cum = np.cumsum(weights)
                u0 = rng.uniform(0, 1.0 / n_particles)
                sample_idx = np.clip(np.searchsorted(cum, u0 + np.arange(n_particles) / n_particles), 0, n_particles - 1)
                pos = pos[sample_idx] + rp * rng.standard_normal(n_particles)
                rate = rate[sample_idx] + rr * rng.standard_normal(n_particles)
                weights = np.ones(n_particles) / n_particles
            pred[i] = float(np.dot(weights, pos - z_values[i]))
            prev_md = md_values[i]
        out_values[list(eval_rows.index)] = pred
        return out_values, float(log_lik)

    def pfg_run_pf_lik_ensemble(hw, tw, n_particles=500, n_seeds=32, scale=5.0):
        preds, log_liks = [], []
        for seed in range(int(n_seeds)):
            pred, log_lik = pfg_run_particle_filter(hw, tw, n_particles=n_particles, seed=seed)
            preds.append(pred)
            log_liks.append(log_lik)
        log_liks = np.asarray(log_liks, dtype=float)
        weights = np.exp((log_liks - log_liks.max()) / float(scale))
        weights = weights / max(weights.sum(), 1e-12)
        return (weights[:, None] * np.stack(preds, axis=0)).sum(axis=0)

    def pfg_build_spatial_map():
        rows = []
        for file in sorted(PFG_TRAIN_DIR.glob('*__horizontal_well.csv')):
            frame = pd.read_csv(file)
            valid = frame.dropna(subset=['TVT'])
            if len(valid) > 10:
                delta_tvt = valid['TVT'].diff()
                delta_md = valid['MD'].diff()
                real_dip = (delta_tvt / delta_md).replace([np.inf, -np.inf], np.nan).mean()
                rows.append({
                    'WELLNAME': file.stem.replace('__horizontal_well', ''),
                    'X': float(valid['X'].mean()),
                    'Y': float(valid['Y'].mean()),
                    'Real_Dip': float(real_dip),
                })
        spatial = pd.DataFrame(rows).replace([np.inf, -np.inf], np.nan).dropna()
        if len(spatial) < 2:
            raise RuntimeError('PF-residual GBDT spatial map needs at least two valid training wells.')
        return spatial, cKDTree(spatial[['X', 'Y']].to_numpy(dtype=float))

    def pfg_process_single_well(args):
        horiz_path, spatial_map, kdtree, n_particles, n_seeds, likelihood_scale = args
        horiz_path = Path(horiz_path)
        well_name = horiz_path.stem.replace('__horizontal_well', '')
        typewell_path = horiz_path.parent / f'{well_name}__typewell.csv'
        if not typewell_path.exists():
            return None
        frame = pd.read_csv(horiz_path)
        typewell = pd.read_csv(typewell_path)
        frame['original_index'] = frame.index.astype(int)
        frame['WELLNAME'] = well_name
        frame['is_blind_zone'] = frame['TVT_input'].isna().astype(int)

        for key, value in pfg_compute_tortuosity(frame).items():
            frame[key] = value

        frame['delta_Z'] = frame['Z'].diff().fillna(0)
        frame['delta_MD'] = frame['MD'].diff().fillna(0)
        frame['delta_X'] = frame['X'].diff().fillna(0)
        frame['delta_Y'] = frame['Y'].diff().fillna(0)
        frame['dip_proxy'] = np.where(frame['delta_MD'] != 0, frame['delta_Z'] / frame['delta_MD'], 0)
        frame['dip_acceleration'] = frame['dip_proxy'].diff().fillna(0)

        for window in [5, 10, 20]:
            roll = frame['GR'].rolling(window=window, min_periods=1)
            frame[f'GR_rolling_mean_{window}'] = roll.mean()
            frame[f'GR_rolling_min_{window}'] = roll.min().fillna(0)
            frame[f'GR_rolling_max_{window}'] = roll.max().fillna(0)

        current_xy = np.array([[float(frame['X'].mean()), float(frame['Y'].mean())]])
        k = min(4, len(spatial_map))
        _, indices = kdtree.query(current_xy, k=k)
        indices = np.atleast_1d(indices[0] if np.ndim(indices) > 1 else indices)
        if len(indices) > 1:
            neighbor_idx = indices[1:]
        else:
            neighbor_idx = indices
        frame['Spatial_Neighbor_Dip'] = float(spatial_map.iloc[neighbor_idx]['Real_Dip'].mean())

        known = frame.dropna(subset=['TVT_input'])
        if not known.empty:
            last_tvt = float(known['TVT_input'].iloc[-1])
            first_tvt = float(known['TVT_input'].iloc[0])
            last_z = float(known['Z'].iloc[-1])
            last_md = float(known['MD'].iloc[-1])
            first_md = float(known['MD'].iloc[0])
            all_slope = (last_tvt - first_tvt) / (last_md - first_md) if (last_md - first_md) != 0 else 0.0
            if len(known) >= 10:
                recent_tvt = float(known['TVT_input'].iloc[-10])
                recent_md = float(known['MD'].iloc[-10])
                recent_slope = (last_tvt - recent_tvt) / (last_md - recent_md) if (last_md - recent_md) != 0 else all_slope
            else:
                recent_slope = all_slope
        else:
            last_tvt = float(typewell['TVT'].median())
            last_z = float(frame['Z'].iloc[0])
            all_slope = 0.0
            recent_slope = 0.0

        frame['last_known_TVT'] = last_tvt
        frame['last_known_Z'] = last_z
        frame['baseline_tvt_all_slope'] = all_slope
        frame['baseline_tvt_recent_slope'] = recent_slope
        frame['Distance_From_Known_Z'] = np.abs(frame['Z'] - frame['last_known_Z'])
        frame['pf_residual'] = 0.0
        frame['dip_pf'] = 0.0
        frame['pf_pred'] = frame['TVT_input'].fillna(last_tvt).astype(float)

        blind_mask = frame['is_blind_zone'] == 1
        if int(blind_mask.sum()) > 0:
            pf_pred = pfg_run_pf_lik_ensemble(
                frame,
                typewell,
                n_particles=int(n_particles),
                n_seeds=int(n_seeds),
                scale=float(likelihood_scale),
            )
            blind_idx = frame.index[blind_mask]
            pf_blind_pred = pf_pred[blind_idx]
            frame['pf_pred'] = pf_pred
            frame.loc[blind_idx, 'pf_residual'] = pf_blind_pred - last_tvt
            dip = np.diff(pf_blind_pred, prepend=pf_blind_pred[0] if len(pf_blind_pred) else 0.0)
            dmd = frame.loc[blind_idx, 'delta_MD'].replace(0, np.nan).to_numpy(dtype=float)
            frame.loc[blind_idx, 'dip_pf'] = np.nan_to_num(dip / dmd, nan=0.0, posinf=0.0, neginf=0.0)

        if 'TVT' in frame.columns:
            frame['target_pf_residual'] = frame['TVT'].astype(float) - frame['pf_pred'].astype(float)
        return frame

    def pfg_process_files(files, spatial_map, kdtree, label):
        args = [
            (
                str(path),
                spatial_map,
                kdtree,
                int(globals().get('PF_RESIDUAL_GBDT_N_PARTICLES', 500)),
                int(globals().get('PF_RESIDUAL_GBDT_N_SEEDS', 32)),
                float(globals().get('PF_RESIDUAL_GBDT_LIKELIHOOD_SCALE', 5.0)),
            )
            for path in files
        ]
        use_mp = bool(globals().get('PF_RESIDUAL_GBDT_USE_MULTIPROCESS', True)) and bool(KAGGLE_NOTEBOOK_RUN)
        try:
            if use_mp:
                workers = min(os.cpu_count() or 2, len(args))
                with ProcessPoolExecutor(max_workers=workers) as executor:
                    return [res for res in tqdm(executor.map(pfg_process_single_well, args), total=len(args), desc=label) if res is not None]
            return [res for res in tqdm(map(pfg_process_single_well, args), total=len(args), desc=label) if res is not None]
        except Exception as exc:
            if use_mp:
                print(f'{label}: multiprocessing failed, retrying serially: {exc!r}')
                return [res for res in tqdm(map(pfg_process_single_well, args), total=len(args), desc=f'{label}_serial') if res is not None]
            raise

    def pfg_rmse(y_true, y_pred):
        return float(np.sqrt(mean_squared_error(np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float))))

    def pfg_prepare_matrix(frame, feature_cols, fill_values=None):
        matrix = frame[feature_cols].replace([np.inf, -np.inf], np.nan).copy()
        for col in matrix.columns:
            matrix[col] = pd.to_numeric(matrix[col], errors='coerce')
        if PFG_FILL_MODE == 'public_native':
            return matrix, fill_values
        if fill_values is None:
            fill_values = matrix.median(numeric_only=True).fillna(0.0)
        matrix = matrix.fillna(fill_values).fillna(0.0)
        return matrix.astype(np.float32), fill_values

    def run_pf_residual_gbdt_submission():
        t0 = time.time()
        sample = pd.read_csv(PFG_SAMPLE)
        spatial_map, kdtree = pfg_build_spatial_map()
        if PFG_PUBLIC_EXACT:
            train_files = [Path(x) for x in glob.glob(str(PFG_TRAIN_DIR / '*__horizontal_well.csv'))]
            test_files = [Path(x) for x in glob.glob(str(PFG_TEST_DIR / '*__horizontal_well.csv'))]
        else:
            train_files = sorted(PFG_TRAIN_DIR.glob('*__horizontal_well.csv'))
            test_files = sorted(PFG_TEST_DIR.glob('*__horizontal_well.csv'))

        print('Building PF-residual training frame...')
        train_parts = pfg_process_files(train_files, spatial_map, kdtree, 'pf_gbdt_train')
        if not train_parts:
            raise RuntimeError(f'No training wells were processed for {PFG_PROFILE}.')
        train_frame = pd.concat(train_parts, ignore_index=True)
        train_blind = train_frame.loc[train_frame['is_blind_zone'].eq(1)].dropna(subset=['target_pf_residual']).copy().reset_index(drop=True)
        if train_blind.empty:
            raise RuntimeError(f'No blind-zone training rows are available for {PFG_PROFILE}.')

        X_train, fill_values = pfg_prepare_matrix(train_blind, PFG_FEATURES)
        y_train = train_blind['target_pf_residual'].astype(float).to_numpy()
        groups = train_blind['WELLNAME'].astype(str).to_numpy()

        n_splits = min(int(globals().get('PF_RESIDUAL_GBDT_N_SPLITS', 5)), len(np.unique(groups)))
        if n_splits < 2:
            raise RuntimeError('PF-residual GBDT needs at least two training groups for GroupKFold.')
        gkf = GroupKFold(n_splits=n_splits)
        boost_round = int(globals().get('PF_RESIDUAL_GBDT_NUM_BOOST_ROUND', 1000))
        oof = np.zeros(len(train_blind), dtype=float)
        fold_rows = []

        print('Running PF-residual GBDT GroupKFold...')
        for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train, y_train, groups), 1):
            X_tr, y_tr = X_train.iloc[tr_idx], y_train[tr_idx]
            X_va, y_va = X_train.iloc[va_idx], y_train[va_idx]
            model_lgb = lgb.train(PFG_BEST_LGB, lgb.Dataset(X_tr, label=y_tr), num_boost_round=boost_round)
            model_xgb = xgb.train(PFG_BEST_XGB, xgb.DMatrix(X_tr, label=y_tr), num_boost_round=boost_round)
            model_cat = CatBoostRegressor(**PFG_BEST_CAT).fit(X_tr, y_tr)
            pred_lgb = model_lgb.predict(X_va)
            pred_xgb = model_xgb.predict(xgb.DMatrix(X_va))
            pred_cat = model_cat.predict(X_va)
            pred = (
                PFG_WEIGHTS.get('lgb', 0.0) * pred_lgb
                + PFG_WEIGHTS.get('xgb', 0.0) * pred_xgb
                + PFG_WEIGHTS.get('catboost', 0.0) * pred_cat
            )
            oof[va_idx] = pred
            residual_rmse = pfg_rmse(y_va, pred)
            abs_rmse = pfg_rmse(train_blind.iloc[va_idx]['TVT'].to_numpy(dtype=float), train_blind.iloc[va_idx]['pf_pred'].to_numpy(dtype=float) + pred)
            fold_rows.append({'fold': fold, 'rows': int(len(va_idx)), 'residual_rmse': residual_rmse, 'absolute_tvt_rmse': abs_rmse})
            print(f'  fold {fold}: residual RMSE={residual_rmse:.4f}, absolute TVT RMSE={abs_rmse:.4f}')

        oof_abs = train_blind['pf_pred'].to_numpy(dtype=float) + oof
        oof_abs_rmse = pfg_rmse(train_blind['TVT'].to_numpy(dtype=float), oof_abs)
        pf_only_rmse = pfg_rmse(train_blind['TVT'].to_numpy(dtype=float), train_blind['pf_pred'].to_numpy(dtype=float))
        print(f'PF-only OOF absolute RMSE: {pf_only_rmse:.4f}')
        print(f'PF + residual GBDT OOF absolute RMSE: {oof_abs_rmse:.4f}')

        print('Training final PF-residual GBDT models...')
        model_lgb = lgb.train(PFG_BEST_LGB, lgb.Dataset(X_train, label=y_train), num_boost_round=boost_round)
        model_xgb = xgb.train(PFG_BEST_XGB, xgb.DMatrix(X_train, label=y_train), num_boost_round=boost_round)
        model_cat = CatBoostRegressor(**PFG_BEST_CAT).fit(X_train, y_train)

        print('Building PF-residual test frame...')
        test_parts = pfg_process_files(test_files, spatial_map, kdtree, 'pf_gbdt_test')
        if not test_parts:
            raise RuntimeError(f'No test wells were processed for {PFG_PROFILE}.')
        test_frame = pd.concat([part.loc[part['is_blind_zone'].eq(1)].copy() for part in test_parts], ignore_index=True)
        if test_frame.empty:
            raise RuntimeError(f'No blind-zone test rows are available for {PFG_PROFILE}.')
        X_test, _ = pfg_prepare_matrix(test_frame, PFG_FEATURES, fill_values=fill_values)
        pred_lgb = model_lgb.predict(X_test)
        pred_xgb = model_xgb.predict(xgb.DMatrix(X_test))
        pred_cat = model_cat.predict(X_test)
        correction = (
            PFG_WEIGHTS.get('lgb', 0.0) * pred_lgb
            + PFG_WEIGHTS.get('xgb', 0.0) * pred_xgb
            + PFG_WEIGHTS.get('catboost', 0.0) * pred_cat
        )
        test_frame['tvt'] = test_frame['pf_pred'].astype(float) + correction
        raw_sub = pd.DataFrame({
            'id': test_frame['WELLNAME'].astype(str) + '_' + test_frame['original_index'].astype(int).astype(str),
            'tvt': test_frame['tvt'].astype(float),
        })
        raw_sub = raw_sub.groupby('id', as_index=False)['tvt'].mean()
        aligned = sample[['id']].merge(raw_sub, on='id', how='left')
        if aligned['tvt'].isna().any():
            missing = aligned.loc[aligned['tvt'].isna(), 'id'].head(10).tolist()
            raise RuntimeError(f'PF-residual GBDT missing sample ids: {missing}')
        if not np.isfinite(aligned['tvt'].to_numpy(dtype=float)).all():
            raise RuntimeError('PF-residual GBDT produced non-finite TVT values.')

        suffix = 'pf_residual_gbdt_exact' if PFG_PUBLIC_EXACT else 'pf_residual_gbdt'
        out_path = OUTPUT_DIR / f'submission_{suffix}.csv'
        aligned[['id', 'tvt']].to_csv(out_path, index=False)
        aligned[['id', 'tvt']].to_csv(PFG_OUT, index=False)
        if PFG_PUBLIC_EXACT:
            aligned[['id', 'tvt']].to_csv(OUTPUT_DIR / 'submission_pf_ensemble.csv', index=False)
        globals()['FINAL_SELECTED_BASE_SOURCE'] = out_path
        globals()['FINAL_BASE_SOURCE_LABEL'] = PFG_PROFILE

        fold_report = pd.DataFrame(fold_rows)
        fold_report.to_csv(OUTPUT_DIR / 'pf_residual_gbdt_fold_report.csv', index=False)
        pd.DataFrame([{'model': k, 'weight': v} for k, v in PFG_WEIGHTS.items()]).to_csv(OUTPUT_DIR / 'pf_residual_gbdt_weights.csv', index=False)
        summary = pd.DataFrame([{
            'candidate': PFG_PROFILE,
            'public_exact_reproduction': bool(PFG_PUBLIC_EXACT),
            'feature_fill_mode': PFG_FILL_MODE,
            'selected': True,
            'pf_only_oof_abs_rmse': float(pf_only_rmse),
            'oof_rmse_used_for_selection': float(oof_abs_rmse),
            'train_rows': int(len(train_blind)),
            'test_rows': int(len(test_frame)),
            'feature_count': int(len(PFG_FEATURES)),
            'n_splits': int(n_splits),
            'n_particles': int(globals().get('PF_RESIDUAL_GBDT_N_PARTICLES', 500)),
            'n_seeds': int(globals().get('PF_RESIDUAL_GBDT_N_SEEDS', 32)),
            'likelihood_scale': float(globals().get('PF_RESIDUAL_GBDT_LIKELIHOOD_SCALE', 5.0)),
            'use_gpu': bool(PFG_USE_GPU),
            'weights_json': json.dumps(PFG_WEIGHTS, sort_keys=True),
            'output_file': out_path.name,
            'runtime_seconds': float(time.time() - t0),
            'tvt_mean': float(aligned['tvt'].mean()),
            'tvt_std': float(aligned['tvt'].std()),
            'tvt_min': float(aligned['tvt'].min()),
            'tvt_max': float(aligned['tvt'].max()),
        }])
        summary.to_csv(OUTPUT_DIR / f'{suffix}_summary.csv', index=False)
        summary.to_csv(OUTPUT_DIR / 'pf_residual_gbdt_summary.csv', index=False)
        summary.rename(columns={'pf_only_oof_abs_rmse': 'pf_baseline_oof_rmse'}).to_csv(OUTPUT_DIR / 'v7_candidate_selection_summary.csv', index=False)
        pd.DataFrame([{
            'final_source': PFG_PROFILE,
            'public_exact_reproduction': bool(PFG_PUBLIC_EXACT),
            'feature_fill_mode': PFG_FILL_MODE,
            'final_output': str(PFG_OUT),
            'submission_rows': int(len(aligned)),
            'submission_tvt_mean': float(aligned['tvt'].mean()),
            'submission_tvt_std': float(aligned['tvt'].std()),
            'submission_tvt_min': float(aligned['tvt'].min()),
            'submission_tvt_max': float(aligned['tvt'].max()),
            'oof_abs_rmse': float(oof_abs_rmse),
            'pf_only_oof_abs_rmse': float(pf_only_rmse),
        }]).to_csv(OUTPUT_DIR / 'submission_contract_guard_summary_v7.csv', index=False)

        print(f'\nSaved {PFG_OUT} with {len(aligned)} rows')
        display(summary)
        display(aligned.head(8))
        gc.collect()
        return aligned[['id', 'tvt']]

    pf_residual_gbdt_submission = run_pf_residual_gbdt_submission()


### 18.2 Artifact-Backed Ridge + Physical/PF Heuristic Profile

This profile builds the artifact-backed Ridge estimate and the physical/PF heuristic estimate, then writes an aligned submission.

The exact reference profile uses:

$$
T_i^{\mathrm{ridge\_ref}} = 0.30\,T_i^{\mathrm{ridge}} + 0.70\,T_i^{\mathrm{heur}}.
$$

The weight-probe profile keeps the same two estimates but changes the ridge weight. The PF-boost profile keeps the exact 0.30/0.70 blend but uses a larger PF ensemble inside the heuristic estimate.

$$
T_i^{\mathrm{final}} = w\,T_i^{\mathrm{ridge}} + (1-w)\,T_i^{\mathrm{heur}}.
$$

When `SUBMISSION_PROFILE = 'ridge_artifact_model_blend'`, the exact ridge reference is used as an anchor and mixed with the PF-residual GBDT prediction:

$$
T_i^{\mathrm{final}} = (1-\lambda)\,T_i^{\mathrm{ridge\_ref}} + \lambda\,T_i^{\mathrm{pfgbdt}}.
$$


In [ ]:
# Artifact-backed ridge / heuristic profile.
if not bool(globals().get('RUN_RIDGE_ARTIFACT_PROFILE', False)):
    print('Ridge artifact profile skipped.')
else:
    from lightgbm import LGBMRegressor, log_evaluation, early_stopping
    from sklearn.metrics import root_mean_squared_error
    from sklearn.model_selection import GroupKFold
    from sklearn.linear_model import Ridge
    from catboost import CatBoostRegressor
    from scipy.spatial import cKDTree
    from scipy.signal import savgol_filter
    from joblib import Parallel, delayed
    try:
        from koolbox import Trainer
    except ModuleNotFoundError as exc:
        raise RuntimeError('The ridge artifact profiles require the original notebook dependency: koolbox.') from exc
    from pathlib import Path
    from numba import njit
    import matplotlib.pyplot as plt
    import multiprocessing
    import seaborn as sns
    import pandas as pd
    import numpy as np
    import warnings
    import joblib
    import time
    import glob
    import os

    warnings.filterwarnings("ignore")


    class CFG:
        dataset_path = next(
            (Path(path) for path in COMPETITION_DATA_ROOTS if Path(path).exists()),
            Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction'),
        )
        artifacts_path = next(
            (Path(path) for path in RIDGE_ARTIFACT_ROOTS if Path(path).exists()),
            Path('/kaggle/input/datasets/ravaghi/wellbore-geology-prediction-artifacts'),
        )

        seed = 42
        n_splits = 5
        cv = GroupKFold(n_splits=n_splits)
        metric = root_mean_squared_error


    SELECTOR_N_EVAL_THRESHOLD = 4840.0
    SELECTOR_Z_SPAN_THRESHOLDS = (136.73000000000016, 185.5133333333342)

    SELECTOR_BIN_VARIANTS = {
        0: 'pf_scale_5_hold_0.2',
        1: 'pf_scale_3_hold_0.15',
        2: 'pf_scale_12_beam_0.2_hold_0.15',
        3: 'pf_scale_5_hold_0.15',
        4: 'pf_scale_5_beam_0.05_hold_0.05',
        5: 'pf_scale_12_beam_0.2_hold_0.05',
    }

    SELECTOR_GLOBAL_VARIANT = 'pf_scale_8_hold_0.2'
    SELECTOR_SCALES = (3.0, 5.0, 8.0, 12.0)

    FORMATION_COLS = ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']

    BEAM_CONFIGS = [
        (10, 20.0, 144.0, 2),
        (10,  8.0,  64.0, 2),
        ( 8, 35.0, 220.0, 1),
        (10, 14.0,  90.0, 5),
        (20,  4.0,  36.0, 3),
        (12, 12.0, 100.0, 3),
        (15, 25.0, 180.0, 2),
        (20, 30.0, 200.0, 2),
        (15, 10.0,  80.0, 4),
        (25,  6.0,  50.0, 3),
        (10, 40.0, 300.0, 1),
        (12, 18.0, 120.0, 5),
        (30,  8.0,  70.0, 2),
        (10, 50.0, 400.0, 0),
    ]


    def tvt_from_contacts(hw_tr, tw_tr, ref_col='EGFDU'):
        tw_g = tw_tr.dropna(subset=['Geology'])
        ref_tvt = tw_g[tw_g['Geology'] == ref_col]['TVT'].min()
        if np.isnan(ref_tvt):
            ref_col = tw_g['Geology'].iloc[0]
            ref_tvt = tw_g[tw_g['Geology'] == ref_col]['TVT'].min()
        offset = (hw_tr['TVT'] - (ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]))).mean()
        return ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]) + offset


    def load_well(wid, split='train'):
        base = CFG.dataset_path / split
        hw = pd.read_csv(base / f'{wid}__horizontal_well.csv')
        tw = pd.read_csv(base / f'{wid}__typewell.csv')
        return hw, tw


    def run_particle_filter(hw, tw, n_particles=500, seed=42):
        tw_s   = tw.sort_values('TVT')
        tw_tvt = tw_s['TVT'].values.astype(float)
        tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

        kn = hw[hw['TVT_input'].notna()]
        ev = hw[hw['TVT_input'].isna()]
        if len(ev) == 0:
            return hw['TVT_input'].values.astype(float).copy(), 0.0

        last     = kn.iloc[-1]
        last_tvt = float(last['TVT_input'])
        last_Z   = float(last['Z'])
        last_MD  = float(last['MD'])

        tw_at_k = np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)
        gs = float(np.clip(np.nanstd(kn['GR'].fillna(0).values - tw_at_k), 10., 60.))

        tail = kn.tail(30)
        dt = np.diff(tail['TVT_input'].values)
        dz = np.diff(tail['Z'].values)
        dm = np.diff(tail['MD'].values)
        m  = dm > 0
        ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.0

        N   = n_particles
        rng = np.random.default_rng(seed)
        ls   = last_tvt + last_Z
        pos  = ls + 2.0 * rng.standard_normal(N)
        rate = ir + 0.01 * rng.standard_normal(N)
        w    = np.ones(N) / N

        MOM = 0.998; VN = 0.002; PN = 0.005; RP = 0.1; RR = 0.001; RESAMP = 0.5

        md_v = ev['MD'].values.astype(float)
        z_v  = ev['Z'].values.astype(float)
        # Interpolate GR gaps before tracking
        gr_interp = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean())
        gr_v = gr_interp.values.astype(float)[ev.index]

        out_vals = hw['TVT_input'].values.astype(float).copy()
        res = np.empty(len(ev))
        prev_MD = last_MD
        log_lik = 0.0

        for i in range(len(ev)):
            dm_step = max(md_v[i] - prev_MD, 1.0)
            rate = MOM * rate + VN * rng.standard_normal(N)
            pos  = pos + rate * dm_step + PN * rng.standard_normal(N)
            tvt_p = pos - z_v[i]
            tvt_p = np.clip(tvt_p, tw_tvt[0] - 100, tw_tvt[-1] + 100)
            pos   = tvt_p + z_v[i]

            eg = np.interp(tvt_p, tw_tvt, tw_gr)
            d  = (gr_v[i] - eg) / gs
            lk = np.exp(-0.5 * np.minimum(d**2, 600.))
            lk = np.maximum(lk, 1e-300)
            avg_lk = float((w * lk).sum())
            log_lik += np.log(max(avg_lk, 1e-300))
            w = w * lk
            ws = w.sum()
            w = w / ws if ws > 0 else np.ones(N) / N

            n_eff = 1.0 / (w**2).sum()
            if n_eff < RESAMP * N:
                cum = np.cumsum(w)
                u0  = rng.uniform(0, 1.0 / N)
                idx = np.clip(np.searchsorted(cum, u0 + np.arange(N) / N), 0, N - 1)
                pos  = pos[idx]  + RP * rng.standard_normal(N)
                rate = rate[idx] + RR * rng.standard_normal(N)
                w    = np.ones(N) / N

            res[i] = float(np.dot(w, pos - z_v[i]))
            prev_MD = md_v[i]

        out_vals[list(ev.index)] = res
        return out_vals, log_lik


    def run_pf_lik_ensemble(hw, tw, n_particles=500, n_seeds=128, scale=5.0):
        preds = []
        liks  = []
        for s in range(n_seeds):
            p, ll = run_particle_filter(hw, tw, n_particles=n_particles, seed=s)
            preds.append(p)
            liks.append(ll)

        liks   = np.array(liks)
        liks_n = liks - liks.max()
        weights = np.exp(liks_n / scale)
        weights /= weights.sum()

        return (weights[:, None] * np.stack(preds, 0)).sum(0)


    def run_pf_lik_ensemble_scales(hw, tw, scales=SELECTOR_SCALES, n_particles=500, n_seeds=128):
        preds = []
        liks = []
        for s in range(n_seeds):
            p, ll = run_particle_filter(hw, tw, n_particles=n_particles, seed=s)
            preds.append(p)
            liks.append(ll)
        pred_arr = np.stack(preds, 0)
        liks = np.array(liks)
        liks_n = liks - liks.max()
        out = {}
        for scale in scales:
            weights = np.exp(liks_n / float(scale))
            weights /= weights.sum()
            out[f'pf_scale_{scale:g}'] = (weights[:, None] * pred_arr).sum(0)
        out['pf_mean'] = pred_arr.mean(0)
        return out


    def beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs=10, mc=20.0, es=144.0, r=2):
        n  = len(hgr)
        nt = len(tw_tvt)
        if n == 0:
            return np.array([last_tvt])

        if r > 0 and n > max(3, 2 * r + 1):
            win = min(2 * r + 1, n if n % 2 == 1 else n - 1)
            sgr = savgol_filter(hgr, win, min(2, win - 1))
        else:
            sgr = hgr.copy()

        si = int(np.argmin(np.abs(tw_tvt - last_tvt)))

        MOVES = np.array([-2, -1, 0, 1, 2], dtype=np.int64)
        MC    = mc * np.array([2., 1., 0., 1., 2.])

        bidx  = np.full(bs, si, dtype=np.int64)
        bcost = np.full(bs, np.inf)
        bcost[0] = 0.
        bn = 1

        result = np.zeros(n)

        for step in range(n):
            gv = sgr[step]
            ni = bidx[:bn, None] + MOVES[None, :]
            ci = np.clip(ni, 0, nt - 1)
            valid = (ni >= 0) & (ni < nt)

            gr_e = (gv - tw_gr[ci])**2 / es
            tot  = bcost[:bn, None] + gr_e + MC[None, :]
            tot  = np.where(valid, tot, np.inf)

            ni_f  = ni.flatten()
            tot_f = tot.flatten()
            vf    = valid.flatten()
            ni_f  = ni_f[vf]
            tot_f = tot_f[vf]

            order = np.argsort(tot_f)
            ni_s  = ni_f[order]
            tot_s = tot_f[order]

            _, first = np.unique(ni_s, return_index=True)
            ni_u  = ni_s[first]
            tot_u = tot_s[first]

            kept = min(bs, len(ni_u))
            top  = np.argpartition(tot_u, min(kept - 1, len(tot_u) - 1))[:kept]
            top  = top[np.argsort(tot_u[top])]

            bidx[:kept]  = ni_u[top]
            bcost[:kept] = tot_u[top]
            if kept < bs:
                bidx[kept:]  = bidx[kept - 1]
                bcost[kept:] = np.inf
            bn = kept

            result[step] = tw_tvt[bidx[0]]

        return result


    def run_beam_ensemble(hw, tw):
        kn = hw[hw['TVT_input'].notna()]
        ev = hw[hw['TVT_input'].isna()]
        if len(ev) == 0:
            return hw['TVT_input'].values.astype(float).copy()

        last_tvt = float(kn.iloc[-1]['TVT_input'])
        tw_s  = tw.sort_values('TVT')
        tw_tvt = tw_s['TVT'].values.astype(float)
        tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

        gr_all = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean()).values.astype(float)
        hgr    = gr_all[ev.index]

        beam_results = [beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r)
                        for (bs, mc, es, r) in BEAM_CONFIGS]

        beam_mean = np.stack(beam_results, 0).mean(0)

        out = hw['TVT_input'].values.astype(float).copy()
        out[list(ev.index)] = beam_mean
        return out


    def selector_well_code(hw):
        eval_mask = hw['TVT_input'].isna().to_numpy()
        n_eval = float(eval_mask.sum())
        z_eval = hw.loc[eval_mask, 'Z'].values.astype(float)
        z_span = float(np.nanmax(z_eval) - np.nanmin(z_eval)) if len(z_eval) else 0.0
        n_bin = int(n_eval > SELECTOR_N_EVAL_THRESHOLD)
        z_bin = int(np.searchsorted(SELECTOR_Z_SPAN_THRESHOLDS, z_span, side='right'))
        code = n_bin + 2 * z_bin
        variant = SELECTOR_BIN_VARIANTS.get(code, SELECTOR_GLOBAL_VARIANT)
        return code, variant, n_eval, z_span


    def parse_selector_variant(name):
        parts = name.split('_')
        scale = float(parts[2])
        beam_weight = 0.0
        hold_weight = 0.0
        if 'beam' in parts:
            beam_weight = float(parts[parts.index('beam') + 1])
        if 'hold' in parts:
            hold_weight = float(parts[parts.index('hold') + 1])
        return scale, beam_weight, hold_weight


    def apply_selector_variant(name, pf_by_scale, tvt_beam, last_known_tvt):
        scale, beam_weight, hold_weight = parse_selector_variant(name)
        base = pf_by_scale.get(f'pf_scale_{scale:g}')
        if base is None:
            base = pf_by_scale[SELECTOR_GLOBAL_VARIANT.split('_beam_')[0].split('_hold_')[0]]
        pred = (1.0 - beam_weight) * base + beam_weight * tvt_beam
        pred = (1.0 - hold_weight) * pred + hold_weight * last_known_tvt
        return pred

    SEED=42
    NCPU=min(4,multiprocessing.cpu_count())

    FORMATIONS=["ANCC","ASTNU","ASTNL","EGFDU","EGFDL","BUDA"]
    PLANE_K=10; DENSE_SPW=60; DENSE_K=20; N_SPLITS=5

    BEAMS=[
        (10,20.0,144.0,2,"cons"),
        (10, 8.0, 64.0,2,"loose"),
        ( 8,35.0,220.0,1,"vcons"),
        (10,14.0, 90.0,5,"sm5"),
        (20, 4.0, 36.0,3,"vloose"),
        (12,12.0,100.0,3,"mid"),
        (15,25.0,180.0,2,"stiff"),
    ]

    PF_N=600; ANCC_N=600
    PF_MOM=0.993; PF_VN=0.005; PF_PN=0.01
    PF_GR_SIG_MIN=10.; PF_GR_SIG_MAX=60.; PF_GR_SIG_DEF=30.
    PF_INIT_V_STD=0.02; PF_INIT_SPR=0.5; PF_RESAMP=0.5
    PF_ROUGH_P=0.2; PF_ROUGH_V=0.003; PF_GR_WIN=5; PF_GR_WT=0.3
    ANCC_ALPHA=0.998; ANCC_RN=0.002; ANCC_PN=0.005
    ANCC_IR=0.01; ANCC_IS=0.3; ANCC_RP=0.1; ANCC_RR=0.001

    @njit(cache=True)
    def _interp1(grid, v, vmin, step):
        i = int((v - vmin) / step)
        if i < 0: return grid[0]
        n = len(grid) - 1
        if i >= n: return grid[n]
        t = (v - vmin) / step - i
        return grid[i]*(1.-t) + grid[i+1]*t

    @njit(cache=True)
    def _resamp(pos, aux, w, N, rp, rv):
        cum = np.zeros(N+1)
        for j in range(N): cum[j+1]=cum[j]+w[j]
        u0=np.random.uniform(0.,1./N)
        np2=np.empty(N); na=np.empty(N); ci=0
        for j in range(N):
            u=u0+j/N
            while ci<N-1 and cum[ci+1]<u: ci+=1
            np2[j]=pos[ci]+rp*np.random.randn()
            na[j] =aux[ci]+rv*np.random.randn()
        return np2,na

    @njit(cache=True)
    def _beam_jit(sgr, tw_gr, si, BS, mc, es):
        """Beam search ±2 delta, Numba JIT."""
        n=len(sgr); nt=len(tw_gr); MAX=BS*6
        bidx=np.zeros(BS,np.int64); bidx[0]=si
        bcost=np.full(BS,1e30);     bcost[0]=0.; bn=np.int64(1)
        hI=np.zeros((n,BS),np.int64); hP=np.zeros((n,BS),np.int64)
        cI=np.zeros(MAX,np.int64); cC=np.full(MAX,1e30); cP=np.zeros(MAX,np.int64)
        for step in range(n):
            gv=sgr[step]; nc=np.int64(0)
            for bi in range(bn):
                idx=bidx[bi]; cost=bcost[bi]
                for d in range(-2,3):            # ±2: TVT can go down
                    ni=idx+d
                    if ni<0 or ni>=nt: continue
                    tot=cost+(gv-tw_gr[ni])**2/es+mc*(d if d>=0 else -d)
                    fnd=np.int64(-1)
                    for ci in range(nc):
                        if cI[ci]==ni: fnd=ci; break
                    if fnd>=0:
                        if tot<cC[fnd]: cC[fnd]=tot; cP[fnd]=bi
                    else:
                        if nc<MAX: cI[nc]=ni; cC[nc]=tot; cP[nc]=bi; nc+=1
            kept=min(BS,nc)
            for i in range(kept):
                mi=i
                for j in range(i+1,nc):
                    if cC[j]<cC[mi]: mi=j
                if mi!=i:
                    cI[i],cI[mi]=cI[mi],cI[i]
                    cC[i],cC[mi]=cC[mi],cC[i]
                    cP[i],cP[mi]=cP[mi],cP[i]
            hI[step,:kept]=cI[:kept]; hP[step,:kept]=cP[:kept]
            bidx[:kept]=cI[:kept]; bcost[:kept]=cC[:kept]; bn=kept
        best=np.int64(0)
        for b in range(1,bn):
            if bcost[b]<bcost[best]: best=b
        path=np.zeros(n,np.int64); b=best
        for s in range(n-1,-1,-1): path[s]=hI[s,b]; b=hP[s,b]
        return path

    @njit(cache=True)
    def _pf_ancc(md_v,z_v,gr_v,gg,vmin,step,gs,ls,ir,N,
                  ALPHA,RN,PN,IS,RP,RR,RESAMP):
        pos=np.empty(N); rate=np.empty(N); w=np.ones(N)/N
        for j in range(N):
            pos[j]=ls+IS*np.random.randn()
            rate[j]=ir+0.01*np.random.randn()
        pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.
        for i in range(len(md_v)):
            dm=md_v[i]-pm; dm=max(dm,1.)
            for j in range(N):
                rate[j]=ALPHA*rate[j]+RN*np.random.randn()
                pos[j]+=rate[j]*dm+PN*np.random.randn()
                tvt_j=pos[j]-z_v[i]
                tvt_j=max(tvt_j,vmin-50.); tvt_j=min(tvt_j,vmin+len(gg)*step+50.)
                pos[j]=tvt_j+z_v[i]
            if not np.isnan(gr_v[i]):
                ws=0.
                for j in range(N):
                    eg=_interp1(gg,pos[j]-z_v[i],vmin,step)
                    d=(gr_v[i]-eg)/gs
                    lk=max(np.exp(-0.5*d*d) if d*d<600. else 0.,1e-300)
                    w[j]*=lk; ws+=w[j]
                if ws>0.:
                    for j in range(N): w[j]/=ws
                else:
                    for j in range(N): w[j]=1./N
            ne=0.
            for j in range(N): ne+=w[j]*w[j]
            if 1./ne<RESAMP*N:
                pos,rate=_resamp(pos,rate,w,N,RP,RR)
                for j in range(N): w[j]=1./N
            tv=0.
            for j in range(N): tv+=w[j]*(pos[j]-z_v[i])
            pts[i]=tv; va=0.
            for j in range(N): va+=w[j]*(pos[j]-z_v[i]-tv)**2
            std_[i]=va**0.5; pm=md_v[i]
        return pts,std_

    @njit(cache=True)
    def _pf_z(md_v,z_v,gr_v,gr_sm_v,gg_p,gg_s,vmin,step,
              gs,ip,iv,beta,icpt,zsig,N,
              MOM,VN,PN,GR_WT,RP,RV,RESAMP):
        pos=np.empty(N); vel=np.empty(N); w=np.ones(N)/N
        for j in range(N):
            pos[j]=ip+0.5*np.random.randn()
            vel[j]=iv+0.02*np.random.randn()
        pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.; pz=z_v[0]-1.
        for i in range(len(md_v)):
            dm=md_v[i]-pm; dm=max(dm,1.)
            dzd=(z_v[i]-pz)/dm; ve=beta*dzd+icpt
            for j in range(N):
                vel[j]=MOM*vel[j]+VN*np.random.randn()
                pos[j]+=vel[j]*dm+PN*np.random.randn()
                pos[j]=max(pos[j],vmin-50.); pos[j]=min(pos[j],vmin+len(gg_p)*step+50.)
            if not np.isnan(gr_v[i]):
                ws=0.
                for j in range(N):
                    ep=_interp1(gg_p,pos[j],vmin,step)
                    dp=(gr_v[i]-ep)/gs
                    lp=max(np.exp(-0.5*dp*dp) if dp*dp<600. else 0.,1e-300)
                    if not np.isnan(gr_sm_v[i]):
                        es=_interp1(gg_s,pos[j],vmin,step)
                        ds=(gr_sm_v[i]-es)/(gs*1.5)
                        ls=max(np.exp(-0.5*ds*ds) if ds*ds<600. else 0.,1e-300)
                        lk=(1.-GR_WT)*lp+GR_WT*ls
                    else: lk=lp
                    lk=max(lk,1e-300); w[j]*=lk; ws+=w[j]
                if ws>0.:
                    for j in range(N): w[j]/=ws
                else:
                    for j in range(N): w[j]=1./N
            ws2=0.
            for j in range(N):
                dv=(vel[j]-ve)/max(zsig*2.,0.005)
                lz=max(np.exp(-0.5*dv*dv) if dv*dv<600. else 0.,1e-300)
                w[j]*=lz; ws2+=w[j]
            if ws2>0.:
                for j in range(N): w[j]/=ws2
            else:
                for j in range(N): w[j]=1./N
            ne=0.
            for j in range(N): ne+=w[j]*w[j]
            if 1./ne<RESAMP*N:
                pos,vel=_resamp(pos,vel,w,N,RP,RV)
                for j in range(N): w[j]=1./N
            wm=0.
            for j in range(N): wm+=w[j]*pos[j]
            pts[i]=wm; va=0.
            for j in range(N): va+=w[j]*(pos[j]-wm)**2
            std_[i]=va**0.5; pm=md_v[i]; pz=z_v[i]
        return pts,std_

    # Dense grid for O(1) typewell lookup
    def _grid(tw_tvt,tw_gr,step=0.2):
        tmin=float(tw_tvt.min()); tmax=float(tw_tvt.max())
        tvt_g=np.arange(tmin,tmax+step,step)
        return np.interp(tvt_g,tw_tvt,tw_gr).astype(np.float64),float(tmin),float(step)

    def _gr_sig(hw,tw_tvt,tw_gr):
        kn=hw[hw['TVT_input'].notna()&hw['GR'].notna()]
        if len(kn)<20: return float(PF_GR_SIG_DEF)
        return float(np.clip(np.std(kn['GR'].values-np.interp(kn['TVT_input'].values,tw_tvt,tw_gr)),
                              PF_GR_SIG_MIN,PF_GR_SIG_MAX))

    def _nn(arr,v):
        i=int(np.searchsorted(arr,v,'left'))
        if i>=len(arr): return len(arr)-1
        if i>0 and abs(arr[i-1]-v)<=abs(arr[i]-v): return i-1
        return i

    def _smooth(vals,fb,r):
        s=pd.Series(vals,dtype='float32').interpolate(limit_direction='both').fillna(fb)
        return (s.rolling(r*2+1,center=True,min_periods=1).mean() if r>0 else s).to_numpy(np.float32)

    def beam_search(gr_h,tw_tvt,tw_gr,start_tvt,bs,mc,es,r):
        si=_nn(tw_tvt,start_tvt)
        sgr=_smooth(gr_h,float(np.nanmean(tw_gr)),r).astype(np.float64)
        path=_beam_jit(sgr,tw_gr.astype(np.float64),si,bs,float(mc),float(es))
        return tw_tvt[path].astype(np.float32)

    def run_pf_ancc(hw,tw_tvt,tw_gr,N=ANCC_N):
        gs=_gr_sig(hw,tw_tvt,tw_gr)
        kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
        if len(ev)==0: return np.array([]),np.array([])
        ls=float(kn['TVT_input'].iloc[-1]+kn['Z'].iloc[-1])
        tail=kn.tail(30); dt=np.diff(tail['TVT_input'].values)
        dz=np.diff(tail['Z'].values); dm=np.diff(tail['MD'].values); m=dm>0
        ir=float(np.median((dt+dz)[m]/dm[m])) if m.sum()>=3 else 0.
        gg,gmin,gst=_grid(tw_tvt,tw_gr)
        pts,std=_pf_ancc(ev['MD'].values.astype(np.float64),ev['Z'].values.astype(np.float64),
                          ev['GR'].values.astype(np.float64),gg,gmin,gst,
                          gs,ls,ir,N,ANCC_ALPHA,ANCC_RN,ANCC_PN,ANCC_IS,ANCC_RP,ANCC_RR,PF_RESAMP)
        return pts.astype(np.float32),std.astype(np.float32)

    def run_pf_z(hw,tw_tvt,tw_gr,N=PF_N):
        gs=_gr_sig(hw,tw_tvt,tw_gr)
        tw_s=pd.Series(tw_gr).rolling(PF_GR_WIN,center=True,min_periods=1).mean().values.astype(np.float32)
        kna=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
        if len(ev)==0: return np.array([]),np.array([])
        dz_k=np.diff(kna['Z'].values); dvt=np.diff(kna['TVT_input'].values)
        dmd_k=np.diff(kna['MD'].values); m2=dmd_k>0
        if m2.sum()>=10:
            vz=dz_k[m2]/dmd_k[m2]; vt=dvt[m2]/dmd_k[m2]
            A=np.column_stack([vz,np.ones_like(vz)]); c,_,_,_=np.linalg.lstsq(A,vt,rcond=None)
            beta,icpt,zsig=float(c[0]),float(c[1]),max(float(np.std(vt-(c[0]*vz+c[1]))),0.001)
        else: beta,icpt,zsig=-1.,0.,0.1
        t2=kna.tail(20); dvt2=np.diff(t2['TVT_input'].values); dmd2=np.diff(t2['MD'].values); m3=dmd2>0
        iv=float(np.median(dvt2[m3]/dmd2[m3])) if m3.sum()>=3 else 0.
        gg,gmin,gst=_grid(tw_tvt,tw_gr)
        gs2,_,_=_grid(tw_tvt,tw_s)
        gr_sm=hw['GR'].rolling(PF_GR_WIN,center=True,min_periods=1).mean()
        pts,std=_pf_z(ev['MD'].values.astype(np.float64),ev['Z'].values.astype(np.float64),
                       ev['GR'].values.astype(np.float64),
                       gr_sm.loc[ev.index].values.astype(np.float64),
                       gg,gs2,gmin,gst,gs,float(kna['TVT_input'].iloc[-1]),iv,
                       beta,icpt,zsig,N,
                       PF_MOM,PF_VN,PF_PN,PF_GR_WT,PF_ROUGH_P,PF_ROUGH_V,PF_RESAMP)
        return pts.astype(np.float32),std.astype(np.float32)


    _md=np.linspace(1,50,20,np.float64); _z=np.zeros(20,np.float64); _gr=np.full(20,50.,np.float64)
    _gg=np.linspace(45,55,100,np.float64)
    _pf_ancc(_md,_z,_gr,_gg,45.,0.1,20.,50.,0.,8,0.998,0.002,0.005,0.3,0.1,0.001,0.5)
    _pf_z(_md,_z,_gr,_gr,_gg,_gg,45.,0.1,20.,50.,0.,-1.,0.,0.1,8,0.993,0.005,0.01,0.3,0.2,0.003,0.5)
    _beam_jit(np.random.randn(30),np.random.randn(50),25,8,15.,100.)

    def robust_slope(x,y,w=None):
        x=np.asarray(x,float); y=np.asarray(y,float)
        m=np.isfinite(x)&np.isfinite(y)
        if m.sum()<2 or np.std(x[m])<1e-6: return 0.
        return float(np.polyfit(x[m],y[m],1)[0])

    def affine_cal(kgr,tw_at_k,min_pts=20):
        v=np.isfinite(kgr)&np.isfinite(tw_at_k)
        if v.sum()<min_pts or np.std(tw_at_k[v])<1e-6:
            return 1.,float(np.nanmean(kgr)-np.nanmean(tw_at_k)) if v.any() else 0.
        a,b=np.polyfit(tw_at_k[v],kgr[v],1); return float(a),float(b)

    def seg_b_well(ktvt,kz,form_col):
        """Segment b_well: early/mid/late thirds + full prefix.
        Returns (b_full, b_early, b_mid, b_late, b_wls) for feature richness."""
        bv=ktvt+kz-form_col; n=len(bv)
        b_full=float(np.median(bv))
        b_late=float(np.median(bv[max(0,n-50):])) if n>=5 else b_full
        t1,t2=n//3, 2*n//3
        b_early=float(np.median(bv[:max(1,t1)])) if t1>0 else b_full
        b_mid  =float(np.median(bv[t1:max(t1+1,t2)])) if t2>t1 else b_full
        # WLS (tail-upweighted)
        w=np.exp(0.02*np.arange(n)); w/=w.sum()
        b_wls=float(np.dot(w,bv))
        return b_full,b_early,b_mid,b_late,b_wls

    def multi_scale_ncc(kgr,ktvt,hgr,hws=(8,15,25),stride=3):
        """Multi-scale NCC. Returns score-weighted ensemble + per-scale signals."""
        out=[]
        for hw in hws:
            win=2*hw+1; nk=len(kgr); nh=len(hgr)
            if nk<win+1 or nh==0:
                out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
            kg=pd.Series(kgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
            hg=pd.Series(hgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
            sts=np.arange(0,nk-win+1,stride,dtype=np.int32); M=len(sts)
            if M==0:
                out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
            C=kg[sts[:,None]+np.arange(win,dtype=np.int32)[None,:]].astype(np.float32)
            Cn=(C-C.mean(1,keepdims=True))/(C.std(1,keepdims=True)+1e-6)
            hp=np.pad(hg,hw,mode='edge')
            H=hp[np.arange(nh)[:,None]+np.arange(win)[None,:]].astype(np.float32)
            Hn=(H-H.mean(1,keepdims=True))/(H.std(1,keepdims=True)+1e-6)
            ncc=Hn@Cn.T/win; best=ncc.argmax(1); score=ncc.max(1).astype(np.float32)
            out.append((ktvt[np.clip(sts[best]+hw,0,nk-1)].astype(np.float32),score))
        # Score-weighted ensemble (NEW: softmax-weighted combination)
        tvts=np.stack([o[0] for o in out],1); scores=np.stack([o[1] for o in out],1)
        sw=np.exp(3.*scores); sw/=sw.sum(1,keepdims=True)+1e-9
        sc_ens=(tvts*sw).sum(1).astype(np.float32)
        return out, sc_ens   # [(tvt8,sc8),(tvt15,sc15),(tvt25,sc25)], ensemble

    class FormationPlaneKNN:
        def __init__(self,well_ids,data_dir):
            rows=[]
            for wid in well_ids:
                p=data_dir/f'{wid}__horizontal_well.csv'
                try: df=pd.read_csv(p,usecols=['X','Y']+FORMATIONS).dropna()
                except: continue
                if len(df)==0: continue
                row={'wid':wid,'x':float(df['X'].median()),'y':float(df['Y'].median())}
                for c in FORMATIONS: row[f'{c}_m']=float(df[c].median())
                rows.append(row)
            self.df=pd.DataFrame(rows); self.wmap={w:i for i,w in enumerate(self.df['wid'])}
            xy=self.df[['x','y']].to_numpy(); self.scale=np.where(xy.std(0)<1e-3,1.,xy.std(0))
            self.tree=cKDTree(xy/self.scale)
            self.xa=self.df['x'].to_numpy(); self.ya=self.df['y'].to_numpy()
            self.fa=self.df[[f'{c}_m' for c in FORMATIONS]].to_numpy(np.float64)

        def impute(self,xy_q,self_wid=None,k=PLANE_K):
            q=xy_q/self.scale; nf=min(k+5,len(self.df))
            dist,idx=self.tree.query(q,k=nf,workers=-1)
            if self_wid in self.wmap: dist=np.where(idx==self.wmap[self_wid],np.inf,dist)
            ord=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
            dk=np.take_along_axis(dist,ord,1); ik=np.take_along_axis(idx,ord,1)
            vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.).astype(np.float64)
            xn=self.xa[ik]; yn=self.ya[ik]; fn=self.fa[ik]; wx=w*xn; wy=w*yn
            A=np.zeros((len(q),3,3))
            A[:,0,0]=(wx*xn).sum(1); A[:,0,1]=(wx*yn).sum(1); A[:,0,2]=wx.sum(1)
            A[:,1,0]=A[:,0,1]; A[:,1,1]=(wy*yn).sum(1); A[:,1,2]=wy.sum(1)
            A[:,2,0]=A[:,0,2]; A[:,2,1]=A[:,1,2]; A[:,2,2]=w.sum(1)
            A[:,0,0]+=1e-9; A[:,1,1]+=1e-9; A[:,2,2]+=1e-9
            rhs=np.stack([(wx[:,:,None]*fn).sum(1),(wy[:,:,None]*fn).sum(1),(w[:,:,None]*fn).sum(1)],1)
            try: coef=np.linalg.solve(A,rhs)
            except:
                coef=np.zeros((len(q),3,6))
                for r in range(len(q)):
                    try: coef[r]=np.linalg.pinv(A[r])@rhs[r]
                    except: pass
            Xq=xy_q[:,0]; Yq=xy_q[:,1]
            pred=(Xq[:,None]*coef[:,0,:]+Yq[:,None]*coef[:,1,:]+coef[:,2,:]).astype(np.float32)
            pred[~vk.any(1)]=self.fa.mean(0)
            return pred,np.where(vk,dk,np.inf).min(1).astype(np.float32)

    class DenseANCCImputer:
        def __init__(self,well_ids,data_dir,spw=DENSE_SPW):
            xs,ys,anccs,wids=[],[],[],[]
            for wid in well_ids:
                p=data_dir/f'{wid}__horizontal_well.csv'
                try: df=pd.read_csv(p,usecols=['X','Y','ANCC']).dropna()
                except: continue
                if len(df)==0: continue
                ix=np.linspace(0,len(df)-1,min(spw,len(df)),dtype=int); s=df.iloc[ix]
                xs.append(s['X'].values); ys.append(s['Y'].values)
                anccs.append(s['ANCC'].values); wids.extend([wid]*len(s))
            self.xy=np.column_stack([np.concatenate(xs),np.concatenate(ys)])
            self.ancc=np.concatenate(anccs).astype(np.float32); self.wids=np.array(wids)
            self.scale=np.where(self.xy.std(0)<1e-3,1.,self.xy.std(0))
            self.tree=cKDTree(self.xy/self.scale)

        def impute(self,xy_q,self_wid=None,k=DENSE_K,nfetch=5000):
            xy_q=np.atleast_2d(xy_q); q=xy_q/self.scale; nf=min(nfetch,len(self.ancc))
            dist,idx=self.tree.query(q,k=nf,workers=-1)
            if self_wid: dist=np.where(self.wids[idx]==self_wid,np.inf,dist)
            ord=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
            dk=np.take_along_axis(dist,ord,1); ik=np.take_along_axis(idx,ord,1)
            vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.)
            sw=w.sum(1); safe=np.where(sw<1e-9,1.,sw); an=self.ancc[ik]
            ap=(an*w).sum(1)/safe; ap=np.where(sw<1e-9,float(self.ancc.mean()),ap)
            var=((an-ap[:,None])**2*w).sum(1)/safe
            return ap.astype(np.float32),np.sqrt(np.maximum(var,0.)).astype(np.float32),np.where(vk,dk,np.inf).min(1).astype(np.float32)

    hw_paths=sorted((CFG.dataset_path / "train").glob('*__horizontal_well.csv'))
    train_wids=[p.stem.replace('__horizontal_well','') for p in hw_paths]
    FI=FormationPlaneKNN(train_wids,CFG.dataset_path / "train")
    DI=DenseANCCImputer(train_wids,CFG.dataset_path / "train")

    _FI=FI; _DI=DI
    ANCH_OFFS=np.array([-80,-40,-20,-10,-5,0,5,10,20,40,80],np.float32)
    BEAM_OFFS=np.array([-40,-20,-10,-5,-3,0,3,5,10,20,40],np.float32)
    SC_OFFS  =np.array([-30,-15,-8,-4,-2,0,2,4,8,15,30],np.float32)
    PF_OFFS  =np.array([-30,-15,-8,-4,-2,0,2,4,8,15,30],np.float32)

    def build_well(hw_path,tw_path,is_train):
        global _FI,_DI
        wid=Path(hw_path).stem.replace('__horizontal_well','')
        try:
            hw=pd.read_csv(hw_path); tw=pd.read_csv(tw_path).sort_values('TVT')
        except: return None
        if is_train and 'TVT' not in hw.columns: return None
        kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
        if len(ev)==0 or len(kn)<10: return None
        if is_train and hw['TVT'].isna().all(): return None
        tw_tvt=tw['TVT'].to_numpy(np.float32); tw_gr=tw['GR'].to_numpy(np.float32)
        if len(tw_tvt)<3: return None

        pf_a,std_a=run_pf_ancc(hw,tw_tvt,tw_gr)
        if len(pf_a)==0: return None
        pf_z,std_z=run_pf_z(hw,tw_tvt,tw_gr)
        pf_use=pf_a.astype(np.float32); std_use=std_a.astype(np.float32)
        has_z=len(pf_z)==len(pf_a) and not np.any(np.isnan(pf_z))

        lk=kn.iloc[-1]; last_tvt=float(lk['TVT_input'])
        gr_full=hw['GR'].astype(float).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
        hgr=gr_full.iloc[ev.index[0]:].to_numpy(np.float32)
        kgr=gr_full.iloc[:len(kn)].to_numpy(np.float32)

        # 7 beams (Numba JIT ±2)
        bpaths={}
        for (bs,mc,es,r,tag) in BEAMS:
            bpaths[tag]=beam_search(hgr,tw_tvt,tw_gr,last_tvt,bs,mc,es,r)
        beam_ref=(bpaths['cons']+bpaths['sm5'])/2.

        # Multi-scale NCC → score-weighted ensemble
        ktvt=kn['TVT_input'].to_numpy(np.float32)
        sc_res,sc_ens=multi_scale_ncc(kgr,ktvt,hgr,hws=(8,15,25),stride=3)
        sc8,sc8s=sc_res[0]; sc15,sc15s=sc_res[1]; sc25,sc25s=sc_res[2]
        sc_cons=(sc8+sc15+sc25)/3.
        sc_trust=float(np.clip(len(kn)/200.,0.,0.6))
        hyb_ref=(1-sc_trust)*beam_ref+sc_trust*sc_ens  # use ensemble not single

        tw_at_k=np.interp(ktvt,tw_tvt,tw_gr).astype(np.float32)
        a_cal,b_cal=affine_cal(kgr,tw_at_k)
        kmd=kn['MD'].to_numpy(np.float32); kz=kn['Z'].to_numpy(np.float32)
        pfx_rmse=float(np.sqrt(np.mean((kgr-tw_at_k)**2)))
        slp_all=robust_slope(kmd,ktvt); slp_50=robust_slope(kmd[-50:],ktvt[-50:])
        slp_z=robust_slope(kz,ktvt)

        swid=wid if is_train else None
        xy_ev=ev[['X','Y']].to_numpy(np.float64); xy_kn=kn[['X','Y']].to_numpy(np.float64)
        form_ev,knn_d=_FI.impute(xy_ev,self_wid=swid)
        form_kn,_   =_FI.impute(xy_kn,self_wid=swid)
        z_kn=kn['Z'].to_numpy(np.float32); z_ev=ev['Z'].to_numpy(np.float32)

        # Per-formation: segment b_well (early/mid/late/wls) + TVT + known-zone RMSE
        tvt_fs={}; form_rmse={}; form_list=[]
        for fi2,fn in enumerate(FORMATIONS):
            b_full,b_early,b_mid,b_late,b_wls=seg_b_well(ktvt,z_kn,form_kn[:,fi2])
            tvt_f  =(-z_ev+form_ev[:,fi2]+b_full ).astype(np.float32)
            tvt_fw =(-z_ev+form_ev[:,fi2]+b_wls  ).astype(np.float32)
            tvt_f50=(-z_ev+form_ev[:,fi2]+b_late ).astype(np.float32)
            tvt_fs[f'tvtF_{fn}']=tvt_f; tvt_fs[f'tvtFw_{fn}']=tvt_fw
            tvt_fs[f'tvtF50_{fn}']=tvt_f50
            tvt_fs[f'bw_{fn}']=np.float32(b_full); tvt_fs[f'bww_{fn}']=np.float32(b_wls)
            tvt_fs[f'bw50_{fn}']=np.float32(b_late)
            tvt_fs[f'bw_early_{fn}']=np.float32(b_early)   # NEW: early segment
            tvt_fs[f'bw_mid_{fn}']=np.float32(b_mid)       # NEW: mid segment
            form_rmse[fn]=float(np.sqrt(np.mean((ktvt-(-z_kn+form_kn[:,fi2]+b_full))**2)))
            form_list.append(tvt_f)

        fs=np.stack(form_list,1)
        form_mean_d=(fs.mean(1)-last_tvt).astype(np.float32)
        form_std_d =fs.std(1).astype(np.float32)
        form_rng_d =(fs.max(1)-fs.min(1)).astype(np.float32)

        d_ancc,d_std,d_dist=_DI.impute(xy_ev,self_wid=swid)
        d_kn,d_std_kn,_=_DI.impute(xy_kn,self_wid=swid)
        b_vd=ktvt+z_kn-d_kn
        _,b_de,b_dm,b_dl,b_dw=seg_b_well(ktvt,z_kn,d_kn)
        b_d=float(np.median(b_vd))
        tvt_dense  =(-z_ev+d_ancc+b_d  ).astype(np.float32)
        tvt_densew =(-z_ev+d_ancc+b_dw ).astype(np.float32)
        tvt_dense50=(-z_ev+d_ancc+b_dl ).astype(np.float32)
        res_kn=ktvt+z_kn-d_kn
        d_rmse=float(np.sqrt(np.mean(res_kn**2))); d_bias=float(np.mean(res_kn)); d_nb_std=float(np.mean(d_std_kn))

        all_sigs=[pf_use]+[p for p in bpaths.values()]+[sc8,sc15,sc25,sc_ens,tvt_fs['tvtF_ANCC'],tvt_dense]
        sig_mat=np.stack(all_sigs,1)
        sig_std=sig_mat.std(1).astype(np.float32)
        sig_mean=(sig_mat.mean(1)-last_tvt).astype(np.float32)

        gr_s=pd.Series(gr_full.values); rolls={}
        for w in [5,21,51,101]:
            r=gr_s.rolling(w,center=True,min_periods=1)
            rolls[f'grm{w}']=r.mean().iloc[ev.index].values.astype(np.float32)
            rolls[f'grs{w}']=r.std().fillna(0).iloc[ev.index].values.astype(np.float32)
        for lag in [1,5,15,30]:
            rolls[f'glag{lag}']=gr_s.shift(lag).bfill().iloc[ev.index].values.astype(np.float32)
            rolls[f'glead{lag}']=gr_s.shift(-lag).ffill().iloc[ev.index].values.astype(np.float32)
        gr_d1=gr_s.diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
        gr_d2=gr_s.diff().diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
        gr_env=gr_s.rolling(21,center=True,min_periods=1).max().iloc[ev.index].values.astype(np.float32)
        gr_nrg=np.sqrt(np.maximum((gr_s**2).rolling(21,center=True,min_periods=1).mean(),0.)
                       ).iloc[ev.index].values.astype(np.float32)

        hmd=ev['MD'].to_numpy(np.float32); md_since=hmd-float(lk['MD'])
        slp_b_all=(last_tvt+slp_all*md_since).astype(np.float32)
        slp_b_50 =(last_tvt+slp_50 *md_since).astype(np.float32)

        mdd=hw['MD'].diff().replace(0,np.nan)
        dzdmd=(hw['Z'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
        dxdmd=(hw['X'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
        dydmd=(hw['Y'].diff()/mdd).iloc[ev.index].values.astype(np.float32)

        nh=len(ev); frac=(np.arange(nh)/max(nh-1,1)).astype(np.float32)
        def sc(v): return np.full(nh,np.float32(v),np.float32)

        feats={
            'well':wid,'id':[f'{wid}_{i}' for i in ev.index],
            'last_known_tvt':sc(last_tvt),
            'pf_ancc':pf_use,'pf_ancc_std':std_use,
            'pf_ancc_delta':(pf_use-last_tvt).astype(np.float32),
            'pf_z':(pf_z.astype(np.float32) if has_z else sc(last_tvt)),
            'pf_z_delta':((pf_z-last_tvt).astype(np.float32) if has_z else sc(0.)),
            'pf_vs_z':((pf_use-pf_z.astype(np.float32)) if has_z else sc(0.)),
            **{f'beam_{t}_d':(p-np.float32(last_tvt)).astype(np.float32) for t,p in bpaths.items()},
            'beam_mean_d':np.stack([(p-last_tvt) for p in bpaths.values()],1).mean(1).astype(np.float32),
            'beam_std_d': np.stack([(p-last_tvt) for p in bpaths.values()],1).std(1).astype(np.float32),
            'beam_med_d': np.median(np.stack([(p-last_tvt) for p in bpaths.values()],1),1).astype(np.float32),
            'sc8_d':(sc8-np.float32(last_tvt)).astype(np.float32),'sc8_sc':sc8s,
            'sc15_d':(sc15-np.float32(last_tvt)).astype(np.float32),'sc15_sc':sc15s,
            'sc25_d':(sc25-np.float32(last_tvt)).astype(np.float32),'sc25_sc':sc25s,
            'sc_cons_d':(sc_cons-np.float32(last_tvt)).astype(np.float32),
            'sc_ens_d':(sc_ens-np.float32(last_tvt)).astype(np.float32),  # score-weighted ensemble
            'sc_trust':sc(sc_trust),'hyb_d':(hyb_ref-np.float32(last_tvt)).astype(np.float32),
            'sig_std':sig_std,'sig_mean_d':sig_mean,
            **tvt_fs,
            **{f'frm_rmse_{fn}':sc(form_rmse[fn]) for fn in FORMATIONS},
            'form_mean_d':form_mean_d,'form_std_d':form_std_d,'form_rng_d':form_rng_d,
            'spatial_ancc_d':(form_ev[:,0]-np.float32(np.interp(last_tvt,tw_tvt,tw_gr))),
            'spatial_knn_dist':knn_d,
            'dense_ancc':d_ancc,'dense_std':d_std,'dense_dist':d_dist,
            'tvt_dense_d' :(tvt_dense -last_tvt).astype(np.float32),
            'tvt_densew_d':(tvt_densew-last_tvt).astype(np.float32),
            'tvt_dense50_d':(tvt_dense50-last_tvt).astype(np.float32),
            'dense_rmse':sc(d_rmse),'dense_bias':sc(d_bias),'dense_nb_std':sc(d_nb_std),
            'pf_vs_spatial':(pf_use-tvt_fs['tvtF_ANCC']).astype(np.float32),
            'pf_vs_dense':(pf_use-tvt_dense).astype(np.float32),
            'spatial_vs_dense':(tvt_fs['tvtF_ANCC']-tvt_dense).astype(np.float32),
            'beam_vs_spatial':(bpaths['cons']-tvt_fs['tvtF_ANCC']).astype(np.float32),
            'sc_vs_beam':(sc_ens-bpaths['cons']).astype(np.float32),
            'cal_a':sc(a_cal),'cal_b':sc(b_cal),
            'pfx_rmse':sc(pfx_rmse),'known_len':sc(len(kn)),'eval_len':sc(nh),
            'slp_all':sc(slp_all),'slp_50':sc(slp_50),'slp_z':sc(slp_z),
            'slp_b_d_all':(slp_b_all-last_tvt).astype(np.float32),
            'slp_b_d_50': (slp_b_50 -last_tvt).astype(np.float32),
            'ktvt_range':sc(float(np.ptp(ktvt))),'ktvt_std':sc(float(ktvt.std())),
            'md_since':md_since,'frac':frac,'frac2':frac**2,'sqrt_frac':np.sqrt(frac),
            'z':z_ev,
            'dx':(ev['X']-float(lk['X'])).to_numpy(np.float32),
            'dy':(ev['Y']-float(lk['Y'])).to_numpy(np.float32),
            'dz':(z_ev-float(lk['Z'])).astype(np.float32),
            'dxy':np.sqrt((ev['X']-float(lk['X']))**2+(ev['Y']-float(lk['Y']))**2).to_numpy(np.float32),
            'dzdmd':dzdmd,'dxdmd':dxdmd,'dydmd':dydmd,
            'gr':hgr,'gr_d1':gr_d1,'gr_d2':gr_d2,'gr_env':gr_env,'gr_nrg':gr_nrg,
            'gr_vs_tw_anc':hgr-np.float32(np.interp(last_tvt,tw_tvt,tw_gr)),
            'gr_vs_slp_all':hgr-np.interp(slp_b_all,tw_tvt,tw_gr).astype(np.float32),
            **{f'tda{int(o)}' :hgr-np.float32(np.interp(last_tvt+o,tw_tvt,tw_gr)) for o in ANCH_OFFS},
            **{f'tdbc{int(o)}':hgr-np.interp(beam_ref+o,tw_tvt,tw_gr).astype(np.float32) for o in BEAM_OFFS},
            **{f'tdsc{int(o)}':hgr-np.interp(sc_ens+o,tw_tvt,tw_gr).astype(np.float32) for o in SC_OFFS},
            **{f'tdpf{int(o)}':hgr-np.interp(pf_use+o,tw_tvt,tw_gr).astype(np.float32) for o in PF_OFFS},
            'tw_range':sc(float(np.ptp(tw_tvt))),'tw_gr_mean':sc(float(tw_gr.mean())),
        }
        for k,v in rolls.items(): feats[k]=v
        result=pd.DataFrame(feats)
        if is_train:
            if 'TVT' not in ev.columns or ev['TVT'].isna().all(): return None
            result['target']=(ev['TVT'].to_numpy(np.float32)-np.float32(last_tvt))
        return result

    def build_dataset(paths,is_train,label):
        args=[(str(p),str(p.parent/f'{p.stem.replace("__horizontal_well","")}__typewell.csv'),is_train)
              for p in paths
              if (p.parent/f'{p.stem.replace("__horizontal_well","")}__typewell.csv').exists()]
        t0=time.time()
        res=Parallel(n_jobs=NCPU,prefer='threads',verbose=3)(
            delayed(build_well)(hp,tp,it) for hp,tp,it in args)
        parts=[r for r in res if r is not None]
        return pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()

    if (CFG.artifacts_path / "data" / "train.csv").exists():
        train_df = pd.read_csv(CFG.artifacts_path / "data" / "train.csv", low_memory=False)
    else:
        train_paths = sorted((CFG.dataset_path / "train").glob('*__horizontal_well.csv'))
        train_df = build_dataset(train_paths, is_train=True, label="train")    

    test_paths = sorted((CFG.dataset_path / "test").glob('*__horizontal_well.csv'))
    test_df = build_dataset(test_paths, is_train=False, label="test")

    features = [c for c in train_df.columns if c not in {'well','id','target'}]

    X = train_df[features]
    y = train_df['target']
    g = train_df['well']

    X_test = test_df[features]

    lgb_params = [
        dict(
            boosting_type="gbdt", 
            num_leaves=255, 
            min_child_samples=15,
            subsample=0.8, 
            subsample_freq=1, 
            colsample_bytree=0.8,
            reg_lambda=3.0, 
            reg_alpha=0.05, 
            objective="regression",
            verbose=-1, 
            n_jobs=-1, 
            device_type="gpu", 
            gpu_use_dp=False, 
            max_bin=255,
            learning_rate=0.030, 
            n_estimators=5000, 
            seed=123
        ),
        dict(
            n_jobs=-1, 
            verbose=-1, 
            reg_alpha=10.788188919840913, 
            subsample=0.47437582748953966, 
            num_leaves=64, 
            reg_lambda=95.75401894533888, 
            n_estimators=10000,
            random_state=0,
            boosting_type='gbdt', 
            learning_rate=0.00934485794382918,
            colsample_bytree=0.39283351290380497,
            min_child_weight=0.24081152127177283, 
            min_child_samples=40,
            device='gpu',
        ),
        dict(
            n_jobs=-1, 
            verbose=-1, 
            reg_alpha=10.788188919840913, 
            subsample=0.47437582748953966, 
            num_leaves=64, 
            reg_lambda=95.75401894533888, 
            n_estimators=10000,
            random_state=29,
            boosting_type='gbdt', 
            learning_rate=0.00934485794382918,
            colsample_bytree=0.39283351290380497,
            min_child_weight=0.24081152127177283, 
            min_child_samples=40,
            device='gpu',
        ),
    ]

    cb_params = [
        dict(
            iterations=8000, 
            depth=7, 
            l2_leaf_reg=2.0,
            min_data_in_leaf=15, 
            border_count=254,
            loss_function="RMSE", 
            task_type="GPU", 
            devices="0",
            od_type="Iter", 
            od_wait=300, 
            verbose=0,
            learning_rate=0.020, 
            random_seed=7
        ),
        dict(
            iterations=8000, 
            depth=7, 
            l2_leaf_reg=2.0,
            min_data_in_leaf=15, 
            border_count=254,
            loss_function="RMSE", 
            task_type="GPU", 
            devices="0",
            od_type="Iter", 
            od_wait=300, 
            verbose=0,
            learning_rate=0.030, 
            random_seed=123
        ),
    ]

    ridge_params = {
        "random_state": 42,
        "alpha": 1.6602834637650032,
        "tol": 0.0005030247295617308,
        "positive": True,
        "fit_intercept": True
    }

    pp_params = {
        'alpha': 1.0,
        'tau': 85,
        'w_pf': 0.09
    }

    oof_preds = {}
    test_preds = {}

    overall_scores = {}
    fold_scores = {}

    for i, params in enumerate(lgb_params):   
        save_path = f"models/lightgbm-{i+1}"
    
        if (CFG.artifacts_path / save_path).exists():
            print(f"Loading lightgbm-{i+1} from disk...")
        
            trainer_path = CFG.artifacts_path / save_path / 'models.pkl'
            if not trainer_path.exists():
                trainer_paths = sorted((CFG.artifacts_path / save_path).glob('*.pkl'))
                if not trainer_paths:
                    raise FileNotFoundError(f'No pickle files found under {CFG.artifacts_path / save_path}')
                trainer_path = trainer_paths[0]
            trainer = joblib.load(trainer_path)
        
            print(f"Loaded lightgbm-{i+1} with overall RMSE: {trainer.overall_score:.4f}\n")
        else:
     
            trainer = Trainer(
                estimator=LGBMRegressor(**params),
                task="regression",
                metric=CFG.metric,
                cv=CFG.cv,
                cv_args={"groups": g},
                use_early_stopping=True,
                verbose=True,
                save=True,
                save_path=save_path
            )
        
            trainer.fit(
                X, 
                y,
                fit_args={
                    "eval_metric": "rmse",
                    "callbacks": [
                        log_evaluation(period=250), 
                        early_stopping(stopping_rounds=250)
                    ]
                }
            )
            print("\n\n")

        oof_preds[f"lightgbm-{i+1}"] = trainer.oof_preds
        test_preds[f"lightgbm-{i+1}"] = trainer.predict(X_test)
        overall_scores[f"lightgbm-{i+1}"] = trainer.overall_score
        fold_scores[f"lightgbm-{i+1}"] = trainer.fold_scores

    for i, params in enumerate(cb_params):    
        save_path = f"models/catboost-{i+1}"
        if (CFG.artifacts_path / save_path).exists():
            print(f"Loading catboost-{i+1} from disk...")
        
            trainer_path = CFG.artifacts_path / save_path / 'models.pkl'
            if not trainer_path.exists():
                trainer_paths = sorted((CFG.artifacts_path / save_path).glob('*.pkl'))
                if not trainer_paths:
                    raise FileNotFoundError(f'No pickle files found under {CFG.artifacts_path / save_path}')
                trainer_path = trainer_paths[0]
            trainer = joblib.load(trainer_path)
        
            print(f"Loaded catboost-{i+1} with overall RMSE: {trainer.overall_score:.4f}\n")
        else:
            trainer = Trainer(
                estimator=CatBoostRegressor(**params),
                task="regression",
                metric=CFG.metric,
                cv=CFG.cv,
                cv_args={"groups": g},
                use_early_stopping=True,
                verbose=True,
                save=True,
                save_path=save_path
            )
        
            trainer.fit(
                X, 
                y,
                fit_args={
                    "verbose": 250,
                    "early_stopping_rounds": 250,
                    "use_best_model": True
                }
            )
            print("\n\n")

        oof_preds[f"catboost-{i+1}"] = trainer.oof_preds
        test_preds[f"catboost-{i+1}"] = trainer.predict(X_test)
        overall_scores[f"catboost-{i+1}"] = trainer.overall_score
        fold_scores[f"catboost-{i+1}"] = trainer.fold_scores

    oof_preds = pd.DataFrame(oof_preds)
    test_preds = pd.DataFrame(test_preds)

    ridge_trainer = Trainer(
        Ridge(**ridge_params),
        task="regression",
        metric=CFG.metric,
        cv=CFG.cv,
        cv_args={"groups": g},
        verbose=True
    )

    ridge_trainer.fit(oof_preds, y)

    ridge_oof_preds = ridge_trainer.oof_preds
    ridge_test_preds = ridge_trainer.predict(test_preds)

    overall_scores["ridge"] = ridge_trainer.overall_score
    fold_scores["ridge"] = ridge_trainer.fold_scores

    def apply_pp(df, md, pd_, alpha, tau, w_pf):
        d = md * (1-w_pf) + pd_ * w_pf
        if tau: 
            d *= (1.-np.exp(-np.maximum(df['md_since'].values,0.) / tau))
        
        return d * alpha

    def sg_smooth(df, col, sg_w=17, sg_p=3):
        df = df.copy()
    
        for _, g in df.groupby('well', sort=False):
            v = g[col].values
            n = len(v)
            wl = min(sg_w, n)
        
            if wl % 2 == 0: 
                wl -= 1
            
            if wl >= sg_p + 2: 
                v = savgol_filter(v, wl, sg_p)
            
            df.loc[g.index,col] = v
        
        return df

    base = train_df['last_known_tvt'].values
    ytrue = y.values + base

    pf_oof = (train_df['pf_ancc'].values - base)

    d = apply_pp(train_df, ridge_oof_preds, pf_oof, **pp_params)
    ridge_score = root_mean_squared_error(ytrue, base + d)

    overall_scores["ridge (pp)"] = ridge_score
    fold_scores["ridge (pp)"] = [ridge_score] * CFG.n_splits

    test_df2 = test_df.copy()
    pf_test = test_df2['pf_ancc'].values - test_df2['last_known_tvt'].values

    test_df2['pred'] = test_df2['last_known_tvt'].values + apply_pp(
        test_df2, 
        ridge_test_preds,
        pf_test, 
        **pp_params
    )
    test_df2 = sg_smooth(test_df2, 'pred')

    sample_sub = pd.read_csv(CFG.dataset_path / "sample_submission.csv")
    sub_1 = (sample_sub[['id']].merge(
        test_df2[['id', 'pred']].rename(columns={'pred':'tvt'}),
        on='id', 
        how='left'
    ))

    sub_1['tvt']=sub_1['tvt'].fillna(float(train_df['last_known_tvt'].mean()+train_df['target'].mean()))
    sub_1

    sample = pd.read_csv(CFG.dataset_path / 'sample_submission.csv')
    sample['well']    = sample['id'].str[:8]
    sample['row_idx'] = sample['id'].str[9:].astype(int)

    train_hw_files = sorted(glob.glob(str(CFG.dataset_path / 'train' / '*__horizontal_well.csv')))
    train_wells = [os.path.basename(f).split('__')[0] for f in train_hw_files]

    test_hw_files = sorted(glob.glob(str(CFG.dataset_path / 'test' / '*__horizontal_well.csv')))
    test_wells = [os.path.basename(f).split('__')[0] for f in test_hw_files]

    rows = []
    for i, wid in enumerate(test_wells):
        print(f'\nProcessing {i + 1}/{len(test_wells)}: {wid}...')
        hw_te, tw_te = load_well(wid, 'test')

        tvt_phys = None
        hw_tr    = None
        tw_tr    = None

        # Physical model for visible wells
        if wid in train_wells:
            try:
                hw_tr, tw_tr = load_well(wid, 'train')
                hw_te['TVT_input'] = hw_tr['TVT_input'].values
                tvt_phys = tvt_from_contacts(hw_tr, tw_tr)
                print(f'  Physical model OK')
            except Exception as e:
                print(f'  Physical model failed: {e}')
                tvt_phys = None

        selector_code, selector_variant, selector_n_eval, selector_z_span = selector_well_code(hw_te)

        # Likelihood-weighted PF ensemble
        try:
            tw_ref = tw_tr if tw_tr is not None else tw_te
            pf_particles = int(globals().get('RIDGE_ARTIFACT_PF_N_PARTICLES', 600))
            pf_seeds = int(globals().get('RIDGE_ARTIFACT_PF_N_SEEDS', 150))
            pf_by_scale = run_pf_lik_ensemble_scales(hw_te, tw_ref, n_particles=pf_particles, n_seeds=pf_seeds)
            tvt_pf = pf_by_scale['pf_scale_8']
            print(f'  PF lik-ensemble OK particles={pf_particles} seeds={pf_seeds} scales={SELECTOR_SCALES}')
        except Exception as e:
            print(f'  PF failed: {e}')
            last_known = hw_te['TVT_input'].dropna()
            last_val   = float(last_known.iloc[-1]) if len(last_known) > 0 else 0.0
            tvt_pf = hw_te['TVT_input'].fillna(last_val).values.astype(float)
            pf_by_scale = {f'pf_scale_{scale:g}': tvt_pf.copy() for scale in SELECTOR_SCALES}

        # Beam search ensemble
        try:
            tw_ref = tw_tr if tw_tr is not None else tw_te
            tvt_beam = run_beam_ensemble(hw_te, tw_ref)
            print(f'  Beam 14-config ensemble OK')
        except Exception as e:
            print(f'  Beam failed: {e}')
            tvt_beam = tvt_pf.copy()

        # Selector blending
        last_known = hw_te['TVT_input'].dropna()
        last_known_tvt = float(last_known.iloc[-1]) if len(last_known) > 0 else float(np.nanmean(tvt_pf))
        tvt_selector = apply_selector_variant(selector_variant, pf_by_scale, tvt_beam, last_known_tvt)
        print(
            f'  Selector code={selector_code} variant={selector_variant} '
            f'n_eval={selector_n_eval:.0f} z_span={selector_z_span:.3f}'
        )

        ws = sample[sample['well'] == wid]
        for _, row in ws.iterrows():
            ridx = int(row['row_idx'])
            if tvt_phys is not None:
                tvt_val = float(tvt_phys.iloc[ridx])
            else:
                tvt_val = float(tvt_selector[ridx])
            rows.append({'id': row['id'], 'tvt': tvt_val})
        print(f'  Added {len(ws)} rows')

    sub_2 = pd.DataFrame(rows)


    # Final artifact-ridge blend. The exact reference uses 0.30 ridge / 0.70 heuristic.
    exact_ridge_weight = float(np.clip(float(RIDGE_ARTIFACT_EXACT_RIDGE_WEIGHT), 0.0, 1.0))
    push_ridge_weight = float(np.clip(float(globals().get('RIDGE_ARTIFACT_PUSH_RIDGE_WEIGHT', 0.25)), 0.0, 1.0))
    merged = sub_1.merge(sub_2, on='id', suffixes=('_ridge', '_heur'))
    sample_order = pd.read_csv(CFG.dataset_path / 'sample_submission.csv')[['id']]

    def _align_to_sample(frame, label):
        frame = frame[['id', 'tvt']].copy()
        frame['_id_key'] = frame['id'].astype(str)
        sample_keys = sample_order.copy()
        sample_keys['_id_key'] = sample_keys['id'].astype(str)
        if frame['_id_key'].duplicated().any():
            dup = frame.loc[frame['_id_key'].duplicated(), 'id'].head(10).tolist()
            raise RuntimeError(f'{label}: duplicated ids: {dup}')
        missing = sorted(set(sample_keys['_id_key']) - set(frame['_id_key']))
        extra = sorted(set(frame['_id_key']) - set(sample_keys['_id_key']))
        if missing or extra:
            raise RuntimeError(f'{label}: id mismatch missing={len(missing)} extra={len(extra)} examples={missing[:5] or extra[:5]}')
        aligned = sample_keys.merge(frame[['_id_key', 'tvt']], on='_id_key', how='left')
        aligned = pd.DataFrame({'id': sample_order['id'].to_numpy(), 'tvt': aligned['tvt'].to_numpy()})
        aligned['tvt'] = pd.to_numeric(aligned['tvt'], errors='coerce')
        if aligned['tvt'].isna().any():
            bad = aligned.loc[aligned['tvt'].isna(), 'id'].head(10).tolist()
            raise RuntimeError(f'{label}: NaN after sample alignment: {bad}')
        if not np.isfinite(aligned['tvt'].to_numpy(dtype=float)).all():
            raise RuntimeError(f'{label}: non-finite TVT values')
        return aligned[['id', 'tvt']]

    def _make_ridge_heuristic_blend(weight, label):
        w = float(np.clip(float(weight), 0.0, 1.0))
        cand = merged.assign(
            tvt=lambda x, ww=w: ww * x['tvt_ridge'] + (1.0 - ww) * x['tvt_heur']
        )[['id', 'tvt']]
        return _align_to_sample(cand, label)

    out_dir = Path(globals().get('OUTPUT_DIR', Path('/kaggle/working')))
    out_dir.mkdir(parents=True, exist_ok=True)
    final_output = Path(globals().get('FINAL_SUBMISSION_OUTPUT', Path('submission.csv')))

    is_pf_boost_profile = bool(globals().get('RUN_RIDGE_ARTIFACT_PF_BOOST', False))
    exact_ref_label = 'ridge_artifact_pf_boost' if is_pf_boost_profile else 'ridge_artifact_exact'
    exact_ref = _make_ridge_heuristic_blend(exact_ridge_weight, exact_ref_label)
    exact_ref_file = out_dir / ('submission_ridge_artifact_pf_boost_base.csv' if is_pf_boost_profile else 'submission_ridge_artifact_exact.csv')
    exact_ref.to_csv(exact_ref_file, index=False)

    selected = exact_ref.copy()
    selected_ridge_weight = exact_ridge_weight
    selected_model_blend_weight = 0.0
    profile_name = 'ridge_artifact_exact'

    if bool(globals().get('RUN_RIDGE_ARTIFACT_PUSH', False)):
        selected_ridge_weight = push_ridge_weight
        selected = _make_ridge_heuristic_blend(selected_ridge_weight, 'ridge_artifact_push')
        profile_name = 'ridge_artifact_push'
    elif bool(globals().get('RUN_RIDGE_ARTIFACT_PF_BOOST', False)):
        profile_name = 'ridge_artifact_pf_boost'

    candidate_rows = [{
        'file': exact_ref_file.name,
        'candidate_type': 'ridge_reference_pf_boost' if is_pf_boost_profile else 'ridge_reference_exact',
        'ridge_weight': exact_ridge_weight,
        'heuristic_weight': 1.0 - exact_ridge_weight,
        'pf_gbdt_blend_weight': 0.0,
        'selected_for_submission_csv': bool(globals().get('RUN_RIDGE_ARTIFACT_EXACT', False) or is_pf_boost_profile),
        'rows': int(len(exact_ref)),
        'tvt_mean': float(exact_ref['tvt'].mean()),
        'tvt_std': float(exact_ref['tvt'].std()),
        'tvt_min': float(exact_ref['tvt'].min()),
        'tvt_max': float(exact_ref['tvt'].max()),
    }]

    ridge_weight_grid = sorted(set(
        [float(w) for w in list(globals().get('RIDGE_ARTIFACT_CANDIDATE_RIDGE_WEIGHTS', [0.20, 0.25, 0.30, 0.35, 0.40]))]
        + [exact_ridge_weight, push_ridge_weight, selected_ridge_weight]
    ))
    for w in ridge_weight_grid:
        w = float(np.clip(float(w), 0.0, 1.0))
        cand = _make_ridge_heuristic_blend(w, f'ridge_artifact_w{w:.3f}')
        cand_name = f'submission_ridge_artifact_w{int(round(w * 1000)):03d}.csv'
        cand.to_csv(out_dir / cand_name, index=False)
        is_selected = profile_name == 'ridge_artifact_push' and abs(w - selected_ridge_weight) < 1e-12
        candidate_rows.append({
            'file': cand_name,
            'candidate_type': 'ridge_heuristic_weight_probe',
            'ridge_weight': w,
            'heuristic_weight': 1.0 - w,
            'pf_gbdt_blend_weight': 0.0,
            'selected_for_submission_csv': bool(is_selected),
            'rows': int(len(cand)),
            'tvt_mean': float(cand['tvt'].mean()),
            'tvt_std': float(cand['tvt'].std()),
            'tvt_min': float(cand['tvt'].min()),
            'tvt_max': float(cand['tvt'].max()),
        })

    if bool(globals().get('RUN_RIDGE_ARTIFACT_MODEL_BLEND', False)):
        pfg_path = out_dir / 'submission_pf_residual_gbdt_exact.csv'
        if not pfg_path.exists():
            fallback = out_dir / 'submission_pf_ensemble.csv'
            pfg_path = fallback if fallback.exists() else pfg_path
        if not pfg_path.exists():
            raise RuntimeError(
                'ridge_artifact_model_blend needs submission_pf_residual_gbdt_exact.csv. '
                'The PF-residual GBDT cell should run before the ridge artifact cell.'
            )
        pfg_sub = _align_to_sample(pd.read_csv(pfg_path), 'pf_residual_gbdt_for_ridge_model_blend')
        base = exact_ref.rename(columns={'tvt': 'tvt_ridge_ref'}).merge(
            pfg_sub.rename(columns={'tvt': 'tvt_pfgbdt'}), on='id', how='left'
        )
        if base['tvt_pfgbdt'].isna().any():
            bad = base.loc[base['tvt_pfgbdt'].isna(), 'id'].head(10).tolist()
            raise RuntimeError(f'PF-residual GBDT missing ids for model blend: {bad}')

        selected_model_blend_weight = float(np.clip(float(RIDGE_ARTIFACT_MODEL_BLEND_PF_GBDT_WEIGHT), 0.0, 1.0))
        selected_ridge_weight = exact_ridge_weight
        profile_name = 'ridge_artifact_model_blend'
        for w in sorted(set(list(globals().get('RIDGE_ARTIFACT_MODEL_BLEND_WEIGHTS', [0.00, 0.03, 0.05, 0.07, 0.10])) + [selected_model_blend_weight])):
            w = float(np.clip(float(w), 0.0, 1.0))
            cand = pd.DataFrame({
                'id': base['id'].to_numpy(),
                'tvt': (1.0 - w) * base['tvt_ridge_ref'].to_numpy(dtype=float) + w * base['tvt_pfgbdt'].to_numpy(dtype=float),
            })
            cand = _align_to_sample(cand, f'ridge_artifact_model_blend_w{w:.3f}')
            cand_name = f'submission_ridge_artifact_modelblend_{int(round(w * 1000)):03d}.csv'
            cand.to_csv(out_dir / cand_name, index=False)
            is_selected = abs(w - selected_model_blend_weight) < 1e-12
            candidate_rows.append({
                'file': cand_name,
                'candidate_type': 'ridge_reference_plus_pf_gbdt',
                'ridge_weight': exact_ridge_weight,
                'heuristic_weight': 1.0 - exact_ridge_weight,
                'pf_gbdt_blend_weight': w,
                'selected_for_submission_csv': bool(is_selected),
                'rows': int(len(cand)),
                'tvt_mean': float(cand['tvt'].mean()),
                'tvt_std': float(cand['tvt'].std()),
                'tvt_min': float(cand['tvt'].min()),
                'tvt_max': float(cand['tvt'].max()),
            })
            if is_selected:
                selected = cand

    profile_out = out_dir / f'submission_{profile_name}.csv'
    selected.to_csv(final_output, index=False)
    selected.to_csv(profile_out, index=False)

    pd.DataFrame(candidate_rows).to_csv(out_dir / 'ridge_artifact_candidate_report.csv', index=False)
    pd.DataFrame([{
        'profile': profile_name,
        'reference_overvalueawareness': 'https://www.kaggle.com/code/overvalueawareness/wellbore-geology-prediction-ridge/notebook',
        'reference_ravaghi': 'https://www.kaggle.com/code/ravaghi/wellbore-geology-prediction-ridge',
        'artifact_root': str(CFG.artifacts_path),
        'ridge_weight': selected_ridge_weight,
        'heuristic_weight': 1.0 - selected_ridge_weight,
        'pf_gbdt_blend_weight': selected_model_blend_weight,
        'ridge_pf_particles': int(globals().get('RIDGE_ARTIFACT_PF_N_PARTICLES', 600)),
        'ridge_pf_seeds': int(globals().get('RIDGE_ARTIFACT_PF_N_SEEDS', 150)),
        'output_file': str(final_output),
        'profile_output_file': profile_out.name,
        'rows': int(len(selected)),
        'tvt_mean': float(selected['tvt'].mean()),
        'tvt_std': float(selected['tvt'].std()),
        'tvt_min': float(selected['tvt'].min()),
        'tvt_max': float(selected['tvt'].max()),
    }]).to_csv(out_dir / 'ridge_artifact_summary.csv', index=False)

    globals()['FINAL_SELECTED_BASE_SOURCE'] = final_output
    globals()['FINAL_BASE_SOURCE_LABEL'] = profile_name
    pf_particles_used = int(globals().get('RIDGE_ARTIFACT_PF_N_PARTICLES', 600))
    pf_seeds_used = int(globals().get('RIDGE_ARTIFACT_PF_N_SEEDS', 150))
    print(
        f'Saved {final_output} using {profile_name}: '
        f'ridge={selected_ridge_weight:.3f}, heuristic={1.0 - selected_ridge_weight:.3f}, '
        f'pf_particles={pf_particles_used}, pf_seeds={pf_seeds_used}, '
        f'pf_gbdt={selected_model_blend_weight:.3f}'
    )
    try:
        display(selected.head())
    except Exception:
        print(selected.head())


In [ ]:
from pathlib import Path
from IPython.display import Image, display

figure_image_path = Path("/kaggle/input/datasets/pilkwang/pilkwang-public-dataset-for-notebooks-figures/ROGII_Graph_Fig10.png")
if figure_image_path.exists():
    display(Image(filename=str(figure_image_path)))


**Figure 16. Physical-estimator disagreement as uncertainty.**  
Hard wells often produce conflicting pseudo-TVT paths from PF, beam, formation, and self-correlation estimators. Their spread and pairwise differences are treated as uncertainty signals and can be used for gated blending.


## Conservative Residual Calibration

After the main alignment stack produces the base trajectory, this section computes an optional auxiliary estimate and applies the small gated correction selected above.


In [ ]:
# Optional external correction profiles are not active in the current profile set.
sidecar_submission = None


In [ ]:
# Preserve the selected profile submission unchanged.
if FINAL_SUBMISSION_OUTPUT.exists():
    FINAL_SIDECAR_SOURCE_LABEL = str(globals().get('FINAL_BASE_SOURCE_LABEL', 'base_only'))
    FINAL_SIDECAR_AVAILABLE = False
    FINAL_SIDECAR_AUTO_DISABLED_REASON = ''


## Final Submission Contract Guard

Final sanity check before writing `submission.csv`:

| Check | Requirement |
|---|---|
| columns | `id,tvt` only |
| rows | same count as `sample_submission.csv` |
| ids | same order as sample |
| tvt | numeric, finite, non-missing |
| source | final TVT trajectory after optional conservative correction |

If any condition fails, the notebook raises an error instead of silently writing a bad file.


In [ ]:
# Final v7 submission contract guard.
FINAL_V7_SOURCE = Path(globals().get('FINAL_SELECTED_BASE_SOURCE', globals().get('SUPER_STACK_SUBMISSION_OUTPUT', FINAL_SUBMISSION_OUTPUT)))
if bool(globals().get('RUN_SUPER_STACK_SOLUTION', False)) and not FINAL_V7_SOURCE.exists():
    raise RuntimeError(f'Expected super-stack submission was not produced: {FINAL_V7_SOURCE}')

if FINAL_SUBMISSION_OUTPUT.exists() and SAMPLE_SUBMISSION.exists():
    sample = pd.read_csv(SAMPLE_SUBMISSION)
    final = pd.read_csv(FINAL_SUBMISSION_OUTPUT)
    if list(final.columns) != ['id', 'tvt']:
        raise RuntimeError(f'Final submission columns must be [id, tvt], got {list(final.columns)}')
    if len(final) != len(sample):
        raise RuntimeError(f'Final submission row mismatch: got {len(final)}, expected {len(sample)}')
    if not final['id'].equals(sample['id']):
        raise RuntimeError('Final submission ids do not match sample_submission order.')
    final['tvt'] = pd.to_numeric(final['tvt'], errors='coerce')
    if final['tvt'].isna().any() or not np.isfinite(final['tvt'].to_numpy(dtype=float)).all():
        raise RuntimeError('Final submission contains missing or non-finite tvt values.')
    final[['id', 'tvt']].to_csv(FINAL_SUBMISSION_OUTPUT, index=False)
    contract_summary = pd.DataFrame([{
        'final_submission': str(FINAL_SUBMISSION_OUTPUT),
        'submission_profile': str(globals().get('SUBMISSION_PROFILE', 'unknown')),
        'source_label': str(globals().get('FINAL_BASE_SOURCE_LABEL', globals().get('FINAL_SIDECAR_SOURCE_LABEL', 'base_only'))),
        'source_file': str(FINAL_V7_SOURCE) if FINAL_V7_SOURCE.exists() else str(FINAL_SUBMISSION_OUTPUT),
        'rows': int(len(final)),
        'columns': ','.join(final.columns),
        'tvt_mean': float(final['tvt'].mean()),
        'tvt_std': float(final['tvt'].std()),
        'tvt_min': float(final['tvt'].min()),
        'tvt_max': float(final['tvt'].max()),
        'contract_pass': True,
    }])
    contract_summary.to_csv(OUTPUT_DIR / 'submission_contract_guard_summary_v7_final.csv', index=False)
    display(contract_summary)
else:
    print('Final submission guard skipped because submission.csv or sample_submission.csv is unavailable.')
